# ARC-AGI-3 — pbg hypothesis agent submission (SungsuHyun)

pbg treats an unseen game as a science problem: explore level 1 briefly (random buttons that never repeat the last one, one click per object class; at most 20 actions), then Qwen3.8-Flash-Next (NVFP4, vLLM in the notebook) writes a hypothesis of the whole game as executable code (roles, rules, win condition, tests); the prediction is checked against the real next frame pixel by pixel, counter-examples go back to the model, and A* plans only on a verified hypothesis. Sources are embedded from `pbg/` by `scripts/build_pbg_notebook.py`; do not edit cells here.

Commit = 2-game offline smoke (12 min); the competition rerun plays every gateway game.

In [ ]:
import json, os, subprocess, sys, time
T0 = time.time()
RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
WORK = '/kaggle/working'
print('competition rerun:', RERUN)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels', 'arc-agi'], check=True)
FILES = json.loads("{\"pbg/__init__.py\": \"\\\"\\\"\\\"pbg \\u2014 pixel board game autonomous reasoning system (docs/027, SW spec).\\n\\nLayers: env -> perception -> {probe, wml, goal} -> planner, driven by the orchestrator state machine.\\nThe LLM is used only inside wml/goal (HYPOTHESIZE state); everything else is deterministic code.\\n\\\"\\\"\\\"\\n__version__ = \\\"0.1.0\\\"\\n\", \"pbg/configs/budget.yaml\": \"# Action budget policy (spec \\u00a712). Fractions are of the total B.\\ntotal: 2000              # default when the environment gives no budget\\ninitial_probe: 0.12      # once per game\\nexperiment: 0.25         # information-gain experiments, cumulative\\nlevel_probe: 0.05        # local probe per level (only after a prediction error / novelty)\\nreprobe: 0.08            # re-exploration after STUCK, cumulative; beyond it -> reset\\nlow_budget_fraction: 0.20 # below this remaining fraction: no experiments, plan on the top hypothesis/goal only\\nllm_calls_max: 60        # per game (spec \\u00a714)\\nper_level:               # absolute caps per level, on top of the fractions above (docs/027 experiment 3)\\n  reprobe: 32\\n  experiment: 12\\n  level_probe: 20\\n  initial_clicks: 16      # initial-probe click targets (one per colour/shape class first)\\n\", \"pbg/configs/perception.yaml\": \"region_min_area_ratio: 0.05   # minimum region area as a fraction of the grid\\nconnectivity: 4               # component connectivity (4 or 8)\\nmerge_confirm_count: 3        # co-movement observations needed to merge components into one object\\ntrack_cost_max: 20.0          # matching cost above which an object counts as appeared/disappeared\\nnoop_pixel_threshold: 0       # changes at or below this many pixels are a no-op\\nui_strip_max_thickness: 3     # strips at most this thick with a periodic pattern get the ui_strip hint\\ncomposite_min_side: 2      # rectangle composites: each part at least this wide/high (1 px lines and marks never fuse)\\ncomposite_min_area: 6      # ... and at least this many pixels\\ncomposite_min_share: 0.2   # ... and each part at least this share of the union (a marker on a bar stays its own object)\\n\", \"pbg/core/__init__.py\": \"from .types import Action, Frame, Region, Object, Scene, Diff, Transition, ActionType  # noqa: F401\\nfrom .contracts import WorldModel, Rule, Goal, Hypothesis, RuleModel, scene_equal  # noqa: F401\\n\", \"pbg/core/budget.py\": \"\\\"\\\"\\\"core/budget.py \\u2014 action budget accounting and the category caps of spec \\u00a712.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nimport yaml\\n\\nPER_LEVEL_FRACTIONS = (\\\"level_probe\\\",)   # fractions that apply per level rather than cumulatively\\n\\nDEFAULT_BUDGET = {\\\"total\\\": 2000, \\\"initial_probe\\\": 0.12, \\\"experiment\\\": 0.25, \\\"level_probe\\\": 0.05, \\\"reprobe\\\": 0.08,\\n                  \\\"low_budget_fraction\\\": 0.20, \\\"llm_calls_max\\\": 60,\\n                  \\\"per_level\\\": {\\\"reprobe\\\": 32, \\\"experiment\\\": 12, \\\"level_probe\\\": 20, \\\"initial_clicks\\\": 16}}\\n\\n\\ndef load_budget_config(path: Optional[Path] = None) -> dict:\\n    cfg = dict(DEFAULT_BUDGET)\\n    p = path or Path(__file__).resolve().parents[1] / \\\"configs\\\" / \\\"budget.yaml\\\"\\n    if p.exists():\\n        cfg.update(yaml.safe_load(p.read_text()) or {})\\n    return cfg\\n\\n\\nclass Budget:\\n    \\\"\\\"\\\"Tracks actions used per category against fractional caps of the total B.\\n\\n    cap(kind) -> remaining actions allowed for that category; allows(kind) -> at least one action is allowed;\\n    charge(kind, n) records usage. Categories: initial_probe (once per game), level_probe (per level), experiment\\n    (cumulative), reprobe (cumulative), plan (everything else, uncapped).\\\"\\\"\\\"\\n\\n    def __init__(self, total: Optional[int] = None, config: Optional[dict] = None):\\n        self.cfg = dict(config or load_budget_config())\\n        self.total = int(total or self.cfg[\\\"total\\\"])\\n        self.used_by: dict[str, int] = {}\\n        self.level_used: dict[str, dict[int, int]] = {}\\n        self.llm_calls = 0\\n\\n    # \\u2500\\u2500 accounting \\u2500\\u2500\\n    def used(self) -> int:\\n        return sum(self.used_by.values())\\n\\n    def remaining(self) -> int:\\n        return max(0, self.total - self.used())\\n\\n    def fraction_left(self) -> float:\\n        return self.remaining() / max(1, self.total)\\n\\n    def low(self) -> bool:\\n        return self.fraction_left() <= float(self.cfg[\\\"low_budget_fraction\\\"])\\n\\n    def charge(self, kind: str, n: int = 1, level: Optional[int] = None) -> None:\\n        self.used_by[kind] = self.used_by.get(kind, 0) + n\\n        if level is not None:\\n            lv = self.level_used.setdefault(kind, {}); lv[level] = lv.get(level, 0) + n\\n\\n    def reset_level(self, level: int, kind: str) -> None:\\n        \\\"\\\"\\\"A new attempt (after RESET) gets the per-level allowance of `kind` again.\\\"\\\"\\\"\\n        self.level_used.setdefault(kind, {})[level] = 0\\n\\n    def sync(self, env_actions_used: int) -> None:\\n        \\\"\\\"\\\"Reconcile with the environment's own action counter (resets and engine-side counting): any actions the\\n        environment counted that we did not charge go to the 'other' category so the total is never exceeded.\\\"\\\"\\\"\\n        extra = env_actions_used - self.used()\\n        if extra > 0:\\n            self.used_by[\\\"other\\\"] = self.used_by.get(\\\"other\\\", 0) + extra\\n\\n    def cap(self, kind: str, level: Optional[int] = None) -> int:\\n        \\\"\\\"\\\"Remaining allowance for `kind`: the fraction-of-B cap (cumulative) AND, when configured, an absolute per-level\\n        cap (spec \\u00a712 rows are fractions of B; per-level absolute caps keep exploration near the level's own scale).\\\"\\\"\\\"\\n        frac = self.cfg.get(kind)\\n        if frac is None:\\n            return self.remaining()\\n        if kind in PER_LEVEL_FRACTIONS and level is not None:\\n            limit = int(frac * self.total) - self.level_used.get(kind, {}).get(level, 0)   # \\\"per level\\\" rows of spec \\u00a712\\n        else:\\n            limit = int(frac * self.total) - self.used_by.get(kind, 0)\\n        per_level = (self.cfg.get(\\\"per_level\\\") or {}).get(kind)\\n        if per_level is not None and level is not None:\\n            limit = min(limit, int(per_level) - self.level_used.get(kind, {}).get(level, 0))\\n        return max(0, min(limit, self.remaining()))\\n\\n    def allows(self, kind: str, n: int = 1, level: Optional[int] = None) -> bool:\\n        return self.cap(kind, level) >= n and self.remaining() >= n\\n\\n    def allows_llm(self) -> bool:\\n        return self.llm_calls < int(self.cfg.get(\\\"llm_calls_max\\\", 60))\\n\\n    def snapshot(self) -> dict:\\n        return {\\\"total\\\": self.total, \\\"used\\\": self.used(), \\\"remaining\\\": self.remaining(), \\\"by_kind\\\": dict(self.used_by), \\\"llm_calls\\\": self.llm_calls}\\n\", \"pbg/core/contracts.py\": \"\\\"\\\"\\\"core/contracts.py \\u2014 interfaces every world model / goal must satisfy (spec \\u00a74). The verifier calls only these.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass, field\\nfrom typing import Callable, Optional, Protocol, runtime_checkable\\n\\nfrom .types import Action, Scene\\n\\n\\n@runtime_checkable\\nclass WorldModel(Protocol):\\n    def assign_roles(self, scene: Scene) -> dict[int, str]: ...\\n    def predict(self, scene: Scene, action: Action) -> Optional[Scene]: ...   # None == UNKNOWN\\n    def rules(self) -> list[\\\"Rule\\\"]: ...\\n\\n\\n@dataclass\\nclass Rule:\\n    name: str\\n    applies: Callable[[Scene, Action], bool]\\n    effect: Callable[[Scene, Action], Optional[Scene]]     # None == UNKNOWN for this transition\\n    confidence: float = 0.5\\n    evidence: list[str] = field(default_factory=list)     # supporting transition ids\\n    violated_by: list[str] = field(default_factory=list)  # failing transition ids (filled by evaluate)\\n    source: str = \\\"llm\\\"                                   # llm | prior | induced | transferred\\n    meta: Optional[dict] = None                           # induction metadata (class-level description for the knowledge asset)\\n\\n    def __repr__(self) -> str:  # pragma: no cover\\n        return f\\\"Rule({self.name}, conf={self.confidence:.2f})\\\"\\n\\n\\n@runtime_checkable\\nclass Goal(Protocol):\\n    name: str\\n    confidence: float\\n\\n    def is_goal(self, scene: Scene) -> bool: ...\\n    def progress(self, scene: Scene) -> float: ...   # 0.0..1.0 heuristic\\n\\n\\nclass RuleModel:\\n    \\\"\\\"\\\"Standard WorldModel: a role assigner + an ordered rule list applied sequentially.\\n\\n    predict() applies every rule whose applies() is true, in order; each effect receives the scene produced by the\\n    previous one (rules therefore compose: move, then collision cancel, then counter). An effect returning None makes\\n    the whole prediction UNKNOWN. If no rule applies the result is `default` (\\\"unknown\\\" -> None, \\\"noop\\\" -> unchanged).\\\"\\\"\\\"\\n\\n    def __init__(self, rules: list[Rule], role_fn: Optional[Callable[[Scene], dict[int, str]]] = None, default: str = \\\"unknown\\\",\\n                 name: str = \\\"rule_model\\\"):\\n        self._rules = list(rules)\\n        self._role_fn = role_fn\\n        self.default = default\\n        self.name = name\\n        self.verified = False\\n        self.level_scoped = False      # True: positional rules learned for the current level -> verified on that level's log only\\n\\n    def assign_roles(self, scene: Scene) -> dict[int, str]:\\n        if self._role_fn is None:\\n            return {o.id: (o.role or \\\"unknown\\\") for o in scene.objects}\\n        try:\\n            return dict(self._role_fn(scene))\\n        except Exception:      # an LLM-written role function that fails on an unexpected scene = no roles, never a crash\\n            return {o.id: \\\"unknown\\\" for o in scene.objects}\\n\\n    def with_roles(self, scene: Scene) -> Scene:\\n        roles = self.assign_roles(scene)\\n        objs = []\\n        for o in scene.objects:\\n            role = roles.get(o.id, o.role)\\n            if role != o.role:\\n                o = o.moved(0, 0); o.role = role\\n            objs.append(o)\\n        return scene.copy(objects=objs)\\n\\n    def predict(self, scene: Scene, action: Action) -> Optional[Scene]:\\n        s = self.with_roles(scene)\\n        s.aux[\\\"_before\\\"] = scene\\n        applied = False\\n        for rule in self._rules:\\n            try:\\n                if rule.applies(s, action):\\n                    applied = True\\n                    nxt = rule.effect(s, action)\\n                    if nxt is None:\\n                        return None\\n                    nxt.aux[\\\"_before\\\"] = scene\\n                    s = nxt\\n            except Exception as e:  # a broken rule = unknown prediction, never a crash\\n                s.aux[\\\"_error\\\"] = f\\\"{rule.name}: {type(e).__name__}: {e}\\\"\\n                return None\\n        if not applied and self.default == \\\"unknown\\\":\\n            return None\\n        s.aux.pop(\\\"_before\\\", None)\\n        return s\\n\\n    def rules(self) -> list[Rule]:\\n        return self._rules\\n\\n    def rule(self, name: str) -> Optional[Rule]:\\n        for r in self._rules:\\n            if r.name == name:\\n                return r\\n        return None\\n\\n\\n@dataclass\\nclass Hypothesis:\\n    model: WorldModel\\n    score: float = 0.0          # transition-level accuracy on the whole log\\n    coverage: float = 0.0       # fraction of transitions with a non-UNKNOWN prediction\\n    violations: list[str] = field(default_factory=list)   # failing transition ids\\n    code: str = \\\"\\\"              # source that built the model (LLM / induction)\\n    name: str = \\\"\\\"\\n    verified: bool = False\\n    prediction_key: str = \\\"\\\"    # hash of all predictions on the log (dedupe)\\n    origin: str = \\\"llm\\\"         # llm | induced | transferred | prior\\n    recent_mismatches: int = 0  # mispredictions during execution since the last re-verification that changed the model\\n    change_score: float = 1.0   # accuracy on the log's non-noop transitions (a model that predicts \\\"nothing happens\\\" scores 0)\\n\\n    def usable(self, min_score: float = 0.6, min_change: float = 0.5, min_coverage: float = 0.5) -> bool:\\n        \\\"\\\"\\\"Good enough to plan and run experiments with (orchestrator quality gate): accurate where it predicts (score\\n        and change accuracy over covered transitions) and predicting at least half of the log.\\\"\\\"\\\"\\n        covered_acc = self.score / self.coverage if self.coverage > 0 else 0.0\\n        return covered_acc >= min_score and self.change_score >= min_change and self.coverage >= min_coverage\\n\\n    def sort_key(self) -> tuple:\\n        return (self.score, self.coverage)\\n\\n    def plan_weight(self) -> float:\\n        \\\"\\\"\\\"Weight used when choosing plans: verified log accuracy, penalised for live mispredictions.\\\"\\\"\\\"\\n        return max(0.05, self.score - 0.15 * self.recent_mismatches)\\n\\n\\ndef scene_equal(a: Optional[Scene], b: Optional[Scene], *, ignore_roles: bool = True) -> bool:\\n    \\\"\\\"\\\"Object-level equality used by evaluation and execution monitoring (spec \\u00a78): same multiset of\\n    (colour, bbox, shape_sig). Roles are ignored because the observed scene has none until the model assigns them.\\\"\\\"\\\"\\n    if a is None or b is None:\\n        return False\\n    return sorted(o.identity() for o in a.objects) == sorted(o.identity() for o in b.objects)\\n\\n\\ndef scene_mismatch(a: Scene, b: Scene) -> dict:\\n    \\\"\\\"\\\"Which objects differ between a predicted and an observed scene (for prompts and diagnostics).\\\"\\\"\\\"\\n    ia = {o.identity(): o for o in a.objects}; ib = {o.identity(): o for o in b.objects}\\n    only_pred = [o for k, o in ia.items() if k not in ib]; only_obs = [o for k, o in ib.items() if k not in ia]\\n    return {\\\"only_predicted\\\": [(o.id, o.color, o.bbox) for o in only_pred], \\\"only_observed\\\": [(o.id, o.color, o.bbox) for o in only_obs]}\\n\", \"pbg/core/events.py\": \"\\\"\\\"\\\"core/events.py \\u2014 state-transition event log (events.jsonl; spec \\u00a712, input of harness/attribution).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport time\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\n\\nclass EventLog:\\n    def __init__(self, path: Optional[Path] = None):\\n        self.path = Path(path) if path else None\\n        self.events: list[dict] = []\\n        if self.path:\\n            self.path.parent.mkdir(parents=True, exist_ok=True)\\n\\n    def emit(self, frm: str, to: str, reason: str, *, transition_id: Optional[str] = None, budget_used: int = 0, **extra) -> dict:\\n        if not reason:\\n            raise ValueError(\\\"every state transition needs a reason (spec \\u00a712 acceptance)\\\")\\n        ev = {\\\"ts\\\": time.time(), \\\"from\\\": frm, \\\"to\\\": to, \\\"reason\\\": reason, \\\"transition_id\\\": transition_id, \\\"budget_used\\\": budget_used, **extra}\\n        self.events.append(ev)\\n        if self.path:\\n            with open(self.path, \\\"a\\\") as f:\\n                f.write(json.dumps(ev, default=str) + \\\"\\\\n\\\")\\n        return ev\\n\\n    @staticmethod\\n    def read(path: Path) -> list[dict]:\\n        p = Path(path)\\n        if not p.exists():\\n            return []\\n        return [json.loads(line) for line in p.read_text().splitlines() if line.strip()]\\n\", \"pbg/core/types.py\": \"\\\"\\\"\\\"core/types.py \\u2014 the only data types modules exchange (spec \\u00a74).\\n\\nRaw grids travel between Env Wrapper and Perception only; everything above sees Scene / Diff / Transition.\\nSerialisation is JSON (grids as lists of hex strings, masks as run-length strings).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nfrom dataclasses import dataclass, field\\nfrom typing import Iterable, Literal, Optional\\n\\nimport numpy as np\\n\\nActionType = Literal[\\\"BUTTON\\\", \\\"CLICK\\\", \\\"RESET\\\"]\\nHEX = \\\"0123456789abcdef\\\"\\n\\n\\ndef grid_hash(grid: np.ndarray) -> str:\\n    return hashlib.sha1(np.ascontiguousarray(grid, dtype=np.int8).tobytes()).hexdigest()[:16]\\n\\n\\ndef grid_to_rows(grid: np.ndarray) -> list[str]:\\n    return [\\\"\\\".join(HEX[int(v) & 15] for v in row) for row in grid]\\n\\n\\ndef rows_to_grid(rows: list[str]) -> np.ndarray:\\n    return np.array([[HEX.index(ch) for ch in r] for r in rows], dtype=np.int8)\\n\\n\\n@dataclass(frozen=True)\\nclass Action:\\n    type: ActionType\\n    id: Optional[int] = None          # BUTTON: 1..N (engine ACTION{id})\\n    row: Optional[int] = None         # CLICK: grid cell coordinates\\n    col: Optional[int] = None\\n\\n    # \\u2500\\u2500 constructors \\u2500\\u2500\\n    @staticmethod\\n    def button(i: int) -> \\\"Action\\\":\\n        return Action(\\\"BUTTON\\\", id=int(i))\\n\\n    @staticmethod\\n    def click(row: int, col: int) -> \\\"Action\\\":\\n        return Action(\\\"CLICK\\\", row=int(row), col=int(col))\\n\\n    @staticmethod\\n    def reset() -> \\\"Action\\\":\\n        return Action(\\\"RESET\\\")\\n\\n    # \\u2500\\u2500 views \\u2500\\u2500\\n    @property\\n    def key(self) -> str:\\n        \\\"\\\"\\\"Action class key used by semantics / rules: 'ACTION1'.. for buttons, 'CLICK', 'RESET'.\\\"\\\"\\\"\\n        if self.type == \\\"BUTTON\\\":\\n            return f\\\"ACTION{self.id}\\\"\\n        return self.type\\n\\n    def label(self) -> str:\\n        if self.type == \\\"CLICK\\\":\\n            return f\\\"CLICK({self.row},{self.col})\\\"\\n        return self.key\\n\\n    def to_json(self) -> dict:\\n        d: dict = {\\\"type\\\": self.type}\\n        if self.id is not None:\\n            d[\\\"id\\\"] = self.id\\n        if self.row is not None:\\n            d[\\\"row\\\"], d[\\\"col\\\"] = self.row, self.col\\n        return d\\n\\n    @staticmethod\\n    def from_json(d: dict) -> \\\"Action\\\":\\n        return Action(d[\\\"type\\\"], id=d.get(\\\"id\\\"), row=d.get(\\\"row\\\"), col=d.get(\\\"col\\\"))\\n\\n    def __str__(self) -> str:  # pragma: no cover\\n        return self.label()\\n\\n\\n@dataclass\\nclass Frame:\\n    grid: np.ndarray                  # (H, W) int8 colour indices\\n    level: int\\n    step_idx: int\\n    hash: str = \\\"\\\"                    # stable hash of grid\\n\\n    def __post_init__(self):\\n        self.grid = np.asarray(self.grid, dtype=np.int8)\\n        if not self.hash:\\n            self.hash = grid_hash(self.grid)\\n\\n    @property\\n    def shape(self) -> tuple[int, int]:\\n        return tuple(self.grid.shape)  # type: ignore[return-value]\\n\\n    def to_json(self) -> dict:\\n        return {\\\"rows\\\": grid_to_rows(self.grid), \\\"level\\\": self.level, \\\"step_idx\\\": self.step_idx, \\\"hash\\\": self.hash}\\n\\n    @staticmethod\\n    def from_json(d: dict) -> \\\"Frame\\\":\\n        return Frame(rows_to_grid(d[\\\"rows\\\"]), int(d[\\\"level\\\"]), int(d[\\\"step_idx\\\"]), d.get(\\\"hash\\\", \\\"\\\"))\\n\\n\\n@dataclass\\nclass Region:\\n    id: str                           # \\\"R0\\\", \\\"R1\\\", ...\\n    bg_color: int\\n    bbox: tuple[int, int, int, int]   # r0, c0, r1, c1 (exclusive)\\n    kind_hint: Literal[\\\"board\\\", \\\"panel\\\", \\\"ui_strip\\\", \\\"unknown\\\"] = \\\"unknown\\\"\\n    mask: Optional[np.ndarray] = None # bbox-sized bool: the region's real pixels when it is not a full rectangle (corridors)\\n\\n    @property\\n    def area(self) -> int:\\n        r0, c0, r1, c1 = self.bbox\\n        return (r1 - r0) * (c1 - c0)\\n\\n    @property\\n    def center(self) -> tuple[int, int]:\\n        r0, c0, r1, c1 = self.bbox\\n        return ((r0 + r1 - 1) // 2, (c0 + c1 - 1) // 2)\\n\\n    def contains(self, r: int, c: int) -> bool:\\n        r0, c0, r1, c1 = self.bbox\\n        if not (r0 <= r < r1 and c0 <= c < c1):\\n            return False\\n        return True if self.mask is None else bool(self.mask[r - r0, c - c0])\\n\\n    def to_json(self) -> dict:\\n        d = {\\\"id\\\": self.id, \\\"bg_color\\\": self.bg_color, \\\"bbox\\\": list(self.bbox), \\\"kind_hint\\\": self.kind_hint}\\n        if self.mask is not None:\\n            d[\\\"mask\\\"] = mask_to_rle(self.mask)\\n        return d\\n\\n    @staticmethod\\n    def from_json(d: dict) -> \\\"Region\\\":\\n        bbox = tuple(d[\\\"bbox\\\"])\\n        m = rle_to_mask(d[\\\"mask\\\"], (bbox[2] - bbox[0], bbox[3] - bbox[1])) if d.get(\\\"mask\\\") else None\\n        return Region(d[\\\"id\\\"], int(d[\\\"bg_color\\\"]), bbox, d.get(\\\"kind_hint\\\", \\\"unknown\\\"), m)\\n\\n\\ndef mask_to_rle(mask: np.ndarray) -> str:\\n    flat = np.asarray(mask, dtype=bool).ravel()\\n    out, cur, run = [], False, 0\\n    for v in flat:\\n        if v == cur:\\n            run += 1\\n        else:\\n            out.append(str(run)); cur, run = v, 1\\n    out.append(str(run))\\n    return \\\",\\\".join(out)\\n\\n\\ndef rle_to_mask(rle: str, shape: tuple[int, int]) -> np.ndarray:\\n    runs = [int(x) for x in rle.split(\\\",\\\")] if rle else []\\n    flat = np.zeros(shape[0] * shape[1], dtype=bool)\\n    pos, cur = 0, False\\n    for run in runs:\\n        if cur:\\n            flat[pos:pos + run] = True\\n        pos += run; cur = not cur\\n    return flat.reshape(shape)\\n\\n\\ndef shape_signature(mask: np.ndarray) -> str:\\n    \\\"\\\"\\\"Normalised mask hash: crop to bbox, top-left align, hash bytes. Colour is not part of it (spec \\u00a76).\\\"\\\"\\\"\\n    m = np.asarray(mask, dtype=bool)\\n    rows = np.where(m.any(axis=1))[0]; cols = np.where(m.any(axis=0))[0]\\n    if len(rows) == 0:\\n        return \\\"empty\\\"\\n    m = m[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]\\n    return hashlib.sha1(np.packbits(m).tobytes() + bytes(m.shape)).hexdigest()[:12]\\n\\n\\ndef shape_signature_d4(mask: np.ndarray) -> str:\\n    \\\"\\\"\\\"Signature invariant under the 8 rotations/reflections (the minimum over the D4 orbit).\\\"\\\"\\\"\\n    m = np.asarray(mask, dtype=bool)\\n    variants = []\\n    for k in range(4):\\n        r = np.rot90(m, k)\\n        variants.append(shape_signature(r)); variants.append(shape_signature(np.fliplr(r)))\\n    return min(variants)\\n\\n\\n@dataclass\\nclass Object:\\n    id: int                           # tracking id across frames\\n    color: int                        # representative colour (multi-colour objects: most frequent)\\n    colors: tuple[int, ...]\\n    bbox: tuple[int, int, int, int]   # r0, c0, r1, c1 (exclusive)\\n    mask: np.ndarray                  # bbox-sized bool\\n    area: int\\n    shape_sig: str\\n    region: str\\n    role: Optional[str] = None\\n    color_mask: Optional[np.ndarray] = None   # bbox-sized int8 (-1 outside) for multi-colour objects\\n    parts: Optional[list] = None              # components a fused object was built from (tracking only, not serialised)\\n    composite: bool = False                   # adjacent components whose union is a filled rectangle (a pattern / canvas / button)\\n    occluded: bool = False                    # a static object currently hidden under a moving one: kept in the scene at its last position\\n\\n    @property\\n    def center(self) -> tuple[int, int]:\\n        r0, c0, r1, c1 = self.bbox\\n        return ((r0 + r1 - 1) // 2, (c0 + c1 - 1) // 2)\\n\\n    @property\\n    def height(self) -> int:\\n        return self.bbox[2] - self.bbox[0]\\n\\n    @property\\n    def width(self) -> int:\\n        return self.bbox[3] - self.bbox[1]\\n\\n    @property\\n    def shape_sig_d4(self) -> str:\\n        return shape_signature_d4(self.mask)\\n\\n    def cells(self) -> list[tuple[int, int]]:\\n        r0, c0 = self.bbox[0], self.bbox[1]\\n        rr, cc = np.nonzero(self.mask)\\n        return [(int(r) + r0, int(c) + c0) for r, c in zip(rr, cc)]\\n\\n    def moved(self, dr: int, dc: int) -> \\\"Object\\\":\\n        r0, c0, r1, c1 = self.bbox\\n        parts = None if not self.parts else [q.moved(dr, dc) for q in self.parts]\\n        return Object(self.id, self.color, self.colors, (r0 + dr, c0 + dc, r1 + dr, c1 + dc), self.mask, self.area,\\n                      self.shape_sig, self.region, self.role, self.color_mask, parts=parts, composite=self.composite)\\n\\n    def recolored(self, color: int) -> \\\"Object\\\":\\n        cm = None if self.color_mask is None else np.where(self.color_mask >= 0, color, -1).astype(np.int8)\\n        return Object(self.id, int(color), (int(color),), self.bbox, self.mask, self.area, self.shape_sig, self.region, self.role, cm)\\n\\n    def overlaps(self, other: \\\"Object\\\") -> bool:\\n        a, b = self.bbox, other.bbox\\n        if a[0] >= b[2] or b[0] >= a[2] or a[1] >= b[3] or b[1] >= a[3]:\\n            return False\\n        r0, c0, r1, c1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])\\n        ma = self.mask[r0 - a[0]:r1 - a[0], c0 - a[1]:c1 - a[1]]\\n        mb = other.mask[r0 - b[0]:r1 - b[0], c0 - b[1]:c1 - b[1]]\\n        return bool((ma & mb).any())\\n\\n    def with_mask(self, mask: np.ndarray, color_mask: Optional[np.ndarray] = None, origin: Optional[tuple[int, int]] = None) -> \\\"Object\\\":\\n        \\\"\\\"\\\"Copy with a new pixel layout (bbox-sized bool mask, optional int8 colour mask with -1 outside) anchored at origin.\\\"\\\"\\\"\\n        r0, c0 = origin if origin is not None else (self.bbox[0], self.bbox[1])\\n        mask = np.asarray(mask, dtype=bool)\\n        cm = None if color_mask is None else np.asarray(color_mask, dtype=np.int8)\\n        if cm is not None:\\n            vals = [int(v) for v in np.unique(cm[mask])] if mask.any() else [self.color]\\n            counts = {v: int(((cm == v) & mask).sum()) for v in vals}\\n            color = max(counts, key=counts.get); colors = tuple(sorted(counts))\\n        else:\\n            color, colors = self.color, (self.color,)\\n        return Object(self.id, int(color), colors, (r0, c0, r0 + mask.shape[0], c0 + mask.shape[1]), mask, int(mask.sum()), shape_signature(mask),\\n                      self.region, self.role, color_mask=cm, parts=None, composite=self.composite)\\n\\n    def rotated(self, k: int = 1) -> \\\"Object\\\":\\n        \\\"\\\"\\\"Copy rotated by k x 90 degrees clockwise about the bbox centre (colour layout rotates too).\\\"\\\"\\\"\\n        k = int(k) % 4\\n        m = np.rot90(self.mask, -k)\\n        cm = None if self.color_mask is None else np.rot90(self.color_mask, -k)\\n        cr, cc = (self.bbox[0] + self.bbox[2]) / 2, (self.bbox[1] + self.bbox[3]) / 2\\n        r0 = int(round(cr - m.shape[0] / 2)); c0 = int(round(cc - m.shape[1] / 2))\\n        return self.with_mask(m, cm, (r0, c0))\\n\\n    def flipped(self, axis: int = 1) -> \\\"Object\\\":\\n        \\\"\\\"\\\"Copy mirrored: axis=1 left-right, axis=0 top-bottom.\\\"\\\"\\\"\\n        m = np.flip(self.mask, axis)\\n        cm = None if self.color_mask is None else np.flip(self.color_mask, axis)\\n        return self.with_mask(m, cm)\\n\\n    def identity(self) -> tuple:\\n        \\\"\\\"\\\"What evaluation compares (spec \\u00a78): role-free object identity.\\\"\\\"\\\"\\n        if self.color_mask is not None and len(self.colors) >= 2:\\n            # multi-colour objects differ by their colour layout too (a half-stamped canvas is not a blank one)\\n            return (int(self.color), tuple(int(x) for x in self.bbox), self.shape_sig, hashlib.sha1(self.color_mask.tobytes()).hexdigest()[:8])\\n        return (int(self.color), tuple(int(x) for x in self.bbox), self.shape_sig)\\n\\n    def to_json(self) -> dict:\\n        d = {\\\"id\\\": self.id, \\\"color\\\": self.color, \\\"colors\\\": list(self.colors), \\\"bbox\\\": list(self.bbox), \\\"mask\\\": mask_to_rle(self.mask),\\n             \\\"area\\\": self.area, \\\"shape_sig\\\": self.shape_sig, \\\"region\\\": self.region, \\\"role\\\": self.role}\\n        if self.color_mask is not None:\\n            d[\\\"color_mask\\\"] = [[int(v) for v in row] for row in self.color_mask]\\n        return d\\n\\n    @staticmethod\\n    def from_json(d: dict) -> \\\"Object\\\":\\n        bbox = tuple(d[\\\"bbox\\\"])\\n        shape = (bbox[2] - bbox[0], bbox[3] - bbox[1])\\n        cm = np.array(d[\\\"color_mask\\\"], dtype=np.int8) if d.get(\\\"color_mask\\\") is not None else None\\n        return Object(int(d[\\\"id\\\"]), int(d[\\\"color\\\"]), tuple(d[\\\"colors\\\"]), bbox, rle_to_mask(d[\\\"mask\\\"], shape), int(d[\\\"area\\\"]),\\n                      d[\\\"shape_sig\\\"], d[\\\"region\\\"], d.get(\\\"role\\\"), cm)\\n\\n\\n@dataclass\\nclass Scene:\\n    frame_hash: str\\n    grid_shape: tuple[int, int]\\n    regions: list[Region]\\n    objects: list[Object]\\n    aux: dict = field(default_factory=dict)   # hidden-state estimates (spec \\u00a78)\\n    # the colour last seen at each cell while no object covered it (-1 = never seen), kept by Perception per level: a\\n    # predicted scene paints the cells a moved object vacates from it, so a corridor / floor / decoration under a piece\\n    # renders as what it is, not as the region background (docs/029: ls20's move model collapsed on exactly this)\\n    backdrop: Optional[np.ndarray] = field(default=None, repr=False, compare=False)\\n\\n    # \\u2500\\u2500 lookups \\u2500\\u2500\\n    def get(self, obj_id: int) -> Optional[Object]:\\n        for o in self.objects:\\n            if o.id == obj_id:\\n                return o\\n        return None\\n\\n    def region(self, rid: str) -> Optional[Region]:\\n        for r in self.regions:\\n            if r.id == rid:\\n                return r\\n        return None\\n\\n    def by_role(self, role: str) -> list[Object]:\\n        return [o for o in self.objects if o.role == role]\\n\\n    def by_color(self, color: int) -> list[Object]:\\n        return [o for o in self.objects if o.color == color]\\n\\n    def in_region(self, rid: str) -> list[Object]:\\n        return [o for o in self.objects if o.region == rid]\\n\\n    def region_at(self, r: int, c: int) -> Optional[Region]:\\n        best = None\\n        for reg in self.regions:\\n            if reg.contains(r, c) and (best is None or reg.area < best.area):\\n                best = reg\\n        return best\\n\\n    def key(self) -> tuple:\\n        \\\"\\\"\\\"Hashable identity of the scene for search / dedupe (objects + aux).\\\"\\\"\\\"\\n        objs = tuple(sorted(o.identity() for o in self.objects))\\n        aux = tuple(sorted((k, repr(v)) for k, v in self.aux.items() if not k.startswith(\\\"_\\\")))\\n        return (objs, aux)\\n\\n    def copy(self, objects: Optional[list[Object]] = None, aux: Optional[dict] = None) -> \\\"Scene\\\":\\n        return Scene(self.frame_hash, self.grid_shape, list(self.regions), list(self.objects if objects is None else objects),\\n                     dict(self.aux if aux is None else aux), self.backdrop)\\n\\n    def render(self) -> np.ndarray:\\n        \\\"\\\"\\\"Rasterise regions + objects back into a grid (background = region colours).\\\"\\\"\\\"\\n        h, w = self.grid_shape\\n        g = np.full((h, w), self.aux.get(\\\"_global_bg\\\", 0), dtype=np.int8)\\n        for reg in sorted(self.regions, key=lambda r: -r.area):\\n            r0, c0, r1, c1 = reg.bbox\\n            if reg.mask is not None:\\n                sub = g[r0:r1, c0:c1]\\n                sub[reg.mask] = reg.bg_color\\n            else:\\n                g[r0:r1, c0:c1] = reg.bg_color\\n        if self.backdrop is not None and self.backdrop.shape == g.shape:\\n            known = self.backdrop >= 0\\n            g[known] = self.backdrop[known]\\n        # z-order: hidden static objects first, then large before small (a floor parsed as one huge object must never paint\\n        # over the piece the model just moved onto it \\u2014 that hid ls20's agent in every prediction after step 27)\\n        for o in sorted(self.objects, key=lambda o: (not o.occluded, -o.area, o.id)):\\n\\n            r0, c0, r1, c1 = o.bbox\\n            rr0, cc0, rr1, cc1 = max(r0, 0), max(c0, 0), min(r1, h), min(c1, w)\\n            if rr1 <= rr0 or cc1 <= cc0:\\n                continue\\n            sub = o.mask[rr0 - r0:rr1 - r0, cc0 - c0:cc1 - c0]\\n            if o.color_mask is not None:\\n                cm = o.color_mask[rr0 - r0:rr1 - r0, cc0 - c0:cc1 - c0]\\n                g[rr0:rr1, cc0:cc1] = np.where(sub, cm, g[rr0:rr1, cc0:cc1])\\n            else:\\n                g[rr0:rr1, cc0:cc1] = np.where(sub, o.color, g[rr0:rr1, cc0:cc1])\\n        return g\\n\\n    def to_json(self) -> dict:\\n        return {\\\"frame_hash\\\": self.frame_hash, \\\"grid_shape\\\": list(self.grid_shape), \\\"regions\\\": [r.to_json() for r in self.regions],\\n                \\\"objects\\\": [o.to_json() for o in self.objects], \\\"aux\\\": {k: v for k, v in self.aux.items() if not k.startswith(\\\"_\\\") or k == \\\"_global_bg\\\"}}\\n\\n    @staticmethod\\n    def from_json(d: dict) -> \\\"Scene\\\":\\n        return Scene(d[\\\"frame_hash\\\"], tuple(d[\\\"grid_shape\\\"]), [Region.from_json(r) for r in d[\\\"regions\\\"]],\\n                     [Object.from_json(o) for o in d[\\\"objects\\\"]], dict(d.get(\\\"aux\\\") or {}))\\n\\n\\n@dataclass\\nclass Diff:\\n    moved: list[tuple[int, tuple[int, int]]] = field(default_factory=list)      # (obj_id, (dr, dc))\\n    appeared: list[Object] = field(default_factory=list)\\n    disappeared: list[Object] = field(default_factory=list)\\n    recolored: list[tuple[int, int, int]] = field(default_factory=list)          # (obj_id, old, new)\\n    reshaped: list[tuple[int, str, str]] = field(default_factory=list)           # (obj_id, old_sig, new_sig)\\n    pixel_changes: int = 0\\n    is_noop: bool = True\\n\\n    def to_json(self) -> dict:\\n        return {\\\"moved\\\": [[i, list(d)] for i, d in self.moved], \\\"appeared\\\": [o.id for o in self.appeared],\\n                \\\"disappeared\\\": [o.id for o in self.disappeared], \\\"recolored\\\": [list(x) for x in self.recolored],\\n                \\\"reshaped\\\": [list(x) for x in self.reshaped], \\\"pixel_changes\\\": self.pixel_changes, \\\"is_noop\\\": self.is_noop}\\n\\n    def kinds(self) -> set[str]:\\n        k = set()\\n        if self.moved: k.add(\\\"moved\\\")\\n        if self.appeared: k.add(\\\"appeared\\\")\\n        if self.disappeared: k.add(\\\"disappeared\\\")\\n        if self.recolored: k.add(\\\"recolored\\\")\\n        if self.reshaped: k.add(\\\"reshaped\\\")\\n        return k\\n\\n\\n@dataclass\\nclass Transition:\\n    before: Scene\\n    action: Action\\n    after: Scene\\n    diff: Diff\\n    status_change: Optional[Literal[\\\"LEVEL_UP\\\", \\\"WIN\\\", \\\"GAME_OVER\\\"]] = None\\n    intermediate_frames: list[Frame] = field(default_factory=list)\\n    id: str = \\\"\\\"                      # \\\"{game_id}:{level}:{step_idx}\\\" (spec \\u00a711)\\n    level: int = 1\\n    step_idx: int = 0\\n    before_frame: Optional[Frame] = None\\n    after_frame: Optional[Frame] = None\\n\\n    def to_json(self) -> dict:\\n        return {\\\"id\\\": self.id, \\\"level\\\": self.level, \\\"step_idx\\\": self.step_idx, \\\"action\\\": self.action.to_json(),\\n                \\\"before\\\": self.before.to_json(), \\\"after\\\": self.after.to_json(), \\\"diff\\\": self.diff.to_json(),\\n                \\\"status_change\\\": self.status_change, \\\"n_intermediate\\\": len(self.intermediate_frames)}\\n\\n\\ndef transitions_by_level(log: Iterable[Transition]) -> dict[int, list[Transition]]:\\n    out: dict[int, list[Transition]] = {}\\n    for t in log:\\n        out.setdefault(t.level, []).append(t)\\n    return out\\n\", \"pbg/env/__init__.py\": \"from .wrapper import EnvWrapper, ArcadeEnv, EnvStatus, RawTransition  # noqa: F401\\nfrom .replay import ReplayEnv, read_raw_log  # noqa: F401\\n\", \"pbg/env/calibrate.py\": \"\\\"\\\"\\\"env/calibrate.py \\u2014 click calibration (spec \\u00a75). For a pixel-space environment two corner clicks would fix scale and\\noffset; the arcengine API already takes cell coordinates, so calibration is a contract check plus a runtime monitor:\\n`ClickMonitor` records which object responded to each click and flags misalignment when the responder is never the\\nobject under the cursor (spec \\u00a715 'click coordinate misalignment').\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Optional\\n\\nfrom ..core.types import Action, Scene, Transition\\n\\n\\ndef object_under(scene: Scene, r: int, c: int) -> Optional[int]:\\n    for o in scene.objects:\\n        r0, c0, r1, c1 = o.bbox\\n        if r0 <= r < r1 and c0 <= c < c1 and o.mask[r - r0, c - c0]:\\n            return o.id\\n    return None\\n\\n\\nclass ClickMonitor:\\n    def __init__(self, window: int = 6):\\n        self.window = window\\n        self.hits: list[bool] = []\\n\\n    def observe(self, t: Transition) -> None:\\n        if t.action.type != \\\"CLICK\\\" or t.diff.is_noop:\\n            return\\n        under = object_under(t.before, t.action.row, t.action.col)\\n        responders = {i for i, _ in t.diff.moved} | {i for i, _, _ in t.diff.recolored} | {o.id for o in t.diff.disappeared} | {i for i, _, _ in t.diff.reshaped}\\n        self.hits.append(under is not None and under in responders)\\n        self.hits = self.hits[-self.window:]\\n\\n    def misaligned(self) -> bool:\\n        return len(self.hits) >= self.window and not any(self.hits)\\n\", \"pbg/env/replay.py\": \"\\\"\\\"\\\"env/replay.py \\u2014 ReplayEnv: the EnvWrapper interface over a recorded raw.jsonl (spec \\u00a75, \\u00a716).\\n\\nThe log is a sequence of records {kind: reset|step, action, frames, state, level, ...}. step(action) serves the next\\nrecord when the requested action equals the recorded one; a different action is an error (RawTransition.error) unless\\n`lenient` is set, in which case the recorded action is served anyway (used to drive module pipelines over human logs).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nfrom ..core.types import Action, Frame, rows_to_grid\\nfrom .wrapper import BUTTON_IDS, CLICK_ID, EnvStatus, EnvWrapper, RawTransition, stable_frames\\n\\n\\ndef read_raw_log(path: Path) -> list[dict]:\\n    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]\\n\\n\\nclass ReplayEnv(EnvWrapper):\\n    def __init__(self, records: list[dict] | Path, game_id: str = \\\"\\\", *, lenient: bool = False):\\n        self.records = read_raw_log(records) if isinstance(records, (str, Path)) else list(records)\\n        self.game_id = game_id or (self.records[0].get(\\\"game_id\\\", \\\"\\\") if self.records else \\\"\\\")\\n        self.lenient = lenient\\n        self.pos = 0\\n        self.frame: Optional[Frame] = None\\n        self.state, self.level, self.levels_total = \\\"NOT_PLAYED\\\", 1, None\\n        self._avail: list[int] = []\\n        self.actions_used = 0\\n        self.step_idx = 0\\n\\n    def _serve(self, rec: dict, action: Action) -> tuple[Frame, list[Frame], Optional[str]]:\\n        self.state = rec.get(\\\"state\\\", \\\"NOT_FINISHED\\\")\\n        self.level = int(rec.get(\\\"level\\\", 1)); self.levels_total = rec.get(\\\"levels_total\\\")\\n        self._avail = [int(a) for a in rec.get(\\\"available_actions\\\", [])]\\n        grids = [rows_to_grid(rows) for rows in rec[\\\"frames\\\"]]\\n        after_grid, inter = stable_frames(grids)\\n        return Frame(after_grid, self.level, self.step_idx), [Frame(g, self.level, self.step_idx) for g in inter], rec.get(\\\"status_change\\\")\\n\\n    def next_action(self) -> Optional[Action]:\\n        \\\"\\\"\\\"The action the log takes next (None at the end). Lets a pipeline follow a human log step by step.\\\"\\\"\\\"\\n        while self.pos < len(self.records) and self.records[self.pos][\\\"kind\\\"] == \\\"reset\\\":\\n            if self.frame is None:\\n                break\\n            self.pos += 1\\n        if self.pos >= len(self.records):\\n            return None\\n        return Action.from_json(self.records[self.pos][\\\"action\\\"])\\n\\n    def reset(self, game_id: Optional[str] = None) -> Frame:\\n        while self.pos < len(self.records) and self.records[self.pos][\\\"kind\\\"] != \\\"reset\\\":\\n            self.pos += 1\\n        if self.pos >= len(self.records):\\n            raise EOFError(\\\"replay log has no further reset\\\")\\n        rec = self.records[self.pos]; self.pos += 1\\n        self.frame, _, _ = self._serve(rec, Action.reset())\\n        return self.frame\\n\\n    def step(self, action: Action) -> RawTransition:\\n        before, st = self.frame, self.status()\\n        if self.pos >= len(self.records):\\n            return RawTransition(before, action, before, st, error=\\\"end of replay log\\\", step_idx=self.step_idx, level=self.level)\\n        rec = self.records[self.pos]\\n        recorded = Action.from_json(rec[\\\"action\\\"])\\n        if rec[\\\"kind\\\"] == \\\"reset\\\":\\n            if action.type != \\\"RESET\\\" and not self.lenient:\\n                return RawTransition(before, action, before, st, error=\\\"replay expects RESET\\\", step_idx=self.step_idx, level=self.level)\\n            frame = self.reset(); self.actions_used += 1\\n            return RawTransition(before, Action.reset(), frame, self.status(), None, [], self.available_actions(), step_idx=self.step_idx, level=self.level)\\n        if recorded != action and not self.lenient:\\n            return RawTransition(before, action, before, st, error=f\\\"replay expects {recorded.label()}, got {action.label()}\\\", step_idx=self.step_idx, level=self.level)\\n        self.pos += 1; self.step_idx += 1; self.actions_used += 1\\n        level_before = self.level\\n        after, inters, change = self._serve(rec, recorded)\\n        self.frame = after\\n        return RawTransition(before, recorded, after, self.status(), change, inters, self.available_actions(), step_idx=self.step_idx, level=level_before)\\n\\n    def available_actions(self) -> list[Action]:\\n        out = [Action.button(i) for i in self._avail if i in BUTTON_IDS]\\n        if CLICK_ID in self._avail:\\n            out.append(Action(\\\"CLICK\\\"))\\n        return out\\n\\n    def status(self) -> EnvStatus:\\n        return EnvStatus(self.state, self.level, self.levels_total, self.actions_used, None, 0.0, None)\\n\\n    def exhausted(self) -> bool:\\n        return self.pos >= len(self.records)\\n\", \"pbg/env/wrapper.py\": \"\\\"\\\"\\\"env/wrapper.py \\u2014 Env Wrapper (spec \\u00a75): wraps the environment API so upper layers never see the raw interface.\\n\\n* multi-frame responses: the last stable frame is `after`, the rest go to intermediate_frames\\n* click coordinates are grid cells (calibrate_click confirms scale/offset)\\n* determinism: stochasticity_score = fraction of (frame_hash, action) repeats whose result hash differed\\n* budget metering: actions used, remaining, whether RESET costs an action\\n* step() never raises: environment errors land in RawTransition.error\\n* every raw request/response is appended to episodic/{game_id}/raw.jsonl (input of ReplayEnv)\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport time\\nfrom dataclasses import dataclass, field\\nfrom pathlib import Path\\nfrom typing import Literal, Optional\\n\\nimport numpy as np\\n\\nfrom ..core.types import Action, Frame, grid_hash, grid_to_rows\\n\\nBUTTON_IDS = (1, 2, 3, 4, 5, 7)\\nCLICK_ID = 6\\n\\n\\n@dataclass\\nclass EnvStatus:\\n    state: Literal[\\\"NOT_PLAYED\\\", \\\"NOT_FINISHED\\\", \\\"WIN\\\", \\\"GAME_OVER\\\"]\\n    level: int\\n    levels_total: Optional[int]\\n    actions_used: int\\n    budget_total: Optional[int]\\n    stochasticity_score: float\\n    reset_costs_action: Optional[bool] = None\\n\\n    def to_json(self) -> dict:\\n        return self.__dict__.copy()\\n\\n\\n@dataclass\\nclass RawTransition:\\n    before: Optional[Frame]\\n    action: Action\\n    after: Optional[Frame]\\n    status: EnvStatus\\n    status_change: Optional[Literal[\\\"LEVEL_UP\\\", \\\"WIN\\\", \\\"GAME_OVER\\\"]] = None\\n    intermediate_frames: list[Frame] = field(default_factory=list)\\n    available_actions: list[Action] = field(default_factory=list)\\n    error: Optional[str] = None\\n    step_idx: int = 0\\n    level: int = 1\\n\\n\\ndef _state_name(raw_state) -> str:\\n    return str(raw_state).split(\\\".\\\")[-1]\\n\\n\\ndef stable_frames(grids: list[np.ndarray]) -> tuple[np.ndarray, list[np.ndarray]]:\\n    \\\"\\\"\\\"Return (after, intermediates): the LAST frame of a multi-frame response is the state after the action; every earlier\\n    frame is animation. (Picking an earlier pair of identical frames chose a flash frame in ls20's death animation.)\\\"\\\"\\\"\\n    if not grids:\\n        raise ValueError(\\\"empty frame list\\\")\\n    return grids[-1], list(grids[:-1])\\n\\n\\nclass EnvWrapper:\\n    \\\"\\\"\\\"Abstract interface (spec \\u00a75). Concrete: ArcadeEnv (offline/online arc_agi engine) and ReplayEnv.\\\"\\\"\\\"\\n\\n    game_id: str = \\\"\\\"\\n\\n    def reset(self, game_id: Optional[str] = None) -> Frame: raise NotImplementedError\\n    def step(self, action: Action) -> RawTransition: raise NotImplementedError\\n    def available_actions(self) -> list[Action]: raise NotImplementedError\\n    def status(self) -> EnvStatus: raise NotImplementedError\\n    def calibrate_click(self) -> dict: return {\\\"scale\\\": 1, \\\"offset\\\": (0, 0), \\\"verified\\\": False}\\n    def close(self) -> None: pass\\n\\n\\nclass ArcadeEnv(EnvWrapper):\\n    \\\"\\\"\\\"arc_agi / arcengine environment. `env` is the object returned by Arcade.make(game_id).\\\"\\\"\\\"\\n\\n    def __init__(self, env, game_id: str, *, log_dir: Optional[Path] = None, budget_total: Optional[int] = None, replay_path: Optional[Path] = None):\\n        self.env, self.game_id = env, game_id\\n        self.log_path = (Path(log_dir) / \\\"raw.jsonl\\\") if log_dir else None\\n        if self.log_path:\\n            self.log_path.parent.mkdir(parents=True, exist_ok=True)\\n        # per-run action record in the rulebook format (results/logs/<run>/<game>.actions.jsonl): one line per reset/step with a\\n        # frame hash, so the eval viewer (scripts/eval_viewer) can replay the game against the offline engine and check every board\\n        self.replay_path = Path(replay_path) if replay_path else None\\n        if self.replay_path:\\n            self.replay_path.parent.mkdir(parents=True, exist_ok=True)\\n        self.attempt = 0\\n        self.budget_total = budget_total\\n        self.frame: Optional[Frame] = None\\n        self.state = \\\"NOT_PLAYED\\\"\\n        self.level, self.levels_total = 1, None\\n        self._avail: list[int] = []\\n        self.actions_used = 0\\n        self.step_idx = 0\\n        self._results: dict[tuple[str, str], set[str]] = {}\\n        self._repeat_total = self._repeat_diff = 0\\n        self.reset_costs_action: Optional[bool] = None\\n        self.t0 = time.time()\\n        self.click_calibration = {\\\"scale\\\": 1, \\\"offset\\\": (0, 0), \\\"verified\\\": False}\\n\\n    # \\u2500\\u2500 engine glue \\u2500\\u2500\\n    @staticmethod\\n    def _engine_action(action: Action):\\n        from arcengine import GameAction\\n        if action.type == \\\"RESET\\\":\\n            return GameAction.RESET, None\\n        if action.type == \\\"CLICK\\\":\\n            ga = GameAction.ACTION6\\n            ga.set_data({\\\"x\\\": int(action.col), \\\"y\\\": int(action.row)})\\n            return ga, ga.action_data.model_dump()\\n        return getattr(GameAction, f\\\"ACTION{action.id}\\\"), None\\n\\n    def _apply(self, raw, action: Action) -> tuple[Frame, list[Frame], Optional[str]]:\\n        prev_level, prev_state = self.level, self.state\\n        self.state = _state_name(raw.state)\\n        self.levels_total = int(raw.win_levels or 0) or self.levels_total\\n        self.level = int(raw.levels_completed) + 1\\n        self._avail = [int(a) for a in (raw.available_actions or [])]\\n        grids = [np.asarray(g, dtype=np.int8) for g in (raw.frame or [])]\\n        if not grids and self.frame is not None:\\n            grids = [self.frame.grid]\\n        after_grid, inter = stable_frames(grids)\\n        after = Frame(after_grid, self.level, self.step_idx)\\n        inters = [Frame(g, self.level, self.step_idx) for g in inter]\\n        change = None\\n        if self.state == \\\"WIN\\\":\\n            change = \\\"WIN\\\"\\n        elif self.state == \\\"GAME_OVER\\\":\\n            change = \\\"GAME_OVER\\\"\\n        elif self.level > prev_level and prev_state != \\\"NOT_PLAYED\\\":\\n            change = \\\"LEVEL_UP\\\"\\n        self._record({\\\"kind\\\": action.type.lower() if action.type == \\\"RESET\\\" else \\\"step\\\", \\\"step_idx\\\": self.step_idx, \\\"action\\\": action.to_json(),\\n                      \\\"frames\\\": [grid_to_rows(g) for g in grids], \\\"state\\\": self.state, \\\"level\\\": self.level, \\\"levels_total\\\": self.levels_total,\\n                      \\\"available_actions\\\": self._avail, \\\"status_change\\\": change, \\\"t\\\": round(time.time() - self.t0, 3)})\\n        self._record_replay(action, grids[-1] if grids else None, prev_level, prev_state, change)\\n        return after, inters, change\\n\\n    _REPLAY_LABEL = {1: \\\"UP\\\", 2: \\\"DOWN\\\", 3: \\\"LEFT\\\", 4: \\\"RIGHT\\\", 5: \\\"SPACE\\\", 7: \\\"ACTION7\\\"}\\n\\n    def _record_replay(self, action: Action, last_grid, prev_level: int, prev_state: str, change: Optional[str]) -> None:\\n        \\\"\\\"\\\"Append one rulebook-compatible line (see rulebook.env.Game._record): the viewer replays these against the engine.\\\"\\\"\\\"\\n        if self.replay_path is None:\\n            return\\n        h = hashlib.sha1(\\\"\\\\n\\\".join(grid_to_rows(last_grid)).encode()).hexdigest()[:12] if last_grid is not None else None\\n        now = time.time()\\n        base = {\\\"t\\\": round(now - self.t0, 2), \\\"clock\\\": time.strftime(\\\"%H:%M:%S\\\", time.localtime(now)), \\\"hash\\\": h, \\\"state\\\": self.state, \\\"levels_total\\\": self.levels_total}\\n        if action.type == \\\"RESET\\\":\\n            self.attempt += 1\\n            rec = {\\\"kind\\\": \\\"reset\\\", \\\"step\\\": self.step_idx, \\\"level\\\": self.level, \\\"attempt\\\": self.attempt, **base}\\n        else:\\n            label = {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": int(action.row), \\\"col\\\": int(action.col)} if action.type == \\\"CLICK\\\" else self._REPLAY_LABEL.get(action.id, f\\\"ACTION{action.id}\\\")\\n            completed = change in (\\\"LEVEL_UP\\\", \\\"WIN\\\")\\n            rec = {\\\"kind\\\": \\\"step\\\", \\\"step\\\": self.step_idx, \\\"action\\\": label, \\\"level\\\": prev_level, \\\"attempt\\\": self.attempt,\\n                   \\\"level_completed\\\": completed, \\\"game_over\\\": change == \\\"GAME_OVER\\\", \\\"won\\\": change == \\\"WIN\\\", **base}\\n            if completed:\\n                self.attempt += 1\\n        with open(self.replay_path, \\\"a\\\") as f:\\n            f.write(json.dumps(rec) + \\\"\\\\n\\\")\\n\\n    def _record(self, rec: dict) -> None:\\n        if self.log_path:\\n            with open(self.log_path, \\\"a\\\") as f:\\n                f.write(json.dumps(rec) + \\\"\\\\n\\\")\\n\\n    # \\u2500\\u2500 interface \\u2500\\u2500\\n    def reset(self, game_id: Optional[str] = None) -> Frame:\\n        from arcengine import GameAction\\n        before_used = self.actions_used\\n        raw = self.env.step(GameAction.RESET)\\n        self.frame, _, _ = self._apply(raw, Action.reset())\\n        # the engine counts RESET as an action in its action counters; we keep our own metering\\n        if self.state != \\\"NOT_PLAYED\\\" and before_used > 0 and self.reset_costs_action is None:\\n            self.reset_costs_action = True\\n        return self.frame\\n\\n    def step(self, action: Action) -> RawTransition:\\n        before = self.frame\\n        st = self.status()\\n        if action.type == \\\"RESET\\\":\\n            try:\\n                frame = self.reset()\\n            except Exception as e:\\n                return RawTransition(before, action, before, st, error=f\\\"{type(e).__name__}: {e}\\\", step_idx=self.step_idx, level=self.level)\\n            self.actions_used += 1\\n            return RawTransition(before, action, frame, self.status(), None, [], self.available_actions(), step_idx=self.step_idx, level=self.level)\\n        if action.type == \\\"CLICK\\\" and CLICK_ID not in self._avail or action.type == \\\"BUTTON\\\" and action.id not in self._avail:\\n            return RawTransition(before, action, before, st, error=f\\\"action {action.label()} not available ({self._avail})\\\", step_idx=self.step_idx, level=self.level)\\n        try:\\n            ga, data = self._engine_action(action)\\n            raw = self.env.step(ga, data=data) if data is not None else self.env.step(ga)\\n        except Exception as e:\\n            return RawTransition(before, action, before, st, error=f\\\"{type(e).__name__}: {e}\\\", step_idx=self.step_idx, level=self.level)\\n        self.step_idx += 1; self.actions_used += 1\\n        level_before = self.level\\n        after, inters, change = self._apply(raw, action)\\n        self.frame = after\\n        if before is not None:\\n            key = (before.hash, action.label())\\n            seen = self._results.setdefault(key, set())\\n            if seen:\\n                self._repeat_total += 1\\n                if after.hash not in seen:\\n                    self._repeat_diff += 1\\n            seen.add(after.hash)\\n        return RawTransition(before, action, after, self.status(), change, inters, self.available_actions(), step_idx=self.step_idx, level=level_before)\\n\\n    def available_actions(self) -> list[Action]:\\n        out = [Action.button(i) for i in self._avail if i in BUTTON_IDS]\\n        if CLICK_ID in self._avail:\\n            out.append(Action(\\\"CLICK\\\"))\\n        return out\\n\\n    def status(self) -> EnvStatus:\\n        score = self._repeat_diff / self._repeat_total if self._repeat_total else 0.0\\n        return EnvStatus(self.state, self.level, self.levels_total, self.actions_used, self.budget_total, round(score, 4), self.reset_costs_action)\\n\\n    def calibrate_click(self) -> dict:\\n        \\\"\\\"\\\"The arcengine click space is the 64x64 grid itself (x=col, y=row, 0..63): scale 1, offset 0.\\n        Verified structurally (no action spent): the API contract states cell units, so corner clicks are not needed.\\\"\\\"\\\"\\n        self.click_calibration = {\\\"scale\\\": 1, \\\"offset\\\": (0, 0), \\\"verified\\\": True, \\\"method\\\": \\\"api-contract\\\"}\\n        return self.click_calibration\\n\", \"pbg/goal/__init__.py\": \"from .templates import GoalInstance, TEMPLATES, instantiate_all  # noqa: F401\\nfrom .filter import filter_by_levelup  # noqa: F401\\nfrom .infer import GoalInference  # noqa: F401\\n\", \"pbg/goal/filter.py\": \"\\\"\\\"\\\"goal/filter.py \\u2014 level-up back-inference filter (spec \\u00a79): keep predicates true right before/after a LEVEL_UP\\nand false in the previous 10 distinct states.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Iterable\\n\\nfrom ..core.types import Transition\\nfrom .templates import GoalInstance\\n\\n\\ndef filter_by_levelup(G: list[GoalInstance], log: Iterable[Transition], *, prior_window: int = 10, roles_fn=None, model=None) -> list[GoalInstance]:\\n    \\\"\\\"\\\"At a LEVEL_UP the frame returned is the NEXT level's board, so the winning state is never observed directly: a goal\\n    is kept when it holds on the last state of the level, or on the model's prediction of the winning action's result.\\\"\\\"\\\"\\n    log = list(log)\\n    for idx, t in enumerate(log):\\n        if t.status_change not in (\\\"LEVEL_UP\\\", \\\"WIN\\\"):\\n            continue\\n        before = roles_fn(t.before) if roles_fn else t.before\\n        win_states = [before]\\n        ext = _extrapolate(t, log[:idx])\\n        if ext is not None:\\n            win_states.append(roles_fn(ext) if roles_fn else ext)\\n        if model is not None:\\n            try:\\n                pred = model.predict(t.before, t.action)\\n                if pred is not None:\\n                    win_states.append(roles_fn(pred) if roles_fn else pred)\\n            except Exception:\\n                pass\\n        G = [g for g in G if any(g.is_goal(w) for w in win_states)]\\n        prior = [x for x in log[:idx] if x.before.frame_hash != t.before.frame_hash and x.level == t.level]\\n        seen = set(); distinct = []\\n        for x in reversed(prior):\\n            if x.before.frame_hash in seen:\\n                continue\\n            seen.add(x.before.frame_hash); distinct.append(x)\\n            if len(distinct) >= prior_window:\\n                break\\n        G = [g for g in G if not any(g.is_goal(roles_fn(x.before) if roles_fn else x.before) for x in distinct)]\\n        for g in G:\\n            g.confidence = min(1.0, g.confidence + 0.3)\\n    return G\\n\\n\\ndef _extrapolate(t: Transition, earlier: list[Transition]):\\n    \\\"\\\"\\\"The winning action's result is never observed; if the same action (same button / same click cell) always moved the\\n    same objects by the same displacement earlier in the level, apply that displacement to the last state.\\\"\\\"\\\"\\n    same = [x for x in earlier if x.level == t.level and x.action.label() == t.action.label() and not x.diff.is_noop and x.diff.moved\\n            and not x.diff.appeared and not x.diff.disappeared]\\n    if len(same) < 2:\\n        return None\\n    disp = {}\\n    for x in same:\\n        for i, v in x.diff.moved:\\n            o = x.before.get(i)\\n            if o is None:\\n                continue\\n            disp.setdefault((o.color, o.shape_sig), set()).add(v)\\n    consistent = {k: next(iter(v)) for k, v in disp.items() if len(v) == 1}\\n    if not consistent:\\n        return None\\n    objs = []; changed = False\\n    for o in t.before.objects:\\n        d = consistent.get((o.color, o.shape_sig))\\n        if d is not None:\\n            objs.append(o.moved(*d)); changed = True\\n        else:\\n            objs.append(o)\\n    return t.before.copy(objects=objs) if changed else None\\n\", \"pbg/goal/infer.py\": \"\\\"\\\"\\\"goal/infer.py \\u2014 Goal Inference (spec \\u00a79): template instantiation, level-up filtering, confidence, LLM template\\nproposals when no candidate exists.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Optional\\n\\nfrom ..core.types import Scene, Transition\\nfrom .filter import filter_by_levelup\\nfrom .templates import GoalInstance, instantiate_all\\n\\n\\nclass GoalInference:\\n    def __init__(self, memory=None, llm=None, sandbox=None, log=None):\\n        self.memory, self.llm, self.sandbox = memory, llm, sandbox\\n        self.log = log or (lambda *a, **k: None)\\n        self.demoted: dict[str, float] = {}\\n        self.knowledge = None                     # LevelKnowledge of the game (set by the orchestrator)\\n        self.llm_calls = 0\\n        self._proposals: dict[int, int] = {}      # level -> LLM goal proposals made (procedural goals the templates cannot express)\\n\\n    def refine(self, log: list[Transition], current: list[GoalInstance], priors: Optional[dict], level: int, scene: Scene,\\n               *, roles_fn=None, ctx: Optional[dict] = None, model=None) -> list[GoalInstance]:\\n        usage = (priors or {}).get(\\\"usage_stats\\\") if priors else None\\n        if self.knowledge is not None:\\n            usage = self.knowledge.goal_stats(usage)      # what won earlier levels of THIS game comes first\\n        s = roles_fn(scene) if roles_fn else scene\\n        fresh = instantiate_all(s, ctx, usage)\\n        by_name = {g.name: g for g in fresh}\\n        # keep confidence / history of goals we already had\\n        for g in current:\\n            if g.name in by_name:\\n                by_name[g.name].confidence = max(by_name[g.name].confidence, g.confidence)\\n                by_name[g.name].progress_weight = g.progress_weight\\n            elif g.origin == \\\"llm\\\":\\n                by_name[g.name] = g\\n        G = list(by_name.values())\\n        if self.knowledge is not None:\\n            for g in G:\\n                g.confidence = min(1.0, g.confidence + self.knowledge.goal_structure_bonus(g))\\n        for name, pen in self.demoted.items():\\n            for g in G:\\n                if g.name == name:\\n                    g.confidence = max(0.0, g.confidence + pen)\\n        # no template goal, or every template goal was demoted (reached without a level-up): the win condition is\\n        # probably procedural (a sequence, a pattern rule) -> ask the LLM, at most twice per level\\n        exhausted = bool(G) and max(g.confidence for g in G) < 0.25 and not any(g.origin == \\\"llm\\\" for g in G)\\n        if (not G or exhausted) and self.llm is not None and self._proposals.get(level, 0) < 2:\\n            self._proposals[level] = self._proposals.get(level, 0) + 1\\n            rejected = [g.name for g in G if self.demoted.get(g.name, 0.0) < 0]\\n            G = G + self.propose_with_llm(s, log, rejected=rejected)\\n        before = list(G)\\n        G = filter_by_levelup(G, log, roles_fn=roles_fn, model=model)\\n        if not G and before:\\n            # the level-up filter removed every candidate (true goal not in the library yet): keep them at half confidence\\n            for g in before:\\n                g.confidence *= 0.5\\n            G = before\\n        G.sort(key=lambda g: (-g.confidence, -g.progress(s), g.name))\\n        return G\\n\\n    def demote(self, G: list[GoalInstance], idx: int = 0, penalty: float = -0.4) -> list[GoalInstance]:\\n        \\\"\\\"\\\"Planner found no plan under a verified model: the top goal is probably wrong (spec \\u00a710).\\\"\\\"\\\"\\n        if not G:\\n            return G\\n        g = G[idx]\\n        g.confidence = max(0.0, g.confidence + penalty)\\n        self.demoted[g.name] = self.demoted.get(g.name, 0.0) + penalty\\n        G.sort(key=lambda x: -x.confidence)\\n        return G\\n\\n    def observe_progress(self, G: list[GoalInstance], scene: Scene) -> None:\\n        \\\"\\\"\\\"Track monotonicity of the top goal's progress (non-monotone -> weight 0.5, spec \\u00a79).\\\"\\\"\\\"\\n        for g in G[:2]:\\n            p = g.progress(scene)\\n            g.history.append(p)\\n            g.history = g.history[-15:]\\n            if len(g.history) >= 6 and any(g.history[i] < g.history[i - 1] - 1e-9 for i in range(1, len(g.history))):\\n                g.progress_weight = 0.5\\n\\n    def propose_with_llm(self, scene: Scene, log: list[Transition], rejected: Optional[list[str]] = None) -> list[GoalInstance]:\\n        from ..wml.prompts import goal_template_prompt\\n        from ..wml.context import observation_lines, select_context\\n        from ..perception.summarize import scene_text\\n        if self.llm is None or self.sandbox is None:\\n            return []\\n        ctx = select_context(log, [], cap=30)\\n        msgs = goal_template_prompt(scene_text(scene), observation_lines(ctx), rejected=rejected or [])\\n        try:\\n            reply = self.llm.chat(msgs, purpose=\\\"goal_template\\\", image=None)\\n            self.llm_calls += 1\\n        except Exception as e:\\n            self.log(f\\\"goal llm failed: {e!r}\\\")\\n            return []\\n        code = self.llm.extract_code(reply)\\n        if not code:\\n            return []\\n        goal = self.sandbox.load_goal(code)\\n        if goal is None:\\n            return []\\n        goal.origin = \\\"llm\\\"; goal.code = code; goal.confidence = 0.3      # a proposal, ranked below template goals until the level-up filter confirms it\\n        if goal.is_goal(scene):\\n            return []\\n        p = goal.progress(scene)\\n        if not (0.0 <= p < 1.0):\\n            return []\\n        return [goal]\\n\\n    @staticmethod\\n    def _summ(t: Transition) -> str:\\n        from ..perception.summarize import summarize\\n        return summarize(t)\\n\", \"pbg/goal/templates/__init__.py\": \"\\\"\\\"\\\"goal/templates \\u2014 the initial goal template library (spec \\u00a79). Each template instantiates GoalInstances on a scene:\\nis_goal(scene) -> bool, progress(scene) -> [0,1], plus a clue bonus from scene structure.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom dataclasses import dataclass, field\\nfrom typing import Callable, Optional\\n\\nimport numpy as np\\n\\nfrom ...core.types import Action, Scene\\n\\n\\n@dataclass\\nclass GoalInstance:\\n    name: str                     # e.g. \\\"reach(agent,target)\\\"\\n    template: str\\n    params: dict\\n    is_goal_fn: Callable[[Scene], bool]\\n    progress_fn: Callable[[Scene], float]\\n    confidence: float = 0.5\\n    clue: float = 0.0\\n    origin: str = \\\"template\\\"      # template | llm\\n    code: str = \\\"\\\"\\n    progress_weight: float = 1.0  # lowered to 0.5 when progress is not monotone during execution\\n    history: list = field(default_factory=list)\\n    estimate_fn: Optional[Callable[[Scene], Optional[float]]] = None   # remaining actions estimate (A* heuristic), if the template can compute one\\n\\n    def estimate(self, scene: Scene) -> Optional[float]:\\n        if self.estimate_fn is None:\\n            return None\\n        try:\\n            return self.estimate_fn(scene)\\n        except Exception:\\n            return None\\n\\n    def is_goal(self, scene: Scene) -> bool:\\n        try:\\n            return bool(self.is_goal_fn(scene))\\n        except Exception:\\n            return False\\n\\n    def progress(self, scene: Scene) -> float:\\n        try:\\n            return max(0.0, min(1.0, float(self.progress_fn(scene))))\\n        except Exception:\\n            return 0.0\\n\\n    def to_json(self) -> dict:\\n        return {\\\"name\\\": self.name, \\\"template\\\": self.template, \\\"params\\\": self.params, \\\"confidence\\\": round(self.confidence, 3), \\\"clue\\\": self.clue, \\\"origin\\\": self.origin}\\n\\n\\n# \\u2500\\u2500 helpers \\u2500\\u2500\\ndef _dist(a, b) -> int:\\n    return abs(a[0] - b[0]) + abs(a[1] - b[1])\\n\\n\\ndef _max_dist(scene: Scene) -> int:\\n    return scene.grid_shape[0] + scene.grid_shape[1]\\n\\n\\ndef _roles(scene: Scene) -> set[str]:\\n    return {o.role for o in scene.objects if o.role and o.role != \\\"unknown\\\"}\\n\\n\\ndef _agents(s: Scene) -> list:\\n    \\\"\\\"\\\"Objects whose role is 'agent' or a custom variant containing 'agent' (agent_left, custom:agent2 ...).\\\"\\\"\\\"\\n    return [o for o in s.objects if o.role and \\\"agent\\\" in o.role]\\n\\n\\ndef _shape_multiset(scene: Scene, region: str) -> Counter:\\n    return Counter((o.color, o.shape_sig) for o in scene.in_region(region))\\n\\n\\n# \\u2500\\u2500 templates \\u2500\\u2500\\ndef _targets(s: Scene, b: str) -> list:\\n    \\\"\\\"\\\"Objects of role b, or of colour c when b == 'color:c' (unknown-role objects grouped by colour).\\\"\\\"\\\"\\n    if b.startswith(\\\"color:\\\"):\\n        c = int(b[6:])\\n        strips = {r.id for r in s.regions if r.kind_hint == \\\"ui_strip\\\"}\\n        return [o for o in s.objects if o.color == c and (o.role in (None, \\\"unknown\\\")) and o.region not in strips]\\n    return s.by_role(b)\\n\\n\\ndef t_reach(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    out = []\\n    roles = _roles(scene)\\n    if not _agents(scene):\\n        return out\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    targets = sorted(r for r in roles - {\\\"wall\\\", \\\"decoration\\\", \\\"indicator\\\"} if \\\"agent\\\" not in r)\\n    targets += sorted({f\\\"color:{o.color}\\\" for o in scene.objects if o.role in (None, \\\"unknown\\\") and o.region not in strips})\\n    a = \\\"agent\\\"\\n    for b in targets:\\n        def is_goal(s, b=b):\\n            return any(x.overlaps(y) or _adjacent(x, y) for x in _agents(s) for y in _targets(s, b))\\n        def progress(s, b=b):\\n            xs, ys = _agents(s), _targets(s, b)\\n            if not xs or not ys:\\n                return 0.0\\n            d = min(_dist(x.center, y.center) for x in xs for y in ys)\\n            return 1.0 - d / _max_dist(s)\\n        conf_penalty = 0.15 if b.startswith(\\\"color:\\\") else 0.0\\n        out.append(GoalInstance(f\\\"reach({a},{b})\\\", \\\"reach\\\", {\\\"a\\\": a, \\\"b\\\": b}, is_goal, progress, clue=-conf_penalty))\\n    return out\\n\\n\\ndef t_match_shapes(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    out = []\\n    regs = [r for r in scene.regions if r.kind_hint != \\\"ui_strip\\\" and r.id != \\\"R0\\\"]\\n    if len(regs) < 2:\\n        return out\\n    for src in regs:\\n        for dst in regs:\\n            if src.id == dst.id:\\n                continue\\n            ms_src, ms_dst = _shape_multiset(scene, src.id), _shape_multiset(scene, dst.id)\\n            if not ms_dst or not ms_src:\\n                continue\\n            shared = {k[1] for k in ms_src if k[1] != \\\"empty\\\"} & {k[1] for k in ms_dst}\\n            clue = 0.2 if len(shared) >= 2 else (0.1 if shared else 0.0)\\n            def is_goal(s, src=src.id, dst=dst.id):\\n                a, b = _shape_multiset(s, src), _shape_multiset(s, dst)\\n                return bool(b) and a == b\\n            def progress(s, src=src.id, dst=dst.id):\\n                a, b = _shape_multiset(s, src), _shape_multiset(s, dst)\\n                if not b:\\n                    return 0.0\\n                return sum((a & b).values()) / sum(b.values())\\n            out.append(GoalInstance(f\\\"match_shapes({src.id},{dst.id})\\\", \\\"match_shapes\\\", {\\\"src\\\": src.id, \\\"dst\\\": dst.id}, is_goal, progress, clue=clue))\\n    return out\\n\\n\\ndef t_all_removed(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    out = []\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    counts = Counter(o.color for o in scene.objects if o.region not in strips)\\n    for color, n in counts.items():\\n        if n == 0:\\n            continue\\n        def is_goal(s, color=color):\\n            return not any(o.color == color for o in s.objects if o.region not in strips)\\n        def progress(s, color=color, n=n):\\n            k = sum(1 for o in s.objects if o.color == color and o.region not in strips)\\n            return 1.0 - min(k, n) / n\\n        out.append(GoalInstance(f\\\"all_removed({color})\\\", \\\"all_removed\\\", {\\\"color\\\": color}, is_goal, progress))\\n    return out\\n\\n\\ndef t_all_collected(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    out = []\\n    for role in (\\\"collectible\\\", \\\"target\\\", \\\"key\\\"):\\n        n = len(scene.by_role(role))\\n        if n == 0:\\n            continue\\n        def is_goal(s, role=role):\\n            return len(s.by_role(role)) == 0\\n        def progress(s, role=role, n=n):\\n            return 1.0 - min(len(s.by_role(role)), n) / n\\n        out.append(GoalInstance(f\\\"all_collected({role})\\\", \\\"all_collected\\\", {\\\"role\\\": role}, is_goal, progress))\\n    return out\\n\\n\\ndef t_fill_region(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    out = []\\n    for reg in scene.regions:\\n        if reg.kind_hint == \\\"ui_strip\\\" or reg.id == \\\"R0\\\":\\n            continue\\n        objs = scene.in_region(reg.id)\\n        if not objs:\\n            continue\\n        for color in {o.color for o in objs}:\\n            def is_goal(s, rid=reg.id, color=color):\\n                objs = s.in_region(rid)\\n                return bool(objs) and all(o.color == color for o in objs) and sum(o.area for o in objs) >= 0.9 * (s.region(rid).area if s.region(rid) else 1)\\n            def progress(s, rid=reg.id, color=color):\\n                reg2 = s.region(rid)\\n                if reg2 is None:\\n                    return 0.0\\n                return min(1.0, sum(o.area for o in s.in_region(rid) if o.color == color) / max(1, reg2.area))\\n            out.append(GoalInstance(f\\\"fill_region({reg.id},{color})\\\", \\\"fill_region\\\", {\\\"region\\\": reg.id, \\\"color\\\": color}, is_goal, progress))\\n    return out\\n\\n\\ndef t_sort_by(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    out = []\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    objs = [o for o in scene.objects if o.region not in strips]\\n    if len(objs) < 3:\\n        return out\\n    for axis in (\\\"row\\\", \\\"col\\\"):\\n        for key in (\\\"color\\\", \\\"size\\\"):\\n            def order(s, axis=axis, key=key):\\n                os_ = [o for o in s.objects if o.region not in strips]\\n                os_.sort(key=lambda o: o.center[0] if axis == \\\"row\\\" else o.center[1])\\n                return [o.color if key == \\\"color\\\" else o.area for o in os_]\\n            def is_goal(s, order=order):\\n                v = order(s)\\n                return len(v) >= 3 and v == sorted(v)\\n            def progress(s, order=order):\\n                v = order(s)\\n                if len(v) < 2:\\n                    return 0.0\\n                return sum(1 for i in range(1, len(v)) if v[i] >= v[i - 1]) / (len(v) - 1)\\n            out.append(GoalInstance(f\\\"sort_by({axis},{key})\\\", \\\"sort_by\\\", {\\\"axis\\\": axis, \\\"key\\\": key}, is_goal, progress))\\n    return out\\n\\n\\ndef t_count_equals(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    out = []\\n    for reg in scene.regions:\\n        if reg.kind_hint != \\\"ui_strip\\\":\\n            continue\\n        n_now = sum(o.area for o in scene.in_region(reg.id))\\n        for n in (0,):\\n            def is_goal(s, rid=reg.id, n=n):\\n                return sum(o.area for o in s.in_region(rid)) == n\\n            def progress(s, rid=reg.id, n=n, n0=n_now):\\n                k = sum(o.area for o in s.in_region(rid))\\n                return 1.0 - abs(k - n) / max(1, abs(n0 - n))\\n            out.append(GoalInstance(f\\\"count_equals({reg.id},{n})\\\", \\\"count_equals\\\", {\\\"indicator\\\": reg.id, \\\"n\\\": n}, is_goal, progress, confidence=0.2))\\n    return out\\n\\n\\ndef t_align(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    out = []\\n    roles = sorted(_roles(scene) - {\\\"wall\\\", \\\"decoration\\\", \\\"indicator\\\"})\\n    for i, a in enumerate(roles):\\n        for b in roles[i + 1:]:\\n            for axis in (\\\"row\\\", \\\"col\\\"):\\n                def is_goal(s, a=a, b=b, axis=axis):\\n                    xs, ys = s.by_role(a), s.by_role(b)\\n                    k = 0 if axis == \\\"row\\\" else 1\\n                    return bool(xs and ys) and all(any(x.center[k] == y.center[k] for y in ys) for x in xs)\\n                def progress(s, a=a, b=b, axis=axis):\\n                    xs, ys = s.by_role(a), s.by_role(b)\\n                    if not xs or not ys:\\n                        return 0.0\\n                    k = 0 if axis == \\\"row\\\" else 1\\n                    d = sum(min(abs(x.center[k] - y.center[k]) for y in ys) for x in xs) / len(xs)\\n                    return 1.0 - d / s.grid_shape[k]\\n                out.append(GoalInstance(f\\\"align({a},{b},{axis})\\\", \\\"align\\\", {\\\"a\\\": a, \\\"b\\\": b, \\\"axis\\\": axis}, is_goal, progress))\\n    return out\\n\\n\\ndef t_enclose(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    out = []\\n    roles = _roles(scene)\\n    if \\\"target\\\" not in roles:\\n        return out\\n    for wall_color in {o.color for o in scene.objects if o.role == \\\"wall\\\"}:\\n        def is_goal(s, wc=wall_color):\\n            g = s.render()\\n            for t in s.by_role(\\\"target\\\"):\\n                r0, c0, r1, c1 = t.bbox\\n                ring = []\\n                for r in range(r0 - 1, r1 + 1):\\n                    for c in (c0 - 1, c1):\\n                        if 0 <= r < g.shape[0] and 0 <= c < g.shape[1]:\\n                            ring.append(g[r, c])\\n                for c in range(c0, c1):\\n                    for r in (r0 - 1, r1):\\n                        if 0 <= r < g.shape[0] and 0 <= c < g.shape[1]:\\n                            ring.append(g[r, c])\\n                if not ring or any(v != wc for v in ring):\\n                    return False\\n            return True\\n        def progress(s, wc=wall_color):\\n            g = s.render(); tot = ok = 0\\n            for t in s.by_role(\\\"target\\\"):\\n                r0, c0, r1, c1 = t.bbox\\n                for r in range(r0 - 1, r1 + 1):\\n                    for c in (c0 - 1, c1):\\n                        if 0 <= r < g.shape[0] and 0 <= c < g.shape[1]:\\n                            tot += 1; ok += g[r, c] == wc\\n            return ok / tot if tot else 0.0\\n        out.append(GoalInstance(f\\\"enclose(target,{wall_color})\\\", \\\"enclose\\\", {\\\"role\\\": \\\"target\\\", \\\"wall_color\\\": wall_color}, is_goal, progress))\\n    return out\\n\\n\\ndef t_sequence(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    \\\"\\\"\\\"Puzzle-type: a specific action sequence (aux 'seq' reaches len(order)); instantiated only with a hint in ctx.\\\"\\\"\\\"\\n    order = ctx.get(\\\"sequence\\\")\\n    if not order:\\n        return []\\n    def is_goal(s, n=len(order)):\\n        return int(s.aux.get(\\\"seq\\\", 0)) >= n\\n    def progress(s, n=len(order)):\\n        return min(1.0, int(s.aux.get(\\\"seq\\\", 0)) / n)\\n    return [GoalInstance(f\\\"sequence({len(order)})\\\", \\\"sequence\\\", {\\\"actions\\\": list(order)}, is_goal, progress)]\\n\\n\\ndef _frames(scene: Scene) -> list:\\n    \\\"\\\"\\\"Frame-like objects: hollow (mask has an interior hole) and larger than 3x3.\\\"\\\"\\\"\\n    out = []\\n    for o in scene.objects:\\n        if o.height >= 3 and o.width >= 3 and o.area < o.height * o.width * 0.8:\\n            inner = o.mask[1:-1, 1:-1]\\n            if inner.size and not inner.all():\\n                out.append(o)\\n    return out\\n\\n\\ndef t_inside_frame(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    \\\"\\\"\\\"Every piece of colour c sits inside a frame (a hollow object whose bbox contains the piece). Pieces = objects of\\n    a colour that also has at least one frame with a matching colour or a frame of any colour.\\\"\\\"\\\"\\n    frames = _frames(scene)\\n    if not frames:\\n        return []\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    frame_ids = {f.id for f in frames}\\n    pieces_by_color: dict = {}\\n    for o in scene.objects:\\n        if o.id in frame_ids or o.region in strips or o.area < 2 or o.height >= 12:\\n            continue\\n        pieces_by_color.setdefault(o.color, []).append(o)\\n    out = []\\n    frame_colors = {f.color for f in frames}\\n    for color, pieces in pieces_by_color.items():\\n        if len(pieces) > 16:\\n            continue\\n        def inside(p, f):\\n            return f.bbox[0] <= p.bbox[0] and f.bbox[1] <= p.bbox[1] and p.bbox[2] <= f.bbox[2] and p.bbox[3] <= f.bbox[3]\\n        def is_goal(s, color=color):\\n            fs = _frames(s); ps = [o for o in s.objects if o.color == color and o.id not in {f.id for f in fs} and o.height < 12]\\n            return bool(ps) and all(any(inside(p, f) for f in fs) for p in ps)\\n        def progress(s, color=color):\\n            fs = _frames(s); ps = [o for o in s.objects if o.color == color and o.id not in {f.id for f in fs} and o.height < 12]\\n            if not ps or not fs:\\n                return 0.0\\n            done = sum(1 for p in ps if any(inside(p, f) for f in fs))\\n            near = sum(min(_dist(p.center, f.center) for f in fs) for p in ps if not any(inside(p, f) for f in fs))\\n            return (done + max(0.0, 1.0 - near / (_max_dist(s) * max(1, len(ps) - done))) * 0.5) / len(ps)\\n        clue = 0.15 if color in frame_colors else 0.0\\n        out.append(GoalInstance(f\\\"inside_frame({color})\\\", \\\"inside_frame\\\", {\\\"color\\\": color}, is_goal, progress, clue=clue))\\n    return out\\n\\n\\ndef _mark_groups(scene: Scene) -> dict:\\n    \\\"\\\"\\\"(colour, size) -> {(top, left)} of small square marks (1x1 .. 3x3) that occur at least four times.\\\"\\\"\\\"\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    marks: dict = {}\\n    for o in scene.objects:\\n        if o.height == o.width and o.height <= 3 and o.area == o.height * o.width and o.region not in strips:\\n            marks.setdefault((o.color, o.height), set()).add((o.bbox[0], o.bbox[1]))\\n    return {k: v for k, v in marks.items() if len(v) >= 4}\\n\\n\\ndef _marked_slots(scene: Scene) -> list[tuple[int, tuple[int, int, int, int]]]:\\n    \\\"\\\"\\\"Slots marked by four small square corner marks of one colour: returns [(colour, interior bbox)].\\\"\\\"\\\"\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    marks = _mark_groups(scene)\\n    mark_keys = set(marks)\\n    piece_dims = {(o.height, o.width) for o in scene.objects if 2 <= o.area <= 64 and o.region not in strips and (o.color, o.height) not in mark_keys}\\n    slots = []\\n    for (color, k), pts in marks.items():\\n        if len(pts) < 4 or len(pts) > 64:\\n            continue\\n        rows = sorted({r for r, _ in pts}); cols = sorted({c for _, c in pts})\\n        for i, r0 in enumerate(rows):\\n            for r1 in rows[i + 1:]:\\n                if r1 - r0 < k + 1 or r1 - r0 > 16:\\n                    continue\\n                for j, c0 in enumerate(cols):\\n                    for c1 in cols[j + 1:]:\\n                        if c1 - c0 < k + 1 or c1 - c0 > 16:\\n                            continue\\n                        if {(r0, c0), (r0, c1), (r1, c0), (r1, c1)} <= pts:\\n                            inner = (r0 + k, c0 + k, r1, c1)\\n                            if (inner[2] - inner[0], inner[3] - inner[1]) in piece_dims:\\n                                slots.append((color, inner))   # interior sized like a piece\\n    return slots\\n\\n\\ndef t_fill_marked_slots(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    \\\"\\\"\\\"Every corner-marked slot holds a piece (of the marker colour when such pieces exist) \\u2014 lp85-style targets.\\\"\\\"\\\"\\n    slots = _marked_slots(scene)\\n    if not slots:\\n        return []\\n    mark_sizes = set()\\n    for (c, k), pts in ((k_, v) for k_, v in _mark_groups(scene).items()):\\n        mark_sizes.add((c, k))\\n    mark_keys = mark_sizes\\n    def pieces(s):\\n        strips = {r.id for r in s.regions if r.kind_hint == \\\"ui_strip\\\"}\\n        return [o for o in s.objects if 2 <= o.area <= 64 and o.region not in strips\\n                and not (o.height == o.width and o.height <= 3 and (o.color, o.height) in mark_keys)]\\n    def inside(p, b):\\n        return b[0] <= p.bbox[0] and b[1] <= p.bbox[1] and p.bbox[2] <= b[2] and p.bbox[3] <= b[3]\\n    colors = {c for c, _ in slots}\\n    piece_colors = {o.color for o in pieces(scene)}\\n    by_color = bool(colors & piece_colors)\\n    def filled(s, slot):\\n        color, b = slot\\n        return any(inside(p, b) and (p.color == color or not by_color) for p in pieces(s))\\n    def is_goal(s):\\n        sl = _marked_slots(s)\\n        return bool(sl) and all(filled(s, x) for x in sl)\\n    dist_cache: dict = {}\\n\\n    def graph_distance(pos, target, graph) -> Optional[int]:\\n        \\\"\\\"\\\"Fewest trigger clicks moving a piece from top-left `pos` to `target` under the learned permutations\\n        (BFS over positions, other pieces ignored: a relaxed plan).\\\"\\\"\\\"\\n        key = (pos, target)\\n        if key in dist_cache:\\n            return dist_cache[key]\\n        from collections import deque\\n        seen = {pos}; q = deque([(pos, 0)])\\n        while q:\\n            cur, d = q.popleft()\\n            if cur == target:\\n                dist_cache[key] = d; return d\\n            for mapping in graph.values():\\n                nxt = mapping.get(cur)\\n                if nxt is not None and nxt not in seen:\\n                    seen.add(nxt); q.append((nxt, d + 1))\\n        dist_cache[key] = None\\n        return None\\n\\n    def progress(s):\\n        sl = _marked_slots(s)\\n        if not sl:\\n            return 0.0\\n        done = sum(1 for x in sl if filled(s, x))\\n        ps = pieces(s); near = 0.0\\n        graph = getattr(getattr(inst, \\\"model_hint\\\", None), \\\"position_graph\\\", None)\\n        for color, b in sl:\\n            if filled(s, (color, b)):\\n                continue\\n            cands = [p for p in ps if p.color == color] if by_color else ps\\n            if not cands:\\n                continue\\n            if graph:\\n                ds = [graph_distance((p.bbox[0], p.bbox[1]), (b[0], b[1]), graph) for p in cands]\\n                ds = [d for d in ds if d is not None]\\n                if ds:\\n                    near += 1.0 / (1.0 + min(ds)); continue\\n            cr, cc = (b[0] + b[2] - 1) // 2, (b[1] + b[3] - 1) // 2\\n            near += 1.0 - min(_dist(p.center, (cr, cc)) for p in cands) / _max_dist(s)\\n        return (done + 0.9 * near) / len(sl)\\n\\n    def estimate(s):\\n        \\\"\\\"\\\"Relaxed plan length: sum over unfilled slots of the fewest clicks that bring a matching piece there.\\\"\\\"\\\"\\n        graph = getattr(getattr(inst, \\\"model_hint\\\", None), \\\"position_graph\\\", None)\\n        if not graph:\\n            return None\\n        total = 0\\n        for color, b in _marked_slots(s):\\n            if filled(s, (color, b)):\\n                continue\\n            cands = [p for p in pieces(s) if p.color == color] if by_color else pieces(s)\\n            ds = [graph_distance((p.bbox[0], p.bbox[1]), (b[0], b[1]), graph) for p in cands]\\n            ds = [d for d in ds if d is not None]\\n            if not ds:\\n                return None\\n            total += min(ds)\\n        return float(total)\\n    def abstract_plan(s, model):\\n        \\\"\\\"\\\"Exact plan over the goal-relevant pieces only: BFS on the joint top-left positions of the pieces that can fill\\n        the slots (other pieces ride along), under the learned per-trigger permutations. Returns trigger top-lefts.\\\"\\\"\\\"\\n        graph = getattr(model, \\\"position_graph\\\", None)\\n        if not graph:\\n            return None\\n        from collections import deque\\n        sl = _marked_slots(s)\\n        open_slots = [(c, b) for c, b in sl if not filled(s, (c, b))]\\n        if not open_slots:\\n            return []\\n        targets = [(b[0], b[1]) for _, b in open_slots]\\n        cands = [p for p in pieces(s) if (p.color in {c for c, _ in open_slots})] if by_color else pieces(s)\\n        cands = cands[:4]\\n        if not cands:\\n            return None\\n        start = tuple((p.bbox[0], p.bbox[1]) for p in cands)\\n        def done(state):\\n            return all(any(pos == tg for pos in state) for tg in targets)\\n        triggers = list(graph.items())\\n        seen = {start: None}; q = deque([start]); parent = {}\\n        while q:\\n            cur = q.popleft()\\n            if done(cur):\\n                path = []\\n                while cur in parent:\\n                    cur, trig = parent[cur]; path.append(trig)\\n                return list(reversed(path))\\n            for tkey, mapping in triggers:\\n                nxt = tuple(mapping.get(pos, pos) for pos in cur)\\n                if nxt not in seen:\\n                    seen[nxt] = True; parent[nxt] = (cur, tkey); q.append(nxt)\\n            if len(seen) > 200_000:\\n                return None\\n        return None\\n    inst = GoalInstance(\\\"fill_marked_slots\\\", \\\"fill_marked_slots\\\", {\\\"slots\\\": len(slots), \\\"by_color\\\": by_color}, is_goal, progress, clue=0.2, estimate_fn=estimate)\\n    inst.abstract_plan = abstract_plan\\n    return [inst]\\n\\n\\ndef t_same_cell(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    \\\"\\\"\\\"Two agent-like objects (or two objects of one colour that both move) end on the same relative position.\\\"\\\"\\\"\\n    ag = _agents(scene)\\n    if len(ag) != 2:\\n        return []\\n    def rel(o, s):\\n        reg = s.region(o.region)\\n        r0, c0 = (reg.bbox[0], reg.bbox[1]) if reg else (0, 0)\\n        return (o.bbox[0] - r0, o.bbox[1] - c0)\\n    def is_goal(s):\\n        a = _agents(s)\\n        return len(a) == 2 and (a[0].overlaps(a[1]) or _adjacent(a[0], a[1]))\\n    def progress(s):\\n        a = _agents(s)\\n        if len(a) != 2:\\n            return 0.0\\n        return 1.0 - _dist(a[0].center, a[1].center) / _max_dist(s)\\n    return [GoalInstance(\\\"same_cell(agents)\\\", \\\"same_cell\\\", {}, is_goal, progress, clue=0.1)]\\n\\n\\ndef _layout(scene: Scene, rid: str) -> frozenset:\\n    \\\"\\\"\\\"Colour layout of a region's objects relative to the region origin (for pattern matching between regions).\\\"\\\"\\\"\\n    reg = scene.region(rid)\\n    if reg is None:\\n        return frozenset()\\n    r0, c0 = reg.bbox[0], reg.bbox[1]\\n    return frozenset((o.color, o.bbox[0] - r0, o.bbox[1] - c0, o.shape_sig) for o in scene.in_region(rid))\\n\\n\\ndef t_pattern_match(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    \\\"\\\"\\\"The objects of one region reproduce the colour/shape layout of another region (board copies the panel).\\\"\\\"\\\"\\n    regs = [r for r in scene.regions if r.kind_hint in (\\\"board\\\", \\\"panel\\\") and r.id != \\\"R0\\\"]\\n    out = []\\n    for src in regs:\\n        for dst in regs:\\n            if src.id == dst.id or not scene.in_region(src.id) or not scene.in_region(dst.id):\\n                continue\\n            def is_goal(s, a=src.id, b=dst.id):\\n                la, lb = _layout(s, a), _layout(s, b)\\n                return bool(lb) and la == lb\\n            def progress(s, a=src.id, b=dst.id):\\n                la, lb = _layout(s, a), _layout(s, b)\\n                return len(la & lb) / len(lb) if lb else 0.0\\n            same_size = abs(src.area - dst.area) < 0.2 * max(src.area, dst.area)\\n            out.append(GoalInstance(f\\\"pattern_match({src.id},{dst.id})\\\", \\\"pattern_match\\\", {\\\"src\\\": src.id, \\\"dst\\\": dst.id}, is_goal, progress, clue=0.15 if same_size else 0.0))\\n    return out\\n\\n\\ndef _color_units(s: Scene, c: int) -> list:\\n    \\\"\\\"\\\"Bounding boxes (r0,c0,r1,c1) of colour c per object outside ui strips: a plain object, or the colour-c pixels of a\\n    multi-colour object (a mover's coloured tip).\\\"\\\"\\\"\\n    strips = {r.id for r in s.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    out = []\\n    for o in s.objects:\\n        if o.region in strips or c not in o.colors:\\n            continue\\n        if o.color_mask is None or len(o.colors) == 1:\\n            out.append(tuple(o.bbox)); continue\\n        ys, xs = np.nonzero(o.color_mask == c)\\n        if len(ys):\\n            out.append((o.bbox[0] + int(ys.min()), o.bbox[1] + int(xs.min()), o.bbox[0] + int(ys.max()) + 1, o.bbox[1] + int(xs.max()) + 1))\\n    return out\\n\\n\\ndef t_align_color(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    \\\"\\\"\\\"Everything of one colour lines up: a mover's coloured tip must sit in the same columns (or rows) as the marker of\\n    that colour (slider / pointer games). One colour, two or three units, at least one of them part of a larger object.\\\"\\\"\\\"\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    colors = {c for o in scene.objects if o.region not in strips for c in o.colors}\\n    pairs = [c for c in colors if _marker_pair(_color_units(scene, c))]\\n    if len(pairs) >= 2:\\n        return []             # several marker/tip pairs: the win is their conjunction (align_all_colors), not one of them\\n    out = []\\n    for c in sorted(colors):\\n        units = _color_units(scene, c)\\n        if not 2 <= len(units) <= 3 or any((u[2] - u[0]) * (u[3] - u[1]) > 100 for u in units):\\n            continue          # a big unit is a bar or panel, not a marker\\n        if len({(u[2] - u[0], u[3] - u[1]) for u in units}) == 1:\\n            continue          # identical units (two buttons of one colour) are a set of controls, not a marker and a tip\\n        owners = [o for o in scene.objects if o.region not in strips and c in o.colors]\\n        if len(owners) < 2:\\n            continue\\n        for axis in (\\\"col\\\", \\\"row\\\"):\\n            k = (1, 3) if axis == \\\"col\\\" else (0, 2)\\n            def is_goal(s, c=c, k=k):\\n                us = _color_units(s, c)\\n                return len(us) >= 2 and len({(u[k[0]], u[k[1]]) for u in us}) == 1\\n            def progress(s, c=c, k=k):\\n                us = _color_units(s, c)\\n                if len(us) < 2:\\n                    return 0.0\\n                lo = [u[k[0]] for u in us]\\n                return 1.0 - (max(lo) - min(lo)) / s.grid_shape[k[0]]\\n            clue = 0.15 if any(len(o.colors) > 1 for o in owners) else 0.0\\n            out.append(GoalInstance(f\\\"align_color({c},{axis})\\\", \\\"align_color\\\", {\\\"color\\\": c, \\\"axis\\\": axis}, is_goal, progress, clue=clue))\\n    return out\\n\\n\\ndef _marker_pair(units: list) -> bool:\\n    \\\"\\\"\\\"Two small units of one colour that differ in size: a marker and a mover's tip (two identical buttons are not).\\\"\\\"\\\"\\n    if len(units) != 2 or any((u[2] - u[0]) * (u[3] - u[1]) > 100 for u in units):\\n        return False\\n    a, b = units\\n    if (a[2] - a[0], a[3] - a[1]) == (b[2] - b[0], b[3] - b[1]):\\n        return False\\n    # two pieces of one bar split by a marker (same band, small gap) are not a pair either\\n    if (a[0], a[2]) == (b[0], b[2]) and max(a[1], b[1]) - min(a[3], b[3]) <= 4:\\n        return False\\n    if (a[1], a[3]) == (b[1], b[3]) and max(a[0], b[0]) - min(a[2], b[2]) <= 4:\\n        return False\\n    return True\\n\\n\\ndef t_align_all_colors(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    \\\"\\\"\\\"Every colour that has exactly two small units (a marker and a mover's tip) lines up on the same axis: the\\n    conjunction of align_color over colours (three movers, three markers). One colour -> same as align_color.\\\"\\\"\\\"\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    colors = sorted({c for o in scene.objects if o.region not in strips for c in o.colors})\\n    pairs = [c for c in colors if _marker_pair(_color_units(scene, c))]\\n    if len(pairs) < 2:\\n        return []\\n    out = []\\n    for axis in (\\\"col\\\", \\\"row\\\"):\\n        k = (1, 3) if axis == \\\"col\\\" else (0, 2)\\n        def aligned(s, c, k=k):\\n            us = _color_units(s, c)\\n            return len(us) >= 2 and len({(u[k[0]], u[k[1]]) for u in us}) == 1\\n        def is_goal(s, cs=tuple(pairs), k=k):\\n            return all(aligned(s, c, k) for c in cs)\\n        def progress(s, cs=tuple(pairs), k=k):\\n            tot = 0.0\\n            for c in cs:\\n                us = _color_units(s, c)\\n                if len(us) < 2:\\n                    continue\\n                lo = [u[k[0]] for u in us]\\n                tot += 1.0 - (max(lo) - min(lo)) / s.grid_shape[k[0]]\\n            return tot / len(cs)\\n        def estimate(s, cs=tuple(pairs), k=k):\\n            tot = 0.0\\n            for c in cs:\\n                us = _color_units(s, c)\\n                if len(us) >= 2:\\n                    lo = [u[k[0]] for u in us]; tot += (max(lo) - min(lo)) / 2.0\\n            return tot\\n        out.append(GoalInstance(f\\\"align_all_colors({axis})\\\", \\\"align_all_colors\\\", {\\\"axis\\\": axis, \\\"colors\\\": list(pairs)}, is_goal, progress, clue=0.2, estimate_fn=estimate))\\n    return out\\n\\n\\ndef _layout_of(o) -> \\\"np.ndarray\\\":\\n    return o.color_mask if o.color_mask is not None else np.where(o.mask, o.color, -1).astype(np.int8)\\n\\n\\ndef _find_by_id_or_bbox(s: Scene, oid: int, bbox):\\n    o = s.get(oid)\\n    if o is not None and (o.height, o.width) == (bbox[2] - bbox[0], bbox[3] - bbox[1]):\\n        return o\\n    return next((x for x in s.objects if tuple(x.bbox) == tuple(bbox)), None)\\n\\n\\ndef t_match_pattern(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    \\\"\\\"\\\"A multi-colour rectangular object (the target pattern) must be reproduced by another object of the same size (the canvas):\\n    stamp / paint / colour-fill games. Both directions are instantiated; the level-up filter and the clue (the canvas starts\\n    uniform) decide which is the target.\\\"\\\"\\\"\\n    pats = [o for o in scene.objects if o.color_mask is not None and len(o.colors) >= 2 and o.area == o.height * o.width\\n            and min(o.height, o.width) >= 3 and scene.region(o.region) is not None and scene.region(o.region).kind_hint != \\\"ui_strip\\\"]\\n    out = []\\n    for tgt in pats:\\n        for cv in scene.objects:\\n            if cv.id == tgt.id or (cv.height, cv.width) != (tgt.height, tgt.width) or cv.area != tgt.area:\\n                continue\\n            if scene.region(cv.region) is not None and scene.region(cv.region).kind_hint == \\\"ui_strip\\\":\\n                continue\\n            def is_goal(s, a=tgt.id, ab=tgt.bbox, b=cv.id, bb=cv.bbox):\\n                A, B = _find_by_id_or_bbox(s, a, ab), _find_by_id_or_bbox(s, b, bb)\\n                return A is not None and B is not None and A.area == B.area and bool(np.array_equal(_layout_of(A), _layout_of(B)))\\n            def progress(s, a=tgt.id, ab=tgt.bbox, b=cv.id, bb=cv.bbox):\\n                A, B = _find_by_id_or_bbox(s, a, ab), _find_by_id_or_bbox(s, b, bb)\\n                if A is None or B is None or A.area != B.area:\\n                    return 0.0\\n                la, lb = _layout_of(A), _layout_of(B)\\n                return float((la == lb).mean()) if la.shape == lb.shape else 0.0\\n            def estimate(s, a=tgt.id, ab=tgt.bbox, b=cv.id, bb=cv.bbox):\\n                A, B = _find_by_id_or_bbox(s, a, ab), _find_by_id_or_bbox(s, b, bb)\\n                if A is None or B is None or A.area != B.area:\\n                    return None\\n                la, lb = _layout_of(A), _layout_of(B)\\n                return float(len({int(v) for v in la[la != lb]})) if la.shape == lb.shape else None   # >= one action per colour still missing\\n            # the canvas is painted with the target's colours (blank or partly done); the name follows the canvas POSITION so the\\n            # goal keeps its confidence when the tracker re-numbers a fully repainted canvas\\n            clue = 0.2 if set(cv.colors) < set(tgt.colors) else (0.1 if set(cv.colors) == set(tgt.colors) else 0.0)\\n            out.append(GoalInstance(f\\\"match_pattern({tgt.id}->@{cv.bbox[0]},{cv.bbox[1]})\\\", \\\"match_pattern\\\", {\\\"target\\\": tgt.id, \\\"canvas\\\": cv.id, \\\"canvas_bbox\\\": list(cv.bbox)},\\n                                    is_goal, progress, clue=clue, estimate_fn=estimate))\\n    return out\\n\\n\\ndef t_explore(scene: Scene, ctx: dict) -> list[GoalInstance]:\\n    \\\"\\\"\\\"Curiosity fallback (not a win condition): bring the agent next to an object it has not touched yet. `ctx['touched']`\\n    is the set of object identities already reached. Used by the orchestrator when no real goal yields a plan.\\\"\\\"\\\"\\n    if not _agents(scene):\\n        return []\\n    touched = set(ctx.get(\\\"touched\\\", ()))\\n    def targets(s):\\n        return [o for o in s.objects if not (o.role and \\\"agent\\\" in o.role) and o.role not in (\\\"wall\\\", \\\"indicator\\\", \\\"decoration\\\") and o.identity() not in touched\\n                and s.region(o.region) is not None and s.region(o.region).kind_hint != \\\"ui_strip\\\"]\\n    def is_goal(s):\\n        ag = _agents(s)\\n        return any(a.overlaps(o) or _adjacent(a, o) for a in ag for o in targets(s))\\n    def progress(s):\\n        ag, ts = _agents(s), targets(s)\\n        if not ag or not ts:\\n            return 0.0\\n        return 1.0 - min(_dist(a.center, o.center) for a in ag for o in ts) / _max_dist(s)\\n    return [GoalInstance(\\\"explore(agent)\\\", \\\"explore\\\", {}, is_goal, progress, confidence=0.0)]\\n\\n\\ndef _adjacent(a, b) -> bool:\\n    ar0, ac0, ar1, ac1 = a.bbox; br0, bc0, br1, bc1 = b.bbox\\n    touch_r = (ar1 == br0 or br1 == ar0) and not (ac1 <= bc0 or bc1 <= ac0)\\n    touch_c = (ac1 == bc0 or bc1 == ac0) and not (ar1 <= br0 or br1 <= ar0)\\n    return touch_r or touch_c\\n\\n\\n# confidence tiers: role-based templates are more specific than colour/geometry ones (spec \\u00a79 usage-stats prior 0.5)\\nBASE_CONFIDENCE = {\\\"reach\\\": 0.5, \\\"all_collected\\\": 0.5, \\\"inside_frame\\\": 0.5, \\\"fill_marked_slots\\\": 0.5, \\\"same_cell\\\": 0.5, \\\"pattern_match\\\": 0.45, \\\"match_pattern\\\": 0.5, \\\"align_color\\\": 0.5, \\\"align_all_colors\\\": 0.5, \\\"match_shapes\\\": 0.45,\\n                   \\\"enclose\\\": 0.4, \\\"align\\\": 0.35, \\\"all_removed\\\": 0.3, \\\"fill_region\\\": 0.3, \\\"count_equals\\\": 0.2, \\\"sort_by\\\": 0.2, \\\"sequence\\\": 0.3, \\\"explore\\\": 0.0}\\n\\nTEMPLATES: dict[str, Callable[[Scene, dict], list[GoalInstance]]] = {\\n    \\\"reach\\\": t_reach, \\\"match_shapes\\\": t_match_shapes, \\\"all_removed\\\": t_all_removed, \\\"all_collected\\\": t_all_collected,\\n    \\\"fill_region\\\": t_fill_region, \\\"sort_by\\\": t_sort_by, \\\"count_equals\\\": t_count_equals, \\\"align\\\": t_align, \\\"enclose\\\": t_enclose,\\n    \\\"sequence\\\": t_sequence, \\\"inside_frame\\\": t_inside_frame, \\\"fill_marked_slots\\\": t_fill_marked_slots, \\\"same_cell\\\": t_same_cell, \\\"pattern_match\\\": t_pattern_match, \\\"match_pattern\\\": t_match_pattern, \\\"align_color\\\": t_align_color, \\\"align_all_colors\\\": t_align_all_colors, \\\"explore\\\": t_explore}\\n\\n\\ndef instantiate_all(scene: Scene, ctx: Optional[dict] = None, usage_stats: Optional[dict] = None) -> list[GoalInstance]:\\n    \\\"\\\"\\\"All template instances that are NOT already true on the scene (a goal achieved at the start is not a goal).\\\"\\\"\\\"\\n    ctx = ctx or {}\\n    out: list[GoalInstance] = []\\n    for name, fn in TEMPLATES.items():\\n        if name == \\\"explore\\\" and not ctx.get(\\\"include_explore\\\"):\\n            continue\\n        try:\\n            insts = fn(scene, ctx)\\n        except Exception:\\n            continue\\n        stats = (usage_stats or {}).get(name, {})\\n        base = stats.get(\\\"games_verified\\\", 0) / stats[\\\"games_used\\\"] if stats.get(\\\"games_used\\\") else BASE_CONFIDENCE.get(name, 0.5)\\n        for g in insts:\\n            if g.is_goal(scene):\\n                continue\\n            g.confidence = min(1.0, base + g.clue)\\n            out.append(g)\\n    return out\\n\", \"pbg/harness/__init__.py\": \"\", \"pbg/harness/attribution.py\": \"\\\"\\\"\\\"harness/attribution.py \\u2014 bottleneck attribution from events.jsonl (spec \\u00a716): tag each failed episode with the\\nmodule responsible.\\n\\n  last state PROBE repeating                    -> Perception / Probe\\n  budget exhausted in HYPOTHESIZE with score<1  -> World Model Lab\\n  model verified but no plan repeatedly         -> Goal Inference\\n  budget exhausted during execution, no mismatch -> Planner (heuristic) or Goal Inference (progress)\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom collections import Counter\\nfrom pathlib import Path\\n\\n\\ndef attribute(events: list[dict], result: dict | None = None) -> str:\\n    if not events:\\n        return \\\"unknown\\\"\\n    if result and result.get(\\\"state\\\") == \\\"WIN\\\":\\n        return \\\"none\\\"\\n    states = [e[\\\"to\\\"] for e in events]\\n    reasons = [e.get(\\\"reason\\\", \\\"\\\") for e in events]\\n    last = events[-1]\\n    tail = states[-12:]\\n    if tail.count(\\\"PROBE\\\") >= 2:\\n        return \\\"perception/probe\\\"\\n    if last[\\\"from\\\"] == \\\"HYPOTHESIZE\\\" or any(\\\"no hypothesis\\\" in r for r in reasons[-5:]):\\n        top_scores = [r for r in reasons if r.startswith(\\\"top h=\\\")]\\n        if top_scores and \\\"score=1.00\\\" not in top_scores[-1]:\\n            return \\\"world_model_lab\\\"\\n        if not top_scores:\\n            return \\\"world_model_lab\\\"\\n    if sum(1 for r in reasons if \\\"verified model -> goal demoted\\\" in r) >= 2:\\n        return \\\"goal_inference\\\"\\n    if any(r.startswith(\\\"plan of\\\") for r in reasons) and not any(\\\"mismatch\\\" in r for r in reasons[-10:]):\\n        return \\\"planner/goal_progress\\\"\\n    if any(\\\"mismatch\\\" in r for r in reasons[-10:]):\\n        return \\\"world_model_lab\\\"\\n    return \\\"unknown\\\"\\n\\n\\ndef attribute_run(run_json: Path) -> dict:\\n    run = json.loads(Path(run_json).read_text())\\n    log_dir = Path(run_json).parent / \\\"logs\\\" / run[\\\"run_id\\\"]\\n    tags = {}\\n    for g in run[\\\"games\\\"]:\\n        p = log_dir / f\\\"{g['game_id']}.events.jsonl\\\"\\n        events = [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []\\n        tags[g[\\\"game_id\\\"]] = attribute(events, g)\\n    dist = Counter(v for v in tags.values())\\n    return {\\\"run_id\\\": run[\\\"run_id\\\"], \\\"tags\\\": tags, \\\"distribution\\\": dict(dist)}\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    import sys\\n    print(json.dumps(attribute_run(Path(sys.argv[1])), indent=1))\\n\", \"pbg/harness/metrics.py\": \"\\\"\\\"\\\"harness/metrics.py \\u2014 integrated metrics (spec \\u00a716): level clear rate, action efficiency vs human, level transition\\ncost, LLM call ratio, bottleneck distribution; holdout games reported separately.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nfrom .attribution import attribute_run\\n\\nROOT = Path(__file__).resolve().parents[2]\\nHOLDOUT = ROOT / \\\"pbg\\\" / \\\"data\\\" / \\\"holdout.txt\\\"\\n\\n\\ndef holdout_games() -> set[str]:\\n    if HOLDOUT.exists():\\n        return {l.strip() for l in HOLDOUT.read_text().splitlines() if l.strip() and not l.startswith(\\\"#\\\")}\\n    return set()\\n\\n\\ndef summarize_run(run_json: Path, human_actions: Optional[dict] = None) -> dict:\\n    run = json.loads(Path(run_json).read_text())\\n    hold = holdout_games()\\n    attr = attribute_run(run_json)\\n    out = {\\\"run_id\\\": run[\\\"run_id\\\"], \\\"tag\\\": run.get(\\\"tag\\\"), \\\"groups\\\": {}}\\n    for group, games in ((\\\"dev\\\", [g for g in run[\\\"games\\\"] if g[\\\"game_id\\\"] not in hold]), (\\\"holdout\\\", [g for g in run[\\\"games\\\"] if g[\\\"game_id\\\"] in hold])):\\n        if not games:\\n            continue\\n        cleared = sum(g.get(\\\"levels_completed\\\") or 0 for g in games); total = sum(g.get(\\\"levels_total\\\") or 0 for g in games)\\n        actions = sum(g.get(\\\"actions\\\") or 0 for g in games); llm = sum(g.get(\\\"llm_calls\\\") or 0 for g in games)\\n        eff = []\\n        for g in games:\\n            la = g.get(\\\"level_actions\\\") or {}; base = g.get(\\\"level_baseline_actions\\\") or []\\n            for i in range(g.get(\\\"levels_completed\\\") or 0):\\n                if i < len(base) and base[i] and str(i + 1) in la:\\n                    eff.append(la[str(i + 1)] / base[i])\\n        out[\\\"groups\\\"][group] = {\\\"games\\\": len(games), \\\"level_clear_rate\\\": round(cleared / total, 3) if total else None, \\\"levels_cleared\\\": cleared, \\\"levels_total\\\": total,\\n                                \\\"action_efficiency_vs_baseline\\\": round(sum(eff) / len(eff), 2) if eff else None, \\\"llm_call_ratio\\\": round(llm / actions, 4) if actions else None,\\n                                \\\"mean_score\\\": round(sum(g.get(\\\"score\\\") or 0 for g in games) / len(games), 4),\\n                                \\\"bottlenecks\\\": {k: v for k, v in attr[\\\"distribution\\\"].items()} if group == \\\"dev\\\" else {g[\\\"game_id\\\"]: attr[\\\"tags\\\"].get(g[\\\"game_id\\\"]) for g in games}}\\n    return out\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    import sys\\n    print(json.dumps(summarize_run(Path(sys.argv[1])), indent=1))\\n\", \"pbg/harness/online_runner.py\": \"\\\"\\\"\\\"harness/online_runner.py \\u2014 play games with the full orchestrator against the arc_agi engine (offline or gateway),\\nwriting an experiments-style result JSON (same schema as rulebook/arcnav runs) + per-game event logs.\\n\\n    .venv/bin/python -m pbg.harness.online_runner --games ls20,vc33 --minutes 10 --jobs 2 [--no-llm] [--budget 2000] [--tag x]\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport datetime as dt\\nimport json\\nimport logging\\nimport os\\nimport subprocess\\nimport threading\\nimport time\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nfrom ..core.budget import load_budget_config\\nfrom ..env.wrapper import ArcadeEnv\\nfrom ..llm.gateway import LLMGateway, load_llm_config\\nfrom ..memory.store import Memory\\nfrom ..orchestrator.loop import Orchestrator\\n\\nROOT = Path(__file__).resolve().parents[2]\\nDEFAULT_OUT = ROOT / \\\"experiments\\\" / \\\"pbg\\\" / \\\"results\\\"\\nDEFAULT_MEMORY = ROOT / \\\"experiments\\\" / \\\"pbg\\\" / \\\"memory\\\"\\n\\n\\ndef _git() -> dict:\\n    try:\\n        rev = subprocess.check_output([\\\"git\\\", \\\"rev-parse\\\", \\\"--short\\\", \\\"HEAD\\\"], cwd=ROOT, text=True, stderr=subprocess.DEVNULL).strip()\\n        dirty = bool(subprocess.check_output([\\\"git\\\", \\\"status\\\", \\\"--porcelain\\\", \\\"pbg\\\"], cwd=ROOT, text=True, stderr=subprocess.DEVNULL).strip())\\n        return {\\\"commit\\\": rev, \\\"dirty\\\": dirty}\\n    except Exception:\\n        return {}\\n\\n\\ndef make_arcade(environments_dir: Optional[str] = None, *, competition: bool = False, base_url: Optional[str] = None):\\n    import arc_agi\\n    from arc_agi import OperationMode\\n    os.environ.setdefault(\\\"ARC_API_KEY\\\", \\\"local-dev\\\")\\n    if competition:\\n        return arc_agi.Arcade(operation_mode=OperationMode.COMPETITION, arc_base_url=base_url or os.environ.get(\\\"ARC_BASE_URL\\\", \\\"http://gateway:8001/\\\"))\\n    return arc_agi.Arcade(operation_mode=OperationMode.OFFLINE, environments_dir=environments_dir or str(ROOT / \\\"environment_files\\\"))\\n\\n\\ndef play_game(arc, game_id: str, cfg: dict, log_dir: Path, memory: Memory, llm, lock: threading.Lock) -> dict:\\n    log_dir.mkdir(parents=True, exist_ok=True)\\n    log_path = log_dir / f\\\"{game_id}.log\\\"\\n\\n    def log(msg: str) -> None:\\n        line = f\\\"{time.strftime('%H:%M:%S')} {msg}\\\"\\n        with open(log_path, \\\"a\\\") as f:\\n            f.write(line + \\\"\\\\n\\\")\\n        if cfg.get(\\\"verbose\\\"):\\n            print(f\\\"[{game_id}] {msg}\\\", flush=True)\\n\\n    with lock:\\n        raw_env = arc.make(game_id)\\n    if raw_env is None:\\n        return {\\\"game_id\\\": game_id, \\\"error\\\": \\\"could not create env\\\"}\\n    env = ArcadeEnv(raw_env, game_id, log_dir=memory.episodic_dir(game_id), budget_total=cfg.get(\\\"budget\\\"), replay_path=log_dir / f\\\"{game_id}.actions.jsonl\\\")\\n    if cfg.get(\\\"policy\\\") == \\\"hypothesis\\\":\\n        from ..hypothesis.policy import HypothesisPolicy\\n        if llm is None:\\n            return {\\\"game_id\\\": game_id, \\\"error\\\": \\\"the hypothesis policy needs an LLM\\\"}\\n        orch = HypothesisPolicy(memory=memory, llm=llm, log=log, events_dir=log_dir, max_seconds=cfg[\\\"max_minutes\\\"] * 60, budget_cfg=cfg.get(\\\"budget_cfg\\\"),\\n                                explore=int(cfg.get(\\\"explore\\\") or 0))\\n    else:\\n        orch = Orchestrator(memory=memory, llm=llm, budget_cfg=cfg.get(\\\"budget_cfg\\\"), log=log, events_dir=log_dir, use_llm=not cfg.get(\\\"no_llm\\\"),\\n                            max_seconds=cfg[\\\"max_minutes\\\"] * 60, planner_kwargs=cfg.get(\\\"planner\\\", {}), max_levels=int(cfg.get(\\\"max_levels\\\", 20)))\\n    try:\\n        r = orch.play(env, game_id, budget_total=cfg.get(\\\"budget\\\"))\\n        d = r.to_json(); d[\\\"llm\\\"] = llm.stats() if llm else {}\\n        return d\\n    except Exception as e:\\n        logging.exception(\\\"game %s crashed\\\", game_id)\\n        log(f\\\"CRASH {e!r}\\\")\\n        st = env.status()\\n        return {\\\"game_id\\\": game_id, \\\"error\\\": f\\\"{type(e).__name__}: {e}\\\", \\\"actions\\\": st.actions_used, \\\"levels_completed\\\": st.level - 1, \\\"levels_total\\\": st.levels_total}\\n\\n\\ndef write_run_meta(log_dir: Path, **meta) -> None:\\n    from scripts.eval_viewer.ids import game_hash\\n    \\\"\\\"\\\"logs/<run>/run.json, written before the first action: lets the eval viewer list a run that is still in progress\\n    (the summary run-<id>.json only exists once every game has finished).\\\"\\\"\\\"\\n    started = meta.pop(\\\"started\\\")\\n    log_dir.mkdir(parents=True, exist_ok=True)\\n    meta[\\\"hashes\\\"] = {g: game_hash(meta[\\\"run_id\\\"], g) for g in meta.get(\\\"game_ids\\\") or []}\\n    meta[\\\"games\\\"] = meta.pop(\\\"game_ids\\\", [])\\n    (log_dir / \\\"run.json\\\").write_text(json.dumps({**meta, \\\"started_at\\\": started.isoformat(), \\\"pid\\\": os.getpid(), \\\"git\\\": _git()}, indent=1, default=str))\\n\\n\\ndef run(game_ids: list[str], cfg: dict, *, out_dir: Path = DEFAULT_OUT, memory_root: Path = DEFAULT_MEMORY, tag: str = \\\"\\\", arc=None) -> dict:\\n    run_id = dt.datetime.now(dt.timezone.utc).strftime(\\\"%Y%m%d-%H%M%S\\\") + f\\\"-{os.getpid()}\\\"\\n    log_dir = out_dir / \\\"logs\\\" / run_id\\n    arc = arc or make_arcade()\\n    memory = Memory(memory_root)\\n    llm = None\\n    if not cfg.get(\\\"no_llm\\\"):\\n        llm_cfg = load_llm_config()\\n        if cfg.get(\\\"llm\\\"):\\n            llm_cfg.update(cfg[\\\"llm\\\"])\\n        llm = LLMGateway(llm_cfg, cache_dir=memory_root / \\\"llm_cache\\\", max_calls=int(cfg.get(\\\"llm_calls_per_game\\\") or load_budget_config().get(\\\"llm_calls_max\\\", 60)) * len(game_ids))\\n    lock = threading.Lock(); started = dt.datetime.now(dt.timezone.utc)\\n    write_run_meta(log_dir, experiment=\\\"pbg\\\", run_id=run_id, tag=tag, started=started, game_ids=game_ids, params={k: v for k, v in cfg.items() if k != \\\"llm\\\"})\\n    import faulthandler\\n    faulthandler.enable()\\n    hard_limit = cfg[\\\"max_minutes\\\"] * 60 + 300          # a game that is still computing this long after its deadline is reported as hung\\n    games = []\\n    ex = ThreadPoolExecutor(max_workers=int(cfg.get(\\\"jobs\\\", 2)))\\n    futures = [ex.submit(play_game, arc, g, cfg, log_dir, memory, llm, lock) for g in game_ids]\\n    for g, fut in zip(game_ids, futures):\\n        try:\\n            games.append(fut.result(timeout=hard_limit))\\n        except Exception as e:      # TimeoutError -> hung game; the thread is abandoned (daemon-like) and the run still writes results\\n            faulthandler.dump_traceback(all_threads=True)\\n            games.append({\\\"game_id\\\": g, \\\"error\\\": f\\\"hung: {type(e).__name__}\\\", \\\"actions\\\": 0, \\\"levels_completed\\\": 0, \\\"stop_reason\\\": \\\"hung\\\"})\\n    ex.shutdown(wait=False, cancel_futures=True)\\n    from scripts.eval_viewer.ids import game_hash\\n    for g in games:\\n        g[\\\"hash\\\"] = game_hash(run_id, g[\\\"game_id\\\"])     # stable per game execution: the Replay viewer's lookup key\\n    try:\\n        sc = arc.get_scorecard(); scd = sc.model_dump() if hasattr(sc, \\\"model_dump\\\") else {}\\n    except Exception as e:\\n        scd = {\\\"error\\\": repr(e)}\\n    by_id = {}\\n    for e in scd.get(\\\"environments\\\", []) or []:\\n        if e.get(\\\"runs\\\"):\\n            by_id[e[\\\"id\\\"].split(\\\"-\\\")[0]] = e[\\\"runs\\\"][-1]\\n    runs_by_id = {e[\\\"id\\\"].split(\\\"-\\\")[0]: e.get(\\\"runs\\\") or [] for e in scd.get(\\\"environments\\\", []) or []}\\n    for g in games:\\n        r = by_id.get(g[\\\"game_id\\\"].split(\\\"-\\\")[0])\\n        if r:\\n            g[\\\"score\\\"] = r.get(\\\"score\\\"); g[\\\"level_scores\\\"] = r.get(\\\"level_scores\\\"); g[\\\"level_baseline_actions\\\"] = r.get(\\\"level_baseline_actions\\\")\\n            runs = runs_by_id.get(g[\\\"game_id\\\"].split(\\\"-\\\")[0], [])\\n            best = max(runs, key=lambda x: x.get(\\\"score\\\") or 0) if runs else r\\n            g[\\\"scorecard_runs\\\"] = len(runs); g[\\\"score_best_run\\\"] = best.get(\\\"score\\\"); g[\\\"levels_best_run\\\"] = best.get(\\\"levels_completed\\\")\\n            if len(runs) > 1:\\n                g[\\\"note\\\"] = f\\\"{len(runs)} scorecard runs (a RESET on a fresh level restarts the game); last run scored, best run {best.get('score')}\\\"\\n    scores = [g.get(\\\"score\\\") or 0.0 for g in games]\\n    result = {\\\"schema\\\": 1, \\\"experiment\\\": \\\"pbg\\\", \\\"run_id\\\": run_id, \\\"started_at\\\": started.isoformat(), \\\"finished_at\\\": dt.datetime.now(dt.timezone.utc).isoformat(),\\n              \\\"git\\\": _git(), \\\"tag\\\": tag, \\\"config\\\": {\\\"description\\\": \\\"pbg: perception -> probe -> world-model lab -> goal inference -> planner (spec docs/027)\\\", \\\"games\\\": game_ids, \\\"params\\\": {k: v for k, v in cfg.items() if k != \\\"llm\\\"}},\\n              \\\"aggregate\\\": {\\\"score\\\": sum(scores) / max(1, len(scores)), \\\"levels_completed\\\": sum(g.get(\\\"levels_completed\\\", 0) for g in games),\\n                            \\\"games_level2plus\\\": sum(1 for g in games if (g.get(\\\"levels_completed\\\") or 0) >= 2), \\\"actions\\\": sum(g.get(\\\"actions\\\", 0) for g in games), \\\"games_played\\\": len(games),\\n                            \\\"llm_calls\\\": sum(g.get(\\\"llm_calls\\\", 0) for g in games)},\\n              \\\"games\\\": games, \\\"scorecard\\\": scd, \\\"llm\\\": llm.stats() if llm else {}}\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    out = out_dir / f\\\"run-{run_id}.json\\\"; out.write_text(json.dumps(result, indent=2, default=str))\\n    print(f\\\"\\\\n========= pbg {tag} =========\\\")\\n    for g in games:\\n        print(f\\\"  {g['game_id']:8} levels={g.get('levels_completed', '?'):>2}/{g.get('levels_total', '?')} actions={g.get('actions', '?'):>5} score={g.get('score', 0) or 0:.3f} \\\"\\n              f\\\"llm={g.get('llm_calls', '?')} runs={g.get('scorecard_runs', 1)} best={g.get('score_best_run', 0) or 0:.2f} stop={g.get('stop_reason', g.get('error'))} per-level={g.get('level_actions')}\\\")\\n    print(f\\\"Aggregate score: {result['aggregate']['score']:.4f} | level>=2: {result['aggregate']['games_level2plus']}/{len(games)}\\\\nResult: {out}\\\\nLogs: {log_dir}\\\")\\n    return result\\n\\n\\ndef main(argv=None) -> None:\\n    ap = argparse.ArgumentParser(description=\\\"Play ARC-AGI-3 games with the pbg system\\\")\\n    ap.add_argument(\\\"--games\\\", default=\\\"ls20\\\"); ap.add_argument(\\\"--tag\\\", default=\\\"\\\")\\n    ap.add_argument(\\\"--policy\\\", default=\\\"orchestrator\\\", choices=[\\\"orchestrator\\\", \\\"hypothesis\\\"], help=\\\"hypothesis: look/hypothesise/test/verify/revise/plan policy (needs an LLM)\\\")\\n    ap.add_argument(\\\"--out\\\", default=str(DEFAULT_OUT)); ap.add_argument(\\\"--memory\\\", default=str(DEFAULT_MEMORY))\\n    ap.add_argument(\\\"--minutes\\\", type=float, default=10); ap.add_argument(\\\"--jobs\\\", type=int, default=2); ap.add_argument(\\\"--budget\\\", type=int)\\n    ap.add_argument(\\\"--no-llm\\\", action=\\\"store_true\\\", help=\\\"priors-only induction, no model calls\\\")\\n    ap.add_argument(\\\"--explore\\\", type=int, default=0, help=\\\"hypothesis policy: explore level 1 for at most N actions before the first hypothesis (docs/029)\\\")\\n    ap.add_argument(\\\"--fresh\\\", action=\\\"store_true\\\", help=\\\"wipe this run's games from memory first\\\")\\n    ap.add_argument(\\\"--quiet\\\", action=\\\"store_true\\\"); ap.add_argument(\\\"--max-levels\\\", type=int, default=20)\\n    a = ap.parse_args(argv)\\n    logging.basicConfig(level=logging.WARNING)\\n    cfg = {\\\"max_minutes\\\": a.minutes, \\\"jobs\\\": a.jobs, \\\"budget\\\": a.budget, \\\"no_llm\\\": a.no_llm, \\\"verbose\\\": not a.quiet, \\\"max_levels\\\": a.max_levels, \\\"policy\\\": a.policy, \\\"explore\\\": a.explore}\\n    arc = make_arcade()\\n    games = sorted(e.game_id.split(\\\"-\\\")[0] for e in arc.get_environments()) if a.games == \\\"all\\\" else [g.strip() for g in a.games.split(\\\",\\\") if g.strip()]\\n    if a.fresh:\\n        m = Memory(Path(a.memory))\\n        for g in games:\\n            m.wipe_game(g)\\n    run(games, cfg, out_dir=Path(a.out), memory_root=Path(a.memory), tag=a.tag, arc=arc)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"pbg/harness/postmortem.py\": \"\\\"\\\"\\\"harness/postmortem.py \\u2014 per-level post-mortem of one game in a pbg run: where the actions and the time went, how the\\nhypotheses/goals evolved, what every plan did, and a verdict on why the level was not cleared (spec \\u00a716 attribution,\\nfiner grained than attribution.py).\\n\\n    .venv/bin/python -m pbg.harness.postmortem <run_id> <game> [--json]\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport re\\nfrom collections import Counter\\nfrom pathlib import Path\\n\\nROOT = Path(__file__).resolve().parents[2]\\nRESULTS = ROOT / \\\"experiments\\\" / \\\"pbg\\\" / \\\"results\\\"\\n\\n\\ndef load(run_id: str, game: str):\\n    run = json.loads((RESULTS / f\\\"run-{run_id}.json\\\").read_text())\\n    g = next(x for x in run[\\\"games\\\"] if x[\\\"game_id\\\"] == game)\\n    ev = [json.loads(l) for l in (RESULTS / \\\"logs\\\" / run_id / f\\\"{game}.events.jsonl\\\").read_text().splitlines() if l.strip()]\\n    log_lines = (RESULTS / \\\"logs\\\" / run_id / f\\\"{game}.log\\\").read_text().splitlines()\\n    return run, g, ev, log_lines\\n\\n\\ndef per_level(ev: list[dict], g: dict) -> list[dict]:\\n    \\\"\\\"\\\"Split the event stream at level-up events; per segment: dwell by state, actions by transition, plans, mismatches.\\\"\\\"\\\"\\n    segments = []; cur = {\\\"level\\\": 1, \\\"events\\\": []}\\n    for e in ev:\\n        m = re.match(r\\\"level (\\\\d+) -> (\\\\d+)\\\", e[\\\"reason\\\"])\\n        cur[\\\"events\\\"].append(e)\\n        if m:\\n            segments.append(cur); cur = {\\\"level\\\": int(m.group(2)), \\\"events\\\": []}\\n    segments.append(cur)\\n    out = []\\n    for seg in segments:\\n        es = seg[\\\"events\\\"]\\n        if not es:\\n            continue\\n        t0, t1 = es[0][\\\"ts\\\"], es[-1][\\\"ts\\\"]\\n        used0, used1 = es[0][\\\"budget_used\\\"], es[-1][\\\"budget_used\\\"]\\n        actions_by = Counter(); prev = used0\\n        for e in es:\\n            actions_by[(e[\\\"from\\\"], e[\\\"to\\\"])] += e[\\\"budget_used\\\"] - prev; prev = e[\\\"budget_used\\\"]\\n        plans = [e for e in es if e[\\\"reason\\\"].startswith(\\\"plan of\\\") or e[\\\"reason\\\"].startswith(\\\"explore plan\\\")]\\n        outcomes = Counter(\\\" \\\".join(e[\\\"reason\\\"].split(\\\" \\\")[:2]) for e in es if e[\\\"from\\\"] == \\\"EXECUTE\\\")\\n        tops = [e[\\\"reason\\\"] for e in es if e[\\\"reason\\\"].startswith(\\\"top h=\\\")]\\n        goals = Counter(re.search(r\\\"top=([^;|]+)\\\", t).group(1) if re.search(r\\\"top=([^;|]+)\\\", t) else \\\"?\\\" for t in tops)\\n        scores = [float(m.group(1)) for t in tops for m in [re.search(r\\\"score=([\\\\d.]+)\\\", t)] if m]\\n        out.append({\\\"level\\\": seg[\\\"level\\\"], \\\"seconds\\\": round(t1 - t0), \\\"actions\\\": used1 - used0, \\\"actions_by\\\": {f\\\"{a}->{b}\\\": n for (a, b), n in actions_by.most_common() if n > 0},\\n                    \\\"hyp_rounds\\\": len(tops), \\\"score_first\\\": scores[0] if scores else None, \\\"score_last\\\": scores[-1] if scores else None, \\\"score_max\\\": max(scores) if scores else None,\\n                    \\\"top_goals\\\": goals.most_common(4), \\\"plans\\\": len(plans), \\\"plan_lengths\\\": [int(re.search(r\\\"of (\\\\d+)\\\", p[\\\"reason\\\"]).group(1)) for p in plans if re.search(r\\\"of (\\\\d+)\\\", p[\\\"reason\\\"])][:30],\\n                    \\\"exec_outcomes\\\": dict(outcomes), \\\"mismatches\\\": sum(1 for e in es if \\\"mismatch\\\" in e[\\\"reason\\\"]), \\\"game_overs\\\": sum(1 for e in es if \\\"game over\\\" in e[\\\"reason\\\"]),\\n                    \\\"llm_jobs\\\": sum(1 for e in es if \\\"llm job started\\\" in e[\\\"reason\\\"]), \\\"stuck\\\": sum(1 for e in es if e[\\\"reason\\\"].startswith(\\\"stuck\\\")),\\n                    \\\"cleared\\\": any(re.match(r\\\"level (\\\\d+) -> (\\\\d+)\\\", e[\\\"reason\\\"]) for e in es)})\\n    return out\\n\\n\\ndef verdict(lv: dict) -> str:\\n    if lv[\\\"cleared\\\"]:\\n        return \\\"cleared\\\"\\n    if lv[\\\"hyp_rounds\\\"] == 0:\\n        return \\\"no hypothesize round (probe/LLM never returned)\\\"\\n    if (lv[\\\"score_max\\\"] or 0) < 0.8:\\n        return f\\\"world model never explained the log (best score {lv['score_max']})\\\"\\n    if lv[\\\"plans\\\"] == 0:\\n        return \\\"model ok but no plan for any goal (goal templates do not express the win condition or search failed)\\\"\\n    if lv[\\\"mismatches\\\"] >= max(3, lv[\\\"plans\\\"] // 2):\\n        return f\\\"plans executed but predictions failed ({lv['mismatches']} mismatches: model wrong outside the observed states)\\\"\\n    if lv[\\\"exec_outcomes\\\"].get(\\\"plan exhausted\\\", 0) >= 3 and not lv[\\\"cleared\\\"]:\\n        return \\\"plans completed without a level-up: the goal reached is not the win condition\\\"\\n    if lv[\\\"game_overs\\\"] >= 2:\\n        return f\\\"repeated game overs ({lv['game_overs']}): action limit / hazard not modelled\\\"\\n    return \\\"budget or time ran out while still exploring\\\"\\n\\n\\ndef report(run_id: str, game: str) -> dict:\\n    run, g, ev, log_lines = load(run_id, game)\\n    levels = per_level(ev, g)\\n    for lv in levels:\\n        lv[\\\"verdict\\\"] = verdict(lv)\\n    llm = [l for l in log_lines if \\\"llm candidate\\\" in l or \\\"llm job\\\" in l]\\n    return {\\\"run_id\\\": run_id, \\\"game\\\": game, \\\"levels_completed\\\": g.get(\\\"levels_completed\\\"), \\\"levels_total\\\": g.get(\\\"levels_total\\\"), \\\"actions\\\": g.get(\\\"actions\\\"),\\n            \\\"score\\\": g.get(\\\"score\\\"), \\\"baseline\\\": g.get(\\\"level_baseline_actions\\\"), \\\"level_actions\\\": g.get(\\\"level_actions\\\"), \\\"stop\\\": g.get(\\\"stop_reason\\\"),\\n            \\\"llm_calls\\\": g.get(\\\"llm_calls\\\"), \\\"llm_lines\\\": llm[-12:], \\\"final_hypotheses\\\": g.get(\\\"hypotheses\\\"), \\\"final_goals\\\": g.get(\\\"goals\\\"), \\\"levels\\\": levels}\\n\\n\\ndef main(argv=None) -> None:\\n    ap = argparse.ArgumentParser(); ap.add_argument(\\\"run_id\\\"); ap.add_argument(\\\"game\\\"); ap.add_argument(\\\"--json\\\", action=\\\"store_true\\\")\\n    a = ap.parse_args(argv)\\n    r = report(a.run_id, a.game)\\n    if a.json:\\n        print(json.dumps(r, indent=1, default=str)); return\\n    print(f\\\"== {r['game']} run {r['run_id']}: levels {r['levels_completed']}/{r['levels_total']} actions {r['actions']} score {r['score']} stop {r['stop']} llm {r['llm_calls']}\\\")\\n    print(f\\\"   baseline per level {r['baseline']} | ours {r['level_actions']}\\\")\\n    for lv in r[\\\"levels\\\"]:\\n        print(f\\\"-- level {lv['level']}: {lv['seconds']}s {lv['actions']} actions | hyp rounds {lv['hyp_rounds']} score {lv['score_first']}->{lv['score_last']} (max {lv['score_max']}) | plans {lv['plans']} lens {lv['plan_lengths'][:12]}\\\")\\n        print(f\\\"   actions by {lv['actions_by']}\\\")\\n        print(f\\\"   goals {lv['top_goals']} | exec {lv['exec_outcomes']} | mismatches {lv['mismatches']} game_overs {lv['game_overs']} stuck {lv['stuck']} llm_jobs {lv['llm_jobs']}\\\")\\n        print(f\\\"   VERDICT: {lv['verdict']}\\\")\\n    print(\\\"   final hypotheses:\\\", [(h['name'], h['score']) for h in (r['final_hypotheses'] or [])][:4])\\n    print(\\\"   final goals:\\\", [(x['name'], x['confidence']) for x in (r['final_goals'] or [])][:5])\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"pbg/harness/replay_runner.py\": \"\\\"\\\"\\\"harness/replay_runner.py \\u2014 module-level evaluation over recorded logs without the live environment (spec \\u00a716).\\n\\nGiven a raw.jsonl (human or agent play), it: replays every frame through Perception (timing, tracking stability),\\nclassifies action semantics, induces prior-driven hypotheses (+ optional LLM refinement), evaluates them on the whole\\nlog, runs goal inference with the level-up filter, and (given a verified model) plans from the level-start scene.\\n\\n    .venv/bin/python -m pbg.harness.replay_runner path/to/raw.jsonl [--llm] [--json out.json]\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport time\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nfrom ..core.contracts import RuleModel\\nfrom ..core.types import Action, Transition\\nfrom ..env.replay import ReplayEnv\\nfrom ..goal.infer import GoalInference\\nfrom ..perception import Perception\\nfrom ..planner.execute import Planner\\nfrom ..probe.semantics import classify_actions\\nfrom ..wml.refine import WorldModelLab\\n\\n\\ndef replay_transitions(records, game_id: str = \\\"\\\") -> tuple[list[Transition], Perception, dict]:\\n    env = ReplayEnv(records, game_id, lenient=True)\\n    p = Perception()\\n    frame = env.reset(); scene = p.parse(frame, None)\\n    ts: list[Transition] = []; step = 0\\n    id_changes = 0; tracked = 0\\n    while True:\\n        a = env.next_action()\\n        if a is None:\\n            break\\n        rt = env.step(a)\\n        if rt.error:\\n            break\\n        step += 1\\n        if a.type == \\\"RESET\\\":\\n            scene = p.parse(rt.after, None); continue\\n        after = p.parse(rt.after, scene)\\n        t = p.transition(scene, a, after, before_frame=rt.before, after_frame=rt.after, status_change=rt.status_change, tid=f\\\"{env.game_id}:{rt.level}:{step}\\\",\\n                         level=rt.level, step_idx=step)\\n        if p.merge_confirmed:\\n            p.refit(ts); after = t.after\\n        ts.append(t)\\n        if p.reparse_needed:\\n            p.reparse_needed = False\\n            lv = [x for x in ts if x.level == rt.level]\\n            re = p.reparse_level(lv)\\n            if re is not None:\\n                after = re\\n        # tracking stability: objects that stayed (same colour+shape at the same place) must keep their id\\n        before_ids = {o.identity(): o.id for o in scene.objects}\\n        for o in after.objects:\\n            if o.identity() in before_ids:\\n                tracked += 1\\n                if before_ids[o.identity()] != o.id:\\n                    id_changes += 1\\n        scene = after\\n        if rt.status_change == \\\"LEVEL_UP\\\":\\n            scene = p.parse(rt.after, None)\\n    stats = {\\\"transitions\\\": len(ts), \\\"parse_ms_mean\\\": round(1000 * sum(p.timings) / max(1, len(p.timings)), 2), \\\"parse_ms_max\\\": round(1000 * max(p.timings or [0]), 2),\\n             \\\"tracking_keep_rate\\\": round(1 - id_changes / tracked, 4) if tracked else None, \\\"levels\\\": sorted({t.level for t in ts}),\\n             \\\"level_ups\\\": sum(1 for t in ts if t.status_change in (\\\"LEVEL_UP\\\", \\\"WIN\\\"))}\\n    return ts, p, stats\\n\\n\\ndef run_pipeline(raw_path: Path, *, llm=None, plan: bool = True, log=print) -> dict:\\n    records = [json.loads(l) for l in Path(raw_path).read_text().splitlines() if l.strip()]\\n    game_id = records[0].get(\\\"game_id\\\", Path(raw_path).parent.name) if records else \\\"?\\\"\\n    ts, perception, stats = replay_transitions(records, game_id)\\n    out = {\\\"game_id\\\": game_id, \\\"perception\\\": stats}\\n    if not ts:\\n        return out\\n    sem = classify_actions(ts)\\n    out[\\\"semantics\\\"] = {k: v for k, v in sem.items()}\\n    avail = sorted({t.action.key for t in ts})\\n    available = [Action.button(int(k[6:])) for k in avail if k.startswith(\\\"ACTION\\\")] + ([Action(\\\"CLICK\\\")] if \\\"CLICK\\\" in avail else [])\\n    lab = WorldModelLab(llm=llm, log=log)\\n    t0 = time.time()\\n    H = lab.refine(ts, [], None, scene=ts[0].before, semantics=sem, available=available, use_llm=llm is not None)\\n    out[\\\"hypotheses\\\"] = [{\\\"name\\\": h.name, \\\"score\\\": round(h.score, 3), \\\"coverage\\\": round(h.coverage, 3), \\\"violations\\\": len(h.violations), \\\"verified\\\": h.verified,\\n                          \\\"rules\\\": {r.name: round(r.confidence, 2) for r in h.model.rules()} if isinstance(h.model, RuleModel) else {}} for h in H]\\n    out[\\\"wml_seconds\\\"] = round(time.time() - t0, 2); out[\\\"llm_calls\\\"] = lab.llm_calls\\n    gi = GoalInference(log=log)\\n    roles_fn = (lambda sc, m=H[0].model: m.with_roles(sc)) if H and isinstance(H[0].model, RuleModel) else None\\n    by_level: dict[int, list[Transition]] = {}\\n    for t in ts:\\n        by_level.setdefault(t.level, []).append(t)\\n    goals_out = {}\\n    for lv, lts in by_level.items():\\n        G = gi.refine(ts, [], None, lv, lts[0].before, roles_fn=roles_fn)\\n        goals_out[lv] = [g.to_json() for g in G[:5]]\\n    out[\\\"goals\\\"] = goals_out\\n    if plan and H and H[0].score >= 1.0:\\n        planner = Planner(log=log)\\n        plans = {}\\n        for lv, lts in by_level.items():\\n            G = gi.refine(ts, [], None, lv, lts[0].before, roles_fn=roles_fn)\\n            p = planner.search(lts[0].before, H[:1], G, available=available, semantics=sem)\\n            human = sum(1 for t in lts)\\n            plans[lv] = {\\\"plan_len\\\": len(p) if p else None, \\\"human_actions\\\": human, \\\"reason\\\": planner.last_info.reason if planner.last_info else \\\"\\\"}\\n        out[\\\"plans\\\"] = plans\\n    return out\\n\\n\\ndef main(argv=None) -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"raw\\\", nargs=\\\"+\\\"); ap.add_argument(\\\"--llm\\\", action=\\\"store_true\\\"); ap.add_argument(\\\"--json\\\"); ap.add_argument(\\\"--no-plan\\\", action=\\\"store_true\\\")\\n    a = ap.parse_args(argv)\\n    llm = None\\n    if a.llm:\\n        from ..llm.gateway import LLMGateway\\n        llm = LLMGateway()\\n    results = [run_pipeline(Path(p), llm=llm, plan=not a.no_plan) for p in a.raw]\\n    for r in results:\\n        print(json.dumps(r, indent=1, default=str)[:4000])\\n    if a.json:\\n        Path(a.json).write_text(json.dumps(results, indent=1, default=str))\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"pbg/harness/scene_at.py\": \"\\\"\\\"\\\"harness/scene_at.py \\u2014 pull the Scene at any (game, level, step) of a recorded log for module-level tests (spec \\u00a716).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nfrom ..core.types import Scene, Transition\\nfrom .replay_runner import replay_transitions\\n\\n\\nclass Harness:\\n    def __init__(self, logs_root: Path):\\n        self.root = Path(logs_root)\\n        self._cache: dict[str, list[Transition]] = {}\\n\\n    def raw_path(self, game: str) -> Path:\\n        p = self.root / game / \\\"raw.jsonl\\\"\\n        if not p.exists():\\n            raise FileNotFoundError(p)\\n        return p\\n\\n    def transitions(self, game: str) -> list[Transition]:\\n        if game not in self._cache:\\n            records = [json.loads(l) for l in self.raw_path(game).read_text().splitlines() if l.strip()]\\n            self._cache[game], _, _ = replay_transitions(records, game)\\n        return self._cache[game]\\n\\n    def scene_at(self, game: str, level: int, step: int) -> Optional[Scene]:\\n        \\\"\\\"\\\"Scene after `step` actions on `level` (step 0 = the level's first scene).\\\"\\\"\\\"\\n        ts = [t for t in self.transitions(game) if t.level == level]\\n        if not ts:\\n            return None\\n        if step <= 0:\\n            return ts[0].before\\n        return ts[min(step, len(ts)) - 1].after\\n\", \"pbg/hypothesis/__init__.py\": \"\\\"\\\"\\\"Game-hypothesis policy: look, hypothesise, test, verify, revise, plan (the same process for every game).\\\"\\\"\\\"\\n\", \"pbg/hypothesis/contract.py\": \"\\\"\\\"\\\"hypothesis/contract.py \\u2014 the game-hypothesis document the reasoning model writes, and the observations it gets.\\n\\nThe policy (hypothesis/policy.py) is the same for every game: look at the board, write a hypothesis of the whole game\\n(roles, controls, dynamics, win condition), name the actions that would test what is uncertain, run those, verify the\\nhypothesis at pixel level, revise with the counter-examples, and only plan on a verified hypothesis. Nothing in this\\nmodule knows a game; the priors of the mechanism library are offered as hints, never applied by themselves.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Optional\\n\\nimport numpy as np\\n\\nfrom ..core.types import Action, Scene, Transition\\nfrom ..wml.prompts import API_SUMMARY\\n\\nCONTRACT = '''You are playing an unknown pixel board game (64x64 grid, colours 0-15) through an agent. You never see the rules:\\nyou infer them from the board and from what actions did. Work like a scientist: state a HYPOTHESIS of the whole game,\\nsay what is uncertain, and name the cheapest actions that would settle it. Every action costs score, so test before\\nyou plan and plan only what the hypothesis supports.\\n\\nReply with exactly ONE ```python block defining:\\n\\nHYPOTHESIS = {\\n  \\\"summary\\\": \\\"one paragraph: what the objects are, what the player controls, what the win condition probably is\\\",\\n  \\\"roles\\\": {\\\"<colour or description>\\\": \\\"<role>\\\", ...},\\n  \\\"controls\\\": \\\"how actions change the board (be concrete: which click targets or buttons do what, by how much)\\\",\\n  \\\"win\\\": \\\"the win condition as you believe it\\\",\\n  \\\"uncertain\\\": [\\\"what you still do not know\\\", ...],\\n}\\n\\ndef role_fn(scene) -> dict:            # object id -> role name (strings; use the vocabulary below or custom:<name>)\\ndef build_model() -> RuleModel         # RuleModel(rules, role_fn, default=\\\"unknown\\\"); rules predict the NEXT scene exactly\\ndef build_goal():                      # object with name, is_goal(scene) -> bool, progress(scene) -> float in [0,1]\\ndef candidate_actions(scene) -> list:  # the actions worth planning with in this state (Action.button(i) / Action.click(row, col));\\n                                       # for click games include the POINTS that matter (targets, gaps), not only objects\\ndef test_actions(scene) -> list:       # 1-3 actions whose outcome best discriminates the \\\"uncertain\\\" items (empty if none)\\ndef attempt_actions(scene) -> list:    # 1-3 actions that, under your hypothesis, move the board TOWARD the win right now\\n                                       # (used while the model is still approximate: act, observe, re-think)\\ndef ignore_boxes(scene) -> list:       # [(r0, c0, r1, c1), ...] display elements (counters, timers) to ignore when checking predictions\\n\\nScene, Object, Action, Rule, RuleModel, numpy (as np) and every mechanism of the library below are already defined in your\\nnamespace: call them directly (`from mechanisms import move_role` is tolerated, nothing else may be imported).\\nEvery loop in your code must be bounded (by the grid, the objects, a step count): a rule that runs longer than 2 seconds on one\\ntransition is killed and counts as a wrong prediction.\\nKeep the code compact (well under 200 lines): simple geometric rules beat elaborate simulations, and a reply cut off by\\nthe length limit is worthless. Rules for the code: numpy only; no game names; no coordinates memorised from this level except through scene queries;\\nno comments that argue -- put reasoning in HYPOTHESIS. The prediction check compares the rendered predicted grid with the\\nreal next grid pixel by pixel outside ignore_boxes: a rule that moves an object must move ALL its pixels correctly.\\nAlways predict: an approximate next board (right object, right direction, roughly right distance) is far more useful\\nthan None -- the check also scores how close the changed region is, and planning works on approximate models by acting\\none step at a time. Return None only for actions you have no idea about; write what is uncertain in \\\"uncertain\\\".\\nAll functions receive the scene WITH your role_fn roles already applied (o.role, scene.by_role work everywhere).\\nRoles vocabulary: agent, mover, button, bar, wall, target, marker, key, door, collectible, hazard, indicator, decoration.\\n'''\\n\\n\\ndef frame_text(grid: np.ndarray, rows: Optional[tuple[int, int]] = None, cols: Optional[tuple[int, int]] = None) -> str:\\n    \\\"\\\"\\\"The grid (or a window of it) as hex rows with a row index, one char per cell.\\\"\\\"\\\"\\n    g = np.asarray(grid)\\n    r0, r1 = rows or (0, g.shape[0]); c0, c1 = cols or (0, g.shape[1])\\n    head = \\\"     \\\" + \\\"\\\".join(str(c % 10) for c in range(c0, c1))\\n    lines = [head] + [f\\\"{r:3d}: \\\" + \\\"\\\".join(format(int(v), \\\"x\\\") for v in g[r, c0:c1]) for r in range(r0, r1)]\\n    return \\\"\\\\n\\\".join(lines)\\n\\n\\ndef scene_objects_text(scene: Scene, limit: int = 120) -> str:\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    lines = [\\\"regions: \\\" + \\\", \\\".join(f\\\"{r.id}(bg={r.bg_color},{r.kind_hint},rows {r.bbox[0]}-{r.bbox[2] - 1},cols {r.bbox[1]}-{r.bbox[3] - 1})\\\" for r in scene.regions)]\\n    objs = sorted(scene.objects, key=lambda o: (o.region in strips, -o.area, o.id))\\n    for o in objs[:limit]:\\n        col = f\\\"colours={list(o.colors)}\\\" if len(o.colors) > 1 else f\\\"colour={o.color}\\\"\\n        lines.append(f\\\"  obj#{o.id} {col} bbox=({o.bbox[0]},{o.bbox[1]})-({o.bbox[2]},{o.bbox[3]}) size={o.height}x{o.width} area={o.area}\\\"\\n                     + (\\\" [ui strip]\\\" if o.region in strips else \\\"\\\"))\\n    if len(objs) > limit:\\n        lines.append(f\\\"  ... {len(objs) - limit} more\\\")\\n    return \\\"\\\\n\\\".join(lines)\\n\\n\\ndef change_window(t: Transition, pad: int = 2) -> Optional[tuple[int, int, int, int]]:\\n    b = np.asarray(t.before_frame.grid); a = np.asarray(t.after_frame.grid)\\n    ys, xs = np.nonzero(a != b)\\n    if len(ys) == 0:\\n        return None\\n    return (max(0, int(ys.min()) - pad), max(0, int(xs.min()) - pad), min(b.shape[0], int(ys.max()) + 1 + pad), min(b.shape[1], int(xs.max()) + 1 + pad))\\n\\n\\ndef transition_text(t: Transition, ignore_boxes: list = (), max_rows: int = 22, max_cols: int = 40) -> str:\\n    \\\"\\\"\\\"What one action did: the action (with click position), the changed pixels, and before/after windows of the change.\\\"\\\"\\\"\\n    if t.action.type == \\\"RESET\\\" or t.before_frame is None or t.after_frame is None:\\n        return f\\\"[{t.id}] RESET -> the level restarted from its initial board (after a GAME_OVER or a deliberate reset)\\\"\\n    b = np.asarray(t.before_frame.grid); a = np.asarray(t.after_frame.grid)\\n    mask = a != b\\n    for (r0, c0, r1, c1) in ignore_boxes:\\n        mask[r0:r1, c0:c1] = False\\n    n = int(mask.sum())\\n    head = f\\\"[{t.id}] {t.action.label()}\\\"\\n    if t.status_change:\\n        head += f\\\"  -> {t.status_change}\\\"\\n    if n == 0:\\n        return head + \\\" -> no change (outside ignored boxes)\\\"\\n    ys, xs = np.nonzero(mask)\\n    r0, c0, r1, c1 = max(0, int(ys.min()) - 1), max(0, int(xs.min()) - 1), min(64, int(ys.max()) + 2), min(64, int(xs.max()) + 2)\\n    lines = [head + f\\\" -> {n} px changed in rows {r0}-{r1 - 1}, cols {c0}-{c1 - 1}\\\"]\\n    if t.action.type == \\\"CLICK\\\":\\n        lines.append(f\\\"    click at ({t.action.row},{t.action.col}); change window centre ({(r0 + r1) // 2},{(c0 + c1) // 2})\\\")\\n    # object-level summary\\n    bo = {o.id: o for o in t.before.objects}; ao = {o.id: o for o in t.after.objects}\\n    parts = []\\n    for oid, (dr, dc) in t.diff.moved[:6]:\\n        o = bo.get(oid)\\n        if o is not None:\\n            parts.append(f\\\"obj#{oid}(colour {o.color}, {o.height}x{o.width}) moved ({dr:+d},{dc:+d})\\\")\\n    for oid, old, new in t.diff.recolored[:4]:\\n        parts.append(f\\\"obj#{oid} recoloured {old}->{new}\\\")\\n    for o in t.diff.appeared[:4]:\\n        parts.append(f\\\"obj#{o.id}(colour {o.color}) appeared at ({o.bbox[0]},{o.bbox[1]})\\\")\\n    for o in t.diff.disappeared[:4]:\\n        parts.append(f\\\"obj#{o.id}(colour {o.color}) disappeared from ({o.bbox[0]},{o.bbox[1]})\\\")\\n    if parts:\\n        lines.append(\\\"    \\\" + \\\"; \\\".join(parts))\\n    if (r1 - r0) <= max_rows and (c1 - c0) <= max_cols:\\n        lines.append(\\\"    before:\\\"); lines += [\\\"      \\\" + l for l in frame_text(b, (r0, r1), (c0, c1)).splitlines()]\\n        lines.append(\\\"    after:\\\"); lines += [\\\"      \\\" + l for l in frame_text(a, (r0, r1), (c0, c1)).splitlines()]\\n        if t.intermediate_frames:\\n            lines.append(f\\\"    ({len(t.intermediate_frames)} intermediate animation frames between them)\\\")\\n    else:\\n        lines.append(f\\\"    (change window {r1 - r0}x{c1 - c0} too large to print)\\\")\\n    return \\\"\\\\n\\\".join(lines)\\n\\n\\ndef hypothesis_prompt(*, scene: Scene, grid: np.ndarray, log: list[Transition], available: list[Action], prior_signatures: str,\\n                      previous_code: Optional[str], counterexamples: list[str], level: int, ignore_boxes: list = (), max_transitions: int = 24,\\n                      rejected: list[str] = ()) -> list[dict]:\\n    buttons = sorted({a.id for a in available if a.type == \\\"BUTTON\\\"})\\n    acts = (\\\"buttons \\\" + \\\", \\\".join(f\\\"ACTION{i}\\\" for i in buttons) if buttons else \\\"\\\") + (\\\" and \\\" if buttons and any(a.type == \\\"CLICK\\\" for a in available) else \\\"\\\") + \\\\\\n           (\\\"clicks anywhere on the grid\\\" if any(a.type == \\\"CLICK\\\" for a in available) else \\\"\\\")\\n    user = [f\\\"# Level {level}. Available actions: {acts}.\\\", \\\"\\\", \\\"# Current board (hex digits = colours)\\\", frame_text(grid), \\\"\\\",\\n            \\\"# Objects the perception found (small marks included)\\\", scene_objects_text(scene), \\\"\\\"]\\n    if log:\\n        recent = log[-max_transitions:]\\n        user += [f\\\"# What actions did so far ({len(log)} actions on this level; last {len(recent)} shown, oldest first)\\\"]\\n        user += [transition_text(t, ignore_boxes) for t in recent]\\n        user.append(\\\"\\\")\\n    else:\\n        user += [\\\"# No action has been taken yet on this level: propose the hypothesis from the board alone and the first tests.\\\", \\\"\\\"]\\n    if counterexamples:\\n        user += [\\\"# Counter-examples: where the previous hypothesis predicted wrongly (predicted vs real)\\\"] + counterexamples + [\\\"\\\"]\\n    if rejected:\\n        user += [\\\"# Rejected earlier (do not repeat): \\\" + \\\"; \\\".join(rejected)]\\n    if previous_code:\\n        user += [\\\"# Previous hypothesis code (revise it; keep what verified)\\\", \\\"```python\\\", previous_code, \\\"```\\\", \\\"\\\"]\\n    if prior_signatures:\\n        user += [\\\"# Mechanism library you may call (signatures)\\\", prior_signatures, \\\"\\\"]\\n    user.append(\\\"Write the hypothesis document and code now.\\\")\\n    return [{\\\"role\\\": \\\"system\\\", \\\"content\\\": CONTRACT + \\\"\\\\n\\\" + API_SUMMARY}, {\\\"role\\\": \\\"user\\\", \\\"content\\\": \\\"\\\\n\\\".join(user)}]\\n\", \"pbg/hypothesis/explore.py\": \"\\\"\\\"\\\"hypothesis/explore.py \\u2014 level-1 exploration before the first hypothesis (docs/029).\\n\\nLevel 1 weighs 1.8-4.8% of a game score and later levels count their actions from the level-up, so level 1 is where\\nactions are cheapest: act first, collect transitions (and the GAME_OVER condition), then hypothesise from the log.\\n\\n    buttons   a random available button, never the one just pressed (repeating the last move shows nothing new)\\n    clicks    one click per object class (colour + shape) not tried yet, then classes that reacted, then any object\\n    stop      the cap, GAME_OVER, a level-up, or `stale` actions in a row with no new change pattern\\n\\nRandom clicks on the whole grid reveal almost nothing in click games (tn36 0/61, vc33 1/50 changed frames), hence the\\nobject classes; movement games are rich under random moves (ls20 96/135).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport random\\nfrom typing import Callable, Optional\\n\\nimport numpy as np\\n\\nfrom ..core.types import Action, Transition\\nfrom ..probe.clickmap import object_key\\n\\n\\ndef _change_key(t: Transition) -> Optional[str]:\\n    \\\"\\\"\\\"Signature of what an action changed (None = no change beyond a counter tick of <= 4 pixels).\\\"\\\"\\\"\\n    if t.before_frame is None or t.after_frame is None:\\n        return None\\n    d = np.asarray(t.before_frame.grid) != np.asarray(t.after_frame.grid)\\n    if int(d.sum()) <= 4:\\n        return None\\n    return t.action.label().split(\\\"(\\\")[0] + \\\":\\\" + hashlib.md5(d.tobytes()).hexdigest()[:8]\\n\\n\\ndef explore_level(s, act: Callable[[Action, str], Optional[Transition]], *, cap: int, stale: int = 12, unreactive: int = 8, seed: int = 0,\\n                  emit: Callable[[str], None] = lambda m: None) -> dict:\\n    \\\"\\\"\\\"Spend at most `cap` actions exploring the current level; returns counts for the event log / result JSON.\\\"\\\"\\\"\\n    rng = random.Random(seed)\\n    level = s.level\\n    last_button: Optional[int] = None\\n    tried: dict[str, int] = {}          # object class -> clicks\\n    reacted: set[str] = set()           # object classes whose click changed the frame\\n    patterns: set[str] = set()\\n    used = since_new = changed = 0\\n    unreactive_clicks = 0               # object-class clicks in a row that changed nothing (a click game that ignores object clicks)\\n    stop = \\\"cap\\\"\\n    while used < cap:\\n        if s.finished() or s.timed_out():\\n            stop = \\\"clock\\\"; break\\n        if s.env.status().state == \\\"GAME_OVER\\\":\\n            stop = \\\"game_over\\\"; break\\n        if s.level != level:\\n            stop = \\\"level_up\\\"; break\\n        if since_new >= stale:\\n            stop = f\\\"no new change in {stale} actions\\\"; break\\n        if unreactive_clicks >= unreactive:\\n            stop = f\\\"{unreactive} object clicks without any reaction\\\"; break\\n        avail = s.available_actions()\\n        buttons = [a for a in avail if a.type == \\\"BUTTON\\\"]\\n        can_click = any(a.type == \\\"CLICK\\\" for a in avail)\\n        a: Optional[Action] = None\\n        klass = None\\n        if buttons and (not can_click or rng.random() < 0.5):\\n            pool = [b for b in buttons if b.id != last_button] or buttons\\n            a = rng.choice(pool)\\n        elif can_click:\\n            strips = {r.id for r in s.scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n            objs = [o for o in s.scene.objects if o.region not in strips]\\n            if objs:\\n                by_class: dict[str, list] = {}\\n                for o in objs:\\n                    by_class.setdefault(object_key(o), []).append(o)\\n                fresh = [k for k in by_class if k not in tried]\\n                live = [k for k in by_class if k in reacted]\\n                klass = rng.choice(fresh) if fresh else (rng.choice(live) if live else rng.choice(list(by_class)))\\n                o = rng.choice(by_class[klass])\\n                r, c = o.cells()[rng.randrange(len(o.cells()))] if o.area > 1 else o.center\\n                a = Action.click(int(r), int(c))\\n            else:\\n                a = Action.click(rng.randrange(64), rng.randrange(64))\\n        if a is None:\\n            stop = \\\"no action\\\"; break\\n        t = act(a, \\\"explore\\\")\\n        if t is None:\\n            stop = \\\"refused\\\"; break\\n        used += 1\\n        if a.type == \\\"BUTTON\\\":\\n            last_button = a.id\\n        if klass is not None:\\n            tried[klass] = tried.get(klass, 0) + 1\\n        k = _change_key(t)\\n        if klass is not None:\\n            unreactive_clicks = 0 if k is not None else unreactive_clicks + 1\\n        if k is not None:\\n            changed += 1\\n            if klass is not None:\\n                reacted.add(klass)\\n            if k not in patterns:\\n                patterns.add(k); since_new = 0\\n                continue\\n        since_new += 1\\n    if s.env.status().state == \\\"GAME_OVER\\\":\\n        stop = \\\"game_over\\\"\\n    out = {\\\"actions\\\": used, \\\"changed\\\": changed, \\\"patterns\\\": len(patterns), \\\"classes_tried\\\": len(tried), \\\"classes_reacted\\\": len(reacted), \\\"stop\\\": stop}\\n    emit(f\\\"level-{level} exploration: {used} actions, {changed} changed the board, {len(patterns)} distinct change patterns, \\\"\\n         f\\\"{len(reacted)}/{len(tried)} clicked classes reacted; stop: {stop}\\\")\\n    return out\\n\", \"pbg/hypothesis/goals.py\": \"\\\"\\\"\\\"hypothesis/goals.py \\u2014 the win condition, inferred from the moment a level was completed (docs/029, task 8).\\n\\nOn level 2+ the models are often exact (lp85, vc33, m0r0 verified at 1.00) and still no template goal yields a plan:\\nthe win condition is not in the template library. What we do have is evidence: the board right before and right after\\nthe previous level ended. The reasoning model is asked for ONE thing \\u2014 a goal predicate as code \\u2014 and the predicate is\\nchecked against that evidence (true after the win, false before it and on earlier boards of that level, false on the\\ncurrent board). Only predicates that pass become goals; they are tried first by the planner and carried to the next\\nlevel, where the new win evidence re-checks them.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport re\\nimport textwrap\\nimport threading\\nfrom dataclasses import dataclass, field\\nfrom typing import Callable, Optional\\n\\nimport numpy as np\\n\\nfrom ..core.types import Scene\\nfrom ..wml.prompts import API_SUMMARY\\nfrom .contract import frame_text\\n\\nMAX_ROUNDS_PER_LEVEL = 3\\n\\n\\n@dataclass\\nclass WinEvidence:\\n    level: int\\n    action: str\\n    before: Scene\\n    after: Scene\\n    before_grid: np.ndarray\\n    after_grid: np.ndarray\\n    earlier: list = field(default_factory=list)     # scenes of that level where the goal was NOT met (negatives)\\n\\n\\nGOAL_CONTRACT = '''You infer the WIN CONDITION of an unknown pixel board game. A level was just completed: you see the board right\\nBEFORE the winning action and right AFTER it, earlier boards of that level (where the level was not yet won), and the\\ncurrent board of the NEXT level. Write the condition as code and nothing else:\\n\\ndef build_goal():\\n    class Goal:\\n        name = \\\"<short name>\\\"\\n        def is_goal(self, scene) -> bool: ...      # TRUE on the after-board, FALSE on the before-board and on every earlier board\\n        def progress(self, scene) -> float: ...   # 0..1, how close the board is to the condition (higher = closer)\\n    return Goal()\\n\\nReply with exactly ONE ```python block of at most 40 lines and NO comments: do your reasoning before the block, never\\ninside it (a reply cut off by the length limit is worthless). Rules: describe the condition through object RELATIONS (roles, colours, counts,\\nalignment, containment, adjacency, equality of positions/shapes), never through coordinates of this level: the same\\npredicate must apply to the next level, where objects sit elsewhere and there may be more of them. Scene, Object and\\nnumpy (as np) are already defined; do not import. Objects carry .role (from the model), .color, .colors, .bbox\\n(r0, c0, r1, c1), .center, .height, .width, .area, .shape_sig, .region; scene.objects, scene.by_role(r), scene.by_color(c),\\nscene.regions (.id, .bg_color, .bbox, .kind_hint). Objects in a ui strip are counters, not part of the condition.\\n'''\\n\\n\\ndef _objects_text(scene: Scene, limit: int = 60) -> str:\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    objs = sorted([o for o in scene.objects if o.region not in strips], key=lambda o: (-o.area, o.id))\\n    lines = []\\n    for o in objs[:limit]:\\n        col = f\\\"colours={list(o.colors)}\\\" if len(o.colors) > 1 else f\\\"colour={o.color}\\\"\\n        lines.append(f\\\"  obj#{o.id} role={o.role or '?'} {col} bbox=({o.bbox[0]},{o.bbox[1]})-({o.bbox[2]},{o.bbox[3]}) size={o.height}x{o.width}\\\")\\n    if len(objs) > limit:\\n        lines.append(f\\\"  ... {len(objs) - limit} more\\\")\\n    return \\\"\\\\n\\\".join(lines) or \\\"  (no objects)\\\"\\n\\n\\ndef _window(a: np.ndarray, b: np.ndarray, pad: int = 2) -> Optional[tuple]:\\n    ys, xs = np.nonzero(np.asarray(a) != np.asarray(b))\\n    if len(ys) == 0:\\n        return None\\n    return (max(0, int(ys.min()) - pad), max(0, int(xs.min()) - pad), min(a.shape[0], int(ys.max()) + 1 + pad), min(a.shape[1], int(xs.max()) + 1 + pad))\\n\\n\\ndef goal_prompt(ev: WinEvidence, roled: Callable[[Scene], Scene], current: Scene, current_grid: np.ndarray, *, rejected: list = (), feedback: list = (),\\n                alternative: int = 0) -> list[dict]:\\n    before, after = roled(ev.before), roled(ev.after)\\n    user = [f\\\"# Level {ev.level} was completed by the action {ev.action}.\\\", \\\"\\\",\\n            \\\"# Board right BEFORE the winning action (objects with roles)\\\", _objects_text(before), \\\"\\\",\\n            \\\"# Board right AFTER it (the level ended here)\\\", _objects_text(after), \\\"\\\"]\\n    w = _window(ev.before_grid, ev.after_grid)\\n    if w is not None and (w[2] - w[0]) <= 30 and (w[3] - w[1]) <= 48:\\n        user += [f\\\"# The cells that changed with the winning action, rows {w[0]}-{w[2] - 1}, cols {w[1]}-{w[3] - 1} (hex colours)\\\",\\n                 \\\"before:\\\", frame_text(ev.before_grid, (w[0], w[2]), (w[1], w[3])), \\\"after:\\\", frame_text(ev.after_grid, (w[0], w[2]), (w[1], w[3])), \\\"\\\"]\\n    for i, sc in enumerate(ev.earlier[:4]):\\n        user += [f\\\"# An earlier board of level {ev.level} where the level was NOT yet won ({i + 1})\\\", _objects_text(roled(sc), limit=40), \\\"\\\"]\\n    user += [f\\\"# Current board (level {ev.level + 1}, not won): the predicate must be FALSE here and reachable by play\\\", _objects_text(roled(current)), \\\"\\\"]\\n    if rejected:\\n        user += [\\\"# Rejected earlier (do not repeat): \\\" + \\\"; \\\".join(str(r)[:60] for r in rejected), \\\"\\\"]\\n    if feedback:\\n        user += [\\\"# Why the previous proposals failed verification\\\"] + [f\\\"- {f}\\\" for f in feedback[-4:]] + [\\\"\\\"]\\n    if alternative:\\n        user += [f\\\"# Alternative {alternative}: consider a different reading of what made the level end (a count reached, a shape completed, an alignment, a containment, a colour match).\\\"]\\n    user.append(\\\"Write build_goal() now.\\\")\\n    return [{\\\"role\\\": \\\"system\\\", \\\"content\\\": GOAL_CONTRACT + \\\"\\\\n\\\" + API_SUMMARY}, {\\\"role\\\": \\\"user\\\", \\\"content\\\": \\\"\\\\n\\\".join(user)}]\\n\\n\\ndef verify_goal(goal, roled: Callable[[Scene], Scene], ev: WinEvidence, current: Optional[Scene]) -> tuple[bool, str]:\\n    \\\"\\\"\\\"True on the after-board, false before it and on the earlier boards, false on the current board, progress in [0,1).\\\"\\\"\\\"\\n    try:\\n        if not goal.is_goal(roled(ev.after)):\\n            return False, f\\\"'{goal.name}' is FALSE on the board right after level {ev.level} was completed\\\"\\n        if goal.is_goal(roled(ev.before)):\\n            return False, f\\\"'{goal.name}' is already TRUE on the board right before the winning action (it must become true BY that action)\\\"\\n        for i, sc in enumerate(ev.earlier):\\n            if goal.is_goal(roled(sc)):\\n                return False, f\\\"'{goal.name}' is TRUE on an earlier board of level {ev.level} where the level was not won\\\"\\n        if current is not None:\\n            if goal.is_goal(roled(current)):\\n                return False, f\\\"'{goal.name}' is already TRUE on the current board of level {ev.level + 1}, which is not won\\\"\\n            p = float(goal.progress(roled(current)))\\n            if not (0.0 <= p <= 1.0):\\n                return False, f\\\"'{goal.name}'.progress returned {p!r} on the current board (must be within 0..1)\\\"\\n        p_after = float(goal.progress(roled(ev.after))); p_before = float(goal.progress(roled(ev.before)))\\n        if p_after < p_before:\\n            return False, f\\\"'{goal.name}'.progress is lower after the win ({p_after:.2f}) than before ({p_before:.2f}): progress must rise toward the condition\\\"\\n    except Exception as e:\\n        return False, f\\\"'{getattr(goal, 'name', '?')}' raised {type(e).__name__}: {str(e)[:100]}\\\"\\n    return True, \\\"ok\\\"\\n\\n\\n_CLASS_RE = re.compile(r\\\"^class\\\\s+(\\\\w+)\\\\b\\\", re.M)\\n\\n\\ndef normalise_goal_code(code: str) -> str:\\n    \\\"\\\"\\\"The local model often writes the predicate without the build_goal() wrapper (v7: 9 of 14 rejections were\\n    'build_goal missing'): a top-level class with is_goal/progress, or bare is_goal(scene)/progress(scene) functions,\\n    is wrapped so the sandbox can load it. Code that already defines build_goal is returned unchanged.\\\"\\\"\\\"\\n    code = textwrap.dedent(code).strip(\\\"\\\\n\\\") + \\\"\\\\n\\\"      # the local model often indents the whole block (IndentationError)\\n    if \\\"def build_goal\\\" in code:\\n        return code\\n    if \\\"def is_goal\\\" in code and \\\"def progress\\\" in code:\\n        classes = [m.group(1) for m in _CLASS_RE.finditer(code)]\\n        if classes:\\n            return code + f\\\"\\\\n\\\\ndef build_goal():\\\\n    return {classes[-1]}()\\\\n\\\"\\n        name = \\\"inferred_goal\\\"\\n        m = re.search(r\\\"^name\\\\s*=\\\\s*['\\\\\\\"]([^'\\\\\\\"]+)['\\\\\\\"]\\\", code, re.M)\\n        if m:\\n            name = m.group(1)\\n        return code + ('\\\\n\\\\nclass _InferredGoal:\\\\n    name = %r\\\\n    def is_goal(self, scene):\\\\n        return bool(is_goal(scene))\\\\n'\\n                       '    def progress(self, scene):\\\\n        return float(progress(scene))\\\\n\\\\ndef build_goal():\\\\n    return _InferredGoal()\\\\n' % name)\\n    return code\\n\\n\\ndef propose_goals(llm, sandbox, ev: WinEvidence, roled: Callable[[Scene], Scene], current: Scene, current_grid: np.ndarray, *, K: int = 2,\\n                  rejected: list = (), feedback: list = (), log=None) -> tuple[list, list[str]]:\\n    \\\"\\\"\\\"Ask the model for K win predicates; return (accepted GoalInstances, verification failure reasons).\\\"\\\"\\\"\\n    log = log or (lambda *a, **k: None)\\n    codes: list = []\\n\\n    def gen(k: int) -> None:\\n        try:\\n            msgs = goal_prompt(ev, roled, current, current_grid, rejected=rejected, feedback=feedback, alternative=k)\\n            reply = llm.chat(msgs, purpose=\\\"goal_template\\\", use_cache=(k == 0))\\n            code = llm.extract_code(reply)\\n            if code:\\n                codes.append(code)\\n            else:\\n                log(f\\\"goal proposal {k}: no code block\\\")\\n        except Exception as e:\\n            log(f\\\"goal proposal {k} failed: {e!r}\\\")\\n    threads = [threading.Thread(target=gen, args=(k,), daemon=True) for k in range(K)]\\n    for t in threads:\\n        t.start()\\n    for t in threads:\\n        t.join(timeout=float(getattr(llm, \\\"cfg\\\", {}).get(\\\"timeout\\\", 300)) + 30)\\n    accepted, reasons = [], []\\n    for code in codes:\\n        code = normalise_goal_code(code)\\n        ns = sandbox.load_namespace(code)                       # sets last_error with the real cause (syntax, static check, ...)\\n        if ns is None:\\n            reasons.append(f\\\"the code did not load: {sandbox.last_error or 'unknown error'} -- reply with one complete, unindented python block\\\"); continue\\n        if \\\"build_goal\\\" not in ns:\\n            reasons.append(\\\"the code defines no build_goal() (and no is_goal/progress to wrap): follow the contract\\\"); continue\\n        goal = sandbox.load_goal(code)\\n        if goal is None:\\n            reasons.append(\\\"build_goal() raised or returned an object without is_goal/progress\\\"); continue\\n        ok, why = verify_goal(goal, roled, ev, current)\\n        if ok:\\n            goal.origin = \\\"llm\\\"; goal.code = code; goal.confidence = 0.9; goal.template = f\\\"win:{goal.name}\\\"\\n            accepted.append(goal)\\n        else:\\n            reasons.append(why)\\n    return accepted, reasons\\n\", \"pbg/hypothesis/guard.py\": \"\\\"\\\"\\\"hypothesis/guard.py \\u2014 a time limit on LLM-written model code (docs/029).\\n\\nAn LLM-written rule with an unbounded loop froze a game thread for 50 minutes (r11l: `slide_movers_toward_click` inside\\nverify()); Python threads cannot be killed, and the sandbox's CPU-limited subprocess only covers the validation run,\\nnot the in-process calls the verifier and the planner make. This module runs every model/goal call under a per-thread\\ntrace function that raises ModelTimeout once the deadline has passed. sys.settrace is per thread, so games running in\\nparallel threads do not affect each other; pure-Python loops trigger 'line' events and are interrupted, C calls (numpy)\\nare bounded anyway.\\n\\n    GuardedModel(model, seconds)     predict / with_roles / assign_roles under the deadline; a timeout is an UNKNOWN\\n                                     prediction (None), counted in .timeouts; after MAX_TIMEOUTS the model is dead\\n                                     and every call returns None at once (no more time is spent on it)\\n    call_with_deadline(fn, seconds)  the same guard for any function (build_model, build_goal, is_goal, ...)\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport sys\\nimport time\\nfrom typing import Any, Callable, Optional\\n\\nMAX_TIMEOUTS = 2\\nCHECK_EVERY = 32          # line events between clock reads (the clock is the expensive part of the tracer)\\n\\n\\nclass ModelTimeout(BaseException):\\n    \\\"\\\"\\\"BaseException on purpose: LLM code and RuleModel.predict catch Exception; the deadline must pass through them.\\\"\\\"\\\"\\n\\n\\ndef call_with_deadline(fn: Callable, *args: Any, seconds: float = 2.0, **kwargs: Any) -> Any:\\n    \\\"\\\"\\\"Call fn(*args, **kwargs); raise ModelTimeout if its Python code runs past `seconds`.\\\"\\\"\\\"\\n    deadline = time.monotonic() + seconds\\n    count = [0]\\n\\n    def local_tracer(frame, event, arg):\\n        if event == \\\"line\\\":\\n            count[0] += 1\\n            if count[0] % CHECK_EVERY == 0 and time.monotonic() > deadline:\\n                raise ModelTimeout(f\\\"model code exceeded {seconds:.1f}s in {frame.f_code.co_name} (line {frame.f_lineno})\\\")\\n        return local_tracer\\n\\n    def global_tracer(frame, event, arg):\\n        if event == \\\"call\\\":\\n            return local_tracer\\n        return None\\n\\n    prev = sys.gettrace()\\n    sys.settrace(global_tracer)\\n    try:\\n        return fn(*args, **kwargs)\\n    finally:\\n        sys.settrace(prev)\\n\\n\\nclass GuardedModel:\\n    \\\"\\\"\\\"Proxy around an LLM-written WorldModel: every call is deadline-limited; timeouts are recorded, never raised.\\\"\\\"\\\"\\n\\n    def __init__(self, model: Any, seconds: float = 2.0):\\n        self._model = model\\n        self.seconds = float(seconds)\\n        self.timeouts = 0\\n        self.last_timeout: str = \\\"\\\"\\n\\n    @property\\n    def dead(self) -> bool:\\n        return self.timeouts >= MAX_TIMEOUTS\\n\\n    def _guarded(self, name: str, *args: Any, default: Any = None) -> Any:\\n        if self.dead:\\n            return default\\n        fn = getattr(self._model, name, None)\\n        if fn is None:\\n            return default\\n        try:\\n            return call_with_deadline(fn, *args, seconds=self.seconds)\\n        except ModelTimeout as e:\\n            self.timeouts += 1\\n            self.last_timeout = str(e)\\n            return default\\n\\n    def predict(self, scene, action):\\n        return self._guarded(\\\"predict\\\", scene, action, default=None)\\n\\n    def with_roles(self, scene):\\n        return self._guarded(\\\"with_roles\\\", scene, default=scene)\\n\\n    def assign_roles(self, scene) -> dict:\\n        return self._guarded(\\\"assign_roles\\\", scene, default={}) or {}\\n\\n    def rules(self) -> list:\\n        try:\\n            return list(self._model.rules())\\n        except Exception:\\n            return []\\n\\n    def __getattr__(self, name: str):      # anything else (level_scoped, verified, name, ...) passes through\\n        return getattr(self._model, name)\\n\", \"pbg/hypothesis/policy.py\": \"\\\"\\\"\\\"hypothesis/policy.py \\u2014 the one policy for every game:\\n\\n    LOOK      the board and what actions did so far\\n    HYPOTHESISE   a document of the whole game (roles, controls, dynamics, win) + the tests that would settle the doubts\\n    TEST      run those 1-3 actions (never a blind walk)\\n    VERIFY    the hypothesis at pixel level on the level's log\\n    REVISE    with counter-examples until it verifies\\n    PLAN      the shortest action sequence to the hypothesised win; execute; a mismatch is a new counter-example\\n\\nThe mechanism priors are hints in the prompt, not a separate process. Everything is logged as events so the trail\\n\\\"hypothesis -> test -> verdict -> revision\\\" is readable in the replay viewer.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport threading\\nimport zlib\\nimport time\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nimport numpy as np\\n\\nfrom ..core.budget import Budget\\nfrom ..core.contracts import RuleModel\\nfrom ..core.events import EventLog\\nfrom ..core.types import Action, Scene, Transition\\nfrom ..memory.priors.mechanisms import mechanism_signatures\\nfrom ..memory.store import Memory\\nfrom ..orchestrator.knowledge import LevelKnowledge\\nfrom ..orchestrator.loop import Result\\nfrom ..orchestrator.session import Session\\nfrom ..perception import Perception\\nfrom ..planner.astar import astar\\nfrom ..planner.common import action_set, state_key\\nfrom ..probe.semantics import classify_actions\\nfrom ..goal.infer import GoalInference\\nfrom ..goal.templates import _adjacent, t_explore\\nfrom ..wml.refine import WorldModelLab\\nfrom ..wml.sandbox import Sandbox\\nfrom .contract import hypothesis_prompt\\nfrom .explore import explore_level\\nfrom .goals import MAX_ROUNDS_PER_LEVEL as GOAL_ROUNDS, WinEvidence, propose_goals, verify_goal\\nfrom .guard import GuardedModel, ModelTimeout, call_with_deadline\\nfrom .verify import Verdict, verify, pixel_equal, _boxes\\n\\n\\nMODEL_SECONDS = 2.0     # deadline per call into LLM-written code (docs/029: an unbounded loop froze a game thread)\\n\\n\\nclass Hypothesis:\\n    def __init__(self, code: str, ns: dict, n: int):\\n        self.code, self.ns, self.n = code, ns, n\\n        self.doc = ns.get(\\\"HYPOTHESIS\\\") or {}\\n        self.model = GuardedModel(call_with_deadline(ns[\\\"build_model\\\"], seconds=MODEL_SECONDS), MODEL_SECONDS)\\n        self.goal = call_with_deadline(ns[\\\"build_goal\\\"], seconds=MODEL_SECONDS) if \\\"build_goal\\\" in ns else None\\n        self.verdict: Optional[Verdict] = None\\n        self.timeouts = 0            # deadline hits in goal / helper functions (model calls are counted by GuardedModel)\\n        self.origin = \\\"llm\\\"\\n\\n    def timeout_note(self) -> str:\\n        \\\"\\\"\\\"Counter-example text when this hypothesis' code hit the deadline (empty otherwise).\\\"\\\"\\\"\\n        n = self.timeouts + getattr(self.model, \\\"timeouts\\\", 0)\\n        if not n:\\n            return \\\"\\\"\\n        where = getattr(self.model, \\\"last_timeout\\\", \\\"\\\") or \\\"a goal/helper function\\\"\\n        return (f\\\"Your code ran past the {MODEL_SECONDS:.0f}-second limit {n} time(s) ({where}): it has an unbounded loop. \\\"\\n                \\\"Every loop must be bounded by the grid size, the object count or a fixed step count; the timed-out calls counted as wrong predictions.\\\")\\n\\n    def roled(self, scene: Scene) -> Scene:\\n        \\\"\\\"\\\"The scene with this hypothesis' roles applied (every hypothesis function expects them).\\\"\\\"\\\"\\n        try:\\n            return self.model.with_roles(scene) if hasattr(self.model, \\\"with_roles\\\") else scene\\n        except Exception:\\n            return scene\\n\\n    def _timed(self, fn, *args, default):\\n        if self.timeouts + getattr(self.model, \\\"timeouts\\\", 0) >= 2:\\n            return default                # dead code: spend no more time on it\\n        try:\\n            out = call_with_deadline(fn, *args, seconds=MODEL_SECONDS)\\n            return out if out is not None else default\\n        except ModelTimeout:\\n            self.timeouts += 1\\n            return default\\n        except Exception:\\n            return default\\n\\n    def call(self, name: str, scene: Scene, default):\\n        fn = self.ns.get(name)\\n        if fn is None:\\n            return default\\n        return self._timed(fn, self.roled(scene), default=default)\\n\\n    def is_goal(self, scene: Scene) -> bool:\\n        if self.goal is None:\\n            return False\\n        return bool(self._timed(self.goal.is_goal, self.roled(scene), default=False))\\n\\n    def progress(self, scene: Scene) -> float:\\n        if self.goal is None:\\n            return 0.0\\n        try:\\n            return float(self._timed(self.goal.progress, self.roled(scene), default=0.0))\\n        except Exception:\\n            return 0.0\\n\\n    def ignore(self, scene: Scene) -> list:\\n        return self.call(\\\"ignore_boxes\\\", scene, [])\\n\\n    def summary(self) -> str:\\n        d = self.doc\\n        v = self.verdict\\n        vs = f\\\"acc={v.accuracy:.2f} chg={v.change_accuracy:.2f}({v.changed}) approx={v.approx:.2f} cov={v.coverage:.2f} n={v.n}\\\" if v else \\\"unverified\\\"\\n        if self.timeouts + getattr(self.model, \\\"timeouts\\\", 0):\\n            vs += f\\\" TIMEOUTS={self.timeouts + getattr(self.model, 'timeouts', 0)}\\\"\\n        return f\\\"h{self.n} [{vs}] {str(d.get('summary', ''))[:220]} | win: {str(d.get('win', ''))[:120]} | uncertain: {d.get('uncertain', [])}\\\"\\n\\n\\nclass InducedHypothesis(Hypothesis):\\n    \\\"\\\"\\\"A prior-composed (deterministic) world model as a hypothesis candidate (docs/029 P1): the induction of wml/\\n    (move / wall / collect / click-permutation / effect-table rules from the action semantics) competes with the LLM's\\n    hypotheses under the same pixel verifier, paired with the goal templates ranked for its roles. This is the model-free\\n    path: on games where the local model writes nothing pixel-exact (lp85), the composed model still plans.\\\"\\\"\\\"\\n\\n    def __init__(self, wh, goals: list, n: int, actions_fn):\\n        self.code, self.ns, self.n = wh.code or \\\"\\\", {}, n\\n        self.name = wh.name\\n        self.doc = {\\\"summary\\\": f\\\"induced model '{wh.name}' composed from the action semantics and the mechanism priors\\\",\\n                    \\\"win\\\": goals[0].name if goals else \\\"(no goal template fits)\\\", \\\"uncertain\\\": []}\\n        self.model = wh.model               # library code composed by the induction: no per-call trace guard (it costs 5-10x in A*)\\n        self.goals = list(goals)\\n        self.goal = self.goals[0] if self.goals else None\\n        self.verdict: Optional[Verdict] = None\\n        self.timeouts = 0\\n        self.origin = \\\"induced\\\"\\n        self._actions_fn = actions_fn\\n\\n    def call(self, name: str, scene: Scene, default):\\n        if name == \\\"candidate_actions\\\":\\n            try:\\n                return self._actions_fn(scene) or default\\n            except Exception:\\n                return default\\n        return default\\n\\n\\nGD_STEPS_PER_LEVEL = 6       # goal-directed steps a verified-but-planless model may take per level before the model is questioned\\nEXPLORE_PLANS_PER_LEVEL = 8  # contact-exploration plans (reach an object not touched yet) per level when no goal plans (ls20: tile, then lock)\\nAPPROX_ACTIONS_PER_LEVEL = 24  # actions an approximate model may spend per level on closed-loop plans (ls20 spent 66 for nothing)\\nLLM_COOLDOWN_ACTIONS = 4     # with a usable model that is merely stuck, at least this many new actions between two LLM rounds\\nPLAN_SECONDS_TOTAL = 24.0    # one plan() call tries every untried goal within this budget (cheaper than one LLM round)\\nPLAN_FULL_TIME_GOALS = 2     # the top-ranked untried goals get the full plan_time each; the rest share what is left\\n\\n\\nclass HypothesisPolicy:\\n    def __init__(self, *, memory: Memory, llm, log=None, events_dir: Optional[Path] = None, max_seconds: Optional[float] = None,\\n                 budget_cfg: Optional[dict] = None, perception_cfg: Optional[dict] = None, K: int = 2, min_acc: float = 0.8,\\n                 max_rounds_per_level: int = 14, plan_time: float = 8.0, explore: int = 0):\\n        self.memory, self.llm = memory, llm\\n        self.log = log or (lambda *a, **k: None)\\n        self.events_dir, self.max_seconds = events_dir, max_seconds\\n        self.budget_cfg, self.perception_cfg = budget_cfg, perception_cfg\\n        self.K, self.min_acc, self.max_rounds, self.plan_time = K, min_acc, max_rounds_per_level, plan_time\\n        self.explore = int(explore)     # level-1 exploration cap before the first hypothesis (0 = off; docs/029)\\n        self.sandbox = Sandbox(log=self.log, allow_getattr=True)\\n        self.load_errors: list[str] = []\\n        # the deterministic induction of wml/ and the goal template library run without an LLM: their models are\\n        # candidates next to the LLM's, verified by the same pixel verifier (docs/029 P1)\\n        # defaults for the helpers; play() creates a fresh pair per game (GoalInference carries per-game demotions)\\n        self.wml = WorldModelLab(llm=None, memory=memory, log=self.log)\\n        self.goal_inf = GoalInference(memory, None, self.wml.sandbox, log=self.log)\\n\\n    # \\u2500\\u2500 hypothesising \\u2500\\u2500\\n    def _generate(self, msgs: list[dict], k: int, out: list) -> None:\\n        try:\\n            kmsgs = msgs if k == 0 else msgs + [{\\\"role\\\": \\\"user\\\", \\\"content\\\": f\\\"Alternative {k}: consider a different reading of the controls or the win condition; still return ONE complete ```python block.\\\"}]\\n            reply = self.llm.chat(kmsgs, purpose=\\\"world_model\\\", use_cache=(k == 0))\\n            code = self.llm.extract_code(reply)\\n            if not code:\\n                self.log(f\\\"hypothesis candidate {k}: no code block\\\"); return\\n            ns = self.sandbox.load_namespace(code)\\n            if ns is None or \\\"build_model\\\" not in ns:\\n                err = self.sandbox.last_error or \\\"build_model missing\\\"\\n                self.log(f\\\"hypothesis candidate {k}: load failed: {err[:160]}\\\")\\n                self.load_errors.append(f\\\"Your previous code failed to load: {err[:300]} -- fix it (numpy only, no imports beyond numpy, define build_model/build_goal/candidate_actions/test_actions/ignore_boxes).\\\")\\n                return\\n            out.append(code)\\n        except Exception as e:\\n            self.log(f\\\"hypothesis candidate {k} failed: {e!r}\\\")\\n\\n    def hypothesise(self, s: Session, log: list[Transition], prev: Optional[Hypothesis], counterexamples: list[str], n: int, rejected: list[str]) -> list[Hypothesis]:\\n        grid = np.asarray(s.frame.grid)\\n        errs = self.load_errors[-2:]; self.load_errors = []\\n        hints = list(counterexamples) + errs\\n        prev_code = prev.code if (prev is not None and prev.origin == \\\"llm\\\") else None\\n        if prev is not None and prev.origin == \\\"induced\\\" and prev.verdict is not None:\\n            hints.append(f\\\"A deterministic model composed from the action semantics ('{prev.name}') already predicts {prev.verdict.accuracy:.0%} of this level's \\\"\\n                         \\\"transitions exactly; your hypothesis must be at least as exact, and must add the win condition and the candidate actions.\\\")\\n        msgs = hypothesis_prompt(scene=s.scene, grid=grid, log=log, available=s.available_actions(), prior_signatures=mechanism_signatures(),\\n                                 previous_code=prev_code, counterexamples=hints, level=s.level,\\n                                 ignore_boxes=prev.ignore(s.scene) if prev else (), rejected=rejected)\\n        codes: list = []\\n        k_count = 1 if (prev is not None and prev.verdict is not None and prev.verdict.change_accuracy >= 0.9) else self.K\\n        threads = [threading.Thread(target=self._generate, args=(msgs, k, codes), daemon=True) for k in range(k_count)]\\n        for t in threads:\\n            t.start()\\n        for t in threads:\\n            t.join(timeout=float(self.llm.cfg.get(\\\"timeout\\\", 300)) + 30)\\n        out = []\\n        for code in codes:\\n            ns = self.sandbox.load_namespace(code)\\n            if ns is None:\\n                continue\\n            try:\\n                out.append(Hypothesis(code, ns, n))\\n            except Exception as e:\\n                self.log(f\\\"hypothesis build failed: {e!r}\\\")\\n                self.load_errors.append(f\\\"Your previous code loaded but build_model()/build_goal() raised {type(e).__name__}: {str(e)[:200]} -- fix it.\\\")\\n        return out\\n\\n    # \\u2500\\u2500 deterministic candidates (docs/029 P1-3) \\u2500\\u2500\\n    def template_goals(self, s: Session, log: list[Transition], model, goal_inf=None) -> list:\\n        \\\"\\\"\\\"Every goal template instantiated for `model`'s roles, ranked (won templates of this game first via LevelKnowledge).\\\"\\\"\\\"\\n        goal_inf = goal_inf or self.goal_inf\\n        roles_fn = getattr(model, \\\"with_roles\\\", None)\\n        try:\\n            return list(goal_inf.refine(log, [], self.memory.priors(\\\"goals\\\"), s.level, s.scene, roles_fn=roles_fn, model=model))\\n        except Exception as e:\\n            self.log(f\\\"goal inference failed: {e!r}\\\")\\n            return []\\n\\n    def induced_candidates(self, s: Session, log: list[Transition], n: int, limit: int = 3, ids: Optional[dict] = None,\\n                           wml=None, goal_inf=None, click_map=None) -> list[InducedHypothesis]:\\n        \\\"\\\"\\\"Prior-composed models induced from this level's log, each paired with every goal template ranked for its roles.\\n        `ids` keeps one number per model name across rounds, so a no-plan verdict against a model sticks to it.\\\"\\\"\\\"\\n        if not log:\\n            return []\\n        ids = ids if ids is not None else {}\\n        wml = wml or self.wml\\n        try:\\n            sem = classify_actions(log)\\n            available = s.available_actions()\\n            H = wml.refine(log, [], self.memory.priors(\\\"mechanisms\\\"), scene=s.scene, semantics=sem, available=available, use_llm=False)\\n        except Exception as e:\\n            self.log(f\\\"induction failed: {e!r}\\\")\\n            return []\\n        out: list[InducedHypothesis] = []\\n        for wh in H[:limit]:\\n            G = self.template_goals(s, log, wh.model, goal_inf)\\n            if getattr(wh.model, \\\"point_game\\\", False):\\n                # a point-select model plans over destinations: steering clicks around each mover plus the usual object clicks\\n                from ..wml.point import point_click_candidates\\n                actions_fn = (lambda scene, sem=sem, av=available, m=wh.model, cm=click_map: point_click_candidates(m.with_roles(scene), getattr(m, \\\"point_step\\\", 1))\\n                              + [a for a in action_set(scene, av, semantics=sem, click_map=cm) if a.type == \\\"CLICK\\\"][:16])\\n            else:\\n                # with a click map that has evidence, only classes that reacted (or were never tried) are candidates: 4-5 buttons\\n                # instead of 40 object centres (vc33 level 2 never finished a plan over 40 clicks)\\n                actions_fn = (lambda scene, sem=sem, av=available, cm=click_map: action_set(scene, av, semantics=sem, click_map=cm))\\n            out.append(InducedHypothesis(wh, G, ids.setdefault(wh.name, 1000 + len(ids)), actions_fn))\\n        return out\\n\\n    def equal_alternatives(self, hyp, s: Session, log: list, ids: dict, wml, goal_inf, click_map, min_acc: float, tol: float = 0.05, limit: int = 3) -> list:\\n        \\\"\\\"\\\"Induced models the log cannot tell apart from `hyp` (usable, change accuracy within `tol`), other than hyp's own\\n        variant. Where the top model has no plan because of an untested rule (ls20: a floor-colour rule forbade stepping onto\\n        the tile), an equally verified variant without that rule may plan; executing its plan tests the difference.\\\"\\\"\\\"\\n        out = []\\n        ref = hyp.verdict.change_accuracy if (hyp is not None and hyp.verdict is not None) else 0.0\\n        for c in self.induced_candidates(s, log, ids.get(\\\"_n\\\", 0), ids=ids, wml=wml, goal_inf=goal_inf, click_map=click_map, limit=6):\\n            if hyp is not None and getattr(c, \\\"name\\\", None) == getattr(hyp, \\\"name\\\", None):\\n                continue\\n            c.verdict = verify(c.model, log, c.ignore) if log else Verdict(0.0, 0.0, 0, 0, 0)\\n            if c.verdict.usable(min_acc) and c.verdict.change_accuracy >= ref - tol:\\n                out.append(c)\\n        # simplest first (fewest rules): among models the log cannot tell apart, the one without the extra rule is the one\\n        # whose untested assumption is not blocking the plan (ls20: move+collect before move+walls+floor+collect+push)\\n        out.sort(key=lambda c: (len(c.model.rules()) if hasattr(c.model, \\\"rules\\\") else 99, -c.verdict.change_accuracy))\\n        return out[:limit]\\n\\n    def goal_directed_step(self, h: Hypothesis, s: Session, observed: dict, tried_here: set, seen_states: set, click_map=None) -> Optional[Action]:\\n        \\\"\\\"\\\"When a usable model yields no plan (docs/029 P1-4): the candidate action whose predicted outcome raises the goal's\\n        progress the most, or leads to a board not seen yet; unknown outcomes count as mildly informative.\\\"\\\"\\\"\\n        avail = s.available_actions()\\n        cands = [a for a in h.call(\\\"candidate_actions\\\", s.scene, []) if isinstance(a, Action) and _available(a, avail)]\\n        for a in action_set(s.scene, avail, click_map=click_map):\\n            if a not in cands:\\n                cands.append(a)\\n        sk = state_key(s.scene); cur = h.progress(s.scene)\\n        best: Optional[Action] = None; best_score = 0.0\\n        for a in cands[:48]:\\n            if (sk, a.label()) in tried_here:\\n                continue\\n            nxt = observed.get((sk, a.label()))\\n            if nxt is None:\\n                nxt = h.model.predict(s.scene, a)\\n            if nxt is None:\\n                score = 0.15\\n            else:\\n                nk = state_key(nxt)\\n                if nk == sk:\\n                    continue                      # predicted no-op\\n                score = (h.progress(nxt) - cur) + (0.3 if nk not in seen_states else 0.0)\\n            if score > best_score:\\n                best, best_score = a, score\\n        return best\\n\\n    def plan_to(self, h: Hypothesis, s: Session, observed: dict, goal, depth: int = 80, time_limit: Optional[float] = None) -> Optional[list[Action]]:\\n        \\\"\\\"\\\"A* under h's model to one given goal object (is_goal/progress on the roled scene).\\\"\\\"\\\"\\n        available = s.available_actions()\\n        def actions_fn(scene: Scene) -> list[Action]:\\n            cands = [a for a in h.call(\\\"candidate_actions\\\", scene, []) if isinstance(a, Action) and _available(a, available)]\\n            return (cands or [a for a in available if a.type == \\\"BUTTON\\\"])[:48]\\n        def successors(scene: Scene):\\n            sk = state_key(scene)\\n            for a in actions_fn(scene):\\n                nxt = observed.get((sk, a.label()))\\n                if nxt is None:\\n                    try:\\n                        nxt = h.model.predict(scene, a)\\n                    except Exception:\\n                        nxt = None\\n                if nxt is not None:\\n                    yield a, nxt\\n        try:\\n            if goal.is_goal(h.roled(s.scene)):\\n                return []\\n            return astar(s.scene, successors, lambda sc: goal.is_goal(h.roled(sc)), lambda sc: goal.progress(h.roled(sc)), depth=depth, time_limit=time_limit or self.plan_time)\\n        except Exception as e:\\n            self.log(f\\\"planning failed: {e!r}\\\")\\n            return None\\n\\n    # \\u2500\\u2500 planning on a verified hypothesis \\u2500\\u2500\\n    def plan(self, h: Hypothesis, s: Session, observed: dict, log: list = (), tried_goals: Optional[set] = None, dead_goals: Optional[set] = None,\\n             goal_inf=None, level_goals: list = (), goals_version: int = 0) -> Optional[list[Action]]:\\n        \\\"\\\"\\\"Shortest path under h's model to a goal. Tries every goal not yet tried from THIS board (an LLM hypothesis' own goal\\n        first, then the goal templates for its roles; an induced one its templates), within PLAN_SECONDS_TOTAL; the goal that\\n        yields a plan becomes h.goal. Goals whose predicate held without a level-up (`dead_goals`) are skipped.\\\"\\\"\\\"\\n        tried_goals = tried_goals if tried_goals is not None else set()\\n        dead_goals = dead_goals if dead_goals is not None else set()\\n        if getattr(h, \\\"goals\\\", None) is None or getattr(h, \\\"_goals_level\\\", None) != s.level or getattr(h, \\\"_goals_version\\\", None) != goals_version:\\n            own = [h.goal] if h.goal is not None else []\\n            first = list(level_goals) + [o for o in own if getattr(o, \\\"name\\\", None) not in {getattr(g, \\\"name\\\", None) for g in level_goals}]\\n            names = {getattr(g, \\\"name\\\", None) for g in first}\\n            h.goals = first + [g for g in self.template_goals(s, log, h.model, goal_inf) if getattr(g, \\\"name\\\", None) not in names]\\n            h._goals_level = s.level; h._goals_version = goals_version\\n        sk = state_key(s.scene)\\n        goals = [g for g in h.goals if g is not None and getattr(g, \\\"name\\\", \\\"?\\\") not in dead_goals and (h.n, getattr(g, \\\"name\\\", \\\"?\\\"), sk) not in tried_goals]\\n        if not goals:\\n            return None\\n        available = s.available_actions()\\n        def actions_fn(scene: Scene) -> list[Action]:\\n            cands = h.call(\\\"candidate_actions\\\", scene, [])\\n            cands = [a for a in cands if isinstance(a, Action) and _available(a, available)]\\n            if not cands:\\n                cands = [a for a in available if a.type == \\\"BUTTON\\\"]\\n            return cands[:48]\\n        def successors(scene: Scene):\\n            sk = state_key(scene)\\n            for a in actions_fn(scene):\\n                nxt = observed.get((sk, a.label()))\\n                if nxt is None:\\n                    try:\\n                        nxt = h.model.predict(scene, a)\\n                    except Exception:\\n                        nxt = None\\n                if nxt is None:\\n                    continue\\n                yield a, nxt\\n        t_end = time.time() + PLAN_SECONDS_TOTAL\\n        for i, g in enumerate(goals):\\n            left = t_end - time.time()\\n            if left <= 0:\\n                break\\n            per_goal = min(self.plan_time, left) if i < PLAN_FULL_TIME_GOALS else max(1.5, min(3.0, left / max(1, len(goals) - i)))\\n            h.goal = g\\n            tried_goals.add((h.n, getattr(g, \\\"name\\\", \\\"?\\\"), sk))\\n            try:\\n                if h.is_goal(s.scene):\\n                    return []\\n                plan = astar(s.scene, successors, h.is_goal, h.progress, depth=80, time_limit=per_goal)\\n            except Exception as e:\\n                self.log(f\\\"planning failed: {e!r}\\\"); plan = None\\n            if plan:\\n                return plan\\n        h.goal = h.goals[0] if h.goals else None\\n        return None\\n\\n    # \\u2500\\u2500 the loop \\u2500\\u2500\\n    def play(self, env, game_id: str, *, budget_total: Optional[int] = None) -> Result:\\n        t_start = time.time()\\n        budget = Budget(budget_total, self.budget_cfg)\\n        perception = Perception(self.perception_cfg)\\n        events = EventLog((self.events_dir / f\\\"{game_id}.events.jsonl\\\") if self.events_dir else None)\\n        s = Session(env, game_id, perception, self.memory, budget, log=self.log, max_seconds=self.max_seconds)\\n        s.start()\\n        wml = WorldModelLab(llm=None, memory=self.memory, log=self.log)\\n        goal_inf = GoalInference(self.memory, None, wml.sandbox, log=self.log)\\n        knowledge = LevelKnowledge.load(self.memory.knowledge_path(game_id))     # goal templates that won earlier levels rank first\\n        goal_inf.knowledge = knowledge; goal_inf.demoted.update(knowledge.demoted)\\n        hyp: Optional[Hypothesis] = None\\n        counterexamples: list[str] = []; rejected: list[str] = []\\n        observed: dict = {}\\n        level = s.level; rounds = 0; n_hyp = 0; stop = \\\"\\\"; llm_calls0 = self.llm.calls\\n        plans_executed = 0; tests_run = 0; tried_labels: set = set(); idle_rounds = 0; approx_budget = 6\\n        tried_here: set = set()             # (board state, action label) already run as a test: never repeat it from the same board\\n        gd_steps = 0                        # goal-directed steps taken on this level (P1-4)\\n        induced_ids: dict = {}              # induced model name -> stable hypothesis number\\n        llm_best: Optional[Hypothesis] = None   # best LLM candidate of the last round: its tests run when an induced model tops the ranking\\n        approx_actions = 0                  # actions spent under approximate plans on this level (P2-6)\\n        no_plan_n: Optional[int] = None     # hypothesis that verified but produced no plan (keep it only while nothing verifies better)\\n        repeats_dropped = 0\\n        tried_goals: set = set(); dead_goals: set = set()   # (hyp, goal, board) planned already / goals that held without a level-up\\n        last_goal = None                    # goal of the last plan / goal-directed action (a level-up credits it)\\n        last_llm_used = -10 ** 6            # budget.used() at the last LLM round (cooldown while a usable model is merely stuck)\\n        llm_rounds_skipped = 0\\n        win_evidence: dict = {}             # level -> WinEvidence (the boards right before / after that level ended)\\n        level_goals: list = []              # win predicates that passed verification on the evidence: planned first, carried to the next level\\n        goals_version = 0                   # bumped when level_goals change so every hypothesis rebuilds its goal list\\n        goal_rounds = 0; goal_feedback: list[str] = []; goal_rejected: list[str] = []\\n        touched: set = set(); explore_plans = 0       # contact exploration: object identities the agent has overlapped / touched on this level\\n        unreachable: set = set()                      # targets no plan reached on this level (skipped until the level changes)\\n        failed_modes: set = set()                     # (target identity, mode) whose plan mispredicted: overlap -> adjacent -> unreachable\\n        rerank = False                                # a plan under an alternative mispredicted: re-rank the induced variants next round\\n\\n        def act(a: Action, kind: str) -> Optional[Transition]:\\n            t = s.act(a, kind)\\n            if t is not None and t.status_change is None and t.action.type != \\\"RESET\\\":\\n                observed[(state_key(t.before), a.label())] = t.after\\n                mark_touched(t.after)\\n            return t\\n\\n        def mark_touched(scene: Scene) -> None:\\n            if hyp is None:\\n                return\\n            sc = hyp.roled(scene)\\n            for ag in [o for o in sc.objects if o.role and \\\"agent\\\" in o.role]:\\n                for o in sc.objects:\\n                    if o.id != ag.id and (ag.overlaps(o) or _adjacent(ag, o)):\\n                        touched.add(o.identity())\\n\\n        def run_plan(plan: list, exact: bool, why: str) -> str:\\n            \\\"\\\"\\\"Execute a plan step by step with the pixel check; returns 'levelup' | 'mismatch' | 'refused' | 'done'.\\\"\\\"\\\"\\n            nonlocal counterexamples, approx_actions\\n            for a in (plan if exact else plan[:2]):\\n                if not exact:\\n                    approx_actions += 1\\n                    if approx_actions == APPROX_ACTIONS_PER_LEVEL:\\n                        events.emit(\\\"EXECUTE\\\", \\\"REVISE\\\", f\\\"approximate model spent {APPROX_ACTIONS_PER_LEVEL} actions on this level without a level-up: exact verification required from now on\\\", budget_used=budget.used())\\n                        counterexamples = (counterexamples + [f\\\"Acting on your approximate model for {APPROX_ACTIONS_PER_LEVEL} actions did not end the level: the dynamics or the win condition are wrong in a way the closeness score hides.\\\"])[-4:]\\n                pred = None\\n                try:\\n                    pred = hyp.model.predict(s.scene, a)\\n                except Exception:\\n                    pred = None\\n                t = act(a, \\\"plan\\\")\\n                if t is None:\\n                    return \\\"refused\\\"\\n                if t.status_change:\\n                    events.emit(\\\"EXECUTE\\\", \\\"LOOK\\\", f\\\"{t.status_change} after {a.label()} ({why})\\\", transition_id=t.id, budget_used=budget.used()); return \\\"levelup\\\"\\n                if pred is not None:\\n                    ok, n, win = pixel_equal(pred, t.after_frame.grid, _boxes(t.before, hyp.ignore))\\n                    if not ok:\\n                        ce = verify(hyp.model, [t], hyp.ignore).counterexamples\\n                        counterexamples = (counterexamples + ce)[-4:]\\n                        events.emit(\\\"EXECUTE\\\", \\\"REVISE\\\", f\\\"mismatch after {a.label()} ({n} px, {why}): counter-example recorded\\\", transition_id=t.id, budget_used=budget.used())\\n                        return \\\"mismatch\\\"\\n            return \\\"done\\\"\\n\\n        events.emit(\\\"START\\\", \\\"LOOK\\\", \\\"hypothesis policy: look, hypothesise, test, verify, revise, plan\\\", budget_used=0)\\n        explored = None\\n        if self.explore > 0 and s.level == 1:\\n            # act first on level 1 (the cheapest level to spend actions on), hypothesise from what happened\\n            explored = explore_level(s, act, cap=self.explore, seed=zlib.crc32(game_id.encode()),\\n                                     emit=lambda m: events.emit(\\\"EXPLORE\\\", \\\"LOOK\\\", m, budget_used=budget.used()))\\n        while not s.finished():\\n            if s.timed_out():\\n                stop = \\\"timeout\\\"; break\\n            if s.env.status().state == \\\"GAME_OVER\\\":\\n                events.emit(\\\"LOOK\\\", \\\"LOOK\\\", \\\"game over -> RESET (hypothesis kept)\\\", budget_used=budget.used())\\n                s.act(Action.reset(), \\\"reset\\\"); continue\\n            if s.level != level:\\n                prev_log = s.level_log(level)\\n                if s.level > level and prev_log and prev_log[-1].before_frame is not None and prev_log[-1].after_frame is not None:\\n                    t_win = prev_log[-1]; n = len(prev_log)\\n                    earlier = [t.before for t in prev_log[:-1]][::max(1, (n - 1) // 4)][:4]\\n                    win_evidence[level] = WinEvidence(level, t_win.action.label(), t_win.before, t_win.after, np.asarray(t_win.before_frame.grid), np.asarray(t_win.after_frame.grid), earlier)\\n                    if level_goals and hyp is not None:\\n                        kept = []\\n                        for g in level_goals:\\n                            ok, why = verify_goal(g, hyp.roled, win_evidence[level], None)\\n                            events.emit(\\\"LOOK\\\", \\\"LOOK\\\", f\\\"carried win predicate '{g.name}' {'confirmed by' if ok else 'dropped:'} level {level}'s win\\\" + (\\\"\\\" if ok else f\\\" ({why[:80]})\\\"), budget_used=budget.used())\\n                            if ok:\\n                                g.confidence = 1.0; kept.append(g)\\n                        level_goals = kept; goals_version += 1\\n                won_name = _record_level_win(knowledge, hyp, prev_log, level, s.level_actions.get(level, 0), last_goal) if s.level > level else None\\n                knowledge.levels_seen.add(s.level); knowledge.demoted = dict(goal_inf.demoted); knowledge.save(self.memory.knowledge_path(game_id))\\n                events.emit(\\\"LOOK\\\", \\\"LOOK\\\", f\\\"level {level} -> {s.level}: the hypothesis must verify again on the new board\\\" + (f\\\"; win credited to goal '{won_name}'\\\" if won_name else \\\"\\\"), budget_used=budget.used())\\n                level = s.level; rounds = 0; idle_rounds = 0; approx_budget = 6; counterexamples = []; observed.clear(); tried_here.clear(); no_plan_n = None; gd_steps = 0; approx_actions = 0\\n                tried_goals.clear(); dead_goals.clear(); last_goal = None; goal_rounds = 0; goal_feedback = []; goal_rejected = []\\n                touched.clear(); explore_plans = 0; unreachable.clear(); failed_modes.clear(); rerank = False\\n                if hyp is not None:\\n                    hyp.verdict = None; hyp.goals = None\\n            log = s.level_log()\\n            knowledge.clicks.extend(log[-8:])       # click response map: which classes react (candidates for plans / goal-directed steps)\\n            # verify the current hypothesis on this level's log\\n            if hyp is not None and log:\\n                hyp.verdict = verify(hyp.model, log, hyp.ignore)\\n            if rerank and hyp is not None and log:\\n                # the alternative's plan mispredicted: the log now tells the variants apart, let the best one lead again\\n                rerank = False\\n                best_alt = None\\n                for c in self.induced_candidates(s, log, n_hyp + 1, ids=induced_ids, wml=wml, goal_inf=goal_inf, click_map=knowledge.clicks, limit=6):\\n                    c.verdict = verify(c.model, log, c.ignore)\\n                    if c.verdict.usable(self.min_acc) and (best_alt is None or c.verdict.rank_key() > best_alt.verdict.rank_key()):\\n                        best_alt = c\\n                if best_alt is not None and best_alt.name != getattr(hyp, \\\"name\\\", None) and best_alt.verdict.rank_key() > hyp.verdict.rank_key():\\n                    events.emit(\\\"VERIFY\\\", \\\"PLAN\\\", f\\\"after the mismatch h{best_alt.n} ('{best_alt.name}') verifies better ({best_alt.verdict.change_accuracy:.2f}/{best_alt.verdict.accuracy:.2f}) than h{hyp.n}: it leads again\\\", budget_used=budget.used())\\n                    best_alt.goals = None; hyp = best_alt\\n            exact = hyp is not None and hyp.verdict is not None and hyp.verdict.usable(self.min_acc)\\n            approx = (not exact) and hyp is not None and hyp.verdict is not None and hyp.verdict.approximate() and approx_budget > 0 and approx_actions < APPROX_ACTIONS_PER_LEVEL\\n            usable = exact or approx\\n            if usable:\\n                if approx:\\n                    approx_budget -= 1\\n                plan = self.plan(hyp, s, observed, log, tried_goals, dead_goals, goal_inf, level_goals, goals_version)\\n                if plan is None or (plan == [] and not hyp.is_goal(s.scene)):\\n                    seen_states = {state_key(t.before) for t in log} | {state_key(t.after) for t in log}\\n                    a = self.goal_directed_step(hyp, s, observed, tried_here, seen_states, knowledge.clicks) if gd_steps < GD_STEPS_PER_LEVEL else None\\n                    if a is not None:\\n                        # act on the model instead of asking the LLM to rethink: one step that raises progress / reaches a new board\\n                        gd_steps += 1; last_goal = hyp.goal\\n                        pred = hyp.model.predict(s.scene, a)\\n                        t = act(a, \\\"plan\\\")\\n                        if t is None:\\n                            # refused by the environment (or the clock ran out): never leave the game loop here; the action is not retried\\n                            tried_here.add((state_key(s.scene), a.label()))\\n                            events.emit(\\\"PLAN\\\", \\\"PLAN\\\", f\\\"goal-directed step {a.label()} refused by the environment\\\", budget_used=budget.used())\\n                            continue\\n                        tried_here.add((state_key(t.before), a.label()))\\n                        events.emit(\\\"PLAN\\\", \\\"EXECUTE\\\", f\\\"no plan under h{hyp.n} -> goal-directed step {a.label()} ({gd_steps}/{GD_STEPS_PER_LEVEL})\\\", transition_id=t.id, budget_used=budget.used())\\n                        if t.status_change:\\n                            events.emit(\\\"EXECUTE\\\", \\\"LOOK\\\", f\\\"{t.status_change} after {a.label()}\\\", transition_id=t.id, budget_used=budget.used())\\n                        elif pred is not None:\\n                            ok, n, win = pixel_equal(pred, t.after_frame.grid, _boxes(t.before, hyp.ignore))\\n                            if not ok:\\n                                counterexamples = (counterexamples + verify(hyp.model, [t], hyp.ignore).counterexamples)[-4:]\\n                                events.emit(\\\"EXECUTE\\\", \\\"REVISE\\\", f\\\"mismatch after goal-directed {a.label()} ({n} px): counter-example recorded\\\", transition_id=t.id, budget_used=budget.used())\\n                        continue\\n                    # an equally verified variant may plan where hyp cannot (its extra rule is an untested assumption)\\n                    switched = False\\n                    for alt in self.equal_alternatives(hyp, s, log, induced_ids, wml, goal_inf, knowledge.clicks, self.min_acc):\\n                        aplan = self.plan(alt, s, observed, log, tried_goals, dead_goals, goal_inf, level_goals, goals_version)\\n                        if aplan:\\n                            events.emit(\\\"PLAN\\\", \\\"EXECUTE\\\", f\\\"h{hyp.n} has no plan but the equally verified h{alt.n} ('{alt.name}') plans {len(aplan)} steps to '{getattr(alt.goal, 'name', '?')}': executing it tests the rule they differ on\\\", budget_used=budget.used())\\n                            hyp = alt; plans_executed += 1; last_goal = alt.goal\\n                            if run_plan(aplan, True, f\\\"goal '{getattr(alt.goal, 'name', '?')}' under alternative\\\") == \\\"mismatch\\\":\\n                                rerank = True\\n                            switched = True; break\\n                    if switched:\\n                        continue\\n                    ev = win_evidence.get(level - 1)\\n                    if ev is not None and goal_rounds < GOAL_ROUNDS and not self.llm.exhausted():\\n                        # an exact model and no goal that plans: the win condition is not in the library. Infer it from the\\n                        # moment the previous level ended and verify it on that evidence before planning with it.\\n                        goal_rounds += 1; t0 = time.time()\\n                        accepted, reasons = propose_goals(self.llm, self.sandbox, ev, hyp.roled, s.scene, np.asarray(s.frame.grid), K=2,\\n                                                          rejected=goal_rejected, feedback=goal_feedback, log=self.log)\\n                        goal_feedback = (goal_feedback + reasons)[-4:]\\n                        for why in reasons:\\n                            events.emit(\\\"GOAL\\\", \\\"GOAL\\\", f\\\"win predicate rejected: {why[:160]}\\\", budget_used=budget.used())\\n                        new = [g for g in accepted if g.name not in {x.name for x in level_goals}]\\n                        if new:\\n                            level_goals = new + level_goals; goals_version += 1\\n                            for g in new:\\n                                events.emit(\\\"GOAL\\\", \\\"PLAN\\\", f\\\"win predicate '{g.name}' verified on level {ev.level}'s win (round {goal_rounds}, {time.time() - t0:.0f}s) -> planned first\\\", budget_used=budget.used())\\n                                if self.events_dir:\\n                                    hd = self.events_dir / f\\\"{game_id}.hypotheses\\\"; hd.mkdir(exist_ok=True)\\n                                    (hd / f\\\"goal_L{level}_r{goal_rounds}_{g.name[:24].replace('/', '_')}.py\\\").write_text(g.code)\\n                            continue\\n                        events.emit(\\\"GOAL\\\", \\\"PLAN\\\", f\\\"no win predicate passed verification (round {goal_rounds}/{GOAL_ROUNDS}, {time.time() - t0:.0f}s)\\\", budget_used=budget.used())\\n                    if explore_plans < EXPLORE_PLANS_PER_LEVEL:\\n                        # contact exploration under the model: reach an object the agent has not touched yet (nearest first). What a\\n                        # touch does (pick up, unlock, nothing) is evidence the model and the goal lack; a sequence win (tile, then\\n                        # lock) falls out of it without a sequence template.\\n                        mark_touched(s.scene)\\n                        eplan = None; egoal = None; emodel = None\\n                        models = [hyp] + self.equal_alternatives(hyp, s, log, induced_ids, wml, goal_inf, knowledge.clicks, self.min_acc)\\n                        for m_ in models:\\n                            for target in contact_targets(m_.roled(s.scene), touched, unreachable)[:4]:\\n                                for mode, tl in ((\\\"overlap\\\", 6.0), (\\\"adjacent\\\", 3.0)):   # walk ONTO it if the model allows (a pick-up needs overlap), else next to it\\n                                    if (target.identity(), mode) in failed_modes:\\n                                        continue                                          # that plan mispredicted before: the model cannot do it\\n                                    g = ContactGoal(target, mode)\\n                                    pl = self.plan_to(m_, s, observed, g, time_limit=tl)\\n                                    if pl:\\n                                        eplan, egoal, emodel = pl, g, m_; break\\n                                if eplan:\\n                                    break\\n                            if eplan:\\n                                break\\n                        if eplan:\\n                            explore_plans += 1\\n                            if emodel is not hyp:\\n                                events.emit(\\\"PLAN\\\", \\\"PLAN\\\", f\\\"contact exploration: h{hyp.n} reaches nothing, the equally verified h{emodel.n} ('{emodel.name}') does\\\", budget_used=budget.used())\\n                                hyp = emodel\\n                            events.emit(\\\"PLAN\\\", \\\"EXECUTE\\\", f\\\"no goal plans -> contact exploration {explore_plans}/{EXPLORE_PLANS_PER_LEVEL}: {len(eplan)} steps, {egoal.name}\\\", budget_used=budget.used())\\n                            r = run_plan(eplan, True, egoal.name)\\n                            mark_touched(s.scene)\\n                            if r == \\\"mismatch\\\":\\n                                failed_modes.add((egoal.identity, egoal.mode)); rerank = True\\n                                if egoal.mode == \\\"adjacent\\\" or (egoal.identity, \\\"adjacent\\\") in failed_modes:\\n                                    unreachable.add(egoal.identity)\\n                                events.emit(\\\"PLAN\\\", \\\"PLAN\\\", f\\\"contact plan mispredicted: {egoal.name} -> {'target dropped' if egoal.identity in unreachable else 'try standing next to it instead'}\\\", budget_used=budget.used())\\n                            continue\\n                        for m_ in models:\\n                            for target in contact_targets(m_.roled(s.scene), touched, unreachable)[:4]:\\n                                unreachable.add(target.identity())\\n                        events.emit(\\\"PLAN\\\", \\\"PLAN\\\", f\\\"contact exploration: no untouched object is reachable under the model ({len(unreachable)} targets tried)\\\", budget_used=budget.used())\\n                        explore_plans = EXPLORE_PLANS_PER_LEVEL\\n                    events.emit(\\\"PLAN\\\", \\\"TEST\\\", f\\\"no plan under h{hyp.n} for goal '{getattr(hyp.goal, 'name', '?')}' -> test the doubts\\\", budget_used=budget.used())\\n                    counterexamples = (counterexamples + [f\\\"Your model verified but the planner found NO action sequence from the current board to your goal '{getattr(hyp.goal, 'name', '?')}' using candidate_actions: the goal, the candidate actions or the dynamics are incomplete.\\\"])[-4:]\\n                    hyp.verdict = None; no_plan_n = hyp.n\\n                    usable = False\\n                elif plan == []:\\n                    gname = getattr(hyp.goal, \\\"name\\\", \\\"?\\\")\\n                    events.emit(\\\"PLAN\\\", \\\"PLAN\\\", f\\\"goal '{gname}' already true but no level-up: not the win condition -> demoted\\\", budget_used=budget.used())\\n                    _demote_goal(goal_inf, knowledge, hyp.goal); dead_goals.add(gname)\\n                    if any(g.name == gname for g in level_goals):\\n                        level_goals = [g for g in level_goals if g.name != gname]; goals_version += 1; goal_rejected.append(gname)\\n                    if hyp.origin == \\\"llm\\\" and hyp.goal is not None and not hasattr(hyp.goal, \\\"template\\\"):\\n                        counterexamples.append(\\\"The goal predicate you wrote is already TRUE on the current board, yet the level did not end: the win condition is different.\\\")\\n                        rejected.append(f\\\"win='{str(hyp.doc.get('win', ''))[:80]}'\\\")\\n                    continue                      # other goals remain for this model: plan again before asking the LLM\\n                else:\\n                    mode = \\\"exact\\\" if exact else f\\\"approximate (approx {hyp.verdict.approx:.2f}): first 2 steps, then re-think\\\"\\n                    events.emit(\\\"PLAN\\\", \\\"EXECUTE\\\", f\\\"plan of {len(plan)} under h{hyp.n} [{mode}]: {[a.label() for a in plan[:10]]}\\\", budget_used=budget.used())\\n                    plans_executed += 1; last_goal = hyp.goal\\n                    run_plan(plan, exact, f\\\"goal '{getattr(hyp.goal, 'name', '?')}'\\\")\\n                    continue\\n            # not usable: hypothesise (or revise), then run its tests\\n            if idle_rounds >= self.max_rounds:\\n                # only rounds that produced NO action count toward giving up: as long as tests or plans act, keep going until the clock ends\\n                stop = \\\"UNRESOLVED\\\"; events.emit(\\\"LOOK\\\", \\\"END\\\", f\\\"{idle_rounds} hypothesis rounds without any action\\\", budget_used=budget.used()); break\\n            if self.llm.exhausted():\\n                stop = \\\"llm_exhausted\\\"; events.emit(\\\"LOOK\\\", \\\"END\\\", \\\"LLM call cap reached\\\", budget_used=budget.used()); break\\n            used_before = budget.used()\\n            # the current hypothesis' accuracy even when its verdict was cleared (no plan): a worse candidate must not replace it\\n            cur_key = verify(hyp.model, log, hyp.ignore).rank_key() if (hyp is not None and log) else (-1.0,)\\n            # 1. the deterministic path first (docs/029 priority 1): a usable prior-composed model plans without any LLM round\\n            induced = self.induced_candidates(s, log, n_hyp + 1, ids=induced_ids, wml=wml, goal_inf=goal_inf, click_map=knowledge.clicks)\\n            for c in induced:\\n                c.verdict = verify(c.model, log, c.ignore) if log else Verdict(0.0, 0.0, 0, 0, 0)\\n            best_ind = max(induced, key=lambda c: c.verdict.rank_key(), default=None)\\n            if best_ind is not None and best_ind.verdict.usable(self.min_acc) and best_ind.n != no_plan_n and (hyp is None or best_ind.verdict.rank_key() >= cur_key):\\n                hyp = best_ind; llm_rounds_skipped += 1\\n                events.emit(\\\"HYPOTHESISE\\\", \\\"PLAN\\\", f\\\"deterministic model h{hyp.n} usable (chg {hyp.verdict.change_accuracy:.2f}/acc {hyp.verdict.accuracy:.2f}) -> plan, no LLM round\\\", budget_used=budget.used())\\n                idle_rounds = idle_rounds + 1 if budget.used() == used_before else 0\\n                continue\\n            # 2. a usable model that is merely stuck (no plan, goal-directed steps spent): act cheaply before the next LLM round\\n            stuck = hyp is not None and no_plan_n == hyp.n\\n            if stuck and budget.used() - last_llm_used < LLM_COOLDOWN_ACTIONS:\\n                probe = _fallback_probe(s, tried_labels)\\n                if probe:\\n                    t = act(probe[0], \\\"test\\\")\\n                    if t is not None:\\n                        tests_run += 1; tried_labels.add(probe[0].label()); tried_here.add((state_key(t.before), probe[0].label())); last_goal = None; llm_rounds_skipped += 1\\n                        events.emit(\\\"TEST\\\", \\\"TEST\\\", f\\\"LLM cooldown ({budget.used() - last_llm_used}/{LLM_COOLDOWN_ACTIONS} actions since the last round) -> probe {probe[0].label()}\\\", transition_id=t.id, budget_used=budget.used())\\n                        if t.status_change:\\n                            events.emit(\\\"TEST\\\", \\\"LOOK\\\", f\\\"{t.status_change} after probe {probe[0].label()}\\\", transition_id=t.id, budget_used=budget.used())\\n                        idle_rounds = 0\\n                        continue\\n            rounds += 1; n_hyp += 1\\n            t0 = time.time()\\n            cands = self.hypothesise(s, log, hyp, counterexamples, n_hyp, rejected)\\n            last_llm_used = budget.used()\\n            cands += induced\\n            for c in cands:\\n                c.verdict = verify(c.model, log, c.ignore) if log else Verdict(0.0, 0.0, 0, 0, 0)\\n                if c.timeout_note():\\n                    c.verdict.counterexamples = [c.timeout_note()] + c.verdict.counterexamples\\n                    events.emit(\\\"VERIFY\\\", \\\"VERIFY\\\", f\\\"h{c.n} hit the {MODEL_SECONDS:.0f}s deadline: {getattr(c.model, 'last_timeout', '')[:120]}\\\", budget_used=budget.used())\\n            cands.sort(key=lambda c: c.verdict.rank_key(), reverse=True)\\n            llm_best = next((c for c in cands if c.origin == \\\"llm\\\"), llm_best)\\n            events.emit(\\\"HYPOTHESISE\\\", \\\"HYPOTHESISE\\\", f\\\"round {rounds}: {len(cands)} candidate(s) in {time.time() - t0:.0f}s\\\", budget_used=budget.used())\\n            for c in cands:\\n                events.emit(\\\"HYPOTHESISE\\\", \\\"HYPOTHESISE\\\", c.summary(), budget_used=budget.used())\\n                if self.events_dir:\\n                    hd = self.events_dir / f\\\"{game_id}.hypotheses\\\"; hd.mkdir(exist_ok=True)\\n                    (hd / f\\\"h{c.n}{'i' if c.origin == 'induced' else ''}_r{rounds}_acc{c.verdict.accuracy:.2f}.py\\\").write_text(c.code + \\\"\\\\n\\\\n# verdict: \\\" + c.summary()\\n                                                                                + \\\"\\\\n\\\" + \\\"\\\\n\\\".join(\\\"# \\\" + l for ce in c.verdict.counterexamples[:2] for l in ce.splitlines()))\\n            if not cands:\\n                continue\\n            best = cands[0]\\n            if hyp is None or best.verdict.rank_key() >= cur_key or (no_plan_n == hyp.n and best.verdict.usable(self.min_acc) and best.n != no_plan_n):\\n                hyp = best\\n            else:\\n                events.emit(\\\"HYPOTHESISE\\\", \\\"HYPOTHESISE\\\", f\\\"kept h{hyp.n} (chg/acc {cur_key[0]:.2f}/{cur_key[1]:.2f}): best new candidate h{best.n} is worse ({best.verdict.change_accuracy:.2f}/{best.verdict.accuracy:.2f})\\\", budget_used=budget.used())\\n                hyp.verdict = verify(hyp.model, log, hyp.ignore)\\n            if hyp.verdict.counterexamples:\\n                counterexamples = hyp.verdict.counterexamples[-4:]\\n            if hyp.verdict.usable(self.min_acc) and hyp.n != no_plan_n:\\n                events.emit(\\\"VERIFY\\\", \\\"PLAN\\\", f\\\"h{hyp.n} verified (acc {hyp.verdict.accuracy:.2f} on {hyp.verdict.n}) -> plan\\\", budget_used=budget.used())\\n                idle_rounds = idle_rounds + 1 if budget.used() == used_before else 0\\n                continue\\n            src = hyp if (hyp.origin == \\\"llm\\\" or llm_best is None) else llm_best     # an induced model names no experiments: the LLM's run\\n            tests = [a for a in src.call(\\\"test_actions\\\", s.scene, []) if isinstance(a, Action)][:2]\\n            attempts = [a for a in src.call(\\\"attempt_actions\\\", s.scene, []) if isinstance(a, Action) and a not in tests][:1]\\n            tests = tests + attempts          # learn AND try: one move toward the hypothesised win per round\\n            avail_now = s.available_actions()\\n            bad = [a.label() for a in tests if not _available(a, avail_now)]\\n            if bad:\\n                counterexamples = (counterexamples + [f\\\"You proposed {bad}, but this game only offers {sorted({x.label() if x.type == 'BUTTON' else 'clicks' for x in avail_now})}: propose actions the game accepts.\\\"])[-4:]\\n                events.emit(\\\"TEST\\\", \\\"TEST\\\", f\\\"dropped unavailable actions {bad}\\\", budget_used=budget.used())\\n            tests = [a for a in tests if _available(a, avail_now)]\\n            sk = state_key(s.scene)\\n            fresh = [a for a in tests if (sk, a.label()) not in tried_here]\\n            if len(fresh) < len(tests):\\n                repeats_dropped += len(tests) - len(fresh)\\n                dropped = [a.label() for a in tests if a not in fresh]\\n                events.emit(\\\"TEST\\\", \\\"TEST\\\", f\\\"dropped tests already run from this board: {dropped}\\\", budget_used=budget.used())\\n                counterexamples = (counterexamples + [f\\\"You proposed {dropped} as tests, but they were already run from this exact board and their outcome is in the log: propose DIFFERENT actions.\\\"])[-4:]\\n            tests = fresh\\n            if not tests:\\n                tests = [a for a in hyp.call(\\\"candidate_actions\\\", s.scene, []) if isinstance(a, Action) and (sk, a.label()) not in tried_here][:2]\\n            if not tests:\\n                tests = _fallback_probe(s, tried_labels)\\n                events.emit(\\\"TEST\\\", \\\"TEST\\\", f\\\"h{hyp.n} names no test -> fallback probe {[a.label() for a in tests]}\\\", budget_used=budget.used())\\n            if not tests:\\n                events.emit(\\\"TEST\\\", \\\"HYPOTHESISE\\\", f\\\"h{hyp.n} names no test and nothing is left to probe: revise\\\", budget_used=budget.used())\\n                continue\\n            events.emit(\\\"VERIFY\\\", \\\"TEST\\\", f\\\"h{hyp.n} not verified (acc {hyp.verdict.accuracy:.2f}, n {hyp.verdict.n}) -> tests+attempt {[a.label() for a in tests]}\\\", budget_used=budget.used())\\n            for a in tests:\\n                t = act(a, \\\"test\\\")\\n                if t is None:\\n                    break\\n                tests_run += 1; tried_labels.add(a.label()); tried_here.add((state_key(t.before), a.label())); last_goal = None\\n                if t.status_change:\\n                    events.emit(\\\"TEST\\\", \\\"LOOK\\\", f\\\"{t.status_change} after test {a.label()}\\\", transition_id=t.id, budget_used=budget.used()); break\\n            idle_rounds = idle_rounds + 1 if budget.used() == used_before else 0\\n        st = s.env.status()\\n        if not stop:\\n            stop = st.state if st.state in (\\\"WIN\\\", \\\"GAME_OVER\\\") else (\\\"timeout\\\" if s.timed_out() else \\\"budget\\\")\\n        events.emit(\\\"LOOK\\\", \\\"END\\\", stop, budget_used=budget.used())\\n        knowledge.demoted = dict(goal_inf.demoted); knowledge.save(self.memory.knowledge_path(game_id))\\n        self.memory.save()\\n        return Result(game_id, st.state, st.level - 1 if st.state != \\\"WIN\\\" else (st.levels_total or st.level), st.levels_total, st.actions_used, budget.snapshot(),\\n                      self.llm.calls - llm_calls0, len(events.events), stop, dict(s.level_actions),\\n                      [{\\\"name\\\": f\\\"h{hyp.n}\\\", \\\"score\\\": round(hyp.verdict.accuracy, 3) if hyp.verdict else 0.0, \\\"coverage\\\": round(hyp.verdict.coverage, 3) if hyp.verdict else 0.0,\\n                        \\\"verified\\\": bool(hyp.verdict and hyp.verdict.usable(self.min_acc)), \\\"origin\\\": hyp.origin}] if hyp else [],\\n                      [{\\\"name\\\": getattr(hyp.goal, \\\"name\\\", \\\"?\\\"), \\\"win\\\": hyp.doc.get(\\\"win\\\", \\\"\\\")}] if hyp and hyp.goal else [],\\n                      {\\\"rounds\\\": n_hyp, \\\"plans\\\": plans_executed, \\\"tests\\\": tests_run, \\\"repeats_dropped\\\": repeats_dropped, \\\"gd_steps\\\": gd_steps, \\\"llm_rounds_skipped\\\": llm_rounds_skipped,\\n                       \\\"win_goals\\\": [g.name for g in level_goals], \\\"goal_rounds\\\": goal_rounds, \\\"explore_plans\\\": explore_plans, \\\"explore\\\": explored}, round(time.time() - t_start, 1))\\n\\n\\nclass ContactGoal:\\n    \\\"\\\"\\\"Contact exploration target: put the agent ON an object (overlap) or, failing that, next to it. Objects are identified\\n    by identity (colour, bbox, shape), so a static target keeps its identity while the agent moves.\\\"\\\"\\\"\\n    template = \\\"explore\\\"\\n\\n    def __init__(self, target, mode: str):\\n        self.identity = target.identity(); self.center = target.center; self.mode = mode\\n        self.name = f\\\"contact:{mode}:{target.color}@{target.bbox[0]},{target.bbox[1]}\\\"\\n\\n    def _agents(self, sc):\\n        return [o for o in sc.objects if o.role and \\\"agent\\\" in o.role]\\n\\n    def _target(self, sc):\\n        return next((o for o in sc.objects if o.identity() == self.identity), None)\\n\\n    def is_goal(self, sc) -> bool:\\n        t = self._target(sc)\\n        if t is None:\\n            return True                                   # it vanished (picked up): contact happened\\n        return any(a.overlaps(t) if self.mode == \\\"overlap\\\" else (a.overlaps(t) or _adjacent(a, t)) for a in self._agents(sc))\\n\\n    def progress(self, sc) -> float:\\n        ag = self._agents(sc)\\n        if not ag:\\n            return 0.0\\n        d = min(abs(a.center[0] - self.center[0]) + abs(a.center[1] - self.center[1]) for a in ag)\\n        return max(0.0, 1.0 - d / 128.0)\\n\\n\\ndef contact_targets(roled_scene, touched: set, unreachable: set) -> list:\\n    \\\"\\\"\\\"Untouched, reachable-so-far objects the agent could go and touch, nearest first.\\\"\\\"\\\"\\n    ag = [o for o in roled_scene.objects if o.role and \\\"agent\\\" in o.role]\\n    if not ag:\\n        return []\\n    strips = {r.id for r in roled_scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    out = [o for o in roled_scene.objects if o.id not in {a.id for a in ag} and o.role not in (\\\"wall\\\", \\\"indicator\\\", \\\"decoration\\\") and o.region not in strips\\n           and o.identity() not in touched and o.identity() not in unreachable and o.area < 400]\\n    a0 = ag[0].center\\n    return sorted(out, key=lambda o: abs(o.center[0] - a0[0]) + abs(o.center[1] - a0[1]))\\n\\n\\ndef _record_level_win(knowledge, hyp, prev_log: list, level: int, actions: int, last_goal) -> Optional[str]:\\n    \\\"\\\"\\\"docs/029 priority 2: which goal predicate held when the level ended (true on the last board, false on the one before)\\n    becomes this game's prior for the next level (LevelKnowledge.goal_wins). Falls back to the goal the plan was executing.\\\"\\\"\\\"\\n    won = None\\n    if hyp is not None and prev_log:\\n        before, after = prev_log[-1].before, prev_log[-1].after\\n        for g in (getattr(hyp, \\\"goals\\\", None) or [hyp.goal]):\\n            if g is None or not hasattr(g, \\\"template\\\"):\\n                continue\\n            try:\\n                if g.is_goal(hyp.roled(after)) and not g.is_goal(hyp.roled(before)):\\n                    won = g; break\\n            except Exception:\\n                continue\\n    if won is None and last_goal is not None and hasattr(last_goal, \\\"template\\\"):\\n        won = last_goal\\n    knowledge.record_win(level, won, actions)\\n    return getattr(won, \\\"name\\\", None)\\n\\n\\ndef _demote_goal(goal_inf, knowledge, goal, penalty: float = -0.6) -> None:\\n    \\\"\\\"\\\"The goal's predicate held and the level did not end: it is not the win condition (stronger than 'unreachable').\\\"\\\"\\\"\\n    name = getattr(goal, \\\"name\\\", None)\\n    if name:\\n        goal_inf.demoted[name] = goal_inf.demoted.get(name, 0.0) + penalty\\n    if hasattr(goal, \\\"template\\\"):\\n        knowledge.record_fail(goal)\\n\\n\\ndef _available(a: Action, avail: list) -> bool:\\n    \\\"\\\"\\\"The environment refuses buttons it did not list and clicks in button-only games; an LLM hypothesis proposes both.\\\"\\\"\\\"\\n    if a.type == \\\"CLICK\\\":\\n        return any(x.type == \\\"CLICK\\\" for x in avail)\\n    if a.type == \\\"BUTTON\\\":\\n        return any(x.type == \\\"BUTTON\\\" and x.id == a.id for x in avail)\\n    return False\\n\\n\\ndef _fallback_probe(s, tried: set) -> list:\\n    \\\"\\\"\\\"One click on an object class not tried yet on this level (buttons first if the game has them).\\\"\\\"\\\"\\n    from ..probe.clickmap import object_key\\n    avail = s.available_actions()\\n    out = [a for a in avail if a.type == \\\"BUTTON\\\" and a.label() not in tried][:1]\\n    if out or not any(a.type == \\\"CLICK\\\" for a in avail):\\n        return out\\n    seen = set()\\n    strips = {r.id for r in s.scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    for o in sorted(s.scene.objects, key=lambda o: o.area):\\n        k = object_key(o)\\n        if o.region in strips or k in seen:\\n            continue\\n        seen.add(k)\\n        a = Action.click(*o.center)\\n        if a.label() not in tried:\\n            return [a]\\n    return []\\n\", \"pbg/hypothesis/verify.py\": \"\\\"\\\"\\\"hypothesis/verify.py \\u2014 pixel-level verification of a game hypothesis on a level's log. No object-size leniency, no\\nrole-based exceptions: the predicted next grid must equal the real next grid outside the hypothesis' own ignore boxes.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass, field\\nfrom typing import Optional\\n\\nimport numpy as np\\n\\nfrom ..core.types import Scene, Transition\\nfrom .contract import frame_text\\n\\n\\n@dataclass\\nclass Verdict:\\n    accuracy: float            # correct / verifiable transitions\\n    coverage: float            # predicted (not None) / verifiable\\n    n: int\\n    correct: int\\n    unknown: int\\n    counterexamples: list[str] = field(default_factory=list)\\n    violations: list[str] = field(default_factory=list)\\n    approx: float = 0.0        # mean closeness over verifiable transitions (1 exact, IoU of changed regions otherwise, 0 unknown)\\n    changed: int = 0           # transitions where the real board changed (outside the ignore boxes)\\n    change_correct: int = 0    # ... of which the model predicted exactly\\n\\n    @property\\n    def change_accuracy(self) -> float:\\n        \\\"\\\"\\\"Accuracy on the transitions that changed the board: a model that predicts \\\"nothing happens\\\" scores 0 here\\n        (it is exact on every no-op and useless for planning \\u2014 the trap that cost tn36 its level, docs/029).\\\"\\\"\\\"\\n        return self.change_correct / self.changed if self.changed else 0.0\\n\\n    def rank_key(self) -> tuple:\\n        return (self.change_accuracy, self.accuracy, self.coverage)\\n\\n    def usable(self, min_acc: float = 0.8, min_n: int = 3, min_change: float = 0.5) -> bool:\\n        return self.n >= min_n and self.accuracy >= min_acc and self.changed >= 1 and self.change_accuracy >= min_change\\n\\n    def approximate(self, min_approx: float = 0.5, min_cov: float = 0.6, min_n: int = 2) -> bool:\\n        \\\"\\\"\\\"Good enough to act on one step at a time (closed loop), not to trust a long plan.\\\"\\\"\\\"\\n        return self.n >= min_n and self.coverage >= min_cov and self.approx >= min_approx\\n\\n\\ndef _boxes(scene: Scene, hyp_ignore) -> list:\\n    boxes = [tuple(r.bbox) for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"]\\n    try:\\n        boxes += [tuple(int(v) for v in b) for b in (hyp_ignore(scene) or [])]\\n    except Exception:\\n        pass\\n    return boxes\\n\\n\\ndef pixel_equal(pred: Scene, obs_grid: np.ndarray, boxes: list, tol: int = 0) -> tuple[bool, int, Optional[tuple]]:\\n    pg = pred.render(); og = np.asarray(obs_grid)\\n    if pg.shape != og.shape:\\n        return False, 10 ** 6, None\\n    mask = pg != og\\n    for (r0, c0, r1, c1) in boxes:\\n        mask[r0:r1, c0:c1] = False\\n    n = int(mask.sum())\\n    if n <= tol:\\n        return True, n, None\\n    ys, xs = np.nonzero(mask)\\n    return False, n, (max(0, int(ys.min()) - 1), max(0, int(xs.min()) - 1), min(og.shape[0], int(ys.max()) + 2), min(og.shape[1], int(xs.max()) + 2))\\n\\n\\ndef closeness(pred: Scene, before_grid, after_grid, boxes: list) -> float:\\n    \\\"\\\"\\\"How close a wrong prediction is: IoU of the predicted and the real changed-pixel sets, times the colour agreement\\n    on the real changed pixels (moving the right thing roughly right scores well; changing nothing scores 0).\\\"\\\"\\\"\\n    pg = pred.render(); bg = np.asarray(before_grid); ag = np.asarray(after_grid)\\n    if pg.shape != ag.shape:\\n        return 0.0\\n    pc = pg != bg; rc = ag != bg\\n    for (r0, c0, r1, c1) in boxes:\\n        pc[r0:r1, c0:c1] = False; rc[r0:r1, c0:c1] = False\\n    if not rc.any():\\n        return 1.0 if not pc.any() else 0.0\\n    union = (pc | rc).sum()\\n    iou = float((pc & rc).sum()) / float(union) if union else 0.0\\n    # spatial tolerance: a change predicted within 3 cells of the real one counts half\\n    if iou < 0.5 and pc.any():\\n        ry, rx = np.nonzero(rc); py, px = np.nonzero(pc)\\n        d = abs(float(ry.mean()) - float(py.mean())) + abs(float(rx.mean()) - float(px.mean()))\\n        iou = max(iou, 0.5 * max(0.0, 1.0 - d / 12.0))\\n    colour = float((pg[rc] == ag[rc]).mean())\\n    return max(0.0, min(1.0, 0.7 * iou + 0.3 * colour))\\n\\n\\ndef verify(model, log: list[Transition], hyp_ignore=None, *, tol: int = 0, max_examples: int = 4) -> Verdict:\\n    usable = [t for t in log if t.action.type != \\\"RESET\\\" and t.before_frame is not None and t.after_frame is not None and not t.status_change]\\n    if not usable:\\n        return Verdict(0.0, 0.0, 0, 0, 0)\\n    correct = unknown = 0; ex: list[str] = []; viol: list[str] = []; close = 0.0\\n    changed = change_correct = 0\\n    for t in usable:\\n        boxes0 = _boxes(t.before, hyp_ignore) if hyp_ignore else [tuple(r.bbox) for r in t.before.regions if r.kind_hint == \\\"ui_strip\\\"]\\n        real = np.asarray(t.before_frame.grid) != np.asarray(t.after_frame.grid)\\n        for (r0, c0, r1, c1) in boxes0:\\n            real[r0:r1, c0:c1] = False\\n        did_change = bool(real.any())\\n        if did_change:\\n            changed += 1\\n        try:\\n            pred = model.predict(t.before, t.action)\\n        except Exception as e:\\n            pred = None\\n            if len(ex) < max_examples:\\n                ex.append(f\\\"[{t.id}] {t.action.label()}: the model raised {type(e).__name__}: {str(e)[:120]}\\\")\\n        if pred is None:\\n            unknown += 1; continue\\n        boxes = _boxes(t.before, hyp_ignore) if hyp_ignore else [tuple(r.bbox) for r in t.before.regions if r.kind_hint == \\\"ui_strip\\\"]\\n        ok, n, win = pixel_equal(pred, t.after_frame.grid, boxes, tol)\\n        if ok:\\n            correct += 1; close += 1.0\\n            if did_change:\\n                change_correct += 1\\n            continue\\n        close += closeness(pred, t.before_frame.grid, t.after_frame.grid, boxes)\\n        viol.append(t.id)\\n        if len(ex) < max_examples and win is not None:\\n            r0, c0, r1, c1 = win\\n            if (r1 - r0) <= 24 and (c1 - c0) <= 40:\\n                ex.append(f\\\"[{t.id}] {t.action.label()}: {n} px wrong in rows {r0}-{r1 - 1}, cols {c0}-{c1 - 1}\\\\n    predicted:\\\\n\\\"\\n                          + \\\"\\\\n\\\".join(\\\"      \\\" + l for l in frame_text(pred.render(), (r0, r1), (c0, c1)).splitlines())\\n                          + \\\"\\\\n    real:\\\\n\\\" + \\\"\\\\n\\\".join(\\\"      \\\" + l for l in frame_text(np.asarray(t.after_frame.grid), (r0, r1), (c0, c1)).splitlines()))\\n            else:\\n                ex.append(f\\\"[{t.id}] {t.action.label()}: {n} px wrong over rows {r0}-{r1 - 1}, cols {c0}-{c1 - 1} (window too large to print)\\\")\\n    n = len(usable)\\n    return Verdict(correct / n, (n - unknown) / n, n, correct, unknown, ex, viol, approx=close / n, changed=changed, change_correct=change_correct)\\n\", \"pbg/llm/__init__.py\": \"from .gateway import LLMGateway, extract_code, extract_json  # noqa: F401\\n\", \"pbg/llm/cache.py\": \"\\\"\\\"\\\"llm/cache.py \\u2014 response cache keyed by the context hash (spec \\u00a714: identical context -> cached reply).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\n\\nclass ResponseCache:\\n    def __init__(self, path: Optional[Path]):\\n        self.path = Path(path) if path else None\\n        if self.path:\\n            self.path.mkdir(parents=True, exist_ok=True)\\n        self.mem: dict[str, str] = {}\\n        self.hits = 0\\n\\n    @staticmethod\\n    def key(messages: list[dict], model: str, params: dict) -> str:\\n        return hashlib.sha256(json.dumps({\\\"m\\\": messages, \\\"model\\\": model, \\\"p\\\": params}, sort_keys=True, default=str).encode()).hexdigest()[:32]\\n\\n    def get(self, key: str) -> Optional[str]:\\n        if key in self.mem:\\n            self.hits += 1\\n            return self.mem[key]\\n        if self.path and (self.path / f\\\"{key}.json\\\").exists():\\n            try:\\n                v = json.loads((self.path / f\\\"{key}.json\\\").read_text())[\\\"text\\\"]\\n                self.mem[key] = v; self.hits += 1\\n                return v\\n            except Exception:\\n                return None\\n        return None\\n\\n    def put(self, key: str, text: str, meta: Optional[dict] = None) -> None:\\n        self.mem[key] = text\\n        if self.path:\\n            (self.path / f\\\"{key}.json\\\").write_text(json.dumps({\\\"text\\\": text, **(meta or {})}))\\n\", \"pbg/llm/gateway.py\": \"\\\"\\\"\\\"llm/gateway.py \\u2014 LLM Gateway (spec \\u00a714): tiered models, retries, 60 s timeout, cost metering, response cache.\\nVendor-neutral: any OpenAI-compatible /chat/completions endpoint. Dependency-free (urllib).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport base64\\nimport json\\nimport os\\nimport re\\nimport time\\nimport urllib.error\\nimport urllib.request\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nimport yaml\\n\\nfrom .cache import ResponseCache\\n\\nCODE_RE = re.compile(r\\\"```(?:python)?\\\\s*\\\\n(.*?)```\\\", re.S)\\nJSON_RE = re.compile(r\\\"```json\\\\s*\\\\n(.*?)```\\\", re.S)\\n\\n\\ndef _dotenv() -> dict:\\n    \\\"\\\"\\\"KEY=value pairs of the repository's .env (never printed, never committed).\\\"\\\"\\\"\\n    root = Path(__file__).resolve().parents[2]\\n    out = {}\\n    p = root / \\\".env\\\"\\n    if p.exists():\\n        for line in p.read_text().splitlines():\\n            if \\\"=\\\" in line and not line.strip().startswith(\\\"#\\\"):\\n                k, v = line.split(\\\"=\\\", 1); out[k.strip()] = v.strip().strip('\\\"').strip(\\\"'\\\")\\n    return out\\n\\n\\ndef load_llm_config(path: Optional[Path] = None) -> dict:\\n    \\\"\\\"\\\"llm.yaml by default; PBG_LLM_CONFIG selects another profile (e.g. pbg/llm/llm-opus.yaml). `api_key: ${VAR}` is\\n    expanded from the environment or the repository .env.\\\"\\\"\\\"\\n    p = path or Path(os.environ.get(\\\"PBG_LLM_CONFIG\\\") or Path(__file__).resolve().parent / \\\"llm.yaml\\\")\\n    cfg = yaml.safe_load(p.read_text()) if p.exists() else {}\\n    key = str(cfg.get(\\\"api_key\\\", \\\"\\\"))\\n    if key.startswith(\\\"${\\\") and key.endswith(\\\"}\\\"):\\n        var = key[2:-1]\\n        cfg[\\\"api_key\\\"] = os.environ.get(var) or _dotenv().get(var) or \\\"\\\"\\n    return cfg\\n\\n\\ndef extract_code(text: str) -> Optional[str]:\\n    \\\"\\\"\\\"The single python code block of a reply (the first one if several; None if none). A block whose closing fence\\n    was cut off by the token limit is salvaged as-is (the sandbox rejects it if it does not parse).\\\"\\\"\\\"\\n    if not text:\\n        return None\\n    if \\\"```python\\\" in text and text.count(\\\"```\\\") % 2 == 1:\\n        text = text + \\\"\\\\n```\\\"\\n    blocks = [b for b in CODE_RE.findall(text) if \\\"def build_model\\\" in b or \\\"def build_goal\\\" in b]\\n    if not blocks:\\n        blocks = CODE_RE.findall(text)\\n    return blocks[0].strip() if blocks else None\\n\\n\\ndef extract_json(text: str) -> Optional[dict]:\\n    if not text:\\n        return None\\n    m = JSON_RE.search(text)\\n    cand = m.group(1) if m else text[text.find(\\\"{\\\"): text.rfind(\\\"}\\\") + 1]\\n    try:\\n        return json.loads(cand)\\n    except Exception:\\n        return None\\n\\n\\nclass LLMGateway:\\n    def __init__(self, config: Optional[dict] = None, *, cache_dir: Optional[Path] = None, log=None, max_calls: int = 60):\\n        self.cfg = config or load_llm_config()\\n        self.log = log or (lambda *a, **k: None)\\n        self.cache = ResponseCache(cache_dir or self.cfg.get(\\\"cache_dir\\\"))\\n        self.calls = self.failures = 0\\n        self.prompt_tokens = self.completion_tokens = 0\\n        self.latency_total = 0.0\\n        self.max_calls = max_calls\\n        self.history: list[dict] = []\\n\\n    def tier(self, purpose: str) -> dict:\\n        name = (self.cfg.get(\\\"purposes\\\") or {}).get(purpose, \\\"reasoning\\\")\\n        return dict((self.cfg.get(\\\"tiers\\\") or {}).get(name) or {\\\"model\\\": \\\"local-qwen\\\", \\\"temperature\\\": 0.6, \\\"top_p\\\": 0.95, \\\"max_tokens\\\": 4000})\\n\\n    def exhausted(self) -> bool:\\n        return self.calls >= self.max_calls\\n\\n    def chat(self, messages: list[dict], *, purpose: str = \\\"world_model\\\", image: Optional[bytes] = None, use_cache: bool = True,\\n             override: Optional[dict] = None) -> str:\\n        tier = self.tier(purpose)\\n        if override:\\n            tier.update(override)\\n        if tier.get(\\\"no_sampling\\\"):\\n            # hosted models that reject sampling parameters: candidates differ by prompt, not by temperature\\n            tier.pop(\\\"temperature\\\", None); tier.pop(\\\"top_p\\\", None)\\n        msgs = [dict(m) for m in messages]\\n        if image is not None:\\n            b64 = base64.b64encode(image).decode()\\n            last = msgs[-1]\\n            last[\\\"content\\\"] = [{\\\"type\\\": \\\"text\\\", \\\"text\\\": last[\\\"content\\\"]}, {\\\"type\\\": \\\"image_url\\\", \\\"image_url\\\": {\\\"url\\\": f\\\"data:image/png;base64,{b64}\\\"}}]\\n        params = {k: tier[k] for k in (\\\"temperature\\\", \\\"top_p\\\", \\\"max_tokens\\\") if k in tier}\\n        key = self.cache.key(msgs, tier[\\\"model\\\"], params)\\n        if use_cache:\\n            hit = self.cache.get(key)\\n            if hit is not None:\\n                self.history.append({\\\"purpose\\\": purpose, \\\"cached\\\": True, \\\"chars\\\": len(hit)})\\n                return hit\\n        if self.exhausted():\\n            raise RuntimeError(f\\\"LLM call cap reached ({self.max_calls})\\\")\\n        body = {\\\"model\\\": tier[\\\"model\\\"], \\\"messages\\\": msgs, **params, **(tier.get(\\\"extra_body\\\") or {})}\\n        data = json.dumps(body).encode()\\n        timeout = float(self.cfg.get(\\\"timeout\\\", 60)); retries = int(self.cfg.get(\\\"retries\\\", 2))\\n        last_err: Exception = RuntimeError(\\\"no attempt\\\")\\n        for attempt in range(retries + 1):\\n            t0 = time.time(); self.calls += 1\\n            req = urllib.request.Request(self.cfg.get(\\\"base_url\\\", \\\"http://127.0.0.1:1234/v1\\\").rstrip(\\\"/\\\") + \\\"/chat/completions\\\", data=data,\\n                                         headers={\\\"Content-Type\\\": \\\"application/json\\\", \\\"Authorization\\\": f\\\"Bearer {self.cfg.get('api_key', 'x')}\\\"})\\n            try:\\n                with urllib.request.urlopen(req, timeout=timeout) as resp:\\n                    out = json.loads(resp.read())\\n                msg = out[\\\"choices\\\"][0][\\\"message\\\"]\\n                text = msg.get(\\\"content\\\") or \\\"\\\"\\n                reasoning = msg.get(\\\"reasoning_content\\\") or msg.get(\\\"reasoning\\\") or \\\"\\\"\\n                if not text.strip() and reasoning:\\n                    # the model spent its budget thinking: salvage a code block from the reasoning if there is one\\n                    text = reasoning if \\\"```\\\" in reasoning else \\\"\\\"\\n                    self.log(f\\\"llm {purpose}: empty content, reasoning {len(reasoning)} chars ({'code found' if text else 'no code'})\\\")\\n                usage = out.get(\\\"usage\\\") or {}\\n                self.prompt_tokens += int(usage.get(\\\"prompt_tokens\\\", 0)); self.completion_tokens += int(usage.get(\\\"completion_tokens\\\", 0))\\n                self.latency_total += time.time() - t0\\n                self.history.append({\\\"purpose\\\": purpose, \\\"cached\\\": False, \\\"latency\\\": round(time.time() - t0, 1), \\\"chars\\\": len(text), \\\"usage\\\": usage})\\n                if text.strip():\\n                    self.cache.put(key, text, {\\\"purpose\\\": purpose, \\\"usage\\\": usage})   # never cache an empty reply\\n                return text\\n            except urllib.error.HTTPError as e:\\n                last_err = RuntimeError(f\\\"HTTP {e.code}: {e.read().decode(errors='replace')[:300]}\\\")\\n            except Exception as e:\\n                last_err = e\\n            self.failures += 1\\n            self.log(f\\\"llm {purpose} attempt {attempt + 1} failed: {last_err!r}\\\")\\n            if self.exhausted():\\n                break\\n            time.sleep(min(10, 2 ** attempt))\\n        raise last_err\\n\\n    @staticmethod\\n    def extract_code(text: str) -> Optional[str]:\\n        return extract_code(text)\\n\\n    @staticmethod\\n    def extract_json(text: str) -> Optional[dict]:\\n        return extract_json(text)\\n\\n    def stats(self) -> dict:\\n        return {\\\"calls\\\": self.calls, \\\"failures\\\": self.failures, \\\"cache_hits\\\": self.cache.hits, \\\"prompt_tokens\\\": self.prompt_tokens,\\n                \\\"completion_tokens\\\": self.completion_tokens, \\\"latency_total\\\": round(self.latency_total, 1)}\\n\", \"pbg/llm/llm-kaggle.yaml\": \"# LLM Gateway profile for the Kaggle notebook: the in-notebook vLLM server (Qwen3.8-Flash-Next NVFP4, served as local-model).\\n# Selected by the notebook with PBG_LLM_CONFIG=<work>/pbg/llm/llm-kaggle.yaml (scripts/build_pbg_notebook.py).\\nbase_url: http://127.0.0.1:1234/v1\\napi_key: x\\ntimeout: 300\\nretries: 1\\ncache_dir: null\\ntiers:\\n  reasoning:\\n    model: local-model\\n    temperature: 0.6\\n    top_p: 0.95\\n    max_tokens: 5000\\n    extra_body: {chat_template_kwargs: {enable_thinking: false}}\\n  small:\\n    model: local-model\\n    temperature: 0.2\\n    top_p: 0.9\\n    max_tokens: 800\\n    extra_body: {chat_template_kwargs: {enable_thinking: false}}\\npurposes:\\n  world_model: reasoning\\n  goal_template: reasoning\\n  diagnose: reasoning\\n  compress: small\\n\", \"pbg/llm/llm.yaml\": \"# LLM Gateway configuration (spec \\u00a714). Vendor-neutral OpenAI-compatible chat endpoint.\\nbase_url: http://127.0.0.1:1234/v1\\napi_key: x\\ntimeout: 300           # seconds per call (spec default 60; the local 27B reasoning model needs ~2-4 min with thinking)\\nretries: 1\\ncache_dir: null        # null -> <memory_root>/llm_cache\\ntiers:\\n  reasoning:           # world-model code generation, goal template proposals, violation diagnosis\\n    model: local-qwen\\n    temperature: 0.6\\n    top_p: 0.95\\n    max_tokens: 5000\\n    # thinking OFF by default: with thinking on, the local 27B spent the whole 6000-token budget in reasoning and\\n    # returned empty content (164 s per call). Turn it on together with max_tokens >= 16000 and timeout >= 600.\\n    extra_body: {chat_template_kwargs: {enable_thinking: false}}\\n  small:               # diff summary compression when the context is too long\\n    model: local-qwen\\n    temperature: 0.2\\n    top_p: 0.9\\n    max_tokens: 800\\n    extra_body: {chat_template_kwargs: {enable_thinking: false}}\\npurposes:\\n  world_model: reasoning\\n  goal_template: reasoning\\n  diagnose: reasoning\\n  compress: small\\n\", \"pbg/memory/__init__.py\": \"from .store import Memory, GameMemory  # noqa: F401\\n\", \"pbg/memory/priors/__init__.py\": \"\\\"\\\"\\\"Prior library: mechanisms (world-model rule builders) and goal templates shared across games (spec \\u00a711).\\\"\\\"\\\"\\nfrom .mechanisms import MECHANISMS, mechanism_signatures  # noqa: F401\\n\", \"pbg/memory/priors/mechanisms/__init__.py\": \"\\\"\\\"\\\"priors/mechanisms \\u2014 reusable rule *effects* over Scene (spec \\u00a711, initial 30 mechanisms).\\n\\nEvery mechanism is a builder returning `effect(scene, action) -> Scene | None`; rules reference objects by ROLE, never\\nby id. LLM-written models import these by name (`from core.mechanisms import *` inside the sandbox maps here).\\nDirection convention: DIRS maps a button id to (dr, dc).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport inspect\\nfrom typing import Callable, Optional\\n\\nfrom ....core.types import Action, Object, Scene\\n\\nEffect = Callable[[Scene, Action], Optional[Scene]]\\nDIRS4 = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}      # ARC-AGI-3 convention: ACTION1 up, 2 down, 3 left, 4 right\\n\\n\\ndef _roleset(x) -> frozenset:\\n    \\\"\\\"\\\"A role argument may be one role or a collection of roles.\\\"\\\"\\\"\\n    if x is None:\\n        return frozenset()\\n    if isinstance(x, str):\\n        return frozenset([x])\\n    return frozenset(x)\\n\\n\\ndef _by_roles(scene: Scene, roles) -> list[Object]:\\n    rs = _roleset(roles)\\n    return [o for o in scene.objects if o.role in rs]\\n\\n\\ndef _replace(scene: Scene, new: dict[int, Object]) -> Scene:\\n    return scene.copy(objects=[new.get(o.id, o) for o in scene.objects])\\n\\n\\ndef _inside(scene: Scene, o: Object) -> bool:\\n    h, w = scene.grid_shape\\n    return 0 <= o.bbox[0] and 0 <= o.bbox[1] and o.bbox[2] <= h and o.bbox[3] <= w\\n\\n\\ndef _region_of(scene: Scene, o: Object):\\n    return scene.region(o.region)\\n\\n\\ndef _within_region(scene: Scene, o: Object) -> bool:\\n    reg = _region_of(scene, o)\\n    if reg is None or reg.id == \\\"R0\\\":\\n        return _inside(scene, o)\\n    r0, c0, r1, c1 = reg.bbox\\n    return o.bbox[0] >= r0 and o.bbox[1] >= c0 and o.bbox[2] <= r1 and o.bbox[3] <= c1\\n\\n\\n# \\u2500\\u2500 movement \\u2500\\u2500\\ndef move_role(role: str, dirs: dict[int, tuple[int, int]] = DIRS4, step: int = 1) -> Effect:\\n    \\\"\\\"\\\"Shift every object of `role` by step * dirs[action.id] on a BUTTON action.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"BUTTON\\\" or action.id not in dirs:\\n            return scene\\n        dr, dc = dirs[action.id]\\n        new = {o.id: o.moved(dr * step, dc * step) for o in _by_roles(scene, role)}\\n        s = _replace(scene, new)\\n        s.aux[\\\"_last_move\\\"] = (dr * step, dc * step, role)\\n        return s\\n    return effect\\n\\n\\ndef move_role_diagonal(role: str, dirs: dict[int, tuple[int, int]], step: int = 1) -> Effect:\\n    \\\"\\\"\\\"Same as move_role with an explicit (possibly diagonal) direction table.\\\"\\\"\\\"\\n    return move_role(role, dirs, step)\\n\\n\\ndef rotate_role(role: str, button: int, k: int = 1) -> Effect:\\n    \\\"\\\"\\\"Rotate the mask of every `role` object by k*90 degrees (about its bbox) on `button`.\\\"\\\"\\\"\\n    import numpy as np\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"BUTTON\\\" or action.id != button:\\n            return scene\\n        from ....core.types import shape_signature\\n        new = {}\\n        for o in _by_roles(scene, role):\\n            m = np.rot90(o.mask, k); r0, c0 = o.bbox[0], o.bbox[1]\\n            new[o.id] = Object(o.id, o.color, o.colors, (r0, c0, r0 + m.shape[0], c0 + m.shape[1]), m, o.area, shape_signature(m), o.region, o.role)\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef teleport_role(role: str, button: int, to_role: str) -> Effect:\\n    \\\"\\\"\\\"On `button`, move each `role` object onto the first `to_role` object's position.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"BUTTON\\\" or action.id != button:\\n            return scene\\n        tgt = _by_roles(scene, to_role)\\n        if not tgt:\\n            return scene\\n        tr, tc = tgt[0].bbox[0], tgt[0].bbox[1]\\n        new = {o.id: o.moved(tr - o.bbox[0], tc - o.bbox[1]) for o in _by_roles(scene, role)}\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef slide_until_blocked(role: str, blockers: tuple[str, ...] = (\\\"wall\\\",), dirs: dict[int, tuple[int, int]] = DIRS4, step: int = 1, max_steps: int = 64) -> Effect:\\n    \\\"\\\"\\\"Inertial movement: keep moving `role` in the pressed direction until a blocker or the region edge (ice).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"BUTTON\\\" or action.id not in dirs:\\n            return scene\\n        dr, dc = dirs[action.id]\\n        new = {}\\n        for o in _by_roles(scene, role):\\n            cur = o\\n            for _ in range(max_steps):\\n                nxt = cur.moved(dr * step, dc * step)\\n                if not _within_region(scene, nxt) or any(nxt.overlaps(b) for b in scene.objects if b.role in _roleset(blockers) and b.id != o.id):\\n                    break\\n                cur = nxt\\n            new[o.id] = cur\\n        return _replace(scene, new)\\n    return effect\\n\\n\\n# \\u2500\\u2500 interaction \\u2500\\u2500\\ndef cancel_move_if_overlap(mover: str, blocker: str) -> Effect:\\n    \\\"\\\"\\\"Undo the last move of `mover` objects that now overlap a `blocker` object (wall collision).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        before = scene.aux.get(\\\"_before\\\")\\n        if before is None:\\n            return scene\\n        blockers = _by_roles(scene, blocker)\\n        new = {}\\n        for o in _by_roles(scene, mover):\\n            if any(o.overlaps(b) for b in blockers if b.id != o.id):\\n                prev = before.get(o.id)\\n                if prev is not None:\\n                    p = prev.moved(0, 0); p.role = o.role; new[o.id] = p\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef cancel_move_if_outside(mover: str) -> Effect:\\n    \\\"\\\"\\\"Undo the last move of `mover` objects that left their region (or the grid).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        before = scene.aux.get(\\\"_before\\\")\\n        if before is None:\\n            return scene\\n        new = {}\\n        for o in _by_roles(scene, mover):\\n            if not _within_region(scene, o):\\n                prev = before.get(o.id)\\n                if prev is not None:\\n                    p = prev.moved(0, 0); p.role = o.role; new[o.id] = p\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef cancel_move_if_off_floor(mover: str, floor_colors: tuple[int, ...]) -> Effect:\\n    \\\"\\\"\\\"Undo the move when any pixel under the mover (in the rendered *before* scene minus the mover) is not a floor colour.\\\"\\\"\\\"\\n    import numpy as np\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        before = scene.aux.get(\\\"_before\\\")\\n        if before is None:\\n            return scene\\n        movers = _by_roles(scene, mover)\\n        if not movers:\\n            return scene\\n        ids = {o.id for o in movers}\\n        floor = before.copy(objects=[o for o in before.objects if o.id not in ids]).render()\\n        h, w = floor.shape\\n        new = {}\\n        for o in movers:\\n            r0, c0, r1, c1 = o.bbox\\n            if r0 < 0 or c0 < 0 or r1 > h or c1 > w:\\n                bad = True\\n            else:\\n                under = floor[r0:r1, c0:c1][o.mask]\\n                bad = not np.isin(under, list(floor_colors)).all()\\n            if bad:\\n                prev = before.get(o.id)\\n                if prev is not None:\\n                    p = prev.moved(0, 0); p.role = o.role; new[o.id] = p\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef push_role(mover: str, pushable: str, blockers: tuple[str, ...] = (\\\"wall\\\",)) -> Effect:\\n    \\\"\\\"\\\"A mover overlapping a pushable object after its move pushes it by the same displacement, unless the pushable\\n    would hit a blocker or another pushable (then the move is cancelled).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        lm = scene.aux.get(\\\"_last_move\\\"); before = scene.aux.get(\\\"_before\\\")\\n        if not lm or before is None:\\n            return scene\\n        dr, dc, _ = lm\\n        new = {}\\n        for o in _by_roles(scene, mover):\\n            for p in _by_roles(scene, pushable):\\n                if o.overlaps(p):\\n                    q = p.moved(dr, dc)\\n                    blocked = (not _within_region(scene, q)) or any(q.overlaps(b) for b in scene.objects if b.id not in (p.id, o.id) and (b.role in _roleset(blockers) or b.role in _roleset(pushable)))\\n                    if blocked:\\n                        prev = before.get(o.id)\\n                        if prev is not None:\\n                            pv = prev.moved(0, 0); pv.role = o.role; new[o.id] = pv\\n                    else:\\n                        new[p.id] = q\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef pull_role(mover: str, pullable: str) -> Effect:\\n    \\\"\\\"\\\"After a move, a pullable object that was adjacent behind the mover follows it.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        lm = scene.aux.get(\\\"_last_move\\\"); before = scene.aux.get(\\\"_before\\\")\\n        if not lm or before is None:\\n            return scene\\n        dr, dc, _ = lm\\n        new = {}\\n        for o in _by_roles(scene, mover):\\n            prev = before.get(o.id)\\n            if prev is None:\\n                continue\\n            behind = prev.moved(-dr, -dc)\\n            for p in _by_roles(scene, pullable):\\n                if p.overlaps(behind):\\n                    new[p.id] = p.moved(dr, dc)\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef gravity(role: str, blockers: tuple[str, ...] = (\\\"wall\\\", \\\"pushable\\\"), step: int = 1, direction: tuple[int, int] = (1, 0)) -> Effect:\\n    \\\"\\\"\\\"After every action, `role` objects fall in `direction` until blocked or at the region edge.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        dr, dc = direction\\n        new = {}\\n        for o in _by_roles(scene, role):\\n            cur = o\\n            for _ in range(64):\\n                nxt = cur.moved(dr * step, dc * step)\\n                if not _within_region(scene, nxt) or any(nxt.overlaps(b) for b in scene.objects if b.id != o.id and (b.role in _roleset(blockers) or b.role in _roleset(role))):\\n                    break\\n                cur = nxt\\n            new[o.id] = cur\\n        return _replace(scene, new)\\n    return effect\\n\\n\\n# \\u2500\\u2500 state \\u2500\\u2500\\ndef toggle_color(role: str, button: int, a: int, b: int) -> Effect:\\n    \\\"\\\"\\\"On `button`, objects of `role` switch colour a<->b.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"BUTTON\\\" or action.id != button:\\n            return scene\\n        new = {o.id: o.recolored(b if o.color == a else a) for o in _by_roles(scene, role) if o.color in (a, b)}\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef recolor_on_click(role: str, cycle: tuple[int, ...]) -> Effect:\\n    \\\"\\\"\\\"Clicking a `role` object advances its colour along `cycle`.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"CLICK\\\":\\n            return scene\\n        new = {}\\n        for o in _by_roles(scene, role):\\n            r0, c0, r1, c1 = o.bbox\\n            if r0 <= action.row < r1 and c0 <= action.col < c1 and o.mask[action.row - r0, action.col - c0] and o.color in cycle:\\n                new[o.id] = o.recolored(cycle[(cycle.index(o.color) + 1) % len(cycle)])\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef move_on_click(trigger_role: str, mover_role: str, dr: int, dc: int) -> Effect:\\n    \\\"\\\"\\\"Clicking a `trigger_role` object shifts every `mover_role` object by (dr, dc) (arrow buttons, conveyor controls).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"CLICK\\\":\\n            return scene\\n        hit = False\\n        for o in _by_roles(scene, trigger_role):\\n            r0, c0, r1, c1 = o.bbox\\n            if r0 <= action.row < r1 and c0 <= action.col < c1 and o.mask[action.row - r0, action.col - c0]:\\n                hit = True; break\\n        if not hit:\\n            return scene\\n        new = {o.id: o.moved(dr, dc) for o in _by_roles(scene, mover_role)}\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef permute_on_click(trigger_role: str, mapping: dict) -> Effect:\\n    \\\"\\\"\\\"Clicking a `trigger_role` object moves every object whose top-left is a key of `mapping` to the mapped top-left\\n    (a learned position permutation: track shifts with wrap-around, rotations, swaps). Objects elsewhere stay.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"CLICK\\\":\\n            return scene\\n        hit = False\\n        for o in _by_roles(scene, trigger_role):\\n            r0, c0, r1, c1 = o.bbox\\n            if r0 <= action.row < r1 and c0 <= action.col < c1 and o.mask[action.row - r0, action.col - c0]:\\n                hit = True; break\\n        if not hit:\\n            return scene\\n        occupied = {(o.bbox[0], o.bbox[1]) for o in scene.objects}\\n        new = {}\\n        for o in scene.objects:\\n            key = (o.bbox[0], o.bbox[1])\\n            if key in mapping:\\n                dst = mapping[key]\\n                if isinstance(dst, dict):        # conditional: {\\\"near\\\": dst_when_next_slot_occupied, \\\"far\\\": dst_when_free, \\\"next\\\": slot}\\n                    dst = dst[\\\"near\\\"] if dst.get(\\\"next\\\") in occupied else dst.get(\\\"far\\\", dst[\\\"near\\\"])\\n                tr, tc = dst\\n                new[o.id] = o.moved(tr - o.bbox[0], tc - o.bbox[1])\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef set_flag_on_click(role: str, key: str = \\\"selected\\\") -> Effect:\\n    \\\"\\\"\\\"Clicking a `role` object records its id in scene.aux[key] (selection state).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"CLICK\\\":\\n            return scene\\n        for o in _by_roles(scene, role):\\n            r0, c0, r1, c1 = o.bbox\\n            if r0 <= action.row < r1 and c0 <= action.col < c1 and o.mask[action.row - r0, action.col - c0]:\\n                s = scene.copy(); s.aux[key] = o.id\\n                return s\\n        return scene\\n    return effect\\n\\n\\ndef move_selected_on_click(key: str = \\\"selected\\\") -> Effect:\\n    \\\"\\\"\\\"Clicking empty space moves the selected object (aux[key]) so that its top-left lands on the click.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"CLICK\\\" or key not in scene.aux:\\n            return scene\\n        o = scene.get(scene.aux[key])\\n        if o is None:\\n            return scene\\n        s = _replace(scene, {o.id: o.moved(action.row - o.bbox[0], action.col - o.bbox[1])})\\n        s.aux.pop(key, None)\\n        return s\\n    return effect\\n\\n\\ndef step_toward_click(mover_role: str, blockers: tuple[str, ...] = (\\\"wall\\\", \\\"hazard\\\"), step: int = 1, diagonal: bool = False) -> Effect:\\n    \\\"\\\"\\\"Point-select navigation: clicking a cell (r, c) moves each `mover_role` object ONE cell toward (r, c). With\\n    `diagonal` it steps by sign on both axes; otherwise it steps along the axis of greater distance (4-connected). A\\n    step that would overlap a `blockers` role or leave the region is cancelled (the piece stays). Models games where\\n    you click a destination and the piece walks there one step per click (r11l-style).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"CLICK\\\":\\n            return scene\\n        blocks = [b for b in scene.objects if b.role in _roleset(blockers)]\\n        new = {}\\n        for o in _by_roles(scene, mover_role):\\n            cr = (o.bbox[0] + o.bbox[2] - 1) / 2.0; cc = (o.bbox[1] + o.bbox[3] - 1) / 2.0\\n            drr = action.row - cr; dcc = action.col - cc\\n            sr = (drr > 0) - (drr < 0); sc = (dcc > 0) - (dcc < 0)\\n            if diagonal:\\n                dr, dc = sr, sc\\n            elif abs(drr) >= abs(dcc):\\n                dr, dc = sr, 0\\n            else:\\n                dr, dc = 0, sc\\n            if dr == 0 and dc == 0:\\n                new[o.id] = o; continue\\n            nxt = o.moved(dr * step, dc * step)\\n            if not _within_region(scene, nxt) or any(nxt.overlaps(b) for b in blocks if b.id != o.id):\\n                new[o.id] = o\\n            else:\\n                new[o.id] = nxt\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef jump_to_click(mover_role: str, blockers: tuple[str, ...] = (\\\"wall\\\", \\\"hazard\\\")) -> Effect:\\n    \\\"\\\"\\\"Point-select teleport: clicking a cell (r, c) moves each `mover_role` object so that its centre lands on (r, c)\\n    (the whole piece is translated). A jump that would overlap a `blockers` role or leave the region is cancelled.\\n    Models games where a piece is placed where you click (r11l nodes).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"CLICK\\\":\\n            return scene\\n        blocks = [b for b in scene.objects if b.role in _roleset(blockers)]\\n        new = {}\\n        for o in _by_roles(scene, mover_role):\\n            cr = (o.bbox[0] + o.bbox[2] - 1) // 2; cc = (o.bbox[1] + o.bbox[3] - 1) // 2\\n            dr, dc = int(action.row) - cr, int(action.col) - cc\\n            if dr == 0 and dc == 0:\\n                new[o.id] = o; continue\\n            nxt = o.moved(dr, dc)\\n            if not _within_region(scene, nxt) or any(nxt.overlaps(b) for b in blocks if b.id != o.id):\\n                new[o.id] = o\\n            else:\\n                new[o.id] = nxt\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef counter_step(key: str = \\\"counter\\\", delta: int = -1, only_when_changed: bool = True) -> Effect:\\n    \\\"\\\"\\\"Hidden counter in aux[key] that changes by delta on every action (optionally only on non-noop ones).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        before = scene.aux.get(\\\"_before\\\")\\n        if only_when_changed and before is not None and sorted(o.identity() for o in before.objects) == sorted(o.identity() for o in scene.objects):\\n            return scene\\n        s = scene.copy(); s.aux[key] = int(s.aux.get(key, 0)) + delta\\n        return s\\n    return effect\\n\\n\\ndef shrink_strip(region_id: str, per_action: int = 1, axis: str = \\\"col\\\", only_when_changed: bool = False) -> Effect:\\n    \\\"\\\"\\\"A ui_strip bar loses `per_action` cells per action (gauge / timer): the largest object inside the strip shrinks\\n    from its end (col axis: right end; row axis: bottom end).\\\"\\\"\\\"\\n    import numpy as np\\n    from ....core.types import shape_signature\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        before = scene.aux.get(\\\"_before\\\")\\n        if only_when_changed and before is not None and sorted(o.identity() for o in before.objects) == sorted(o.identity() for o in scene.objects):\\n            return scene\\n        bars = [o for o in scene.objects if o.region == region_id]\\n        if not bars:\\n            return scene\\n        bar = max(bars, key=lambda o: o.area)\\n        r0, c0, r1, c1 = bar.bbox\\n        if axis == \\\"col\\\":\\n            if c1 - c0 <= per_action:\\n                return scene.copy(objects=[o for o in scene.objects if o.id != bar.id])\\n            m = bar.mask[:, :c1 - c0 - per_action]; bbox = (r0, c0, r1, c1 - per_action)\\n        else:\\n            if r1 - r0 <= per_action:\\n                return scene.copy(objects=[o for o in scene.objects if o.id != bar.id])\\n            m = bar.mask[:r1 - r0 - per_action, :]; bbox = (r0, c0, r1 - per_action, c1)\\n        nb = Object(bar.id, bar.color, bar.colors, bbox, m, int(m.sum()), shape_signature(m), bar.region, bar.role)\\n        return _replace(scene, {bar.id: nb})\\n    return effect\\n\\n\\n# \\u2500\\u2500 creation / removal \\u2500\\u2500\\ndef collect_on_overlap(mover: str, collectible: str, counter_key: Optional[str] = \\\"collected\\\") -> Effect:\\n    \\\"\\\"\\\"A collectible overlapped by the mover disappears (and aux[counter_key] increments).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        movers = _by_roles(scene, mover)\\n        gone = {c.id for c in _by_roles(scene, collectible) if any(m.overlaps(c) for m in movers)}\\n        if not gone:\\n            return scene\\n        s = scene.copy(objects=[o for o in scene.objects if o.id not in gone])\\n        if counter_key:\\n            s.aux[counter_key] = int(s.aux.get(counter_key, 0)) + len(gone)\\n        return s\\n    return effect\\n\\n\\ndef remove_on_click(role: str) -> Effect:\\n    \\\"\\\"\\\"Clicking a `role` object removes it.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"CLICK\\\":\\n            return scene\\n        keep = []\\n        for o in scene.objects:\\n            r0, c0, r1, c1 = o.bbox\\n            if o.role in _roleset(role) and r0 <= action.row < r1 and c0 <= action.col < c1 and o.mask[action.row - r0, action.col - c0]:\\n                continue\\n            keep.append(o)\\n        return scene.copy(objects=keep)\\n    return effect\\n\\n\\ndef spawn_on_button(template_role: str, button: int, offset: tuple[int, int], new_role: str = \\\"spawned\\\") -> Effect:\\n    \\\"\\\"\\\"On `button`, a copy of each `template_role` object appears at `offset` from it.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"BUTTON\\\" or action.id != button:\\n            return scene\\n        nid = max([o.id for o in scene.objects] + [-1]) + 1\\n        added = []\\n        for o in _by_roles(scene, template_role):\\n            c = o.moved(*offset); c.id = nid; c.role = new_role; nid += 1; added.append(c)\\n        return scene.copy(objects=scene.objects + added)\\n    return effect\\n\\n\\ndef hazard_kills(mover: str, hazard: str, key: str = \\\"dead\\\") -> Effect:\\n    \\\"\\\"\\\"Overlapping a hazard marks aux[key] = True (the planner treats it as a dead end).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if any(m.overlaps(h) for m in _by_roles(scene, mover) for h in _by_roles(scene, hazard)):\\n            s = scene.copy(); s.aux[key] = True\\n            return s\\n        return scene\\n    return effect\\n\\n\\n# \\u2500\\u2500 conditions \\u2500\\u2500\\ndef key_opens_door(mover: str, key_role: str, door_role: str, flag: str = \\\"has_key\\\") -> Effect:\\n    \\\"\\\"\\\"Collecting a key sets aux[flag]; with the flag, doors are removed when the mover touches them.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        movers = _by_roles(scene, mover)\\n        s = scene\\n        keys = [k for k in _by_roles(scene, key_role) if any(m.overlaps(k) for m in movers)]\\n        if keys:\\n            s = s.copy(objects=[o for o in s.objects if o.id not in {k.id for k in keys}]); s.aux[flag] = True\\n        if s.aux.get(flag):\\n            doors = [d for d in _by_roles(s, door_role) if any(m.overlaps(d) for m in movers)]\\n            if doors:\\n                s = s.copy(objects=[o for o in s.objects if o.id not in {d.id for d in doors}])\\n        return s\\n    return effect\\n\\n\\ndef door_blocks_without_key(mover: str, door_role: str, flag: str = \\\"has_key\\\") -> Effect:\\n    \\\"\\\"\\\"Doors act as walls until aux[flag] is set.\\\"\\\"\\\"\\n    inner = cancel_move_if_overlap(mover, door_role)\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if scene.aux.get(flag):\\n            return scene\\n        return inner(scene, action)\\n    return effect\\n\\n\\ndef switch_toggles_role(mover: str, switch_role: str, target_role: str, a: int, b: int) -> Effect:\\n    \\\"\\\"\\\"Stepping on a switch recolours every target object a<->b.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        before = scene.aux.get(\\\"_before\\\")\\n        movers = _by_roles(scene, mover)\\n        on = [sw for sw in _by_roles(scene, switch_role) if any(m.overlaps(sw) for m in movers)]\\n        if not on:\\n            return scene\\n        if before is not None:\\n            was_on = [sw for sw in _by_roles(before, switch_role) if any(m.overlaps(sw) for m in _by_roles(before, mover))]\\n            if was_on:\\n                return scene\\n        new = {o.id: o.recolored(b if o.color == a else a) for o in _by_roles(scene, target_role) if o.color in (a, b)}\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef color_match_pass(mover: str, gate_role: str) -> Effect:\\n    \\\"\\\"\\\"A gate blocks the mover unless they share a colour.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        before = scene.aux.get(\\\"_before\\\")\\n        if before is None:\\n            return scene\\n        new = {}\\n        for o in _by_roles(scene, mover):\\n            for g in _by_roles(scene, gate_role):\\n                if o.overlaps(g) and g.color != o.color:\\n                    prev = before.get(o.id)\\n                    if prev is not None:\\n                        p = prev.moved(0, 0); p.role = o.role; new[o.id] = p\\n        return _replace(scene, new)\\n    return effect\\n\\n\\ndef sequence_lock(role: str, order: tuple[int, ...], key: str = \\\"seq\\\") -> Effect:\\n    \\\"\\\"\\\"Clicking `role` objects in colour `order` advances aux[key]; a wrong click resets it.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"CLICK\\\":\\n            return scene\\n        for o in _by_roles(scene, role):\\n            r0, c0, r1, c1 = o.bbox\\n            if r0 <= action.row < r1 and c0 <= action.col < c1 and o.mask[action.row - r0, action.col - c0]:\\n                s = scene.copy(); i = int(s.aux.get(key, 0))\\n                s.aux[key] = i + 1 if i < len(order) and o.color == order[i] else 0\\n                return s\\n        return scene\\n    return effect\\n\\n\\n# \\u2500\\u2500 ui \\u2500\\u2500\\ndef lives_decrement_on_flag(flag: str = \\\"dead\\\", key: str = \\\"lives\\\", start: int = 3) -> Effect:\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if scene.aux.get(flag):\\n            s = scene.copy(); s.aux[key] = int(s.aux.get(key, start)) - 1; s.aux.pop(flag, None)\\n            return s\\n        return scene\\n    return effect\\n\\n\\ndef highlight_selected(key: str = \\\"selected\\\", color: int = 10) -> Effect:\\n    \\\"\\\"\\\"The selected object (aux[key]) is drawn in `color`.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        oid = scene.aux.get(key)\\n        o = scene.get(oid) if oid is not None else None\\n        if o is None or o.color == color:\\n            return scene\\n        s = _replace(scene, {o.id: o.recolored(color)}); s.aux[f\\\"_{key}_orig_color\\\"] = o.color\\n        return s\\n    return effect\\n\\n\\ndef timer_tick(key: str = \\\"timer\\\", start: int = 100) -> Effect:\\n    return counter_step(key, -1, only_when_changed=False)\\n\\n\\ndef noop_for(buttons: tuple[int, ...]) -> Effect:\\n    \\\"\\\"\\\"Explicit no-op for the given buttons (keeps the prediction covered instead of UNKNOWN).\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        return scene\\n    return effect\\n\\n\\ndef unknown_for(buttons: tuple[int, ...]) -> Effect:\\n    \\\"\\\"\\\"Explicit UNKNOWN for buttons the model does not explain.\\\"\\\"\\\"\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        return None if action.type == \\\"BUTTON\\\" and action.id in buttons else scene\\n    return effect\\n\\n\\ndef split_on_button(role: str, button: int) -> Effect:\\n    \\\"\\\"\\\"On `button`, each `role` object splits into its left/top and right/bottom halves.\\\"\\\"\\\"\\n    import numpy as np\\n    from ....core.types import shape_signature\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        if action.type != \\\"BUTTON\\\" or action.id != button:\\n            return scene\\n        nid = max([o.id for o in scene.objects] + [-1]) + 1\\n        out = []\\n        for o in scene.objects:\\n            if o.role not in _roleset(role) or o.width < 2:\\n                out.append(o); continue\\n            r0, c0, r1, c1 = o.bbox; mid = (c1 - c0) // 2\\n            for j, (a, b) in enumerate(((0, mid), (mid, c1 - c0))):\\n                m = o.mask[:, a:b]\\n                if m.any():\\n                    out.append(Object(o.id if j == 0 else nid, o.color, o.colors, (r0, c0 + a, r1, c0 + b), m, int(m.sum()), shape_signature(m), o.region, o.role))\\n                    if j: nid += 1\\n        return scene.copy(objects=out)\\n    return effect\\n\\n\\ndef merge_touching(role: str) -> Effect:\\n    \\\"\\\"\\\"Objects of `role` that touch after an action become one object (id = smallest).\\\"\\\"\\\"\\n    import numpy as np\\n    from ....core.types import shape_signature\\n    def effect(scene: Scene, action: Action) -> Optional[Scene]:\\n        objs = _by_roles(scene, role)\\n        if len(objs) < 2:\\n            return scene\\n        groups: list[list[Object]] = []\\n        for o in objs:\\n            placed = None\\n            for g in groups:\\n                if any(o.moved(0, 0).overlaps(x.moved(1, 0)) or o.overlaps(x.moved(-1, 0)) or o.overlaps(x.moved(0, 1)) or o.overlaps(x.moved(0, -1)) or o.overlaps(x) for x in g):\\n                    if placed is None:\\n                        g.append(o); placed = g\\n                    else:\\n                        placed.extend(g); g.clear()\\n            if placed is None:\\n                groups.append([o])\\n        groups = [g for g in groups if g]\\n        if all(len(g) == 1 for g in groups):\\n            return scene\\n        out = [o for o in scene.objects if o.role not in _roleset(role)]\\n        for g in groups:\\n            if len(g) == 1:\\n                out.append(g[0]); continue\\n            r0 = min(o.bbox[0] for o in g); c0 = min(o.bbox[1] for o in g); r1 = max(o.bbox[2] for o in g); c1 = max(o.bbox[3] for o in g)\\n            m = np.zeros((r1 - r0, c1 - c0), dtype=bool)\\n            for o in g:\\n                m[o.bbox[0] - r0:o.bbox[2] - r0, o.bbox[1] - c0:o.bbox[3] - c0] |= o.mask\\n            out.append(Object(min(o.id for o in g), g[0].color, g[0].colors, (r0, c0, r1, c1), m, int(m.sum()), shape_signature(m), g[0].region, role))\\n        return scene.copy(objects=out)\\n    return effect\\n\\n\\nMECHANISMS: dict[str, Callable] = {f.__name__: f for f in (\\n    move_role, move_role_diagonal, rotate_role, teleport_role, slide_until_blocked,\\n    cancel_move_if_overlap, cancel_move_if_outside, cancel_move_if_off_floor, push_role, pull_role, gravity,\\n    toggle_color, recolor_on_click, move_on_click, step_toward_click, jump_to_click, permute_on_click, set_flag_on_click, move_selected_on_click, counter_step, shrink_strip,\\n    collect_on_overlap, remove_on_click, spawn_on_button, hazard_kills, split_on_button, merge_touching,\\n    key_opens_door, door_blocks_without_key, switch_toggles_role, color_match_pass, sequence_lock,\\n    lives_decrement_on_flag, highlight_selected, timer_tick, noop_for, unknown_for)}\\n\\n\\ndef mechanism_signatures() -> str:\\n    \\\"\\\"\\\"Signatures + one-line descriptions for prompts (bodies excluded, spec \\u00a714).\\\"\\\"\\\"\\n    lines = []\\n    for name, f in MECHANISMS.items():\\n        doc = ((inspect.getdoc(f) or \\\"\\\").splitlines() or [\\\"\\\"])[0]\\n        lines.append(f\\\"{name}{inspect.signature(f)}  # {doc}\\\")\\n    return \\\"\\\\n\\\".join(lines)\\n\\n\\n__all__ = list(MECHANISMS) + [\\\"MECHANISMS\\\", \\\"DIRS4\\\", \\\"Effect\\\", \\\"mechanism_signatures\\\"]\\n\", \"pbg/memory/priors/roles.json\": \"{\\\"roles\\\": [\\\"agent\\\", \\\"wall\\\", \\\"target\\\", \\\"key\\\", \\\"door\\\", \\\"pushable\\\", \\\"collectible\\\", \\\"hazard\\\", \\\"indicator\\\", \\\"decoration\\\"],\\n \\\"custom_prefix\\\": \\\"custom:\\\", \\\"note\\\": \\\"roles outside the vocabulary are allowed as custom:<name> and reviewed at prior promotion\\\"}\\n\", \"pbg/memory/promote.py\": \"\\\"\\\"\\\"memory/promote.py \\u2014 prior promotion (spec \\u00a711): a verified rule reused in >= 2 games becomes a prior after game-specific\\nconstants are parameterised. The static filter rejects colour-index and coordinate hard-coding.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport re\\n\\nCOLOR_LITERAL = re.compile(r\\\"\\\\b(color|colour)\\\\s*==\\\\s*\\\\d+\\\")\\nCOORD_LITERAL = re.compile(r\\\"\\\\(\\\\s*\\\\d{1,2}\\\\s*,\\\\s*\\\\d{1,2}\\\\s*\\\\)\\\")\\n\\n\\ndef parameterise(code: str) -> tuple[str, list[str]]:\\n    \\\"\\\"\\\"Replace literal colours / coordinates with named parameters; returns (code, parameter names).\\\"\\\"\\\"\\n    params: list[str] = []\\n    def sub_color(m):\\n        params.append(f\\\"COLOR_{len(params)}\\\")\\n        return f\\\"{m.group(1)} == {params[-1]}\\\"\\n    code = COLOR_LITERAL.sub(sub_color, code)\\n    if COORD_LITERAL.search(code):\\n        params.append(\\\"COORDS\\\")\\n    return code, params\\n\\n\\ndef eligible(rule_meta: dict) -> bool:\\n    return len(set(rule_meta.get(\\\"games_verified\\\", []))) >= 2 and rule_meta.get(\\\"confidence\\\", 0) >= 1.0\\n\", \"pbg/memory/store.py\": \"\\\"\\\"\\\"memory/store.py \\u2014 three-layer memory (spec \\u00a711): episodic (raw logs), semantic (per-game verified knowledge),\\npriors (cross-game mechanisms / goal templates + usage stats). Everything stored is machine-readable and re-verifiable.\\n\\nmemory_root/\\n  episodic/{game_id}/raw.jsonl, transitions.jsonl, frames/{step_idx}.npy\\n  semantic/{game_id}/action_semantics.json, world_model.py, world_model.meta.json, goal.py, hypotheses/, level_notes.json\\n  priors/ -> the package library (pbg/memory/priors) + usage_stats.json + roles.json + promoted/ (auto-added)\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport shutil\\nimport time\\nfrom dataclasses import dataclass, field\\nfrom pathlib import Path\\nfrom typing import Literal, Optional\\n\\nimport numpy as np\\n\\nfrom ..core.types import Transition\\n\\nPRIORS_PKG = Path(__file__).resolve().parent / \\\"priors\\\"\\nROLES = [\\\"agent\\\", \\\"wall\\\", \\\"target\\\", \\\"key\\\", \\\"door\\\", \\\"pushable\\\", \\\"collectible\\\", \\\"hazard\\\", \\\"indicator\\\", \\\"decoration\\\"]\\nMAX_TRANSITIONS = 50_000\\n\\n\\n@dataclass\\nclass GameMemory:\\n    game_id: str\\n    transitions: list[Transition] = field(default_factory=list)\\n    action_semantics: dict = field(default_factory=dict)\\n    world_model_code: Optional[str] = None\\n    world_model_meta: dict = field(default_factory=dict)\\n    goal_code: Optional[str] = None\\n    goal_meta: dict = field(default_factory=dict)\\n    hypotheses: list[dict] = field(default_factory=list)     # unpromoted: [{code, score, coverage, name}]\\n    level_notes: dict = field(default_factory=dict)\\n    aux: dict = field(default_factory=dict)\\n\\n    @property\\n    def world_model(self) -> Optional[str]:\\n        return self.world_model_code\\n\\n\\nclass Memory:\\n    def __init__(self, root: Path):\\n        self.root = Path(root)\\n        for sub in (\\\"episodic\\\", \\\"semantic\\\", \\\"priors\\\"):\\n            (self.root / sub).mkdir(parents=True, exist_ok=True)\\n        self._games: dict[str, GameMemory] = {}\\n        self._step_counter: dict[str, int] = {}\\n\\n    # \\u2500\\u2500 paths \\u2500\\u2500\\n    def episodic_dir(self, game_id: str) -> Path:\\n        p = self.root / \\\"episodic\\\" / game_id; p.mkdir(parents=True, exist_ok=True); (p / \\\"frames\\\").mkdir(exist_ok=True)\\n        return p\\n\\n    def semantic_dir(self, game_id: str) -> Path:\\n        p = self.root / \\\"semantic\\\" / game_id; p.mkdir(parents=True, exist_ok=True); (p / \\\"hypotheses\\\").mkdir(exist_ok=True)\\n        return p\\n\\n    # \\u2500\\u2500 load / save \\u2500\\u2500\\n    def load(self, game_id: str) -> GameMemory:\\n        if game_id in self._games:\\n            return self._games[game_id]\\n        gm = GameMemory(game_id)\\n        sd = self.semantic_dir(game_id)\\n        if (sd / \\\"action_semantics.json\\\").exists():\\n            gm.action_semantics = json.loads((sd / \\\"action_semantics.json\\\").read_text())\\n        if (sd / \\\"world_model.py\\\").exists():\\n            gm.world_model_code = (sd / \\\"world_model.py\\\").read_text()\\n            if (sd / \\\"world_model.meta.json\\\").exists():\\n                gm.world_model_meta = json.loads((sd / \\\"world_model.meta.json\\\").read_text())\\n        if (sd / \\\"goal.py\\\").exists():\\n            gm.goal_code = (sd / \\\"goal.py\\\").read_text()\\n            if (sd / \\\"goal.meta.json\\\").exists():\\n                gm.goal_meta = json.loads((sd / \\\"goal.meta.json\\\").read_text())\\n        for f in sorted((sd / \\\"hypotheses\\\").glob(\\\"*.json\\\")):\\n            try:\\n                gm.hypotheses.append(json.loads(f.read_text()))\\n            except Exception:\\n                pass\\n        if (sd / \\\"level_notes.json\\\").exists():\\n            gm.level_notes = json.loads((sd / \\\"level_notes.json\\\").read_text())\\n        self._games[game_id] = gm\\n        return gm\\n\\n    def append(self, t: Transition, game_id: Optional[str] = None) -> str:\\n        gid = game_id or t.id.split(\\\":\\\")[0]\\n        gm = self.load(gid)\\n        if not t.id:\\n            n = self._step_counter.get(gid, len(gm.transitions)); t.id = f\\\"{gid}:{t.level}:{t.step_idx or n}\\\"\\n        gm.transitions.append(t)\\n        ed = self.episodic_dir(gid)\\n        with open(ed / \\\"transitions.jsonl\\\", \\\"a\\\") as f:\\n            f.write(json.dumps(t.to_json()) + \\\"\\\\n\\\")\\n        if t.after_frame is not None:\\n            np.save(ed / \\\"frames\\\" / f\\\"{t.step_idx}.npy\\\", t.after_frame.grid)\\n        if len(gm.transitions) > MAX_TRANSITIONS:\\n            # keep transitions, drop the oldest frame files only (spec \\u00a711)\\n            for f in sorted((ed / \\\"frames\\\").glob(\\\"*.npy\\\"), key=lambda p: int(p.stem))[: len(gm.transitions) - MAX_TRANSITIONS]:\\n                f.unlink(missing_ok=True)\\n        return t.id\\n\\n    def extend(self, ts: list[Transition], game_id: Optional[str] = None) -> None:\\n        for t in ts:\\n            self.append(t, game_id)\\n\\n    def knowledge_path(self, game_id: str) -> Path:\\n        return self.semantic_dir(game_id) / \\\"knowledge.json\\\"\\n\\n    def save_semantics(self, game_id: str, semantics: dict) -> None:\\n        gm = self.load(game_id); gm.action_semantics = dict(semantics)\\n        (self.semantic_dir(game_id) / \\\"action_semantics.json\\\").write_text(json.dumps(gm.action_semantics, indent=1, default=str))\\n\\n    def promote_model(self, game_id: str, code: str, meta: dict) -> None:\\n        sd = self.semantic_dir(game_id); gm = self.load(game_id)\\n        if gm.world_model_code and gm.world_model_code != code:\\n            n = len(list(sd.glob(\\\"world_model.v*.py\\\"))) + 1\\n            (sd / f\\\"world_model.v{n}.py\\\").write_text(gm.world_model_code)\\n        gm.world_model_code = code; gm.world_model_meta = {**meta, \\\"promoted_at\\\": time.time()}\\n        (sd / \\\"world_model.py\\\").write_text(code)\\n        (sd / \\\"world_model.meta.json\\\").write_text(json.dumps(gm.world_model_meta, indent=1, default=str))\\n\\n    def promote_goal(self, game_id: str, code: str, meta: dict) -> None:\\n        sd = self.semantic_dir(game_id); gm = self.load(game_id)\\n        gm.goal_code = code; gm.goal_meta = {**meta, \\\"promoted_at\\\": time.time()}\\n        (sd / \\\"goal.py\\\").write_text(code); (sd / \\\"goal.meta.json\\\").write_text(json.dumps(gm.goal_meta, indent=1, default=str))\\n\\n    def save_hypotheses(self, game_id: str, hyps: list[dict]) -> None:\\n        sd = self.semantic_dir(game_id); gm = self.load(game_id)\\n        for f in (sd / \\\"hypotheses\\\").glob(\\\"*.json\\\"):\\n            f.unlink()\\n        gm.hypotheses = []\\n        for i, h in enumerate(hyps[:10]):\\n            (sd / \\\"hypotheses\\\" / f\\\"h{i:02d}.json\\\").write_text(json.dumps(h, indent=1, default=str)); gm.hypotheses.append(h)\\n\\n    def record_level_note(self, game_id: str, level: int, note: dict) -> None:\\n        gm = self.load(game_id); gm.level_notes[str(level)] = note\\n        (self.semantic_dir(game_id) / \\\"level_notes.json\\\").write_text(json.dumps(gm.level_notes, indent=1, default=str))\\n\\n    def save(self) -> None:\\n        for gid, gm in self._games.items():\\n            if gm.action_semantics:\\n                (self.semantic_dir(gid) / \\\"action_semantics.json\\\").write_text(json.dumps(gm.action_semantics, indent=1, default=str))\\n            if gm.aux:\\n                (self.semantic_dir(gid) / \\\"aux.json\\\").write_text(json.dumps(gm.aux, indent=1, default=str))\\n\\n    # \\u2500\\u2500 priors \\u2500\\u2500\\n    def priors(self, kind: Literal[\\\"mechanisms\\\", \\\"goals\\\"]) -> dict:\\n        if kind == \\\"mechanisms\\\":\\n            from .priors.mechanisms import MECHANISMS, mechanism_signatures\\n            promoted = {p.stem: p.read_text() for p in (self.root / \\\"priors\\\" / \\\"promoted\\\").glob(\\\"*.py\\\")} if (self.root / \\\"priors\\\" / \\\"promoted\\\").exists() else {}\\n            return {\\\"mechanisms\\\": MECHANISMS, \\\"signatures\\\": mechanism_signatures(), \\\"promoted\\\": promoted, \\\"usage_stats\\\": self.usage_stats()}\\n        from ..goal.templates import TEMPLATES\\n        return {\\\"templates\\\": TEMPLATES, \\\"usage_stats\\\": self.usage_stats()}\\n\\n    def usage_stats(self) -> dict:\\n        p = self.root / \\\"priors\\\" / \\\"usage_stats.json\\\"\\n        return json.loads(p.read_text()) if p.exists() else {}\\n\\n    def record_usage(self, name: str, *, used: bool = True, verified: bool = False, failed: bool = False) -> None:\\n        stats = self.usage_stats(); s = stats.setdefault(name, {\\\"games_used\\\": 0, \\\"games_verified\\\": 0, \\\"games_failed\\\": 0})\\n        if used: s[\\\"games_used\\\"] += 1\\n        if verified: s[\\\"games_verified\\\"] += 1\\n        if failed: s[\\\"games_failed\\\"] += 1\\n        (self.root / \\\"priors\\\" / \\\"usage_stats.json\\\").write_text(json.dumps(stats, indent=1))\\n\\n    def promote_prior(self, code: str, evidence_games: list[str], name: Optional[str] = None) -> bool:\\n        \\\"\\\"\\\"Auto-add a rule to the prior library only when verified in >= 2 games (spec \\u00a711) and it passes static checks.\\\"\\\"\\\"\\n        if len(set(evidence_games)) < 2:\\n            return False\\n        from ..wml.sandbox import static_check, StaticCheckError\\n        try:\\n            static_check(code if \\\"def build_model\\\" in code else code + \\\"\\\\n\\\\ndef build_model():\\\\n    return None\\\\n\\\")\\n        except StaticCheckError:\\n            return False\\n        d = self.root / \\\"priors\\\" / \\\"promoted\\\"; d.mkdir(parents=True, exist_ok=True)\\n        n = name or f\\\"prior_{len(list(d.glob('*.py'))) + 1:03d}\\\"\\n        (d / f\\\"{n}.py\\\").write_text(code)\\n        (d / f\\\"{n}.meta.json\\\").write_text(json.dumps({\\\"evidence_games\\\": sorted(set(evidence_games)), \\\"promoted_at\\\": time.time()}))\\n        return True\\n\\n    # \\u2500\\u2500 cross-game transfer helpers (spec \\u00a713) \\u2500\\u2500\\n    def semantic_games(self) -> list[str]:\\n        return sorted(p.name for p in (self.root / \\\"semantic\\\").iterdir() if (p / \\\"world_model.py\\\").exists())\\n\\n    def structure_vector(self, game_id: str) -> Optional[list[float]]:\\n        p = self.semantic_dir(game_id) / \\\"structure.json\\\"\\n        return json.loads(p.read_text()) if p.exists() else None\\n\\n    def save_structure(self, game_id: str, vec: list[float]) -> None:\\n        (self.semantic_dir(game_id) / \\\"structure.json\\\").write_text(json.dumps(vec))\\n\\n    def wipe_game(self, game_id: str) -> None:\\n        for sub in (\\\"episodic\\\", \\\"semantic\\\"):\\n            shutil.rmtree(self.root / sub / game_id, ignore_errors=True)\\n        self._games.pop(game_id, None)\\n\", \"pbg/orchestrator/__init__.py\": \"from .loop import Orchestrator, Result  # noqa: F401\\nfrom .session import Session  # noqa: F401\\nfrom .states import STATES  # noqa: F401\\n\", \"pbg/orchestrator/knowledge.py\": \"\\\"\\\"\\\"orchestrator/knowledge.py \\u2014 per-game knowledge asset that ACCUMULATES across levels. Later levels of a game build on\\nthe earlier ones (same buttons, same kind of goal, new elements added), so what a level taught must not be re-learned\\nwith actions: which click classes react, which goal template won, which goal predicates held without a level-up.\\nPersisted in the memory store (semantic/<game>/knowledge.json) so a second run of the game starts with it.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom collections import Counter\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nfrom ..probe.clickmap import ClickMap\\n\\n\\nclass LevelKnowledge:\\n    def __init__(self) -> None:\\n        self.clicks = ClickMap()\\n        self.goal_wins: Counter = Counter()        # template -> levels it completed\\n        self.goal_fails: Counter = Counter()       # template -> times its predicate held without a level-up\\n        self.won_goals: list[dict] = []            # [{level, name, template, actions}]\\n        self.demoted: dict[str, float] = {}        # goal name -> penalty (carried into the next level)\\n        self.levels_seen: set[int] = set()\\n        # class-level click rules verified on earlier levels: trigger class -> {mover colours, displacements seen, n}.\\n        # Positions change between levels, classes do not: on a new level ONE click per trigger object is enough to\\n        # re-instantiate the rule (the displacement must be one already seen for the class)\\n        self.click_rules: dict[str, dict] = {}\\n        self.effect_rules: dict[str, dict] = {}    # trigger class -> {signatures: [...], n}: joint effects seen on earlier levels\\n        self.hyp_triggers: set = set()             # this level: trigger bboxes confirmed by their test click\\n        self.hyp_inert: set = set()                # this level: click classes hypothesised passive\\n\\n    # \\u2500\\u2500 goal confidence prior for THIS game (stronger than the global usage stats) \\u2500\\u2500\\n    def goal_stats(self, global_stats: Optional[dict] = None) -> dict:\\n        out = dict(global_stats or {})\\n        wins_by = dict(self.goal_wins)\\n        for a, b in ((\\\"align_color\\\", \\\"align_all_colors\\\"), (\\\"align_all_colors\\\", \\\"align_color\\\")):\\n            if a in wins_by and b not in wins_by:\\n                wins_by[b] = wins_by[a]          # one family: a single pair and the conjunction of pairs\\n        for name, wins in wins_by.items():\\n            fails = self.goal_fails.get(name, 0)\\n            # a template that completed a level of this game starts the next level at 0.75 (+clue, +structure bonus),\\n            # a failed one lower; kept below 1.0 so the structure bonus (same axis as the winning goal) can still rank\\n            out[name] = {\\\"games_used\\\": 4 * (wins + fails), \\\"games_verified\\\": 3 * (wins + fails) if wins else 0}\\n        return out\\n\\n    def apply_hypotheses(self, res) -> None:\\n        hyp = getattr(res, \\\"hypotheses\\\", None)\\n        if hyp is None:\\n            return\\n        self.hyp_triggers = set(hyp.confirmed_triggers())\\n        self.hyp_inert = set(hyp.non_interactive_classes()) if hyp.confirmed_triggers() else set()\\n        self.clicks.skip_untried = set(self.hyp_inert)\\n\\n    def record_rules(self, model) -> int:\\n        \\\"\\\"\\\"Harvest class-level click rules from a verified model (rules carry `meta` set by the induction).\\\"\\\"\\\"\\n        n = 0\\n        for r in getattr(model, \\\"rules\\\", lambda: [])():\\n            meta = getattr(r, \\\"meta\\\", None)\\n            if meta and meta.get(\\\"kind\\\") == \\\"effect\\\" and r.confidence >= 0.9:\\n                e = self.effect_rules.setdefault(meta[\\\"trigger_class\\\"], {\\\"signatures\\\": [], \\\"n\\\": 0})\\n                if meta[\\\"signature\\\"] not in e[\\\"signatures\\\"]:\\n                    e[\\\"signatures\\\"].append(meta[\\\"signature\\\"])\\n                e[\\\"n\\\"] += 1; n += 1\\n                continue\\n            if not meta or meta.get(\\\"kind\\\") != \\\"click_shift\\\" or r.confidence < 0.9:\\n                continue\\n            e = self.click_rules.setdefault(meta[\\\"trigger_class\\\"], {\\\"mover_colors\\\": [], \\\"displacements\\\": [], \\\"n\\\": 0})\\n            for c in meta.get(\\\"mover_colors\\\", []):\\n                if c not in e[\\\"mover_colors\\\"]:\\n                    e[\\\"mover_colors\\\"].append(c)\\n            d = list(meta[\\\"displacement\\\"])\\n            if d not in e[\\\"displacements\\\"]:\\n                e[\\\"displacements\\\"].append(d)\\n            e[\\\"n\\\"] += 1; n += 1\\n        return n\\n\\n    def goal_structure_bonus(self, goal) -> float:\\n        \\\"\\\"\\\"+0.1 for a goal with the same template AND the same structural parameters (axis, kind) as a goal that won\\n        an earlier level; ids and colours are level-specific and ignored.\\\"\\\"\\\"\\n        for w in self.won_goals:\\n            if w.get(\\\"template\\\") == goal.template and w.get(\\\"structure\\\") == _structure(goal.params):\\n                return 0.1\\n        return 0.0\\n\\n    def record_win(self, level: int, goal, actions: int) -> None:\\n        if goal is None:\\n            return\\n        self.goal_wins[goal.template] += 1\\n        self.won_goals.append({\\\"level\\\": level, \\\"name\\\": goal.name, \\\"template\\\": goal.template, \\\"actions\\\": actions, \\\"structure\\\": _structure(goal.params)})\\n\\n    def record_fail(self, goal) -> None:\\n        if goal is not None:\\n            self.goal_fails[goal.template] += 1\\n\\n    # \\u2500\\u2500 persistence \\u2500\\u2500\\n    def to_json(self) -> dict:\\n        return {\\\"clicks\\\": self.clicks.to_json(), \\\"goal_wins\\\": dict(self.goal_wins), \\\"goal_fails\\\": dict(self.goal_fails),\\n                \\\"won_goals\\\": self.won_goals, \\\"demoted\\\": self.demoted, \\\"levels_seen\\\": sorted(self.levels_seen), \\\"click_rules\\\": self.click_rules, \\\"effect_rules\\\": self.effect_rules}\\n\\n    @classmethod\\n    def from_json(cls, d: dict) -> \\\"LevelKnowledge\\\":\\n        k = cls()\\n        k.clicks = ClickMap.from_json(d.get(\\\"clicks\\\", {}))\\n        k.goal_wins = Counter(d.get(\\\"goal_wins\\\", {})); k.goal_fails = Counter(d.get(\\\"goal_fails\\\", {}))\\n        k.won_goals = list(d.get(\\\"won_goals\\\", [])); k.demoted = dict(d.get(\\\"demoted\\\", {}))\\n        k.levels_seen = set(d.get(\\\"levels_seen\\\", [])); k.click_rules = dict(d.get(\\\"click_rules\\\", {})); k.effect_rules = dict(d.get(\\\"effect_rules\\\", {}))\\n        return k\\n\\n    def save(self, path: Path) -> None:\\n        path.write_text(json.dumps(self.to_json(), indent=1, default=str))\\n\\n    @classmethod\\n    def load(cls, path: Path) -> \\\"LevelKnowledge\\\":\\n        try:\\n            return cls.from_json(json.loads(path.read_text())) if path.exists() else cls()\\n        except Exception:\\n            return cls()\\n\\n    def summary(self) -> str:\\n        return f\\\"clicks[{self.clicks.summary()}] wins={dict(self.goal_wins)} fails={dict(self.goal_fails)} rules={list(self.click_rules)}\\\"\\n\\n\\ndef _structure(params: dict) -> dict:\\n    \\\"\\\"\\\"Structural (level-independent) goal parameters: string values that are not roles/regions/ids.\\\"\\\"\\\"\\n    out = {}\\n    for k, v in (params or {}).items():\\n        if k in (\\\"color\\\", \\\"colors\\\", \\\"target\\\", \\\"canvas\\\", \\\"canvas_bbox\\\", \\\"a\\\", \\\"b\\\", \\\"src\\\", \\\"dst\\\", \\\"id\\\"):\\n            continue\\n        if isinstance(v, str) and not v.startswith((\\\"custom:\\\", \\\"R\\\")):\\n            out[k] = v\\n    return out\\n\", \"pbg/orchestrator/loop.py\": \"\\\"\\\"\\\"orchestrator/loop.py \\u2014 the control loop (spec \\u00a712): a state machine PROBE -> HYPOTHESIZE -> PLAN -> EXECUTE driven\\nonly by events (mismatch, level-up, stuck, no plan). LLM calls happen only in HYPOTHESIZE. Every transition is logged\\nto events.jsonl with a reason; budget caps come from budget.yaml.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport time\\nfrom dataclasses import dataclass, field\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nfrom ..core.budget import Budget\\nfrom ..core.contracts import Hypothesis, RuleModel\\nfrom ..core.events import EventLog\\nfrom ..core.types import Action, Scene, Transition\\nfrom ..goal.infer import GoalInference\\nfrom ..goal.templates import GoalInstance\\nfrom ..memory.store import Memory\\nfrom ..perception import Perception\\nfrom ..planner.execute import Planner\\nfrom ..probe.protocol import Probe\\nfrom ..probe.semantics import ActionSemantics, classify_actions\\nfrom ..wml.evaluate import evaluate\\nfrom ..wml.refine import WorldModelLab\\nfrom .session import Session\\nfrom .knowledge import LevelKnowledge\\nfrom .transfer import level_novelty, structure_vector, transfer_hypotheses\\n\\n\\n@dataclass\\nclass Result:\\n    game_id: str\\n    state: str\\n    levels_completed: int\\n    levels_total: Optional[int]\\n    actions: int\\n    budget: dict\\n    llm_calls: int\\n    events: int\\n    stop_reason: str\\n    level_actions: dict = field(default_factory=dict)\\n    hypotheses: list = field(default_factory=list)\\n    goals: list = field(default_factory=list)\\n    semantics: dict = field(default_factory=dict)\\n    elapsed: float = 0.0\\n\\n    def to_json(self) -> dict:\\n        return self.__dict__.copy()\\n\\n\\nclass Orchestrator:\\n    def __init__(self, *, memory: Memory, llm=None, budget_cfg: Optional[dict] = None, perception_cfg: Optional[dict] = None, log=None,\\n                 events_dir: Optional[Path] = None, use_llm: bool = True, max_seconds: Optional[float] = None, planner_kwargs: Optional[dict] = None,\\n                 max_levels: int = 20, max_resets: int = 6, min_plan_score: float = 0.6):\\n        self.memory, self.llm = memory, llm\\n        self.max_resets = max_resets\\n        self.min_plan_score = min_plan_score\\n        self.budget_cfg, self.perception_cfg = budget_cfg, perception_cfg\\n        self.log = log or (lambda *a, **k: None)\\n        self.events_dir = events_dir\\n        self.use_llm = use_llm and llm is not None\\n        self.max_seconds = max_seconds\\n        self.planner_kwargs = planner_kwargs or {}\\n        self.max_levels = max_levels\\n\\n    def play(self, env, game_id: str, *, budget_total: Optional[int] = None) -> Result:\\n        t_start = time.time()\\n        budget = Budget(budget_total, self.budget_cfg)\\n        perception = Perception(self.perception_cfg)\\n        events = EventLog((self.events_dir / f\\\"{game_id}.events.jsonl\\\") if self.events_dir else None)\\n        s = Session(env, game_id, perception, self.memory, budget, log=self.log, max_seconds=self.max_seconds)\\n        mem = self.memory.load(game_id)\\n        wml = WorldModelLab(llm=self.llm if self.use_llm else None, memory=self.memory, log=self.log)\\n        goal_inf = GoalInference(self.memory, self.llm if self.use_llm else None, wml.sandbox, log=self.log)\\n        # knowledge asset of this game: click response map, goal template wins/fails, demoted goals; accumulates across\\n        # levels (and runs) so nothing a level taught is re-learned with actions\\n        knowledge = LevelKnowledge.load(self.memory.knowledge_path(game_id))\\n        goal_inf.knowledge = knowledge; goal_inf.demoted.update(knowledge.demoted)\\n        self.log(f\\\"knowledge: {knowledge.summary()}\\\")\\n        planner = Planner(log=self.log, **self.planner_kwargs)\\n        planner.click_map = knowledge.clicks\\n        probe = Probe(s)\\n        scene = s.start()\\n        env.calibrate_click()\\n        semantics = ActionSemantics(mem.action_semantics or {})\\n        H: list[Hypothesis] = []; G: list[GoalInstance] = []\\n        # transfer: verified model of this game (retry) or of similar games (spec \\u00a713)\\n        H += transfer_hypotheses(self.memory, game_id, scene, wml.sandbox, log=self.log)\\n        self.memory.save_structure(game_id, structure_vector(scene))\\n        state = \\\"PLAN\\\" if (H and mem.world_model_code) else \\\"PROBE\\\"\\n        events.emit(\\\"START\\\", state, \\\"verified model in memory\\\" if state == \\\"PLAN\\\" else \\\"no model\\\", budget_used=0)\\n        stop = \\\"\\\"; experiments_this_round = 0; hypothesize_rounds = 0; no_plan_rounds = 0; reprobe_rounds = 0\\n        last_level = s.level; level_start_step = 0; last_mismatch: Optional[Transition] = None\\n        prev_level_scene: Optional[Scene] = scene\\n        plan: Optional[list[Action]] = None\\n        touched: set = set(); exploring = False\\n        last_refine_n = -1; resets_without_progress = 0; game_overs = 0\\n        tried_experiments: set = set(); bumped: set = set(); clicked: set = set(); gated_rounds = 0\\n        idle_iters = 0; last_used = -1\\n        walk_dry = 0; last_action = None; fresh_level = False; llm_waits_this_level = 0; level_note = \\\"\\\"\\n        forced_tried: set = set(); gd_last = \\\"\\\"; gd_repeat = 0\\n        while not s.finished():\\n            knowledge.clicks.extend(s.transitions[-8:])\\n            if budget.used() == last_used:\\n                idle_iters += 1\\n                if wml.job_running() and idle_iters >= 4:\\n                    # an LLM job is thinking in the background: wait on it instead of busy-spinning the planner\\n                    # (a no-plan state used to re-run planner.search ~30x/second for the whole ~80 s job -> 15k idle loops)\\n                    wml.wait_job(5.0); idle_iters = 0\\n                elif idle_iters >= 8 and state in (\\\"PLAN\\\", \\\"HYPOTHESIZE\\\"):\\n                    events.emit(state, \\\"PROBE\\\", \\\"no action for 8 loop iterations -> walk probe (liveness)\\\", budget_used=budget.used())\\n                    state = \\\"PROBE\\\"; idle_iters = 0\\n                if idle_iters >= 60 and not wml.job_running():\\n                    # hard anti-spin governor: probing bounced and no plan is forthcoming. Take one forced exploratory\\n                    # action to generate new evidence; if every action class is exhausted, end the level rather than\\n                    # spin the CPU (and starve the shared LLM) for the rest of the budget.\\n                    a = _forced_explore(knowledge, s, forced_tried)\\n                    if a is not None:\\n                        t_ = s.act(a, \\\"reprobe\\\")\\n                        events.emit(state, \\\"HYPOTHESIZE\\\", f\\\"anti-spin: forced exploratory {a.label()}\\\", transition_id=t_.id if t_ else None, budget_used=budget.used())\\n                        idle_iters = 0; last_used = budget.used(); state = \\\"HYPOTHESIZE\\\"; experiments_this_round = 0; continue\\n                    stop = \\\"stuck (no untried action; ending to free the budget)\\\"; break\\n            else:\\n                idle_iters = 0; last_used = budget.used()\\n            if s.env.status().state == \\\"GAME_OVER\\\":\\n                game_overs += 1\\n                if game_overs > 20:\\n                    stop = \\\"GAME_OVER\\\"; break\\n                events.emit(state, \\\"PLAN\\\", f\\\"game over #{game_overs} (during {state}) -> RESET\\\", budget_used=budget.used())\\n                s.act(Action.reset(), \\\"plan\\\"); planner.stuck.reset(); touched.clear(); state = \\\"PLAN\\\"\\n                continue\\n            if s.level < last_level:\\n                # the game restarted from level 1 (engine semantics of a RESET on a fresh level): start the level bookkeeping over\\n                events.emit(state, \\\"PLAN\\\", f\\\"game restarted at level {s.level} (was {last_level}); level bookkeeping reset\\\", budget_used=budget.used())\\n                last_level = s.level; prev_level_scene = s.scene; planner.stuck.reset(); touched.clear(); bumped.clear(); tried_experiments.clear()\\n                no_plan_rounds = 0; reprobe_rounds = 0; resets_without_progress = 0; state = \\\"PLAN\\\"; continue\\n            if s.level > last_level:\\n                # level transition procedure (spec \\u00a713)\\n                novelty, novel_ids = level_novelty(prev_level_scene, s.scene)\\n                self.memory.record_level_note(game_id, last_level, {\\\"level\\\": last_level, \\\"actions_used\\\": s.level_actions.get(last_level, 0),\\n                                                                     \\\"novelty\\\": novelty, \\\"replaced_rules\\\": [], \\\"added_rules\\\": []})\\n                events.emit(state, \\\"PLAN\\\" if novelty == 0 else \\\"PROBE\\\", f\\\"level {last_level} -> {s.level}, novelty {novelty}\\\", budget_used=budget.used())\\n                fresh_level = novelty > 0; knowledge.hyp_triggers = set(); knowledge.hyp_inert = set(); knowledge.clicks.skip_untried = set(); llm_waits_this_level = 0\\n                last_level = s.level; prev_level_scene = s.scene; level_start_step = s.step_idx; walk_dry = 0\\n                knowledge.levels_seen.add(s.level); knowledge.demoted = dict(goal_inf.demoted); knowledge.save(self.memory.knowledge_path(game_id))\\n                planner.stuck.reset(); no_plan_rounds = 0; reprobe_rounds = 0; resets_without_progress = 0; touched.clear(); clicked.clear(); forced_tried.clear(); gd_last = \\\"\\\"; gd_repeat = 0\\n                if s.level > self.max_levels:\\n                    stop = \\\"max_levels\\\"; break\\n                if novelty == 0 and H:\\n                    state = \\\"PLAN\\\"\\n                else:\\n                    state = \\\"LOCAL_PROBE\\\"\\n                    self._novel_ids = novel_ids\\n            if state == \\\"PROBE\\\":\\n                cap = budget.cap(\\\"initial_probe\\\") if not s.transitions else budget.cap(\\\"reprobe\\\", s.level)\\n                kind = \\\"initial\\\" if not s.transitions else \\\"reprobe\\\"\\n                if kind == \\\"reprobe\\\":\\n                    reprobe_rounds += 1\\n                    if walk_dry >= 2:\\n                        cap = 0     # the click map has nothing untried or responsive left: do not walk, wait/reset instead\\n                    if cap > 0 and not _walk_useful(knowledge, s) and H:\\n                        # nothing new to click: one goal-directed step (the responsive action whose predicted outcome raises the\\n                        # top goal's progress the most, never the inverse of the last action) instead of a round-robin walk\\n                        a = _goal_directed_action(knowledge, s, H, G, last_action)\\n                        if a is not None:\\n                            t_ = s.act(a, \\\"reprobe\\\")\\n                            if t_ is not None:\\n                                last_action = a\\n                                events.emit(\\\"PROBE\\\", \\\"HYPOTHESIZE\\\", f\\\"goal-directed probe {a.label()}\\\", transition_id=t_.id, budget_used=budget.used())\\n                                semantics = classify_actions(s.transitions, semantics); knowledge.clicks.extend(s.transitions[-3:])\\n                                state = \\\"HYPOTHESIZE\\\"; experiments_this_round = 0; continue\\n                        walk_dry += 1; cap = 0\\n                    if cap <= 0 and wml.job_running():\\n                        # nothing cheap left to try: wait for the pending LLM candidates rather than spend actions (spec \\u00a715)\\n                        events.emit(state, \\\"HYPOTHESIZE\\\", \\\"reprobe budget exhausted, waiting for the llm job\\\", budget_used=budget.used())\\n                        wml.wait_job(min(240.0, max(0.0, (s.deadline - time.time()) if s.deadline else 240.0)))\\n                        state = \\\"HYPOTHESIZE\\\"; continue\\n                    if cap <= 0 and self.use_llm and not wml.job_running() and llm_waits_this_level < 2 and budget.allows_llm():\\n                        # nothing cheap left and no LLM job in flight: ask (K=2) and wait once before resetting\\n                        best = H[0] if H else None\\n                        if wml.start_llm_job(s.transitions, best, self.memory.priors(\\\"mechanisms\\\"), scene=s.scene, semantics=semantics, level_note=level_note, K=2):\\n                            llm_waits_this_level += 1\\n                            events.emit(state, \\\"HYPOTHESIZE\\\", \\\"reprobe budget exhausted -> llm job started, waiting\\\", budget_used=budget.used())\\n                            wml.wait_job(120.0); state = \\\"HYPOTHESIZE\\\"; continue\\n                    if cap <= 0:\\n                        # re-exploration budget exhausted: reset as the last resort (spec \\u00a715); the reprobe allowance\\n                        # starts again after the reset (a new attempt), and the game is only abandoned after 3 resets\\n                        # without a level-up\\n                        if s.reset_allowed and resets_without_progress < int(self.max_resets):\\n                            if s.actions_since_reset == 0:\\n                                # a RESET now would restart the WHOLE game (refused). Don't re-attempt it every loop\\n                                # (that was 188+ refused resets); take a forced exploratory action so we make progress\\n                                # (and a later RESET becomes a real attempt), or end cleanly when nothing is left to try.\\n                                a = _forced_explore(knowledge, s, forced_tried)\\n                                if a is not None:\\n                                    t_ = s.act(a, \\\"reprobe\\\")\\n                                    events.emit(state, \\\"HYPOTHESIZE\\\", f\\\"reset would restart the game -> forced exploratory {a.label()}\\\", transition_id=t_.id if t_ else None, budget_used=budget.used())\\n                                    budget.reset_level(s.level, \\\"reprobe\\\"); walk_dry = 0; state = \\\"HYPOTHESIZE\\\"; continue\\n                                if wml.job_running():\\n                                    events.emit(state, \\\"HYPOTHESIZE\\\", \\\"reset unsafe, nothing to explore -> waiting for the llm job\\\", budget_used=budget.used())\\n                                    wml.wait_job(60.0); budget.reset_level(s.level, \\\"reprobe\\\"); state = \\\"HYPOTHESIZE\\\"; continue\\n                                stop = \\\"UNRESOLVED (reset unsafe, nothing to explore)\\\"; events.emit(state, \\\"END\\\", stop, budget_used=budget.used()); break\\n                            events.emit(state, \\\"HYPOTHESIZE\\\", \\\"reprobe budget exhausted -> RESET\\\", budget_used=budget.used())\\n                            s.act(Action.reset(), \\\"reprobe\\\"); planner.stuck.reset(); resets_without_progress += 1\\n                            budget.reset_level(s.level, \\\"reprobe\\\"); touched.clear(); walk_dry = 0; state = \\\"HYPOTHESIZE\\\"; continue\\n                        stop = \\\"UNRESOLVED\\\"; events.emit(state, \\\"END\\\", f\\\"no progress after {self.max_resets} resets\\\", budget_used=budget.used()); break\\n                if kind == \\\"initial\\\":\\n                    res = probe.run_initial(cap, do_reset_probe=False)\\n                elif fresh_level and any(a.type == \\\"CLICK\\\" for a in s.available_actions()):\\n                    # new board: hypothesise roles/relations from the frame and test them with one click per trigger candidate\\n                    fresh_level = False\\n                    res = probe.run_hypothesis_probe(min(cap, 12))\\n                else:\\n                    agent_ids = set()\\n                    if H and isinstance(H[0].model, RuleModel):\\n                        agent_ids = {o.id for o in H[0].model.with_roles(s.scene).objects if o.role and \\\"agent\\\" in o.role}\\n                    dirs = {int(k[6:]): (int(v[\\\"displacement\\\"][0]) and (v[\\\"displacement\\\"][0] > 0) - (v[\\\"displacement\\\"][0] < 0), (v[\\\"displacement\\\"][1] > 0) - (v[\\\"displacement\\\"][1] < 0))\\n                            for k, v in semantics.items() if k.startswith(\\\"ACTION\\\") and isinstance(v, dict) and v.get(\\\"class\\\") == \\\"MOVE\\\" and v.get(\\\"displacement\\\")}\\n                    step = max([abs(v[\\\"displacement\\\"][0]) + abs(v[\\\"displacement\\\"][1]) for k, v in semantics.items() if k.startswith(\\\"ACTION\\\") and isinstance(v, dict) and v.get(\\\"displacement\\\")] or [1])\\n                    dirs = {k: (d[0] * step, d[1] * step) for k, d in dirs.items()}\\n                    res = probe.run_walk(cap, agent_ids=agent_ids, dirs=dirs, bumped=bumped, clicked=clicked, click_map=knowledge.clicks)\\n                    walk_dry = walk_dry + 1 if res.actions_used == 0 else 0\\n                semantics = classify_actions(s.transitions, semantics)\\n                knowledge.clicks.extend(res.transitions); knowledge.save(self.memory.knowledge_path(game_id))\\n                if getattr(res, \\\"hypotheses\\\", None) is not None:\\n                    knowledge.apply_hypotheses(res)\\n                    events.emit(\\\"PROBE\\\", \\\"PROBE\\\", f\\\"hypotheses: {res.hypotheses.summary()}\\\", budget_used=budget.used())\\n                    for v in (res.verdicts or [])[:12]:\\n                        events.emit(\\\"PROBE\\\", \\\"PROBE\\\", f\\\"verdict: {v}\\\", budget_used=budget.used())\\n                self.memory.save_semantics(game_id, semantics.to_json())\\n                events.emit(\\\"PROBE\\\", \\\"HYPOTHESIZE\\\", f\\\"{kind} probe done: {res.actions_used} actions, steps {res.steps_done}\\\", budget_used=budget.used())\\n                state = \\\"HYPOTHESIZE\\\"; experiments_this_round = 0\\n            elif state == \\\"LOCAL_PROBE\\\":\\n                cap = budget.cap(\\\"level_probe\\\", s.level)\\n                res = probe.run_local(cap, novel_ids=getattr(self, \\\"_novel_ids\\\", []), mismatch=last_mismatch, navigate=None)\\n                semantics = classify_actions(s.transitions, semantics)\\n                events.emit(\\\"LOCAL_PROBE\\\", \\\"HYPOTHESIZE\\\", f\\\"local probe: {res.actions_used} actions\\\", budget_used=budget.used())\\n                state = \\\"HYPOTHESIZE\\\"; experiments_this_round = 0\\n            elif state == \\\"HYPOTHESIZE\\\":\\n                hypothesize_rounds += 1\\n                for t_ in s.transitions[len(planner.observed_seen):]:\\n                    planner.observed_seen.append(t_.id)\\n                    if t_.status_change is None and t_.action.type != \\\"RESET\\\":\\n                        planner.observed[(_state_key(t_.before), t_.action.label())] = t_.after\\n                level_note = self._level_note(mem, s.level)\\n                allow_llm = self.use_llm and budget.allows_llm() and not budget.low()\\n                if len(s.transitions) != last_refine_n or (wml._job is not None and not wml.job_running()):\\n                    semantics = classify_actions(s.transitions, semantics)\\n                    H = wml.refine(s.transitions, H, self.memory.priors(\\\"mechanisms\\\"), scene=s.scene, semantics=semantics, available=s.available_actions(),\\n                                   level_note=level_note, use_llm=False, knowledge=knowledge)\\n                    last_refine_n = len(s.transitions)\\n                    best = H[0] if H else None\\n                    if allow_llm and (best is None or best.score < 1.0 or best.coverage < 1.0) and not wml.job_running():\\n                        if wml.start_llm_job(s.transitions, best, self.memory.priors(\\\"mechanisms\\\"), scene=s.scene, semantics=semantics, level_note=level_note,\\n                                             K=(2 if budget.fraction_left() < 0.5 else 4)):\\n                            events.emit(\\\"HYPOTHESIZE\\\", \\\"HYPOTHESIZE\\\", f\\\"llm job started in background (round {wml.async_rounds}, log {len(s.transitions)})\\\", budget_used=budget.used())\\n                budget.llm_calls = wml.llm_calls + goal_inf.__dict__.get(\\\"llm_calls\\\", 0)\\n                roles_fn = (lambda sc, m=H[0].model: m.with_roles(sc)) if H and isinstance(H[0].model, RuleModel) else None\\n                G = goal_inf.refine(s.transitions, G, self.memory.priors(\\\"goals\\\"), s.level, s.scene, roles_fn=roles_fn, model=H[0].model if H else None)\\n                exp = None\\n                model_ok = bool(H) and H[0].usable(self.min_plan_score)\\n                if model_ok and not budget.low() and budget.allows(\\\"experiment\\\", level=s.level) and experiments_this_round < 6:\\n                    exp = wml.most_informative_action(H, s.scene, s.available_actions(), semantics=semantics, extra_clicks=planner.responsive, click_map=knowledge.clicks,\\n                                                      exclude=tried_experiments, state_key=_state_key(s.scene))   # never repeat an experiment from the same state\\n                if exp is not None:\\n                    tried_experiments.add((_state_key(s.scene), exp.label()))\\n                    t = s.act(exp, \\\"experiment\\\"); experiments_this_round += 1\\n                    events.emit(\\\"HYPOTHESIZE\\\", \\\"HYPOTHESIZE\\\", f\\\"experiment {exp.label()} (gain over {len(H)} hypotheses)\\\", transition_id=t.id if t else None, budget_used=budget.used())\\n                    continue\\n                if H and H[0].verified and H[0].code and H[0].origin != \\\"induced\\\":\\n                    self.memory.promote_model(game_id, H[0].code, {\\\"score\\\": H[0].score, \\\"coverage\\\": H[0].coverage, \\\"n\\\": len(s.transitions)})\\n                self.memory.save_hypotheses(game_id, [{\\\"name\\\": h.name, \\\"score\\\": h.score, \\\"coverage\\\": h.coverage, \\\"code\\\": h.code, \\\"origin\\\": h.origin} for h in H])\\n                top = f\\\"top h={H[0].name} score={H[0].score:.2f} chg={H[0].change_score:.2f} cov={H[0].coverage:.2f}\\\" if H else \\\"no hypothesis\\\"\\n                events.emit(\\\"HYPOTHESIZE\\\", \\\"PLAN\\\", f\\\"{top}; goals={len(G)} top={G[0].name if G else None}\\\", budget_used=budget.used())\\n                state = \\\"PLAN\\\"\\n            elif state == \\\"PLAN\\\":\\n                if wml._job is not None and not wml.job_running():\\n                    events.emit(\\\"PLAN\\\", \\\"HYPOTHESIZE\\\", \\\"llm job finished -> merge candidates\\\", budget_used=budget.used()); state = \\\"HYPOTHESIZE\\\"; continue\\n                if H and not H[0].usable(self.min_plan_score) and not H[0].verified:\\n                    # quality gate: a model that explains < 60% of the log is not worth executing plans on; gather evidence instead\\n                    if wml.job_running() and budget.cap(\\\"reprobe\\\", s.level) <= 0 and llm_waits_this_level < 1:\\n                        # one wait per level at most: after that the loop acts (goal-directed step) instead of idling\\n                        llm_waits_this_level += 1\\n                        events.emit(\\\"PLAN\\\", \\\"HYPOTHESIZE\\\", f\\\"model score {H[0].score:.2f}/change {H[0].change_score:.2f} below gate; waiting for the llm job\\\", budget_used=budget.used())\\n                        wml.wait_job(120.0); state = \\\"HYPOTHESIZE\\\"; continue\\n                    if budget.cap(\\\"reprobe\\\", s.level) > 0:\\n                        # walk while the click map can still learn; otherwise PROBE takes one goal-directed step (an action\\n                        # whose predicted outcome raises the top goal's progress) -- idling on the LLM is the last resort\\n                        why = \\\"walk probe\\\" if _walk_useful(knowledge, s) else \\\"goal-directed step\\\"\\n                        events.emit(\\\"PLAN\\\", \\\"PROBE\\\", f\\\"model score {H[0].score:.2f}/change {H[0].change_score:.2f} below gate -> {why}\\\", budget_used=budget.used()); state = \\\"PROBE\\\"; continue\\n                    # anti-repeat: after the same click 3x without a plan emerging, forbid it and pick another; if the\\n                    # ranked candidates are exhausted, fall back to a forced untried action (r11l hammered one click 33x)\\n                    avoid = frozenset({gd_last}) if gd_repeat >= 3 else frozenset()\\n                    a = _goal_directed_action(knowledge, s, H, G, last_action, avoid=avoid) if H else None\\n                    if a is not None and a.label() == gd_last and gd_repeat >= 3:\\n                        a = _forced_explore(knowledge, s, forced_tried) or a\\n                    if a is not None:\\n                        t_ = s.act(a, \\\"reprobe\\\")\\n                        if t_ is not None:\\n                            gd_repeat = gd_repeat + 1 if a.label() == gd_last else 0\\n                            gd_last = a.label(); last_action = a\\n                            events.emit(\\\"PLAN\\\", \\\"HYPOTHESIZE\\\", f\\\"below gate, reprobe spent -> goal-directed step {a.label()}\\\", transition_id=t_.id, budget_used=budget.used())\\n                            semantics = classify_actions(s.transitions, semantics); knowledge.clicks.extend(s.transitions[-3:])\\n                            state = \\\"HYPOTHESIZE\\\"; experiments_this_round = 0; continue\\n                    if wml.job_running():\\n                        llm_waits_this_level += 1\\n                        events.emit(\\\"PLAN\\\", \\\"HYPOTHESIZE\\\", \\\"below gate, reprobe spent -> waiting for the llm job\\\", budget_used=budget.used())\\n                        wml.wait_job(120.0); state = \\\"HYPOTHESIZE\\\"; continue\\n                    if not self.use_llm or not wml.job_running():\\n                        # nothing left to learn from cheaply: ask the LLM for a goal/model or let the low-quality plan through once\\n                        gated_rounds += 1\\n                        if gated_rounds > 3:\\n                            events.emit(\\\"PLAN\\\", \\\"PLAN\\\", \\\"gate lifted after 3 rounds without new evidence\\\", budget_used=budget.used())\\n                        else:\\n                            events.emit(\\\"PLAN\\\", \\\"HYPOTHESIZE\\\", \\\"below gate, reprobe spent, no llm job -> hypothesize\\\", budget_used=budget.used()); state = \\\"HYPOTHESIZE\\\"; continue\\n                if not H or not G:\\n                    if not H and s.transitions:\\n                        # nothing explains the log: fall back to probing with untried actions / clicks\\n                        state = \\\"PROBE\\\" if budget.cap(\\\"reprobe\\\", s.level) > 0 else \\\"HYPOTHESIZE\\\"\\n                        events.emit(\\\"PLAN\\\", state, \\\"no hypothesis\\\" if not H else \\\"no goal\\\", budget_used=budget.used())\\n                        if state == \\\"HYPOTHESIZE\\\" and hypothesize_rounds > 3:\\n                            stop = \\\"no_hypothesis\\\"; break\\n                        continue\\n                    if not G:\\n                        G = goal_inf.refine(s.transitions, G, self.memory.priors(\\\"goals\\\"), s.level, s.scene,\\n                                            roles_fn=(lambda sc, m=H[0].model: m.with_roles(sc)) if H and isinstance(H[0].model, RuleModel) else None)\\n                        if not G:\\n                            no_plan_rounds += 1\\n                            state = \\\"PROBE\\\" if no_plan_rounds > 1 else \\\"HYPOTHESIZE\\\"\\n                            events.emit(\\\"PLAN\\\", state, \\\"no goal candidates\\\", budget_used=budget.used())\\n                            continue\\n                plan = planner.search(s.scene, H, G, available=s.available_actions(), semantics=semantics, visited=planner.stuck.visited)\\n                if plan is None and H and not H[0].verified:\\n                    # exploration fallback (spec \\u00a710 'model unverified -> experiment'): reach an untouched object to gather evidence\\n                    roles_fn = (lambda sc, m=H[0].model: m.with_roles(sc)) if isinstance(H[0].model, RuleModel) else (lambda sc: sc)\\n                    from ..goal.templates import t_explore\\n                    ex = t_explore(roles_fn(s.scene), {\\\"touched\\\": touched})\\n                    if ex:\\n                        plan = planner.search(s.scene, H[:1], ex, available=s.available_actions(), semantics=semantics, visited=planner.stuck.visited)\\n                        if plan:\\n                            events.emit(\\\"PLAN\\\", \\\"EXECUTE\\\", f\\\"explore plan of {len(plan)} toward an untouched object\\\", budget_used=budget.used())\\n                            state = \\\"EXECUTE\\\"; exploring = True\\n                            continue\\n                if plan is None:\\n                    no_plan_rounds += 1\\n                    if H and H[0].verified and not getattr(H[0].model, \\\"level_scoped\\\", False):\\n                        # (a verified permutation table only knows the positions it saw: no plan there says nothing about the goal)\\n                        G = goal_inf.demote(G, 0)\\n                        reason = \\\"no plan under verified model -> goal demoted\\\"\\n                    elif no_plan_rounds % 2 == 0:\\n                        planner.depth = min(240, planner.depth * 2); reason = f\\\"no plan -> depth {planner.depth}\\\"\\n                    else:\\n                        reason = \\\"no plan under unverified model -> experiment\\\"\\n                    events.emit(\\\"PLAN\\\", \\\"HYPOTHESIZE\\\", reason, budget_used=budget.used())\\n                    state = \\\"HYPOTHESIZE\\\"\\n                    if no_plan_rounds >= 6:\\n                        if wml.job_running() and budget.cap(\\\"reprobe\\\", s.level) <= 0:\\n                            events.emit(\\\"PLAN\\\", \\\"HYPOTHESIZE\\\", \\\"no plan, reprobe spent -> waiting for the llm job\\\", budget_used=budget.used())\\n                            wml.wait_job(120.0); no_plan_rounds = 0; continue\\n                        if _walk_useful(knowledge, s):\\n                            events.emit(\\\"PLAN\\\", \\\"PROBE\\\", \\\"no plan after 6 rounds -> reprobe\\\", budget_used=budget.used()); state = \\\"PROBE\\\"; no_plan_rounds = 0\\n                        else:\\n                            walk_dry += 1\\n                            events.emit(\\\"PLAN\\\", \\\"HYPOTHESIZE\\\", \\\"no plan after 6 rounds, nothing new to click -> hypothesize\\\", budget_used=budget.used()); no_plan_rounds = 0\\n                    continue\\n                events.emit(\\\"PLAN\\\", \\\"EXECUTE\\\", f\\\"plan of {len(plan)}: {[a.label() for a in plan[:8]]}\\\", budget_used=budget.used())\\n                state = \\\"EXECUTE\\\"; exploring = False\\n            elif state == \\\"EXECUTE\\\":\\n                r = planner.execute(plan or [], H, s, G)\\n                goal_inf.observe_progress(G, s.scene)\\n                _mark_touched(touched, s.scene, H)\\n                if exploring and r.kind == \\\"EXHAUSTED\\\":\\n                    exploring = False\\n                    events.emit(\\\"EXECUTE\\\", \\\"HYPOTHESIZE\\\", f\\\"explore plan done ({r.executed} actions)\\\", budget_used=budget.used()); state = \\\"HYPOTHESIZE\\\"; continue\\n                if r.kind == \\\"PROGRESS\\\":\\n                    events.emit(\\\"EXECUTE\\\", \\\"PLAN\\\", f\\\"level up after {r.executed} actions\\\", transition_id=r.t.id if r.t else None, budget_used=budget.used())\\n                    won = planner.last_info.goal if planner.last_info else None\\n                    knowledge.record_win(last_level, won, s.step_idx - level_start_step)\\n                    # harvest class-level rules from every prior-composed model that verified on this level (the plan may\\n                    # have used an LLM model whose rules carry no class description)\\n                    n_rules = sum(knowledge.record_rules(h.model) for h in H if h.origin == \\\"induced\\\" and isinstance(h.model, RuleModel) and h.score / max(h.coverage, 1e-9) >= 0.9)\\n                    self.log(f\\\"knowledge: {n_rules} class-level click rule(s) harvested; asset now {list(knowledge.click_rules)}\\\")\\n                    if won is not None:\\n                        self.memory.record_usage(won.template, used=True, verified=True)\\n                    G = goal_inf.refine(s.transitions, G, self.memory.priors(\\\"goals\\\"), s.level, s.scene)\\n                    state = \\\"PLAN\\\"; last_mismatch = None\\n                elif r.kind == \\\"MISMATCH\\\":\\n                    last_mismatch = r.t\\n                    self._mark_violations(H, r.t)\\n                    events.emit(\\\"EXECUTE\\\", \\\"HYPOTHESIZE\\\", f\\\"prediction mismatch at step {r.executed}: {str(r.detail.get('mismatch'))[:120]}\\\",\\n                                transition_id=r.t.id if r.t else None, budget_used=budget.used())\\n                    state = \\\"HYPOTHESIZE\\\"; experiments_this_round = 0\\n                elif r.kind == \\\"STUCK\\\":\\n                    events.emit(\\\"EXECUTE\\\", \\\"PROBE\\\", f\\\"stuck: {r.detail.get('reason')}\\\", transition_id=r.t.id if r.t else None, budget_used=budget.used())\\n                    planner.stuck.reset(); state = \\\"PROBE\\\"\\n                elif r.kind == \\\"EXHAUSTED\\\":\\n                    reached = planner.last_info.goal if planner.last_info else None\\n                    if reached is not None and reached.template != \\\"explore\\\":\\n                        roles_fn = (lambda sc, m=H[0].model: m.with_roles(sc)) if H and isinstance(H[0].model, RuleModel) else (lambda sc: sc)\\n                        if reached.is_goal(roles_fn(s.scene)):\\n                            # the predicate holds and nothing happened: it is not the win condition (stronger than 'unreachable')\\n                            idx = next((i for i, g in enumerate(G) if g.name == reached.name), None)\\n                            if idx is not None:\\n                                goal_inf.demote(G, idx, penalty=-0.6)\\n                            knowledge.record_fail(reached)\\n                            events.emit(\\\"EXECUTE\\\", \\\"PLAN\\\", f\\\"plan exhausted ({r.executed} actions): goal {reached.name} reached without level-up -> demoted\\\", budget_used=budget.used())\\n                            state = \\\"PLAN\\\"; continue\\n                    events.emit(\\\"EXECUTE\\\", \\\"PLAN\\\", f\\\"plan exhausted ({r.executed} actions)\\\", budget_used=budget.used()); state = \\\"PLAN\\\"\\n                elif r.kind == \\\"FAILED\\\":\\n                    # GAME_OVER: the level restarts on RESET; retry from PLAN with memory kept (spec \\u00a712)\\n                    game_overs += 1\\n                    events.emit(\\\"EXECUTE\\\", \\\"PLAN\\\", f\\\"game over #{game_overs} -> RESET, retry from PLAN with memory kept\\\", transition_id=r.t.id if r.t else None, budget_used=budget.used())\\n                    if game_overs > 20:\\n                        stop = \\\"GAME_OVER\\\"; break\\n                    s.act(Action.reset(), \\\"plan\\\"); planner.stuck.reset(); touched.clear()\\n                    state = \\\"PLAN\\\"\\n                else:\\n                    events.emit(\\\"EXECUTE\\\", \\\"PLAN\\\", f\\\"execution error: {r.detail}\\\", budget_used=budget.used()); state = \\\"PLAN\\\"\\n                    if s.errors > 20:\\n                        stop = \\\"env_errors\\\"; break\\n        st = s.env.status()\\n        if not stop:\\n            stop = st.state if st.state in (\\\"WIN\\\", \\\"GAME_OVER\\\") else (\\\"timeout\\\" if s.timed_out() else \\\"budget\\\")\\n        events.emit(state, \\\"END\\\", stop, budget_used=budget.used())\\n        self.memory.save_semantics(game_id, semantics.to_json())\\n        knowledge.clicks.extend(s.transitions[-8:]); knowledge.demoted = dict(goal_inf.demoted); knowledge.save(self.memory.knowledge_path(game_id))\\n        self.memory.save()\\n        return Result(game_id, st.state, st.level - 1 if st.state != \\\"WIN\\\" else (st.levels_total or st.level), st.levels_total, st.actions_used, budget.snapshot(),\\n                      wml.llm_calls, len(events.events), stop, dict(s.level_actions),\\n                      [{\\\"name\\\": h.name, \\\"score\\\": round(h.score, 3), \\\"coverage\\\": round(h.coverage, 3), \\\"verified\\\": h.verified, \\\"origin\\\": h.origin} for h in H],\\n                      [g.to_json() for g in G[:5]], {k: v for k, v in semantics.items() if not k.startswith(\\\"_\\\")}, round(time.time() - t_start, 1))\\n\\n    # \\u2500\\u2500 helpers \\u2500\\u2500\\n    @staticmethod\\n    def _mark_violations(H: list[Hypothesis], t: Optional[Transition]) -> None:\\n        if t is None:\\n            return\\n        for h in H:\\n            if t.id not in h.violations:\\n                h.violations.append(t.id)\\n            if isinstance(h.model, RuleModel):\\n                s = h.model.with_roles(t.before)\\n                for r in h.model.rules():\\n                    try:\\n                        if r.applies(s, t.action):\\n                            r.confidence = 0.0; r.violated_by.append(t.id)\\n                    except Exception:\\n                        pass\\n\\n    @staticmethod\\n    def _level_note(mem, level: int) -> str:\\n        n = mem.level_notes.get(str(level - 1)) if mem.level_notes else None\\n        return f\\\"previous level note: {n}\\\" if n else \\\"\\\"\\n\\n\\ndef _mark_touched(touched: set, scene: Scene, H: list[Hypothesis]) -> None:\\n    \\\"\\\"\\\"Objects the agent overlaps or touches are 'touched' (for the exploration goal).\\\"\\\"\\\"\\n    if not H or not isinstance(H[0].model, RuleModel):\\n        return\\n    from ..goal.templates import _adjacent\\n    s = H[0].model.with_roles(scene)\\n    for a in [o for o in s.objects if o.role and \\\"agent\\\" in o.role]:\\n        for o in s.objects:\\n            if o.id != a.id and (a.overlaps(o) or _adjacent(a, o)):\\n                touched.add(o.identity())\\n\\n\\ndef _state_key(scene: Scene) -> tuple:\\n    \\\"\\\"\\\"State identity for experiment bookkeeping: objects outside ui strips (counters must not make states look new).\\\"\\\"\\\"\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    return tuple(sorted(o.identity() for o in scene.objects if o.region not in strips))\\n\\n\\ndef _walk_useful(knowledge, s) -> bool:\\n    \\\"\\\"\\\"A walk probe is worth its actions only when the click map still has an untried class on this scene, or a\\n    responsive class not yet re-checked on this level; button games always walk (buttons are cheap to re-press).\\\"\\\"\\\"\\n    if not any(a.type == \\\"CLICK\\\" for a in s.available_actions()) or any(a.type == \\\"BUTTON\\\" for a in s.available_actions()):\\n        return True\\n    from ..probe.clickmap import click_key\\n    cmap = knowledge.clicks\\n    tried_here = cmap.tried_level.get(s.level, set())\\n    # clicks per trigger OBJECT on this level (two buttons of one class are two triggers to learn)\\n    per_obj: dict = {}\\n    for t in s.level_log():\\n        if t.action.type == \\\"CLICK\\\":\\n            for o in t.before.objects:\\n                if o.bbox[0] <= t.action.row < o.bbox[2] and o.bbox[1] <= t.action.col < o.bbox[3]:\\n                    per_obj[tuple(o.bbox)] = per_obj.get(tuple(o.bbox), 0) + 1\\n    for a in cmap.rank(s.scene, level=s.level, include_inert=False, limit=64):\\n        key = click_key(s.scene, a.row, a.col)\\n        st = cmap.status(key)\\n        if st == \\\"untried\\\" or (st == \\\"responsive\\\" and key not in tried_here):\\n            return True\\n        if st == \\\"responsive\\\":\\n            box = next((tuple(o.bbox) for o in s.scene.objects if o.bbox[0] <= a.row < o.bbox[2] and o.bbox[1] <= a.col < o.bbox[3]), None)\\n            if box is not None and per_obj.get(box, 0) < 2:\\n                return True    # induction needs repeated observations of each reacting trigger (one click is not a rule)\\n    return False\\n\\n\\ndef _forced_explore(knowledge, s, tried: set):\\n    \\\"\\\"\\\"Anti-spin last resort: one untried action to break a no-plan loop and generate fresh evidence. Untried click\\n    classes first (ranked, most promising), then any simple/button action not yet forced this level. None when every\\n    action class has been exhausted -> the caller ends the level rather than spinning the CPU.\\\"\\\"\\\"\\n    from ..probe.clickmap import click_key\\n    if any(a.type == \\\"CLICK\\\" for a in s.available_actions()):\\n        for a in knowledge.clicks.rank(s.scene, level=s.level, include_inert=True, limit=64):\\n            if a.type != \\\"CLICK\\\":\\n                continue\\n            key = (\\\"C\\\", click_key(s.scene, a.row, a.col))\\n            if key in tried:\\n                continue\\n            if knowledge.clicks.status(click_key(s.scene, a.row, a.col)) == \\\"untried\\\":\\n                tried.add(key); return a\\n    for a in s.available_actions():\\n        if a.type != \\\"CLICK\\\":\\n            key = (\\\"B\\\", a.id)\\n            if key not in tried:\\n                tried.add(key); return a\\n    return None\\n\\n\\ndef _goal_directed_action(knowledge, s, H, G, last_action, avoid: frozenset = frozenset()):\\n    \\\"\\\"\\\"The responsive action (click map) whose predicted next state has the highest top-goal progress; unknown\\n    outcomes count as informative (progress 0.5); the inverse of the last action (same objects moved back) is skipped.\\n    `avoid` holds action labels to skip -- used to stop hammering the same click when it is not making progress.\\\"\\\"\\\"\\n    from ..probe.clickmap import click_key\\n    if not G:\\n        return None\\n    goal = G[0]; model = H[0].model\\n    cands = [a for a in s.available_actions() if a.type == \\\"BUTTON\\\"] + knowledge.clicks.rank(s.scene, level=s.level, include_inert=False, limit=16)\\n    cands = [a for a in cands if a.label() not in avoid] or cands\\n    if not cands:\\n        return None\\n    scored = []\\n    for a in cands:\\n        if a.type == \\\"CLICK\\\" and knowledge.clicks.status(click_key(s.scene, a.row, a.col)) == \\\"untried\\\":\\n            scored.append((0.6, a)); continue\\n        try:\\n            nxt = model.predict(s.scene, a)\\n        except Exception:\\n            nxt = None\\n        if nxt is None:\\n            scored.append((0.5, a)); continue\\n        if nxt is s.scene or _state_key(nxt) == _state_key(s.scene):\\n            continue                       # predicted no-op: nothing to learn or gain\\n        if last_action is not None:\\n            try:\\n                back = model.predict(nxt, last_action)\\n            except Exception:\\n                back = None\\n            if back is not None and _state_key(back) == _state_key(s.scene) and a.label() != last_action.label():\\n                pass                       # a cancels last_action: allowed only if it raises progress (checked below)\\n        scored.append((goal.progress(nxt), a))\\n    if not scored:\\n        return None\\n    scored.sort(key=lambda x: -x[0])\\n    best = scored[0][1]\\n    if last_action is not None and best.label() != last_action.label():\\n        # never undo the previous step unless it is the only option\\n        try:\\n            nxt = model.predict(s.scene, best)\\n            if nxt is not None and last_action is not None:\\n                undo = model.predict(nxt, last_action)\\n                if undo is not None and _state_key(undo) == _state_key(s.scene) and len(scored) > 1:\\n                    best = scored[1][1]\\n        except Exception:\\n            pass\\n    return best\\n\", \"pbg/orchestrator/session.py\": \"\\\"\\\"\\\"orchestrator/session.py \\u2014 one game session: env + perception + memory + budget glue. `act()` is the single place\\nan action is executed: it parses the frame, builds the Transition, appends it to memory, charges the budget and keeps\\nthe current scene. Probe and Planner only see this object.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport time\\nfrom typing import Optional\\n\\nfrom ..core.budget import Budget\\nfrom ..core.types import Action, Scene, Transition\\nfrom ..env.calibrate import ClickMonitor\\nfrom ..env.wrapper import EnvWrapper\\nfrom ..memory.store import Memory\\nfrom ..perception import Perception\\n\\n\\nclass Session:\\n    def __init__(self, env: EnvWrapper, game_id: str, perception: Perception, memory: Memory, budget: Budget, *, log=None,\\n                 max_seconds: Optional[float] = None, deadline: Optional[float] = None):\\n        self.env, self.game_id, self.perception, self.memory, self.budget = env, game_id, perception, memory, budget\\n        self.log = log or (lambda *a, **k: None)\\n        self.scene: Optional[Scene] = None\\n        self.frame = None\\n        self.level = 1\\n        self.step_idx = 0\\n        self.transitions: list[Transition] = []\\n        self.level_actions: dict[int, int] = {}\\n        self.click_monitor = ClickMonitor()\\n        self.t0 = time.time()\\n        self.deadline = deadline or (self.t0 + max_seconds if max_seconds else None)\\n        self.reset_allowed = True\\n        self.errors = 0\\n        self.actions_since_reset = 0\\n        self.initial_click_cap = int((budget.cfg.get(\\\"per_level\\\") or {}).get(\\\"initial_clicks\\\", 16))\\n\\n    # \\u2500\\u2500 lifecycle \\u2500\\u2500\\n    def start(self) -> Scene:\\n        self.frame = self.env.reset(self.game_id)\\n        self.level = self.env.status().level\\n        self.scene = self.perception.parse(self.frame, None)\\n        self.budget.sync(self.env.status().actions_used)\\n        return self.scene\\n\\n    def status(self):\\n        return self.env.status()\\n\\n    def finished(self) -> bool:\\n        st = self.env.status().state\\n        return st == \\\"WIN\\\" or self.budget.remaining() <= 0 or self.timed_out()\\n\\n    def timed_out(self) -> bool:\\n        return self.deadline is not None and time.time() > self.deadline\\n\\n    def available_actions(self) -> list[Action]:\\n        return self.env.available_actions()\\n\\n    def actions_used(self) -> int:\\n        return self.env.status().actions_used\\n\\n    # \\u2500\\u2500 the one action path \\u2500\\u2500\\n    def act(self, action: Action, kind: str = \\\"plan\\\") -> Optional[Transition]:\\n        if self.budget.remaining() <= 0 or self.timed_out():\\n            return None\\n        if action.type == \\\"RESET\\\":\\n            if self.actions_since_reset == 0:\\n                self.log(\\\"RESET refused: no action since the last reset (a second RESET would restart the whole game)\\\")\\n                return None\\n            self.actions_since_reset = 0\\n            level = self.level\\n            frame = self.env.reset(self.game_id)\\n            self.budget.charge(kind, 1, level)\\n            before = self.scene\\n            self.frame = frame\\n            self.scene = self.perception.parse(frame, None)   # tracking restarts after a reset\\n            self.step_idx += 1\\n            t = self.perception.transition(before, action, self.scene, before_frame=None, after_frame=frame, status_change=None,\\n                                           tid=f\\\"{self.game_id}:{level}:{self.step_idx}\\\", level=level, step_idx=self.step_idx)\\n            self.transitions.append(t); self.memory.append(t, self.game_id)\\n            self.level = self.env.status().level\\n            self.budget.sync(self.env.status().actions_used)\\n            return t\\n        if self.env.status().state == \\\"GAME_OVER\\\":\\n            return None          # the caller must RESET first\\n        rt = self.env.step(action)\\n        if rt.error:\\n            self.errors += 1\\n            self.log(f\\\"env error on {action.label()}: {rt.error}\\\")\\n            if \\\"not available\\\" in rt.error:\\n                return None\\n            return None\\n        level = rt.level\\n        self.actions_since_reset += 1\\n        self.budget.charge(kind, 1, level)\\n        self.level_actions[level] = self.level_actions.get(level, 0) + 1\\n        self.step_idx += 1\\n        before = self.scene\\n        after = self.perception.parse(rt.after, before)\\n        t = self.perception.transition(before, action, after, before_frame=rt.before, after_frame=rt.after, status_change=rt.status_change,\\n                                       intermediate=rt.intermediate_frames, tid=f\\\"{self.game_id}:{level}:{self.step_idx}\\\", level=level, step_idx=self.step_idx)\\n        if self.perception.merge_confirmed:\\n            self.perception.refit(self.transitions)\\n            after = t.after\\n        self.transitions.append(t); self.memory.append(t, self.game_id)\\n        if self.perception.reparse_needed:\\n            # a background panel changed extent: it is a resizable bar; re-parse this level so every scene has it as an object\\n            self.perception.reparse_needed = False\\n            lv = [x for x in self.transitions if x.level == level]\\n            re = self.perception.reparse_level(lv)\\n            if re is not None:\\n                after = re\\n        self.click_monitor.observe(t)\\n        self.budget.sync(self.env.status().actions_used)\\n        self.scene, self.frame = after, rt.after\\n        if rt.status_change == \\\"LEVEL_UP\\\":\\n            self.level = self.env.status().level\\n            self.actions_since_reset = 0        # a RESET on a fresh level restarts the whole game: treat the new level like a fresh reset\\n            # new level: tracking ids restart with the new board\\n            self.scene = self.perception.parse(rt.after, None)\\n        return t\\n\\n    def level_log(self, level: Optional[int] = None) -> list[Transition]:\\n        lv = self.level if level is None else level\\n        return [t for t in self.transitions if t.level == lv]\\n\", \"pbg/orchestrator/states.py\": \"\\\"\\\"\\\"orchestrator/states.py \\u2014 the four states and the events that move between them (spec \\u00a712).\\\"\\\"\\\"\\nSTATES = (\\\"PROBE\\\", \\\"HYPOTHESIZE\\\", \\\"PLAN\\\", \\\"EXECUTE\\\")\\nTERMINAL = (\\\"WIN\\\", \\\"GAME_OVER\\\", \\\"BUDGET\\\", \\\"UNRESOLVED\\\", \\\"TIMEOUT\\\")\\n\", \"pbg/orchestrator/transfer.py\": \"\\\"\\\"\\\"orchestrator/transfer.py \\u2014 level-to-level and game-to-game transfer (spec \\u00a713).\\n\\n* level novelty = new colours + new shape signatures + region-structure change (0/1)\\n* game transfer: structure vector of the first scene -> cosine similarity with games in semantic memory -> the top-2\\n  games' verified models become initial hypotheses with confidence 0.3 (discarded on their first failed verification)\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom typing import Optional\\n\\nfrom ..core.contracts import Hypothesis\\nfrom ..core.types import Scene\\n\\n\\ndef structure_vector(scene: Scene) -> list[float]:\\n    strips = [r for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"]\\n    kinds = {\\\"board\\\": 0, \\\"panel\\\": 0, \\\"ui_strip\\\": 0, \\\"unknown\\\": 0}\\n    for r in scene.regions:\\n        kinds[r.kind_hint] = kinds.get(r.kind_hint, 0) + 1\\n    hist = [0.0] * 16\\n    for o in scene.objects:\\n        hist[o.color & 15] += 1\\n    tot = sum(hist) or 1.0\\n    return [len(scene.regions), kinds[\\\"board\\\"], kinds[\\\"panel\\\"], kinds[\\\"ui_strip\\\"], len(scene.objects) / 10.0, float(bool(strips)),\\n            scene.grid_shape[0] / 64.0, scene.grid_shape[1] / 64.0] + [h / tot for h in hist]\\n\\n\\ndef cosine(a: list[float], b: list[float]) -> float:\\n    num = sum(x * y for x, y in zip(a, b)); da = math.sqrt(sum(x * x for x in a)); db = math.sqrt(sum(y * y for y in b))\\n    return num / (da * db) if da and db else 0.0\\n\\n\\ndef level_novelty(prev: Optional[Scene], cur: Scene) -> tuple[int, list[int]]:\\n    if prev is None:\\n        return 0, []\\n    strips = {r.id for r in cur.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    pc = {o.color for o in prev.objects}; ps = {o.shape_sig for o in prev.objects}\\n    new_colors = {o.color for o in cur.objects if o.region not in strips} - pc\\n    new_shapes = {o.shape_sig for o in cur.objects if o.region not in strips} - ps\\n    region_change = int(sorted((r.kind_hint, r.bg_color) for r in prev.regions) != sorted((r.kind_hint, r.bg_color) for r in cur.regions))\\n    novel_ids = [o.id for o in cur.objects if (o.color in new_colors or o.shape_sig in new_shapes) and o.region not in strips]\\n    return len(new_colors) + len(new_shapes) + region_change, novel_ids\\n\\n\\ndef transfer_hypotheses(memory, game_id: str, scene: Scene, sandbox, *, log=None, top_k: int = 2) -> list[Hypothesis]:\\n    log = log or (lambda *a, **k: None)\\n    out: list[Hypothesis] = []\\n    mem = memory.load(game_id)\\n    # 1. this game's own verified model / unpromoted hypotheses (retry)\\n    if mem.world_model_code:\\n        m = sandbox.load(mem.world_model_code)\\n        if m is not None:\\n            out.append(Hypothesis(m, 1.0, 1.0, [], mem.world_model_code, \\\"memory:verified\\\", False, \\\"\\\", \\\"llm\\\"))   # re-verified on this run's log before it counts as verified\\n    for h in mem.hypotheses[:3]:\\n        if h.get(\\\"code\\\") and \\\"def build_model\\\" in h[\\\"code\\\"]:\\n            m = sandbox.load(h[\\\"code\\\"])\\n            if m is not None:\\n                out.append(Hypothesis(m, float(h.get(\\\"score\\\", 0)), float(h.get(\\\"coverage\\\", 0)), [], h[\\\"code\\\"], f\\\"memory:{h.get('name')}\\\", False, \\\"\\\", h.get(\\\"origin\\\", \\\"llm\\\")))\\n    # 2. similar games (cosine similarity of the structure vectors)\\n    vec = structure_vector(scene)\\n    sims = []\\n    for other in memory.semantic_games():\\n        if other == game_id:\\n            continue\\n        v = memory.structure_vector(other)\\n        if v:\\n            sims.append((cosine(vec, v), other))\\n    sims.sort(reverse=True)\\n    for sim, other in sims[:top_k]:\\n        code = memory.load(other).world_model_code\\n        if code:\\n            m = sandbox.load(code)\\n            if m is not None:\\n                out.append(Hypothesis(m, 0.3, 0.0, [], code, f\\\"transfer:{other}:{sim:.2f}\\\", False, \\\"\\\", \\\"transferred\\\"))\\n                log(f\\\"transfer from {other} (sim {sim:.2f})\\\")\\n    return out\\n\", \"pbg/perception/__init__.py\": \"\\\"\\\"\\\"perception/ \\u2014 pixels -> Scene (regions + tracked objects), Diff, summaries. Deterministic, no LLM (spec \\u00a76).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport time\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nimport numpy as np\\nimport yaml\\n\\nfrom ..core.types import Frame, Object, Scene, Transition\\nfrom .diff import compute_diff\\nfrom .regions import find_regions, stabilize_regions\\nfrom .render import grid_png, render_scene\\nfrom .segment import fuse, rect_composites, segment_objects\\nfrom .summarize import scene_text, summarize\\nfrom .track import Tracker\\n\\nDEFAULT_CONFIG = {\\\"region_min_area_ratio\\\": 0.05, \\\"connectivity\\\": 4, \\\"merge_confirm_count\\\": 3, \\\"track_cost_max\\\": 20.0,\\n                  \\\"noop_pixel_threshold\\\": 0, \\\"ui_strip_max_thickness\\\": 3}\\n\\n\\ndef load_perception_config(path: Optional[Path] = None) -> dict:\\n    cfg = dict(DEFAULT_CONFIG)\\n    p = path or Path(__file__).resolve().parents[1] / \\\"configs\\\" / \\\"perception.yaml\\\"\\n    if p.exists():\\n        cfg.update(yaml.safe_load(p.read_text()) or {})\\n    return cfg\\n\\n\\nclass Perception:\\n    \\\"\\\"\\\"Stateful per game: keeps the tracker, the merge candidates and the confirmed merge log.\\\"\\\"\\\"\\n\\n    def __init__(self, config: Optional[dict] = None):\\n        self.cfg = {**load_perception_config(), **(config or {})}\\n        self.tracker = Tracker(float(self.cfg[\\\"track_cost_max\\\"]))\\n        self._pair_moves: dict[tuple[int, int], int] = {}   # (id_a, id_b) -> co-movement count\\n        self._merged: list[tuple[int, int]] = []             # confirmed merges (id_a, id_b)\\n        self._groups: list[set[int]] = []\\n        self._solo: set[int] = set()                         # part ids seen moving on their own: never part of a composite\\n        self._dynamic: set[tuple] = set()                    # (colour, axis, lo, hi): panels seen changing extent -> objects, not regions\\n        self.reparse_needed = False                          # a panel was just found dynamic: callers re-parse the level's transitions\\n        self.timings: list[float] = []\\n        self._backdrop: Optional[np.ndarray] = None       # per level: colour last seen at each uncovered cell (Scene.backdrop)\\n        self._ever_moved: set[int] = set()                # per level: tracking ids that changed position at least once\\n        self.merge_confirmed = False                         # a merge was confirmed OR a composite dissolved: callers refit past transitions\\n\\n    # \\u2500\\u2500 main entry points \\u2500\\u2500\\n    def parse(self, frame: Frame, prev: Optional[Scene]) -> Scene:\\n        t0 = time.perf_counter()\\n        grid = np.asarray(frame.grid, dtype=np.int8)\\n        regions, global_bg, meta = find_regions(grid, min_area_ratio=float(self.cfg[\\\"region_min_area_ratio\\\"]),\\n                                               ui_strip_max_thickness=int(self.cfg[\\\"ui_strip_max_thickness\\\"]))\\n        raw_regions = [(r.bg_color, tuple(r.bbox)) for r in regions if r.id != \\\"R0\\\" and r.kind_hint != \\\"ui_strip\\\"]\\n        regions = [r for r in regions if not self._is_dynamic(r)]\\n        if prev is not None:\\n            regions = stabilize_regions(regions, [r for r in prev.regions if not self._is_dynamic(r)])\\n        min_area = max(4, int(float(self.cfg[\\\"region_min_area_ratio\\\"]) * grid.shape[0] * grid.shape[1]))\\n        objs, adj = segment_objects(grid, regions, global_bg, meta[\\\"region_masks\\\"], connectivity=int(self.cfg[\\\"connectivity\\\"]), min_region_area=min_area)\\n        prev_parts = [p for o in prev.objects for p in (o.parts or [o])] if prev else None   # fused objects are tracked by their parts\\n        assign = self.tracker.match(prev_parts, objs)\\n        tracked: list[Object] = []\\n        for i, o in enumerate(objs):\\n            o.id = assign[i]; tracked.append(o)\\n        # merge candidates: adjacent different-colour components, by tracking id\\n        cand = {(min(assign[a], assign[b]), max(assign[a], assign[b])) for a, b in adj}\\n        # rectangle composites (target pattern, canvas, framed button): fused right away, dissolved if a part moves alone\\n        grouped = {i for g in self._groups for i in g}\\n        tracked = self._fuse_composites(tracked, [(assign[a], assign[b]) for a, b in adj], grid, regions, grouped)\\n        # confirmed merges: fuse groups whose members are all present\\n        if self._groups:\\n            by_id = {o.id: o for o in tracked}\\n            fused: list[Object] = []; consumed: set[int] = set()\\n            for grp in self._groups:\\n                members = [by_id[i] for i in grp if i in by_id]\\n                if len(members) >= 2:\\n                    fused.append(fuse(members, grid, regions)); consumed |= {m.id for m in members}\\n            tracked = [o for o in tracked if o.id not in consumed] + fused\\n        tracked.sort(key=lambda o: o.id)\\n        if prev is None:\\n            self._ever_moved = set()\\n        all_cover = np.zeros(grid.shape, dtype=bool)\\n        def paint(cover, o):\\n            r0, c0, r1, c1 = o.bbox\\n            rr0, cc0, rr1, cc1 = max(r0, 0), max(c0, 0), min(r1, grid.shape[0]), min(c1, grid.shape[1])\\n            if rr1 > rr0 and cc1 > cc0:\\n                cover[rr0:rr1, cc0:cc1] |= o.mask[rr0 - r0:rr1 - r0, cc0 - c0:cc1 - c0]\\n        for o in tracked:\\n            paint(all_cover, o)\\n        movers: set[int] = set()\\n        if prev is not None:\\n            prev_by_id = {o.id: o for o in prev.objects}\\n            for o in tracked:\\n                p_ = prev_by_id.get(o.id)\\n                if p_ is None:\\n                    movers.add(o.id)                        # appeared: not scenery until it proves static\\n                elif tuple(p_.bbox) != tuple(o.bbox):\\n                    self._ever_moved.add(o.id)\\n            movers |= self._ever_moved\\n            mover_cover_now = np.zeros(grid.shape, dtype=bool)     # hidden = under a MOVER (a static object split by one is not hidden)\\n            for o in tracked:\\n                if o.id in movers:\\n                    paint(mover_cover_now, o)\\n            # a static object (never moved on this level) whose pixels all lie under other objects now is occluded, not gone:\\n            # it stays in the scene at its position so a predicted scene shows it again once the mover leaves (docs/029)\\n            present = {o.id for o in tracked}\\n            for p_ in prev.objects:\\n                if p_.id in present or p_.id in self._ever_moved:\\n                    continue\\n                r0, c0, r1, c1 = p_.bbox\\n                rr0, cc0, rr1, cc1 = max(r0, 0), max(c0, 0), min(r1, grid.shape[0]), min(c1, grid.shape[1])\\n                if rr1 <= rr0 or cc1 <= cc0:\\n                    continue\\n                m = p_.mask[rr0 - r0:rr1 - r0, cc0 - c0:cc1 - c0]\\n                if m.any() and bool(np.all(mover_cover_now[rr0:rr1, cc0:cc1][m])):\\n                    o = p_.moved(0, 0); o.occluded = True; o.role = None; tracked.append(o)\\n            tracked.sort(key=lambda o: o.id)\\n        # backdrop = the colour last seen at each cell while no MOVING object covered it: static scenery (floors, corridors,\\n        # walls, decorations) is recorded, cells under a mover keep what was there before it arrived. The first frame of a\\n        # level records only uncovered cells (nothing has moved yet, so any object may be a mover). A predicted scene paints vacated cells from it.\\n        if prev is None or self._backdrop is None or self._backdrop.shape != grid.shape:\\n            # first frame of a level: nothing has moved yet, so what is under any object is unknown (-1 -> region colour)\\n            self._backdrop = grid.astype(np.int8).copy(); self._backdrop[all_cover] = -1\\n        else:\\n            mover_cover = np.zeros(grid.shape, dtype=bool)\\n            for o in tracked:\\n                if o.id in movers:\\n                    paint(mover_cover, o)\\n            self._backdrop[~mover_cover] = grid[~mover_cover]\\n        scene = Scene(frame.hash, tuple(grid.shape), regions, tracked, {\\\"_global_bg\\\": global_bg, \\\"_merge_candidates\\\": sorted(cand), \\\"_raw_regions\\\": raw_regions},\\n                      self._backdrop.copy())\\n        self.timings.append(time.perf_counter() - t0)\\n        return scene\\n\\n    def diff(self, before: Scene, after: Scene, before_grid: Optional[np.ndarray] = None, after_grid: Optional[np.ndarray] = None):\\n        d = compute_diff(before, after, before_grid=before_grid, after_grid=after_grid, noop_pixel_threshold=int(self.cfg[\\\"noop_pixel_threshold\\\"]))\\n        self.merge_confirmed = self._observe_comovement(before, d)\\n        return d\\n\\n    def _observe_comovement(self, before: Scene, d) -> bool:\\n        \\\"\\\"\\\"Merge confirmation (spec \\u00a76 step 3): adjacent pairs that moved by the same displacement 3 times become one object.\\n        Returns True when a new merge was confirmed.\\\"\\\"\\\"\\n        if not d.moved:\\n            return False\\n        reshaped = {x[0] for x in d.reshaped}\\n        moved = {i: v for i, v in d.moved if i not in reshaped}     # a bar whose end moved with the mover was resized, not carried\\n        new = False\\n        for a, b in before.aux.get(\\\"_merge_candidates\\\", []):\\n            if a in moved and b in moved and moved[a] == moved[b]:\\n                self._pair_moves[(a, b)] = self._pair_moves.get((a, b), 0) + 1\\n                if self._pair_moves[(a, b)] >= int(self.cfg[\\\"merge_confirm_count\\\"]) and (a, b) not in self._merged:\\n                    self._merged.append((a, b)); self._add_group(a, b); new = True\\n        return new\\n\\n    def _is_dynamic(self, r) -> bool:\\n        if r.id == \\\"R0\\\" or r.kind_hint == \\\"ui_strip\\\" or not self._dynamic:\\n            return False\\n        r0, c0, r1, c1 = r.bbox\\n        for color, axis, lo, hi in self._dynamic:\\n            if color != r.bg_color:\\n                continue\\n            a, b = (r0, r1) if axis == \\\"row\\\" else (c0, c1)\\n            ov = min(b, hi) - max(a, lo)\\n            if ov > 0 and ov >= 0.5 * min(b - a, hi - lo):\\n                return True      # same bar band (a cap drawn over its end may clip the extent seen this frame)\\n        return False\\n\\n    def _detect_dynamic(self, before: Scene, after: Scene) -> set[tuple]:\\n        \\\"\\\"\\\"Raw background panels whose extent changed between two consecutive frames (same colour, same fixed axis) are\\n        resizable bars: they carry state and must be objects.\\\"\\\"\\\"\\n        new: set[tuple] = set()\\n        rb = before.aux.get(\\\"_raw_regions\\\", []); ra = after.aux.get(\\\"_raw_regions\\\", [])\\n        h, w = before.grid_shape\\n        for color, (r0, c0, r1, c1) in rb:\\n            if (r1 - r0) * (c1 - c0) > 0.25 * h * w:\\n                continue          # a board whose fringe changes when bars grow into it is still the board\\n            same = [(s0, d0, s1, d1) for col, (s0, d0, s1, d1) in ra if col == color and not (s1 <= r0 or r1 <= s0 or d1 <= c0 or c1 <= d0)]\\n            if not same:\\n                continue\\n            for (s0, d0, s1, d1) in same:\\n                if (s0, s1) == (r0, r1) and (d0, d1) != (c0, c1):\\n                    new.add((color, \\\"row\\\", r0, r1))\\n                elif (d0, d1) == (c0, c1) and (s0, s1) != (r0, r1):\\n                    new.add((color, \\\"col\\\", c0, c1))\\n        return new - self._dynamic\\n\\n    def reparse_level(self, transitions: list[Transition]):\\n        \\\"\\\"\\\"Re-parse a level's transitions from their frames (after a panel turned out to be dynamic) with tracking run\\n        again in order. Returns the final scene (the caller's current scene).\\\"\\\"\\\"\\n        scene = None\\n        for t in transitions:\\n            if t.before_frame is None or t.action.type == \\\"RESET\\\":\\n                scene = self.parse(t.after_frame, None) if t.after_frame is not None else scene\\n                if scene is not None:\\n                    t.after = scene\\n                continue\\n            before = scene if (scene is not None and scene.frame_hash == t.before_frame.hash) else self.parse(t.before_frame, None)\\n            after = self.parse(t.after_frame, before)\\n            d = compute_diff(before, after, before_grid=t.before_frame.grid, after_grid=t.after_frame.grid, noop_pixel_threshold=int(self.cfg[\\\"noop_pixel_threshold\\\"]))\\n            t.before, t.after, t.diff = before, after, d\\n            scene = after\\n        return scene\\n\\n    def _fuse_composites(self, objs: list[Object], adj, grid, regions, extra_excluded=()) -> list[Object]:\\n        comps = rect_composites(objs, adj, regions, self._solo | set(extra_excluded), min_side=int(self.cfg.get(\\\"composite_min_side\\\", 2)),\\n                                min_area=int(self.cfg.get(\\\"composite_min_area\\\", 6)), min_share=float(self.cfg.get(\\\"composite_min_share\\\", 0.2)))\\n        if not comps:\\n            return objs\\n        consumed = {m.id for c in comps for m in c}\\n        fused = []\\n        for c in comps:\\n            f = fuse(c, grid, regions); f.composite = True; fused.append(f)\\n        return sorted([o for o in objs if o.id not in consumed] + fused, key=lambda o: o.id)\\n\\n    def _dissolve(self, scene: Scene) -> Scene:\\n        \\\"\\\"\\\"Break composites that contain a part seen moving on its own; the remaining parts may re-form a composite.\\\"\\\"\\\"\\n        if not self._solo or not any(o.composite and any(p.id in self._solo for p in o.parts) for o in scene.objects):\\n            return scene\\n        objs: list[Object] = []; freed: list[Object] = []\\n        for o in scene.objects:\\n            if o.composite and any(p.id in self._solo for p in o.parts):\\n                freed.extend(o.parts)\\n            else:\\n                objs.append(o)\\n        objs = self._fuse_composites(objs + freed, None, None, scene.regions)\\n        return scene.copy(objects=sorted(objs, key=lambda o: o.id))\\n\\n    def _solo_movers(self, before: Scene, after: Scene) -> set[int]:\\n        \\\"\\\"\\\"Parts of a composite that moved while another part of the same composite stayed put.\\\"\\\"\\\"\\n        def parts_map(scene):\\n            return {p.id: p for o in scene.objects for p in (o.parts or [o])}\\n        pb, pa = parts_map(before), parts_map(after)\\n        new: set[int] = set()\\n        for scene_c, other in ((before, pa), (after, pb)):\\n            for o in scene_c.objects:\\n                if not o.composite:\\n                    continue\\n                disp = {}; changed = set()\\n                for p in o.parts:\\n                    q = other.get(p.id)\\n                    if q is None:\\n                        continue\\n                    if q.shape_sig == p.shape_sig and q.area == p.area:\\n                        disp[p.id] = (q.bbox[0] - p.bbox[0], q.bbox[1] - p.bbox[1])\\n                    else:\\n                        changed.add(p.id)          # resized / redrawn part (a bar next to a button is not a pattern half)\\n                if len(disp) >= 2 and any(v == (0, 0) for v in disp.values()):\\n                    new |= {i for i, v in disp.items() if v != (0, 0)}\\n                if changed and any(v == (0, 0) for v in disp.values()):\\n                    new |= changed\\n        return new - self._solo\\n\\n    def apply_groups(self, scene: Scene) -> Scene:\\n        \\\"\\\"\\\"Fuse confirmed groups inside an already-parsed scene (used to refit earlier scenes after a confirmation).\\\"\\\"\\\"\\n        if not self._groups:\\n            return scene\\n        by_id = {o.id: o for o in scene.objects}\\n        fused: list[Object] = []; consumed: set[int] = set()\\n        for grp in self._groups:\\n            members = [by_id[i] for i in grp if i in by_id]\\n            if len(members) >= 2:\\n                fused.append(fuse(members, None, scene.regions)); consumed |= {m.id for m in members}\\n        if not consumed:\\n            return scene\\n        objs = sorted([o for o in scene.objects if o.id not in consumed] + fused, key=lambda o: o.id)\\n        return scene.copy(objects=objs)\\n\\n    def refit(self, transitions: list[Transition]) -> int:\\n        \\\"\\\"\\\"Re-fuse confirmed groups in past transitions (spec \\u00a715: reparse past scenes after a merge). Returns #changed.\\\"\\\"\\\"\\n        n = 0\\n        for t in transitions:\\n            b, a = self._dissolve(self.apply_groups(t.before)), self._dissolve(self.apply_groups(t.after))\\n            if len(b.objects) != len(t.before.objects) or len(a.objects) != len(t.after.objects):\\n                t.before, t.after = b, a\\n                t.diff = compute_diff(b, a, before_grid=None if t.before_frame is None else t.before_frame.grid,\\n                                      after_grid=None if t.after_frame is None else t.after_frame.grid, noop_pixel_threshold=int(self.cfg[\\\"noop_pixel_threshold\\\"]))\\n                n += 1\\n        return n\\n\\n    def _add_group(self, a: int, b: int) -> None:\\n        ga = next((g for g in self._groups if a in g), None); gb = next((g for g in self._groups if b in g), None)\\n        if ga and gb and ga is not gb:\\n            ga |= gb; self._groups.remove(gb)\\n        elif ga:\\n            ga.add(b)\\n        elif gb:\\n            gb.add(a)\\n        else:\\n            self._groups.append({a, b})\\n\\n    def summarize(self, t: Transition, max_len: int = 200) -> str:\\n        return summarize(t, max_len)\\n\\n    def render(self, scene: Scene, scale: int = 8) -> bytes:\\n        return render_scene(scene, scale)\\n\\n    def merge_log(self) -> list[tuple[int, int]]:\\n        return list(self._merged)\\n\\n    def transition(self, before: Scene, action, after: Scene, *, before_frame: Optional[Frame] = None, after_frame: Optional[Frame] = None,\\n                   status_change=None, intermediate=(), tid: str = \\\"\\\", level: int = 1, step_idx: int = 0) -> Transition:\\n        \\\"\\\"\\\"Build the Transition (diff + merge bookkeeping). Callers must continue from `t.after`, not from the scene they\\n        passed in: when a merge is confirmed here both scenes are re-fused and `t.after` is the canonical current scene.\\\"\\\"\\\"\\n        bg = None if before_frame is None else before_frame.grid; ag = None if after_frame is None else after_frame.grid\\n        d = self.diff(before, after, bg, ag)\\n        if status_change is None:\\n            dyn = self._detect_dynamic(before, after)\\n            if dyn:\\n                self._dynamic |= dyn; self.reparse_needed = True\\n        solo = self._solo_movers(before, after)\\n        if solo:\\n            # a composite's part moved alone (an agent next to a wall, a marker under a button): it was never a pattern\\n            self._solo |= solo\\n            before, after = self._dissolve(before), self._dissolve(after)\\n            d = compute_diff(before, after, before_grid=bg, after_grid=ag, noop_pixel_threshold=int(self.cfg[\\\"noop_pixel_threshold\\\"]))\\n            self.merge_confirmed = True\\n        if self.merge_confirmed:\\n            # a merge was just confirmed: fuse both scenes of this transition so the diff sees one object\\n            before, after = self.apply_groups(before), self.apply_groups(after)\\n            d = compute_diff(before, after, before_grid=bg, after_grid=ag, noop_pixel_threshold=int(self.cfg[\\\"noop_pixel_threshold\\\"]))\\n        return Transition(before, action, after, d, status_change, list(intermediate), tid, level, step_idx, before_frame, after_frame)\\n\\n\\n__all__ = [\\\"Perception\\\", \\\"load_perception_config\\\", \\\"summarize\\\", \\\"scene_text\\\", \\\"grid_png\\\", \\\"render_scene\\\", \\\"compute_diff\\\"]\\n\", \"pbg/perception/diff.py\": \"\\\"\\\"\\\"perception/diff.py \\u2014 Diff between two tracked scenes (spec \\u00a76 step 5).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Optional\\n\\nimport numpy as np\\n\\nfrom ..core.types import Diff, Scene\\n\\n\\ndef compute_diff(before: Scene, after: Scene, *, before_grid: Optional[np.ndarray] = None, after_grid: Optional[np.ndarray] = None,\\n                 noop_pixel_threshold: int = 0) -> Diff:\\n    b = {o.id: o for o in before.objects}; a = {o.id: o for o in after.objects}\\n    d = Diff()\\n    for oid, ob in b.items():\\n        oa = a.get(oid)\\n        if oa is None:\\n            d.disappeared.append(ob); continue\\n        if oa.bbox != ob.bbox and oa.shape_sig == ob.shape_sig:\\n            d.moved.append((oid, (oa.bbox[0] - ob.bbox[0], oa.bbox[1] - ob.bbox[1])))\\n        elif oa.bbox != ob.bbox or oa.shape_sig != ob.shape_sig:\\n            if oa.shape_sig != ob.shape_sig:\\n                d.reshaped.append((oid, ob.shape_sig, oa.shape_sig))\\n            if oa.bbox[:2] != ob.bbox[:2]:\\n                d.moved.append((oid, (oa.bbox[0] - ob.bbox[0], oa.bbox[1] - ob.bbox[1])))\\n        if oa.color != ob.color or oa.colors != ob.colors:\\n            d.recolored.append((oid, ob.color, oa.color))\\n    for oid, oa in a.items():\\n        if oid not in b:\\n            d.appeared.append(oa)\\n    if before_grid is not None and after_grid is not None:\\n        d.pixel_changes = int((np.asarray(before_grid) != np.asarray(after_grid)).sum())\\n    else:\\n        d.pixel_changes = int((before.render() != after.render()).sum())\\n    d.is_noop = d.pixel_changes <= noop_pixel_threshold and not (d.moved or d.appeared or d.disappeared or d.recolored or d.reshaped)\\n    return d\\n\", \"pbg/perception/regions.py\": \"\\\"\\\"\\\"perception/regions.py \\u2014 region segmentation (spec \\u00a76 step 1).\\n\\nRegions are large single-colour blocks: every 4-connected component whose area is >= region_min_area_ratio of the grid\\nbecomes a Region over its bounding box (R0 is always the global background covering the whole grid). Thin periodic\\nstrips (thickness <= ui_strip_max_thickness) get their own Region with kind_hint 'ui_strip'.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom typing import Optional\\n\\nimport numpy as np\\n\\nfrom ..core.types import Region\\n\\n\\ndef label_components(grid: np.ndarray, mask: Optional[np.ndarray] = None, connectivity: int = 4) -> tuple[np.ndarray, int]:\\n    \\\"\\\"\\\"Label same-colour connected components (colour-uniform). Returns (labels (-1 = unlabelled), n).\\\"\\\"\\\"\\n    h, w = grid.shape\\n    labels = np.full((h, w), -1, dtype=np.int32)\\n    use = np.ones((h, w), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)\\n    g = grid\\n    n = 0\\n    nbrs4 = ((-1, 0), (1, 0), (0, -1), (0, 1))\\n    nbrs = nbrs4 if connectivity == 4 else nbrs4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))\\n    rows, cols = np.nonzero(use)\\n    for sr, sc in zip(rows.tolist(), cols.tolist()):\\n        if labels[sr, sc] >= 0:\\n            continue\\n        color = g[sr, sc]\\n        labels[sr, sc] = n\\n        stack = [(sr, sc)]\\n        while stack:\\n            r, c = stack.pop()\\n            for dr, dc in nbrs:\\n                nr, nc = r + dr, c + dc\\n                if 0 <= nr < h and 0 <= nc < w and labels[nr, nc] < 0 and use[nr, nc] and g[nr, nc] == color:\\n                    labels[nr, nc] = n\\n                    stack.append((nr, nc))\\n        n += 1\\n    return labels, n\\n\\n\\ndef component_stats(labels: np.ndarray, n: int) -> list[dict]:\\n    \\\"\\\"\\\"Per component: bbox (exclusive), area, and the pixel index arrays.\\\"\\\"\\\"\\n    out = []\\n    flat = labels.ravel()\\n    order = np.argsort(flat, kind=\\\"stable\\\")\\n    sorted_labels = flat[order]\\n    starts = np.searchsorted(sorted_labels, np.arange(n))\\n    ends = np.searchsorted(sorted_labels, np.arange(n), side=\\\"right\\\")\\n    w = labels.shape[1]\\n    for i in range(n):\\n        idx = order[starts[i]:ends[i]]\\n        rr, cc = idx // w, idx % w\\n        out.append({\\\"id\\\": i, \\\"area\\\": int(len(idx)), \\\"bbox\\\": (int(rr.min()), int(cc.min()), int(rr.max()) + 1, int(cc.max()) + 1), \\\"rows\\\": rr, \\\"cols\\\": cc})\\n    return out\\n\\n\\ndef _is_periodic(strip: np.ndarray, min_repeats: int = 2) -> bool:\\n    \\\"\\\"\\\"A 1-D colour sequence is periodic if some period p in 2..len/min_repeats repeats at least min_repeats times.\\\"\\\"\\\"\\n    seq = strip.tolist()\\n    n = len(seq)\\n    for p in range(2, n // min_repeats + 1):\\n        ok = all(seq[i] == seq[i - p] for i in range(p, n))\\n        if ok and n // p >= min_repeats and len(set(seq[:p])) > 1:\\n            return True\\n    return False\\n\\n\\ndef find_regions(grid: np.ndarray, *, min_area_ratio: float = 0.05, ui_strip_max_thickness: int = 3) -> tuple[list[Region], int, dict]:\\n    \\\"\\\"\\\"Returns (regions, global_bg, meta). meta has 'region_masks' {region_id: (bbox, mask)} for non-global regions.\\\"\\\"\\\"\\n    h, w = grid.shape\\n    counts = Counter(grid.ravel().tolist())\\n    # the global background is the colour that dominates the grid border (a framed board keeps its frame colour as\\n    # background even when the board colour covers more pixels); ties -> overall count\\n    border = np.concatenate([grid[0, :], grid[-1, :], grid[:, 0], grid[:, -1]]).tolist()\\n    bc = Counter(border)\\n    global_bg = int(max(bc, key=lambda c: (bc[c], counts[c])))\\n    labels, n = label_components(grid)\\n    stats = component_stats(labels, n)\\n    min_area = max(4, int(min_area_ratio * h * w))\\n    regions: list[Region] = [Region(\\\"R0\\\", global_bg, (0, 0, h, w), \\\"board\\\")]\\n    masks: dict[str, tuple] = {}\\n    # a block that touches the grid border is background-like (a side panel, a column next to a mover): it becomes a\\n    # region at half the size threshold, so widening/narrowing it by a few pixels does not flip it between region and object\\n    def touches_border(s) -> bool:\\n        r0, c0, r1, c1 = s[\\\"bbox\\\"]\\n        return r0 == 0 or c0 == 0 or r1 == h or c1 == w\\n    big = [s for s in stats if s[\\\"area\\\"] >= min_area or (touches_border(s) and s[\\\"area\\\"] >= min_area // 2 and s[\\\"area\\\"] >= 64)]\\n    big.sort(key=lambda s: -s[\\\"area\\\"])\\n    rid = 1\\n    for s in big:\\n        color = int(grid[s[\\\"rows\\\"][0], s[\\\"cols\\\"][0]])\\n        r0, c0, r1, c1 = s[\\\"bbox\\\"]\\n        if color == global_bg and (r1 - r0) * (c1 - c0) >= 0.5 * h * w:\\n            continue  # the global background itself (a same-colour sub-block smaller than half the grid stays a panel)\\n        fill = s[\\\"area\\\"] / ((r1 - r0) * (c1 - c0))\\n        if fill < 0.25:\\n            continue  # sparse (a frame/outline, not a block): treat as an object instead\\n        kind = \\\"board\\\" if (r1 - r0) * (c1 - c0) >= 0.5 * h * w else \\\"panel\\\"\\n        m = np.zeros((r1 - r0, c1 - c0), dtype=bool); m[s[\\\"rows\\\"] - r0, s[\\\"cols\\\"] - c0] = True\\n        m |= grid[r0:r1, c0:c1] == color            # small same-colour gaps inside the bbox belong to the region too\\n        reg = Region(f\\\"R{rid}\\\", color, (r0, c0, r1, c1), kind, None if fill >= 0.98 else m)\\n        masks[reg.id] = ((r0, c0, r1, c1), m, fill)\\n        regions.append(reg); rid += 1\\n    # nested boards: a large area of the global background colour enclosed inside a region (the floor inside a framed\\n    # panel) is a region of its own, so rendering and floor rules see the floor colour there\\n    nested = []\\n    for s in stats:\\n        color = int(grid[s[\\\"rows\\\"][0], s[\\\"cols\\\"][0]])\\n        if color != global_bg or s[\\\"area\\\"] < min_area:\\n            continue\\n        r0, c0, r1, c1 = s[\\\"bbox\\\"]\\n        if r0 == 0 or c0 == 0 or r1 == h or c1 == w:\\n            continue          # touches the grid border: this is the outer background itself\\n        m = np.zeros((r1 - r0, c1 - c0), dtype=bool); m[s[\\\"rows\\\"] - r0, s[\\\"cols\\\"] - c0] = True\\n        fill = s[\\\"area\\\"] / ((r1 - r0) * (c1 - c0))\\n        reg = Region(f\\\"R{rid}\\\", global_bg, (r0, c0, r1, c1), \\\"board\\\", None if fill >= 0.98 else m)\\n        masks[reg.id] = ((r0, c0, r1, c1), m, fill)\\n        nested.append(reg); rid += 1\\n    regions += nested\\n    # ui strips: thin, long components with a periodic pattern along their axis (any colour but the containing bg)\\n    for s in stats:\\n        r0, c0, r1, c1 = s[\\\"bbox\\\"]\\n        th, ln = min(r1 - r0, c1 - c0), max(r1 - r0, c1 - c0)\\n        color = int(grid[s[\\\"rows\\\"][0], s[\\\"cols\\\"][0]])\\n        if color == global_bg:\\n            continue\\n        # anything lying entirely in the outermost row/column is a HUD element (counters that start at 1 px and grow)\\n        outermost = (r1 - r0 == 1 and (r0 == 0 or r1 == h)) or (c1 - c0 == 1 and (c0 == 0 or c1 == w))\\n        if not outermost and (th > ui_strip_max_thickness or ln < 12 or s[\\\"area\\\"] < 6):\\n            continue\\n        line = grid[r0, c0:c1] if (c1 - c0) >= (r1 - r0) else grid[r0:r1, c0]\\n        at_edge = r0 <= 2 or c0 <= 2 or r1 >= h - 2 or c1 >= w - 2\\n        # periodic along the axis (tick marks), or a solid bar hugging a grid edge (gauge / timer)\\n        if outermost or _is_periodic(line) or at_edge:\\n            # a strip region spans the whole edge band so every piece of the gauge (bar, ticks, remainder) shares it\\n            if (c1 - c0) >= (r1 - r0):\\n                bbox = (r0, 0, r1, w)\\n            else:\\n                bbox = (0, c0, h, c1)\\n            if any(reg.kind_hint == \\\"ui_strip\\\" and reg.bbox == bbox for reg in regions):\\n                continue\\n            regions.append(Region(f\\\"R{rid}\\\", global_bg, bbox, \\\"ui_strip\\\")); rid += 1\\n    # dotted / alternating HUD lines: a thin band (1-3 px) whose non-background pixels form a periodic multi-colour\\n    # pattern of length >= 12 (a row of 1-px ticks is not one component, so the component test above misses it)\\n    for axis in (0, 1):\\n        n_lines = h if axis == 0 else w\\n        for i in range(n_lines):\\n            line = grid[i, :] if axis == 0 else grid[:, i]\\n            fg_line = line != global_bg\\n            if fg_line.sum() < 6:\\n                continue\\n            idx = np.nonzero(fg_line)[0]\\n            runs = np.split(idx, np.nonzero(np.diff(idx) > 2)[0] + 1)      # gaps of <= 2 background pixels stay in one run (dotted lines)\\n            for run in runs:\\n                if run[-1] - run[0] + 1 < 12 or len(run) < 6:\\n                    continue\\n                seq = line[run[0]:run[-1] + 1]\\n                if len(set(seq.tolist())) >= 2 and _is_periodic(seq, min_repeats=3):\\n                    # the band must be thin: the same pattern must not continue above/below for more than 3 lines\\n                    thick = 1\\n                    for j in (i + 1, i + 2, i + 3):\\n                        if j < n_lines and np.array_equal((grid[j, :] if axis == 0 else grid[:, j])[run[0]:run[-1] + 1], seq):\\n                            thick += 1\\n                    if thick > 3:\\n                        continue\\n                    bbox = (i, 0, i + thick, w) if axis == 0 else (0, i, h, i + thick)\\n                    if not any(reg.kind_hint == \\\"ui_strip\\\" and reg.bbox == bbox for reg in regions):\\n                        regions.append(Region(f\\\"R{rid}\\\", global_bg, bbox, \\\"ui_strip\\\")); rid += 1\\n                    break\\n    # merge overlapping strip bands into one\\n    strips = [r for r in regions if r.kind_hint == \\\"ui_strip\\\"]\\n    merged: list[Region] = []\\n    def orient(b):\\n        return \\\"h\\\" if (b[3] - b[1]) >= (b[2] - b[0]) else \\\"v\\\"\\n    for r in sorted(strips, key=lambda r: r.bbox):\\n        for m in merged:\\n            if orient(m.bbox) == orient(r.bbox) and _bands_overlap(m.bbox, r.bbox):\\n                m.bbox = (min(m.bbox[0], r.bbox[0]), min(m.bbox[1], r.bbox[1]), max(m.bbox[2], r.bbox[2]), max(m.bbox[3], r.bbox[3]))\\n                break\\n        else:\\n            merged.append(r)\\n    regions = [r for r in regions if r.kind_hint != \\\"ui_strip\\\"] + merged\\n    remap = {}\\n    for i, r in enumerate(regions):\\n        remap[r.id] = f\\\"R{i}\\\"; r.id = f\\\"R{i}\\\"\\n    masks = {remap.get(k, k): v for k, v in masks.items()}\\n    return regions, global_bg, {\\\"region_masks\\\": masks, \\\"labels\\\": labels, \\\"stats\\\": stats}\\n\\n\\ndef _bands_overlap(a, b) -> bool:\\n    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])\\n\\n\\ndef smallest_region_for_bbox(regions: list[Region], bbox: tuple[int, int, int, int]) -> Region:\\n    r0, c0, r1, c1 = bbox\\n    best = regions[0]\\n    for reg in regions:\\n        if reg.kind_hint == \\\"ui_strip\\\":\\n            R0, C0, R1, C1 = reg.bbox\\n            if R0 <= r0 and C0 <= c0 and r1 <= R1 and c1 <= C1 and reg.area < best.area:\\n                best = reg\\n            continue\\n        R0, C0, R1, C1 = reg.bbox\\n        cr, cc = (r0 + r1 - 1) // 2, (c0 + c1 - 1) // 2\\n        if R0 <= cr < R1 and C0 <= cc < C1 and reg.area < best.area:\\n            best = reg\\n    return best\\n\\n\\ndef stabilize_regions(cur: list[Region], prev: list[Region]) -> list[Region]:\\n    \\\"\\\"\\\"Temporal stability (spec \\u00a715 occlusion): a region of the same colour seen in the previous scene keeps at least its\\n    previous extent (an object crossing a board can split its background component). Regions never shrink within a level.\\\"\\\"\\\"\\n    out = []\\n    used = set()\\n    for r in cur:\\n        if r.id == \\\"R0\\\" or r.kind_hint == \\\"ui_strip\\\":\\n            out.append(r); continue\\n        best = None\\n        for q in prev:\\n            if q.id == \\\"R0\\\" or q.kind_hint == \\\"ui_strip\\\" or q.bg_color != r.bg_color or q.id in used:\\n                continue\\n            if _bands_overlap(q.bbox, r.bbox):\\n                best = q; break\\n        if best is not None:\\n            used.add(best.id)\\n            bb = (min(r.bbox[0], best.bbox[0]), min(r.bbox[1], best.bbox[1]), max(r.bbox[2], best.bbox[2]), max(r.bbox[3], best.bbox[3]))\\n            r = Region(r.id, r.bg_color, bb, \\\"board\\\" if best.kind_hint == \\\"board\\\" else r.kind_hint, _union_mask(bb, r, best))\\n        out.append(r)\\n    # previous regions with no counterpart this frame (fully occluded / split into small parts) are kept\\n    for q in prev:\\n        if q.id == \\\"R0\\\" or q.id in used:\\n            continue\\n        if q.kind_hint == \\\"ui_strip\\\":\\n            # a gauge that shrank below the detection length is still a gauge: strip bands persist within a level\\n            if not any(o.kind_hint == \\\"ui_strip\\\" and _bands_overlap(o.bbox, q.bbox) for o in out):\\n                out.append(Region(q.id, q.bg_color, q.bbox, \\\"ui_strip\\\"))\\n            continue\\n        if not any(o.bg_color == q.bg_color and _bands_overlap(o.bbox, q.bbox) for o in out if o.id != \\\"R0\\\"):\\n            out.append(Region(q.id, q.bg_color, q.bbox, q.kind_hint, q.mask))\\n    # re-number non-strip regions by size, strips last (ids must be stable-ish: sort by bbox for determinism)\\n    strips = [r for r in out if r.kind_hint == \\\"ui_strip\\\"]; rest = [r for r in out if r.kind_hint != \\\"ui_strip\\\" and r.id != \\\"R0\\\"]\\n    rest.sort(key=lambda r: (-r.area, r.bbox))\\n    final = [out[0]] + rest + strips\\n    for i, r in enumerate(final):\\n        r.id = f\\\"R{i}\\\"\\n    return final\\n\\n\\ndef _union_mask(bb, a: Region, b: Region):\\n    \\\"\\\"\\\"Union of two regions' pixel masks inside the union bbox (None when both are full rectangles and equal to bb).\\\"\\\"\\\"\\n    if a.mask is None and b.mask is None and a.bbox == bb and b.bbox == bb:\\n        return None\\n    m = np.zeros((bb[2] - bb[0], bb[3] - bb[1]), dtype=bool)\\n    for reg in (a, b):\\n        r0, c0, r1, c1 = reg.bbox\\n        sub = m[r0 - bb[0]:r1 - bb[0], c0 - bb[1]:c1 - bb[1]]\\n        sub |= True if reg.mask is None else reg.mask\\n    return m\\n\", \"pbg/perception/render.py\": \"\\\"\\\"\\\"perception/render.py \\u2014 PNG rendering of a scene / grid (LLM auxiliary input). Pure zlib PNG writer, no PIL needed.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport struct\\nimport zlib\\n\\nimport numpy as np\\n\\nPALETTE = [(0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64), (255, 220, 0), (170, 170, 170), (240, 18, 190), (255, 133, 27),\\n           (127, 219, 255), (135, 12, 37), (255, 255, 255), (177, 13, 201), (57, 204, 204), (1, 255, 112), (255, 153, 204), (60, 120, 120)]\\n\\n\\ndef _chunk(tag: bytes, data: bytes) -> bytes:\\n    return struct.pack(\\\">I\\\", len(data)) + tag + data + struct.pack(\\\">I\\\", zlib.crc32(tag + data) & 0xFFFFFFFF)\\n\\n\\ndef grid_png(grid: np.ndarray, scale: int = 8, gridlines: bool = False) -> bytes:\\n    g = np.asarray(grid)\\n    h, w = g.shape\\n    img = np.zeros((h * scale, w * scale, 3), dtype=np.uint8)\\n    pal = np.array(PALETTE, dtype=np.uint8)\\n    rgb = pal[np.clip(g, 0, 15)]\\n    img[:] = np.repeat(np.repeat(rgb, scale, axis=0), scale, axis=1)\\n    if gridlines and scale >= 4:\\n        img[::scale, :, :] = img[::scale, :, :] // 2\\n        img[:, ::scale, :] = img[:, ::scale, :] // 2\\n    raw = b\\\"\\\".join(b\\\"\\\\x00\\\" + img[y].tobytes() for y in range(h * scale))\\n    return (b\\\"\\\\x89PNG\\\\r\\\\n\\\\x1a\\\\n\\\" + _chunk(b\\\"IHDR\\\", struct.pack(\\\">IIBBBBB\\\", w * scale, h * scale, 8, 2, 0, 0, 0))\\n            + _chunk(b\\\"IDAT\\\", zlib.compress(raw, 6)) + _chunk(b\\\"IEND\\\", b\\\"\\\"))\\n\\n\\ndef render_scene(scene, scale: int = 8) -> bytes:\\n    return grid_png(scene.render(), scale)\\n\", \"pbg/perception/segment.py\": \"\\\"\\\"\\\"perception/segment.py \\u2014 object segmentation (spec \\u00a76 steps 2-3): foreground pixels -> per-colour 4-connected\\ncomponents; adjacent multi-colour components are kept as merge candidates; confirmed merges (from track.py) fuse\\ncomponents into one multi-colour object.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom typing import Optional\\n\\nimport numpy as np\\n\\nfrom ..core.types import Object, Region, shape_signature\\nfrom .regions import component_stats, label_components, smallest_region_for_bbox\\n\\n\\ndef foreground_mask(grid: np.ndarray, regions: list[Region], global_bg: int, region_masks: dict, *, min_region_area: int = 0) -> np.ndarray:\\n    \\\"\\\"\\\"A pixel is foreground when its colour differs from the background of the smallest region containing it.\\n    Global-background-coloured components inside a region that touch that region's bbox border are holes, not objects;\\n    enclosed ones are objects when small and background (a nested board) when at least min_region_area.\\\"\\\"\\\"\\n    h, w = grid.shape\\n    fg = grid != global_bg\\n    # per non-global region: pixels of the region colour are background; global-bg pixels inside are holes unless enclosed\\n    for reg in regions:\\n        if reg.id == \\\"R0\\\" or reg.kind_hint == \\\"ui_strip\\\":\\n            continue\\n        r0, c0, r1, c1 = reg.bbox\\n        sub = grid[r0:r1, c0:c1]\\n        fg[r0:r1, c0:c1] &= sub != reg.bg_color\\n        if reg.bg_color == global_bg:\\n            continue          # nested floor of the background colour: its pixels are already background\\n        holes = sub == global_bg\\n        if holes.any():\\n            labels, n = label_components(sub, holes)\\n            if n:\\n                border = np.zeros(n, dtype=bool)\\n                for edge in (labels[0, :], labels[-1, :], labels[:, 0], labels[:, -1]):\\n                    e = edge[edge >= 0]\\n                    border[e] = True\\n                if min_region_area > 0:\\n                    sizes = np.bincount(labels[labels >= 0].ravel(), minlength=n)\\n                    border |= sizes >= min_region_area          # a large enclosed area is a nested board, not an object\\n                enclosed = holes & ~border[np.clip(labels, 0, None)] & (labels >= 0)\\n                fg[r0:r1, c0:c1] |= enclosed\\n    return fg\\n\\n\\ndef checker_mask(grid: np.ndarray) -> np.ndarray:\\n    \\\"\\\"\\\"Pixels that belong to a 2x2-periodic two-colour texture (checkerboard hazards, dotted floors): every 2x2 block\\n    whose diagonals agree and whose rows differ marks its four pixels.\\\"\\\"\\\"\\n    h, w = grid.shape\\n    a, b, c, d = grid[:-1, :-1], grid[:-1, 1:], grid[1:, :-1], grid[1:, 1:]\\n    blk = (a == d) & (b == c) & (a != b)\\n    m = np.zeros((h, w), dtype=bool)\\n    m[:-1, :-1] |= blk; m[:-1, 1:] |= blk; m[1:, :-1] |= blk; m[1:, 1:] |= blk\\n    return m\\n\\n\\ndef dot_lattice_groups(grid: np.ndarray, fg: np.ndarray, regions: list[Region], *, dot_max_area: int = 2,\\n                       min_dots: int = 5, max_link: int = 6) -> tuple[list[Object], np.ndarray]:\\n    \\\"\\\"\\\"Many small same-colour components lying on a regular lattice are ONE object: a dotted ring, a 1-px diagonal outline\\n    (which 4-connectivity shatters into isolated cells), a scatter of marks forming one figure. Two dots of the same colour\\n    link when their Chebyshev distance is within ~1.5x the lattice's typical spacing; a connected run of >= min_dots dots\\n    becomes one multi-cell object of that colour. Returns (objects, mask of pixels consumed). Scale-free: unrelated far\\n    marks never link, and the same rule catches both spaced dots and a broken diagonal line.\\\"\\\"\\\"\\n    used = np.zeros(grid.shape, dtype=bool)\\n    labels, n = label_components(grid, fg, 4)\\n    if n == 0:\\n        return [], used\\n    stats = component_stats(labels, n)\\n    by_color: dict[int, list[dict]] = {}\\n    for s in stats:\\n        if s[\\\"area\\\"] <= dot_max_area:\\n            s[\\\"cen\\\"] = (float(s[\\\"rows\\\"].mean()), float(s[\\\"cols\\\"].mean()))\\n            by_color.setdefault(int(grid[s[\\\"rows\\\"][0], s[\\\"cols\\\"][0]]), []).append(s)\\n    objs: list[Object] = []\\n    for color, dots in by_color.items():\\n        if len(dots) < min_dots:\\n            continue\\n        cen = np.array([d[\\\"cen\\\"] for d in dots])\\n        cheb = np.maximum(np.abs(cen[:, None, 0] - cen[None, :, 0]), np.abs(cen[:, None, 1] - cen[None, :, 1]))\\n        np.fill_diagonal(cheb, np.inf)\\n        nn = cheb.min(axis=1)\\n        link = int(min(max_link, max(2, round(1.5 * float(np.median(nn))))))\\n        adj = cheb <= link\\n        seen = np.zeros(len(dots), dtype=bool)\\n        for start in range(len(dots)):\\n            if seen[start]:\\n                continue\\n            stack = [start]; comp = []\\n            while stack:\\n                x = stack.pop()\\n                if seen[x]:\\n                    continue\\n                seen[x] = True; comp.append(x)\\n                stack.extend(np.nonzero(adj[x] & ~seen)[0].tolist())\\n            if len(comp) < min_dots:\\n                continue\\n            members = [dots[i] for i in comp]\\n            r0 = min(m[\\\"bbox\\\"][0] for m in members); c0 = min(m[\\\"bbox\\\"][1] for m in members)\\n            r1 = max(m[\\\"bbox\\\"][2] for m in members); c1 = max(m[\\\"bbox\\\"][3] for m in members)\\n            m = np.zeros((r1 - r0, c1 - c0), dtype=bool)\\n            area = 0\\n            for mem in members:\\n                m[mem[\\\"rows\\\"] - r0, mem[\\\"cols\\\"] - c0] = True; area += mem[\\\"area\\\"]\\n                used[mem[\\\"rows\\\"], mem[\\\"cols\\\"]] = True\\n            reg = smallest_region_for_bbox(regions, (r0, c0, r1, c1))\\n            objs.append(Object(len(objs), color, (color,), (r0, c0, r1, c1), m, area, shape_signature(m), reg.id))\\n    return objs, used\\n\\n\\ndef segment_objects(grid: np.ndarray, regions: list[Region], global_bg: int, region_masks: dict, *, connectivity: int = 4,\\n                    max_objects: int = 200, min_region_area: int = 0, group_dots: bool = True) -> tuple[list[Object], list[tuple[int, int]]]:\\n    \\\"\\\"\\\"Returns (objects with provisional ids 0..n-1, adjacency pairs between different-colour components).\\\"\\\"\\\"\\n    h, w = grid.shape\\n    fg = foreground_mask(grid, regions, global_bg, region_masks, min_region_area=min_region_area)\\n    objs: list[Object] = []\\n    # 2x2-periodic textures become ONE multi-colour object each (a checkerboard is not 100 one-pixel objects)\\n    tex = checker_mask(grid) & (fg | (grid != global_bg))\\n    tex_labels, tn = label_components(np.zeros_like(grid), tex, 8)\\n    tex_stats = component_stats(tex_labels, tn) if tn else []\\n    tex_used = np.zeros((h, w), dtype=bool)\\n    a_, b_, c_, d_ = grid[:-1, :-1], grid[:-1, 1:], grid[1:, :-1], grid[1:, 1:]\\n    blk = (a_ == d_) & (b_ == c_) & (a_ != b_)\\n    for s in tex_stats:\\n        r0, c0, r1, c1 = s[\\\"bbox\\\"]\\n        # a real texture is dense, at least 3x3, and MOST of its 2x2 windows are checker blocks (a piece inside four\\n        # corner marks only has checker blocks at the corners)\\n        if s[\\\"area\\\"] < 12 or (r1 - r0) < 3 or (c1 - c0) < 3 or s[\\\"area\\\"] < 0.75 * (r1 - r0) * (c1 - c0):\\n            continue\\n        windows = blk[r0:r1 - 1, c0:c1 - 1]\\n        if windows.size == 0 or windows.mean() < 0.6:\\n            continue\\n        m = np.zeros((r1 - r0, c1 - c0), dtype=bool); m[s[\\\"rows\\\"] - r0, s[\\\"cols\\\"] - c0] = True\\n        cm = np.where(m, grid[r0:r1, c0:c1], -1).astype(np.int8)\\n        cols = sorted({int(v) for v in grid[s[\\\"rows\\\"], s[\\\"cols\\\"]]})\\n        color = max(cols, key=lambda cc: int((cm == cc).sum()))\\n        reg = smallest_region_for_bbox(regions, (r0, c0, r1, c1))\\n        objs.append(Object(len(objs), color, tuple(cols), (r0, c0, r1, c1), m, s[\\\"area\\\"], shape_signature(m), reg.id, color_mask=cm))\\n        tex_used[s[\\\"rows\\\"], s[\\\"cols\\\"]] = True\\n    fg = fg & ~tex_used\\n    # a dotted ring / broken diagonal outline / scatter of marks on a lattice is ONE object, not N one-pixel objects\\n    if group_dots:\\n        dots, dot_used = dot_lattice_groups(grid, fg, regions)\\n        for o in dots:\\n            o.id = len(objs); objs.append(o)\\n        fg = fg & ~dot_used\\n    labels, n = label_components(grid, fg, connectivity)\\n    stats = component_stats(labels, n)\\n    base = len(objs)\\n    for s in stats:\\n        r0, c0, r1, c1 = s[\\\"bbox\\\"]\\n        color = int(grid[s[\\\"rows\\\"][0], s[\\\"cols\\\"][0]])\\n        m = np.zeros((r1 - r0, c1 - c0), dtype=bool); m[s[\\\"rows\\\"] - r0, s[\\\"cols\\\"] - c0] = True\\n        reg = smallest_region_for_bbox(regions, (r0, c0, r1, c1))\\n        objs.append(Object(base + s[\\\"id\\\"], color, (color,), (r0, c0, r1, c1), m, s[\\\"area\\\"], shape_signature(m), reg.id))\\n    # adjacency between different-colour components (merge candidates)\\n    adj: set[tuple[int, int]] = set()\\n    a = labels[:, :-1]; b = labels[:, 1:]\\n    sel = (a >= 0) & (b >= 0) & (a != b)\\n    for x, y in zip(a[sel].tolist(), b[sel].tolist()):\\n        adj.add((base + min(x, y), base + max(x, y)))\\n    a = labels[:-1, :]; b = labels[1:, :]\\n    sel = (a >= 0) & (b >= 0) & (a != b)\\n    for x, y in zip(a[sel].tolist(), b[sel].tolist()):\\n        adj.add((base + min(x, y), base + max(x, y)))\\n    if len(objs) > max_objects:\\n        objs.sort(key=lambda o: -o.area); objs = objs[:max_objects]\\n        keep = {o.id for o in objs}\\n        adj = {p for p in adj if p[0] in keep and p[1] in keep}\\n    return objs, sorted(adj)\\n\\n\\ndef fuse(objects: list[Object], grid, regions: list[Region]) -> Object:\\n    \\\"\\\"\\\"Fuse several components into one multi-colour object (keeps the smallest id).\\\"\\\"\\\"\\n    r0 = min(o.bbox[0] for o in objects); c0 = min(o.bbox[1] for o in objects)\\n    r1 = max(o.bbox[2] for o in objects); c1 = max(o.bbox[3] for o in objects)\\n    mask = np.zeros((r1 - r0, c1 - c0), dtype=bool)\\n    cmask = np.full((r1 - r0, c1 - c0), -1, dtype=np.int8)\\n    for o in objects:\\n        a0, b0, a1, b1 = o.bbox\\n        sub = mask[a0 - r0:a1 - r0, b0 - c0:b1 - c0]\\n        sub |= o.mask\\n        cm = cmask[a0 - r0:a1 - r0, b0 - c0:b1 - c0]\\n        cm[o.mask] = o.color if o.color_mask is None else o.color_mask[o.mask]\\n    counts = Counter()\\n    for o in objects:\\n        counts[o.color] += o.area\\n    color = counts.most_common(1)[0][0]\\n    colors = tuple(sorted(counts))\\n    reg = smallest_region_for_bbox(regions, (r0, c0, r1, c1))\\n    parts = [p for o in objects for p in (o.parts or [o])]\\n    return Object(min(o.id for o in objects), int(color), colors, (r0, c0, r1, c1), mask, int(mask.sum()), shape_signature(mask), reg.id,\\n                  color_mask=cmask, parts=parts)\\n\\n\\ndef parts_adjacent(a: Object, b: Object) -> bool:\\n    \\\"\\\"\\\"Do two components touch (4-neighbourhood) anywhere?\\\"\\\"\\\"\\n    r0 = max(a.bbox[0], b.bbox[0]) - 1; c0 = max(a.bbox[1], b.bbox[1]) - 1\\n    r1 = min(a.bbox[2], b.bbox[2]) + 1; c1 = min(a.bbox[3], b.bbox[3]) + 1\\n    if r1 <= r0 or c1 <= c0:\\n        return False\\n    def paste(o):\\n        m = np.zeros((r1 - r0, c1 - c0), dtype=bool)\\n        ar0, ac0 = max(o.bbox[0], r0), max(o.bbox[1], c0); ar1, ac1 = min(o.bbox[2], r1), min(o.bbox[3], c1)\\n        if ar1 > ar0 and ac1 > ac0:\\n            m[ar0 - r0:ar1 - r0, ac0 - c0:ac1 - c0] = o.mask[ar0 - o.bbox[0]:ar1 - o.bbox[0], ac0 - o.bbox[1]:ac1 - o.bbox[1]]\\n        return m\\n    ma, mb = paste(a), paste(b)\\n    grown = ma.copy()\\n    grown[1:, :] |= ma[:-1, :]; grown[:-1, :] |= ma[1:, :]; grown[:, 1:] |= ma[:, :-1]; grown[:, :-1] |= ma[:, 1:]\\n    return bool((grown & mb).any())\\n\\n\\ndef rect_composites(objs: list[Object], adj: Optional[list[tuple[int, int]]], regions: list[Region], excluded=(), *,\\n                    min_side: int = 2, min_area: int = 6, max_parts: int = 6, min_share: float = 0.2) -> list[list[Object]]:\\n    \\\"\\\"\\\"Adjacent different-colour components whose union is a FILLED rectangle: a two-colour target pattern, a half-stamped\\n    canvas, a framed button. Thin parts (1 px lines, corner marks), parts under `min_share` of the union (a marker on a bar)\\n    and ui strips never take part; `excluded` are parts seen moving on their own (an agent standing next to a wall is not a pattern). Adjacency is taken from the segmentation\\n    when given, else from the masks.\\\"\\\"\\\"\\n    by = {o.id: o for o in objs}\\n    strip = {r.id for r in regions if r.kind_hint == \\\"ui_strip\\\"}\\n    def ok(o: Object) -> bool:\\n        return o.id not in excluded and o.parts is None and o.area >= min_area and min(o.height, o.width) >= min_side and o.region not in strip\\n    elig = [o for o in objs if ok(o)]\\n    nbr: dict[int, set[int]] = {}\\n    if adj is None:\\n        pairs = [(a.id, b.id) for i, a in enumerate(elig) for b in elig[i + 1:] if parts_adjacent(a, b)]\\n    else:\\n        pairs = [(a, b) for a, b in adj if a in by and b in by and ok(by[a]) and ok(by[b])]\\n    for a, b in pairs:\\n        nbr.setdefault(a, set()).add(b); nbr.setdefault(b, set()).add(a)\\n    def is_rect(ids) -> bool:\\n        os_ = [by[i] for i in ids]\\n        if len({o.color for o in os_}) < len(os_):\\n            return False      # a repeated colour (button, gap, button) is several things, not one pattern\\n        r0 = min(o.bbox[0] for o in os_); c0 = min(o.bbox[1] for o in os_); r1 = max(o.bbox[2] for o in os_); c1 = max(o.bbox[3] for o in os_)\\n        total = (r1 - r0) * (c1 - c0)\\n        # a small end-cap (a marker on a slider bar, a tab on a panel) is its own object, not a part of a pattern\\n        return sum(o.area for o in os_) == total and all(o.area >= min_share * total for o in os_)\\n    seen: set[int] = set(); out: list[list[int]] = []\\n    for start in sorted(nbr):\\n        if start in seen:\\n            continue\\n        stack = [start]; cl: set[int] = set()\\n        while stack:\\n            x = stack.pop()\\n            if x in cl:\\n                continue\\n            cl.add(x); stack.extend(nbr[x] - cl)\\n        seen |= cl\\n        if 2 <= len(cl) <= max_parts and is_rect(cl):\\n            out.append(sorted(cl)); continue\\n        used: set[int] = set()\\n        for a in sorted(cl):\\n            if a in used:\\n                continue\\n            for b in sorted(nbr[a]):\\n                if b not in used and is_rect({a, b}):\\n                    out.append([a, b]); used |= {a, b}; break\\n    return [[by[i] for i in ids] for ids in out]\\n\", \"pbg/perception/summarize.py\": \"\\\"\\\"\\\"perception/summarize.py \\u2014 one-line natural-language Diff summary for LLM context (spec \\u00a76 step 6).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\nfrom ..core.types import Object, Scene, Transition\\n\\nCOLOR_NAMES = {0: \\\"black\\\", 1: \\\"blue\\\", 2: \\\"red\\\", 3: \\\"green\\\", 4: \\\"yellow\\\", 5: \\\"grey\\\", 6: \\\"magenta\\\", 7: \\\"orange\\\", 8: \\\"sky\\\", 9: \\\"brown\\\",\\n               10: \\\"white\\\", 11: \\\"purple\\\", 12: \\\"cyan\\\", 13: \\\"lime\\\", 14: \\\"pink\\\", 15: \\\"teal\\\"}\\n\\n\\ndef obj_tag(o: Object) -> str:\\n    role = f\\\" {o.role}\\\" if o.role and o.role != \\\"unknown\\\" else \\\"\\\"\\n    colors = f\\\"{COLOR_NAMES.get(o.color, o.color)}={o.color}\\\" if len(o.colors) == 1 else \\\"colors=\\\" + \\\"+\\\".join(str(c) for c in o.colors)\\n    return f\\\"obj#{o.id}({colors}{role}, {o.height}x{o.width}@{o.bbox[0]},{o.bbox[1]})\\\"\\n\\n\\ndef summarize(t: Transition, max_len: int = 200) -> str:\\n    parts = []\\n    b = {o.id: o for o in t.before.objects}; a = {o.id: o for o in t.after.objects}\\n    for oid, (dr, dc) in t.diff.moved:\\n        o = b.get(oid) or a.get(oid)\\n        if o:\\n            parts.append(f\\\"{obj_tag(o)} moved ({dr:+d},{dc:+d})\\\")\\n    for oid, old, new in t.diff.recolored:\\n        o = b.get(oid) or a.get(oid)\\n        if o:\\n            parts.append(f\\\"obj#{oid} recolored {old}->{new}\\\")\\n    for oid, old, new in t.diff.reshaped:\\n        o = a.get(oid) or b.get(oid)\\n        if o:\\n            parts.append(f\\\"obj#{oid} reshaped (now {o.height}x{o.width}, area {o.area})\\\")\\n    for o in t.diff.appeared:\\n        parts.append(f\\\"{obj_tag(o)} appeared\\\")\\n    for o in t.diff.disappeared:\\n        parts.append(f\\\"{obj_tag(o)} disappeared\\\")\\n    if not parts:\\n        body = \\\"no change\\\" if t.diff.is_noop else f\\\"{t.diff.pixel_changes} pixels changed (no object-level change)\\\"\\n    else:\\n        body = \\\"; \\\".join(parts) + \\\"; others unchanged\\\"\\n    status = f\\\" [{t.status_change}]\\\" if t.status_change else \\\"\\\"\\n    text = f\\\"{t.action.label()} -> {body}{status}\\\"\\n    if len(text) > max_len:\\n        text = text[:max_len - 3] + \\\"...\\\"\\n    return text\\n\\n\\ndef scene_text(scene: Scene, max_objects: int = 40) -> str:\\n    \\\"\\\"\\\"Structured scene description for prompts (regions + objects).\\\"\\\"\\\"\\n    lines = [f\\\"grid {scene.grid_shape[0]}x{scene.grid_shape[1]}; regions: \\\" + \\\", \\\".join(\\n        f\\\"{r.id}(bg={r.bg_color},{r.kind_hint},rows {r.bbox[0]}-{r.bbox[2] - 1},cols {r.bbox[1]}-{r.bbox[3] - 1})\\\" for r in scene.regions)]\\n    objs = sorted(scene.objects, key=lambda o: (-o.area, o.id))[:max_objects]\\n    for o in objs:\\n        lines.append(f\\\"  {obj_tag(o)} area={o.area} sig={o.shape_sig[:6]} region={o.region}\\\" + (f\\\" colors={list(o.colors)}\\\" if len(o.colors) > 1 else \\\"\\\"))\\n    if len(scene.objects) > max_objects:\\n        lines.append(f\\\"  ... {len(scene.objects) - max_objects} more objects\\\")\\n    if scene.aux:\\n        aux = {k: v for k, v in scene.aux.items() if not k.startswith(\\\"_\\\")}\\n        if aux:\\n            lines.append(f\\\"  aux={aux}\\\")\\n    return \\\"\\\\n\\\".join(lines)\\n\\n\\ndef crop_text(o, max_rows: int = 14, max_cols: int = 20) -> list[str]:\\n    \\\"\\\"\\\"Rows of an object's pixels as hex digits ('.' outside its mask), clipped to max_rows x max_cols.\\\"\\\"\\\"\\n    cm = o.color_mask if o.color_mask is not None else np.where(o.mask, o.color, -1)\\n    rows = []\\n    for r in range(min(o.height, max_rows)):\\n        rows.append(\\\"\\\".join(\\\".\\\" if cm[r, c] < 0 else format(int(cm[r, c]), \\\"x\\\") for c in range(min(o.width, max_cols))) + (\\\"~\\\" if o.width > max_cols else \\\"\\\"))\\n    if o.height > max_rows:\\n        rows.append(\\\"~\\\")\\n    return rows\\n\\n\\ndef transform_crops(t: Transition, max_area: int = 400) -> list[str]:\\n    \\\"\\\"\\\"Before/after crops of objects that changed shape or colour layout in a transition (what a summary line hides:\\n    a rotation, a flip, a stamped canvas).\\\"\\\"\\\"\\n    b = {o.id: o for o in t.before.objects}; a = {o.id: o for o in t.after.objects}\\n    ids = [x[0] for x in t.diff.reshaped] + [x[0] for x in t.diff.recolored if x[0] not in {y[0] for y in t.diff.reshaped}]\\n    out = []\\n    for oid in ids[:3]:\\n        ob, oa = b.get(oid), a.get(oid)\\n        if ob is None or oa is None or max(ob.area, oa.area) > max_area:\\n            continue\\n        out.append(f\\\"  obj#{oid} before @({ob.bbox[0]},{ob.bbox[1]}) {ob.height}x{ob.width}:\\\")\\n        out += [\\\"    \\\" + r for r in crop_text(ob)]\\n        out.append(f\\\"  obj#{oid} after @({oa.bbox[0]},{oa.bbox[1]}) {oa.height}x{oa.width}:\\\")\\n        out += [\\\"    \\\" + r for r in crop_text(oa)]\\n    return out\\n\", \"pbg/perception/track.py\": \"\\\"\\\"\\\"perception/track.py \\u2014 object tracking across frames (spec \\u00a76 step 4): Hungarian matching with\\ncost = 10 * colour mismatch + 5 * shape mismatch + centroid distance (per cell); above track_cost_max -> unmatched.\\nTie-break: the previous displacement direction (spec \\u00a715 'tracking id swap').\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Optional\\n\\nimport numpy as np\\n\\nfrom ..core.types import Object\\n\\n\\ndef hungarian(cost: np.ndarray) -> list[tuple[int, int]]:\\n    \\\"\\\"\\\"Minimum-cost assignment for a rectangular cost matrix (rows <= cols after padding). Returns (row, col) pairs.\\\"\\\"\\\"\\n    cost = np.asarray(cost, dtype=float)\\n    if cost.size == 0:\\n        return []\\n    n, m = cost.shape\\n    transposed = False\\n    if n > m:\\n        cost = cost.T; n, m = m, n; transposed = True\\n    INF = float(\\\"inf\\\")\\n    u = [0.0] * (n + 1); v = [0.0] * (m + 1); p = [0] * (m + 1); way = [0] * (m + 1)\\n    for i in range(1, n + 1):\\n        p[0] = i; j0 = 0\\n        minv = [INF] * (m + 1); used = [False] * (m + 1)\\n        while True:\\n            used[j0] = True\\n            i0 = p[j0]; delta = INF; j1 = 0\\n            for j in range(1, m + 1):\\n                if not used[j]:\\n                    cur = cost[i0 - 1, j - 1] - u[i0] - v[j]\\n                    if cur < minv[j]:\\n                        minv[j] = cur; way[j] = j0\\n                    if minv[j] < delta:\\n                        delta = minv[j]; j1 = j\\n            for j in range(m + 1):\\n                if used[j]:\\n                    u[p[j]] += delta; v[j] -= delta\\n                else:\\n                    minv[j] -= delta\\n            j0 = j1\\n            if p[j0] == 0:\\n                break\\n        while True:\\n            j1 = way[j0]; p[j0] = p[j1]; j0 = j1\\n            if j0 == 0:\\n                break\\n    pairs = []\\n    for j in range(1, m + 1):\\n        if p[j]:\\n            pairs.append((p[j] - 1, j - 1))\\n    if transposed:\\n        pairs = [(c, r) for r, c in pairs]\\n    return pairs\\n\\n\\nclass Tracker:\\n    def __init__(self, cost_max: float = 20.0):\\n        self.cost_max = cost_max\\n        self.next_id = 0\\n        self.last_disp: dict[int, tuple[int, int]] = {}\\n\\n    def match(self, prev: Optional[list[Object]], cur: list[Object]) -> dict[int, int]:\\n        \\\"\\\"\\\"Returns {current provisional index -> tracking id}; unmatched current objects get fresh ids.\\\"\\\"\\\"\\n        assign: dict[int, int] = {}\\n        if not prev:\\n            for i, _ in enumerate(cur):\\n                assign[i] = self.next_id; self.next_id += 1\\n            return assign\\n        n, m = len(prev), len(cur)\\n        cost = np.zeros((n, m))\\n        for i, a in enumerate(prev):\\n            ar, ac = a.center\\n            for j, b in enumerate(cur):\\n                br, bc = b.center\\n                c = 10.0 * (a.color != b.color) + 5.0 * (a.shape_sig != b.shape_sig) + abs(ar - br) + abs(ac - bc)\\n                if a.color != b.color and abs(ar - br) + abs(ac - bc) > 2:\\n                    c = 1e6          # a different colour somewhere else is another object (a button never becomes the panel next to it)\\n                if a.colors != b.colors and a.color == b.color:\\n                    c += 2.0\\n                d = self.last_disp.get(a.id)\\n                if d is not None and (d[0] or d[1]):\\n                    # tie-break: prefer continuing the previous displacement direction\\n                    dr, dc = br - ar, bc - ac\\n                    if (dr, dc) != (0, 0) and (np.sign(dr) != np.sign(d[0]) or np.sign(dc) != np.sign(d[1])):\\n                        c += 0.5\\n                cost[i, j] = c\\n        used_ids = set()\\n        for i, j in hungarian(cost):\\n            if cost[i, j] <= self.cost_max:\\n                assign[j] = prev[i].id; used_ids.add(prev[i].id)\\n                ar, ac = prev[i].center; br, bc = cur[j].center\\n                self.last_disp[prev[i].id] = (br - ar, bc - ac)\\n        for j in range(m):\\n            if j not in assign:\\n                assign[j] = self.next_id; self.next_id += 1\\n        return assign\\n\", \"pbg/planner/__init__.py\": \"from .astar import astar, plan_key  # noqa: F401\\nfrom .beam import beam_search  # noqa: F401\\nfrom .mcts import mcts  # noqa: F401\\nfrom .execute import ExecResult, Planner  # noqa: F401\\nfrom .stuck import StuckDetector  # noqa: F401\\n\", \"pbg/planner/astar.py\": \"\\\"\\\"\\\"planner/astar.py \\u2014 A* over model-predicted scenes (spec \\u00a710). Heuristic = weight * (1 - progress); revisiting a\\nstate already seen during execution costs +5.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport heapq\\nimport itertools\\nimport time\\nfrom typing import Callable, Iterable, Optional\\n\\nfrom ..core.types import Action, Scene\\n\\n\\ndef plan_key(scene: Scene) -> tuple:\\n    return scene.key()\\n\\n\\ndef astar(start: Scene, successors: Callable[[Scene], Iterable[tuple[Action, Scene]]], is_goal: Callable[[Scene], bool],\\n          progress: Callable[[Scene], float], *, depth: int = 60, weight: float = 8.0, max_nodes: int = 100_000,\\n          visited_penalty: Optional[set] = None, time_limit: float = 0.5, heuristic: Optional[Callable[[Scene], Optional[float]]] = None) -> Optional[list[Action]]:\\n    counter = itertools.count()\\n    start_key = plan_key(start)\\n    if is_goal(start):\\n        return []\\n\\n    h_cache: dict = {}\\n\\n    def h_of(s: Scene) -> float:\\n        k = plan_key(s)\\n        if k in h_cache:\\n            return h_cache[k]\\n        if heuristic is not None:\\n            e = heuristic(s)\\n            if e is not None:\\n                h_cache[k] = float(e); return h_cache[k]\\n        h_cache[k] = weight * (1.0 - progress(s))\\n        return h_cache[k]\\n    open_heap = [(h_of(start), next(counter), 0.0, 0, start_key)]\\n    nodes = {start_key: (start, None, None, 0.0)}   # key -> (scene, parent_key, action, g)\\n    closed: set = set()\\n    expanded = 0; t0 = time.perf_counter()\\n    while open_heap:\\n        f, _, g, d, key = heapq.heappop(open_heap)\\n        if key in closed:\\n            continue\\n        closed.add(key)\\n        scene = nodes[key][0]\\n        if is_goal(scene):\\n            return _path(nodes, key)\\n        if d >= depth:\\n            continue\\n        expanded += 1\\n        if expanded > max_nodes or time.perf_counter() - t0 > time_limit:\\n            return None\\n        for a, nxt in successors(scene):\\n            if time.perf_counter() - t0 > time_limit:\\n                return None\\n            k = plan_key(nxt)\\n            if k in closed:\\n                continue\\n            step = 1.0 + (5.0 if visited_penalty and k in visited_penalty else 0.0)\\n            g2 = g + step\\n            old = nodes.get(k)\\n            if old is not None and old[3] <= g2:\\n                continue\\n            nodes[k] = (nxt, key, a, g2)\\n            h = h_of(nxt)\\n            heapq.heappush(open_heap, (g2 + h, next(counter), g2, d + 1, k))\\n    return None\\n\\n\\ndef _path(nodes: dict, key) -> list[Action]:\\n    out = []\\n    while True:\\n        scene, parent, action, _ = nodes[key]\\n        if parent is None:\\n            break\\n        out.append(action); key = parent\\n    out.reverse()\\n    return out\\n\", \"pbg/planner/beam.py\": \"\\\"\\\"\\\"planner/beam.py \\u2014 beam search (width 32) for large deterministic state spaces (spec \\u00a710).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Callable, Iterable, Optional\\n\\nfrom ..core.types import Action, Scene\\nfrom .astar import plan_key\\n\\n\\ndef beam_search(start: Scene, successors: Callable[[Scene], Iterable[tuple[Action, Scene]]], is_goal: Callable[[Scene], bool],\\n                progress: Callable[[Scene], float], *, depth: int = 60, width: int = 32, time_limit: float = 2.0) -> Optional[list[Action]]:\\n    import time\\n    t0 = time.perf_counter()\\n    if is_goal(start):\\n        return []\\n    beam = [(progress(start), start, [])]\\n    seen = {plan_key(start)}\\n    for _ in range(depth):\\n        if time.perf_counter() - t0 > time_limit:\\n            return None\\n        cand = []\\n        for _, scene, path in beam:\\n            for a, nxt in successors(scene):\\n                if time.perf_counter() - t0 > time_limit:\\n                    return None\\n                k = plan_key(nxt)\\n                if k in seen:\\n                    continue\\n                seen.add(k)\\n                p2 = path + [a]\\n                if is_goal(nxt):\\n                    return p2\\n                cand.append((progress(nxt), nxt, p2))\\n        if not cand:\\n            return None\\n        cand.sort(key=lambda x: -x[0])\\n        beam = cand[:width]\\n    return None\\n\", \"pbg/planner/common.py\": \"\\\"\\\"\\\"planner/common.py \\u2014 shared search utilities: action candidates, successor generation, progress heuristic.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Callable, Optional\\n\\nfrom ..core.types import Action, Scene\\n\\n\\nPASSIVE_ROLES = (\\\"wall\\\", \\\"indicator\\\", \\\"decoration\\\", \\\"unknown\\\", None)\\n\\n\\ndef click_candidates(scene: Scene, extra: list[tuple[int, int]] = (), limit: int = 40, click_map=None) -> list[Action]:\\n    \\\"\\\"\\\"Click coordinates: object centres (objects with an active role first) + region centres + previously responsive\\n    coordinates (spec \\u00a710). With a click response map that has evidence, only classes known to react or never tried.\\\"\\\"\\\"\\n    seen: set[tuple[int, int]] = set(); out: list[Action] = []\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    if click_map is not None and click_map.total() >= 3:\\n        return click_map.rank(scene, include_inert=False, limit=limit, extra=extra)\\n    triggers = [o for o in scene.objects if o.role and \\\"trigger\\\" in o.role]\\n    if triggers:\\n        # the model knows which objects react to clicks: plan over those (plus known responsive coordinates) only\\n        for o in triggers:\\n            if o.center not in seen:\\n                seen.add(o.center); out.append(Action.click(*o.center))\\n        for rc in extra:\\n            rc = (int(rc[0]), int(rc[1]))\\n            if rc not in seen:\\n                seen.add(rc); out.append(Action.click(*rc))\\n        return out[:limit]\\n    for o in sorted(scene.objects, key=lambda o: (o.role in PASSIVE_ROLES, -o.area, o.id)):\\n        if o.region in strips:\\n            continue\\n        c = o.center\\n        if c not in seen:\\n            seen.add(c); out.append(Action.click(*c))\\n    for r in scene.regions:\\n        if r.kind_hint != \\\"ui_strip\\\" and r.center not in seen:\\n            seen.add(r.center); out.append(Action.click(*r.center))\\n    for rc in extra:\\n        rc = (int(rc[0]), int(rc[1]))\\n        if rc not in seen:\\n            seen.add(rc); out.append(Action.click(*rc))\\n    return out[:limit]\\n\\n\\ndef action_set(scene: Scene, available: list[Action], *, semantics: Optional[dict] = None, responsive: list[tuple[int, int]] = (),\\n               max_clicks: int = 40, click_map=None) -> list[Action]:\\n    noop = set()\\n    if semantics:\\n        noop = {k for k, v in semantics.items() if not k.startswith(\\\"_\\\") and isinstance(v, dict) and v.get(\\\"class\\\") == \\\"NOOP\\\" and v.get(\\\"n\\\", 0) >= 2}\\n    out: list[Action] = []\\n    for a in available:\\n        if a.type == \\\"BUTTON\\\" and a.key not in noop:\\n            out.append(a)\\n        elif a.type == \\\"CLICK\\\":\\n            out.extend(click_candidates(scene, responsive, max_clicks, click_map))\\n    return out\\n\\n\\ndef dead(scene: Scene) -> bool:\\n    return bool(scene.aux.get(\\\"dead\\\"))\\n\\n\\ndef state_key(scene: Scene) -> tuple:\\n    \\\"\\\"\\\"Identity of a scene for observed-transition lookup: objects outside ui strips.\\\"\\\"\\\"\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    return tuple(sorted(o.identity() for o in scene.objects if o.region not in strips))\\n\\n\\ndef make_successor(model, actions_fn: Callable[[Scene], list[Action]], observed: Optional[dict] = None):\\n    \\\"\\\"\\\"Successor generator over model predictions. `observed` = {(state_key, action label): after scene} from real\\n    transitions: where the environment has already answered, the answer replaces the prediction (a model that\\n    mispredicted a state cannot keep proposing the same plan).\\\"\\\"\\\"\\n    def successors(scene: Scene):\\n        sk = state_key(scene) if observed else None\\n        for a in actions_fn(scene):\\n            nxt = None\\n            if observed:\\n                nxt = observed.get((sk, a.label()))\\n            if nxt is None:\\n                try:\\n                    nxt = model.predict(scene, a)\\n                except Exception:\\n                    nxt = None\\n            if nxt is None or dead(nxt):\\n                continue\\n            yield a, nxt\\n    return successors\\n\", \"pbg/planner/execute.py\": \"\\\"\\\"\\\"planner/execute.py \\u2014 Planner facade (spec \\u00a710): algorithm selection, multi-hypothesis planning (common prefix of\\nthe top plans), execution monitoring with immediate stop on mismatch.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport time\\nfrom dataclasses import dataclass, field\\nfrom typing import Optional\\n\\nfrom ..core.contracts import Hypothesis, scene_equal, scene_mismatch\\nfrom ..core.types import Action, Scene, Transition\\nfrom .astar import astar\\nfrom .beam import beam_search\\nfrom .common import action_set, make_successor, state_key\\nfrom .mcts import mcts\\nfrom .stuck import StuckDetector\\n\\n\\n@dataclass\\nclass ExecResult:\\n    kind: str                     # PROGRESS | FAILED | MISMATCH | STUCK | EXHAUSTED | ERROR\\n    scene: Scene\\n    t: Optional[Transition] = None\\n    executed: int = 0\\n    detail: dict = field(default_factory=dict)\\n\\n\\n@dataclass\\nclass PlanInfo:\\n    plan: list[Action]\\n    hypothesis: Optional[Hypothesis] = None\\n    goal=None\\n    scored: list = field(default_factory=list)\\n    algorithm: str = \\\"\\\"\\n    reason: str = \\\"\\\"\\n\\n\\nclass Planner:\\n    def __init__(self, *, depth: int = 60, weight: float = 8.0, beam_width: int = 32, time_limit: float = 3.0, stochastic: bool = False,\\n                 log=None):\\n        self.depth, self.weight, self.beam_width, self.time_limit = depth, weight, beam_width, time_limit\\n        self.stochastic = stochastic\\n        self.log = log or (lambda *a, **k: None)\\n        self.stuck = StuckDetector()\\n        self.responsive: list[tuple[int, int]] = []\\n        self.last_info: Optional[PlanInfo] = None\\n        self.click_map = None             # click response map (orchestrator knowledge): restricts click candidates\\n        self.observed: dict = {}          # (state_key, action label) -> observed after scene (ground truth for planning)\\n        self.observed_seen: list = []\\n\\n    def _actions_fn(self, available: list[Action], semantics):\\n        def fn(scene: Scene) -> list[Action]:\\n            return action_set(scene, available, semantics=semantics, responsive=self.responsive, click_map=self.click_map)\\n        return fn\\n\\n    def _abstract(self, scene: Scene, model, goal) -> Optional[list[Action]]:\\n        \\\"\\\"\\\"Goal-provided abstract plan (over the goal-relevant objects only), verified by full simulation with the model.\\\"\\\"\\\"\\n        ap = getattr(goal, \\\"abstract_plan\\\", None)\\n        if ap is None:\\n            return None\\n        try:\\n            rs = model.with_roles(scene) if hasattr(model, \\\"with_roles\\\") else scene\\n            trig_keys = ap(rs, model)\\n        except Exception:\\n            return None\\n        if not trig_keys:\\n            return None\\n        plan = []; s_ = scene\\n        for tk in trig_keys:\\n            r0, c0, r1, c1 = tk\\n            a = Action.click((r0 + r1 - 1) // 2, (c0 + c1 - 1) // 2)\\n            nxt = model.predict(s_, a)\\n            if nxt is None:\\n                return None\\n            plan.append(a); s_ = nxt\\n        return plan if goal.is_goal(model.with_roles(s_) if hasattr(model, \\\"with_roles\\\") else s_) else None\\n\\n    def _search_one(self, scene: Scene, model, goal, available, semantics, depth: int, visited: set) -> tuple[Optional[list[Action]], str]:\\n        actions_fn = self._actions_fn(available, semantics)\\n        succ = make_successor(model, actions_fn, self.observed)\\n        setattr(goal, \\\"model_hint\\\", model)\\n        ab = self._abstract(scene, model, goal)\\n        if ab:\\n            return ab, \\\"abstract\\\"\\n\\n        # a goal's progress heuristic, damped when it proved non-monotone (spec \\u00a79)\\n        w = self.weight * getattr(goal, \\\"progress_weight\\\", 1.0)\\n        if self.stochastic:\\n            p = mcts(scene, lambda s, a: model.predict(s, a), actions_fn, goal.is_goal, goal.progress)\\n            return p, \\\"mcts\\\"\\n        est = getattr(goal, \\\"estimate\\\", None)\\n        p = astar(scene, succ, goal.is_goal, goal.progress, depth=depth, weight=w, visited_penalty=visited, time_limit=self.time_limit,\\n                  heuristic=(lambda s_, e=est: e(s_)) if est else None)\\n        if p is not None:\\n            return p, \\\"astar\\\"\\n        p = beam_search(scene, succ, goal.is_goal, goal.progress, depth=depth, width=self.beam_width, time_limit=self.time_limit)\\n        return p, \\\"beam\\\" if p is not None else \\\"none\\\"\\n\\n    def search(self, scene: Scene, H: list[Hypothesis], G: list, *, available: list[Action], semantics=None, depth: Optional[int] = None,\\n               visited: Optional[set] = None) -> Optional[list[Action]]:\\n        \\\"\\\"\\\"Multi-hypothesis planning (spec \\u00a710): top-3 hypotheses x top-2 goals, execute only the common prefix.\\\"\\\"\\\"\\n        depth = depth or self.depth\\n        plans = []\\n        info = PlanInfo([], algorithm=\\\"\\\", reason=\\\"\\\")\\n        t0 = time.perf_counter()\\n        ranked = sorted(H, key=lambda h: -h.plan_weight())\\n        for h in ranked[:3]:\\n            for g in G[:3]:\\n                p, alg = self._search_one(scene, h.model, g, available, semantics, depth, visited or set())\\n                if p:\\n                    plans.append((h.plan_weight() * max(g.confidence, 0.05), p, h, g, alg))\\n        if not plans:\\n            # a verified model can still be useless for planning (UNKNOWN outside the observed states): try the rest\\n            for h in ranked[3:]:\\n                for g in G[:2]:\\n                    p, alg = self._search_one(scene, h.model, g, available, semantics, depth, visited or set())\\n                    if p:\\n                        plans.append((h.plan_weight() * max(g.confidence, 0.05) * 0.5, p, h, g, alg))\\n        info.scored = [(round(s, 3), len(p), h.name, g.name, alg) for s, p, h, g, alg in plans]\\n        if not plans:\\n            info.reason = \\\"no plan under any (hypothesis, goal) pair\\\"\\n            self.last_info = info\\n            # diagnostics: which hypotheses were tried and how many successors the root state has under each\\n            diag = []\\n            for h in ranked[:5]:\\n                try:\\n                    succ = make_successor(h.model, self._actions_fn(available, semantics), self.observed)\\n                    n_succ = sum(1 for _ in succ(scene))\\n                except Exception as e:\\n                    n_succ = f\\\"err:{type(e).__name__}\\\"\\n                diag.append(f\\\"{h.name}:w={h.plan_weight():.2f},succ={n_succ}\\\")\\n            self.log(f\\\"planner: no plan ({time.perf_counter() - t0:.2f}s) H={len(H)} G={len(G)} top_goals={[g.name for g in G[:3]]} {' '.join(diag)}\\\")\\n            return None\\n        plans.sort(key=lambda x: (-x[0], len(x[1])))\\n        best = plans[0]\\n        # the common prefix is taken across HYPOTHESES for the best goal (where the plans diverge, the next action is\\n        # the experiment that separates them); different goals are not averaged\\n        same_goal = [p for _, p, _, g, _ in plans if g is best[3]][:3]\\n        prefix = _common_prefix(same_goal)\\n        info.plan = prefix if prefix else best[1][:1]\\n        info.hypothesis, info.goal, info.algorithm = best[2], best[3], best[4]\\n        info.reason = f\\\"{len(plans)} plans; prefix {len(prefix)} of best {len(best[1])}\\\"\\n        self.last_info = info\\n        self.log(f\\\"planner: {info.reason} via {info.algorithm} goal={best[3].name} h={best[2].name} ({time.perf_counter() - t0:.2f}s)\\\")\\n        return info.plan\\n\\n    def execute(self, plan: list[Action], H: list[Hypothesis], session, G=None, *, kind: str = \\\"plan\\\") -> ExecResult:\\n        \\\"\\\"\\\"Run a plan step by step; stop immediately on level change, game over, prediction mismatch or stuck (spec \\u00a710).\\\"\\\"\\\"\\n        current = session.scene\\n        chosen = self.last_info.hypothesis if self.last_info and self.last_info.hypothesis in H else (H[0] if H else None)\\n        model = chosen.model if chosen else None\\n        top_goal = G[0] if G else None\\n        for i, a in enumerate(plan):\\n            pred = None\\n            if model is not None:\\n                try:\\n                    pred = model.predict(current, a)\\n                except Exception:\\n                    pred = None\\n            t = session.act(a, kind)\\n            if t is None:\\n                return ExecResult(\\\"ERROR\\\", current, None, i, {\\\"reason\\\": \\\"env error or budget\\\"})\\n            if t.status_change is None:\\n                self.observed[(state_key(t.before), a.label())] = t.after\\n            if a.type == \\\"CLICK\\\" and not t.diff.is_noop:\\n                self.responsive.append((a.row, a.col)); self.responsive = self.responsive[-20:]\\n            if t.status_change in (\\\"LEVEL_UP\\\", \\\"WIN\\\"):\\n                return ExecResult(\\\"PROGRESS\\\", t.after, t, i + 1)\\n            if t.status_change == \\\"GAME_OVER\\\":\\n                return ExecResult(\\\"FAILED\\\", t.after, t, i + 1)\\n            if pred is not None and not _equal_ignoring_ui(pred, t.after, model):\\n                if chosen is not None:\\n                    chosen.recent_mismatches += 1\\n                return ExecResult(\\\"MISMATCH\\\", t.after, t, i + 1, {\\\"mismatch\\\": scene_mismatch(pred, t.after)})\\n            if pred is None and model is not None:\\n                return ExecResult(\\\"MISMATCH\\\", t.after, t, i + 1, {\\\"mismatch\\\": \\\"unknown prediction\\\"})\\n            prog = top_goal.progress(t.after) if top_goal is not None else None\\n            why = self.stuck.check(t.after.frame_hash, prog)\\n            if why:\\n                return ExecResult(\\\"STUCK\\\", t.after, t, i + 1, {\\\"reason\\\": why})\\n            current = t.after\\n        return ExecResult(\\\"EXHAUSTED\\\", current, None, len(plan))\\n\\n\\ndef _equal_ignoring_ui(pred: Scene, obs: Scene, model=None) -> bool:\\n    from ..wml.evaluate import _equal\\n    return _equal(pred, obs, True, None, model)\\n\\n\\ndef _common_prefix(plans: list[list[Action]]) -> list[Action]:\\n    if not plans:\\n        return []\\n    out = []\\n    for i in range(min(len(p) for p in plans)):\\n        a = plans[0][i]\\n        if all(p[i] == a for p in plans):\\n            out.append(a)\\n        else:\\n            break\\n    return out\\n\", \"pbg/planner/mcts.py\": \"\\\"\\\"\\\"planner/mcts.py \\u2014 MCTS for the stochastic-transition mode (spec \\u00a710): maximise expected progress, 200 rollouts.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport math\\nimport random\\nfrom typing import Callable, Iterable, Optional\\n\\nfrom ..core.types import Action, Scene\\n\\n\\nclass _Node:\\n    __slots__ = (\\\"scene\\\", \\\"parent\\\", \\\"action\\\", \\\"children\\\", \\\"n\\\", \\\"value\\\", \\\"untried\\\")\\n\\n    def __init__(self, scene, parent, action, actions):\\n        self.scene, self.parent, self.action = scene, parent, action\\n        self.children: list[_Node] = []; self.n = 0; self.value = 0.0; self.untried = list(actions)\\n\\n\\ndef mcts(start: Scene, step: Callable[[Scene, Action], Optional[Scene]], actions_fn: Callable[[Scene], list[Action]],\\n         is_goal: Callable[[Scene], bool], progress: Callable[[Scene], float], *, rollouts: int = 200, horizon: int = 20,\\n         c: float = 1.0, rng: Optional[random.Random] = None) -> Optional[list[Action]]:\\n    rng = rng or random.Random(0)\\n    root = _Node(start, None, None, actions_fn(start))\\n    if is_goal(start):\\n        return []\\n    for _ in range(rollouts):\\n        node = root; depth = 0\\n        while not node.untried and node.children:\\n            node = max(node.children, key=lambda ch: ch.value / max(1, ch.n) + c * math.sqrt(math.log(max(1, node.n)) / max(1, ch.n)))\\n            depth += 1\\n        if node.untried:\\n            a = node.untried.pop(rng.randrange(len(node.untried)))\\n            nxt = step(node.scene, a)\\n            if nxt is None:\\n                continue\\n            child = _Node(nxt, node, a, actions_fn(nxt)); node.children.append(child); node = child; depth += 1\\n        # rollout\\n        s = node.scene; reward = progress(s)\\n        for _ in range(max(0, horizon - depth)):\\n            if is_goal(s):\\n                reward = 1.0; break\\n            acts = actions_fn(s)\\n            if not acts:\\n                break\\n            nxt = step(s, rng.choice(acts))\\n            if nxt is None:\\n                break\\n            s = nxt; reward = max(reward, progress(s))\\n        while node is not None:\\n            node.n += 1; node.value += reward; node = node.parent\\n    if not root.children:\\n        return None\\n    path = []; node = root\\n    while node.children:\\n        node = max(node.children, key=lambda ch: ch.n)\\n        path.append(node.action)\\n        if is_goal(node.scene):\\n            break\\n    return path[:1] if path else None\\n\", \"pbg/planner/stuck.py\": \"\\\"\\\"\\\"planner/stuck.py \\u2014 stuck detector (spec \\u00a710): <= 5 distinct frame hashes in the last 20 steps, or no top-goal\\nprogress increase in 15 steps.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import deque\\nfrom typing import Optional\\n\\n\\nclass StuckDetector:\\n    def __init__(self, window: int = 20, min_distinct: int = 5, progress_window: int = 15):\\n        self.hashes: deque = deque(maxlen=window)\\n        self.progress: deque = deque(maxlen=progress_window)\\n        self.min_distinct = min_distinct\\n        self.visited: set[str] = set()\\n\\n    def reset(self) -> None:\\n        self.hashes.clear(); self.progress.clear()\\n\\n    def check(self, frame_hash: str, progress: Optional[float] = None) -> Optional[str]:\\n        self.hashes.append(frame_hash); self.visited.add(frame_hash)\\n        if progress is not None:\\n            self.progress.append(progress)\\n        if len(self.hashes) == self.hashes.maxlen and len(set(self.hashes)) <= self.min_distinct:\\n            return \\\"few_distinct_states\\\"\\n        if len(self.progress) == self.progress.maxlen and max(self.progress) <= self.progress[0] + 1e-9:\\n            return \\\"no_progress\\\"\\n        return None\\n\", \"pbg/probe/__init__.py\": \"from .protocol import Probe, ProbeResult  # noqa: F401\\nfrom .semantics import ActionSemantics, classify_actions  # noqa: F401\\n\", \"pbg/probe/clickmap.py\": \"\\\"\\\"\\\"probe/clickmap.py \\u2014 click response map: every click in the log clustered by its TARGET CLASS (colour + shape of the\\nobject under the cursor, or the background cell), with how often that class reacted. Shared by the walk probe, the\\ninformation-gain experiments and the planner so that none of them keeps clicking things already known to be inert.\\n\\nThe class key is level-independent (an arrow button of colour 9 stays an arrow button on the next level), so the map is\\na knowledge asset that accumulates across levels: known-responsive classes are tried first on a new level, known-inert\\nclasses at most once.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom typing import Iterable, Optional\\n\\nfrom ..core.types import Action, Scene, Transition\\nfrom .semantics import core_diff\\n\\nBG_CELL = 8   # background clicks are clustered by 8x8 cell\\n\\n\\ndef click_key(scene: Scene, row: int, col: int) -> str:\\n    \\\"\\\"\\\"Class of a click target: 'c<colour>:<shape>' for an object, 'bg:<region>:<r8>,<c8>' for empty space.\\\"\\\"\\\"\\n    for o in sorted(scene.objects, key=lambda o: o.area):          # smallest object first (a marker on a bar)\\n        r0, c0, r1, c1 = o.bbox\\n        if r0 <= row < r1 and c0 <= col < c1:\\n            return object_key(o)\\n    reg = next((r.id for r in scene.regions if r.id != \\\"R0\\\" and r.bbox[0] <= row < r.bbox[2] and r.bbox[1] <= col < r.bbox[3]), \\\"R0\\\")\\n    return f\\\"bg:{reg}:{row // BG_CELL},{col // BG_CELL}\\\"\\n\\n\\ndef object_key(o) -> str:\\n    \\\"\\\"\\\"Class of an object for the click map: colour + shape, but a solid rectangle is keyed by colour + thickness so a\\n    bar whose length changes stays one class.\\\"\\\"\\\"\\n    if o.mask is not None and min(o.height, o.width) >= 2 and float(o.mask.mean()) >= 0.75:\\n        return f\\\"c{o.color}:rect\\\"\\n    return f\\\"c{o.color}:{o.shape_sig[:8]}\\\"\\n\\n\\nclass ClickMap:\\n    def __init__(self) -> None:\\n        self.n: Counter = Counter()             # class -> clicks\\n        self.responsive: Counter = Counter()    # class -> clicks that changed something outside ui strips\\n        self.effects: dict[str, Counter] = {}   # class -> Counter of (moved colour, displacement) / 'recolor' / 'remove' / 'spawn'\\n        self.seen_ids: set[str] = set()         # transition ids already counted\\n        self.tried_level: dict[int, set[str]] = {}   # level -> classes clicked on that level\\n        self.skip_untried: set[str] = set()          # classes hypothesised passive on this level: never probed as 'untried'\\n\\n    # \\u2500\\u2500 building \\u2500\\u2500\\n    def add(self, t: Transition) -> None:\\n        if t.action.type != \\\"CLICK\\\" or t.id in self.seen_ids:\\n            return\\n        self.seen_ids.add(t.id)\\n        key = click_key(t.before, t.action.row, t.action.col)\\n        self.n[key] += 1\\n        self.tried_level.setdefault(t.level, set()).add(key)\\n        c = core_diff(t)\\n        if c[\\\"moved\\\"] or c[\\\"recolored\\\"] or c[\\\"reshaped\\\"] or c[\\\"appeared\\\"] or c[\\\"disappeared\\\"]:\\n            self.responsive[key] += 1\\n            eff = self.effects.setdefault(key, Counter())\\n            for i, v in c[\\\"moved\\\"]:\\n                o = t.before.get(i)\\n                eff[(\\\"move\\\", o.color if o else -1, tuple(v))] += 1\\n            for x in c[\\\"recolored\\\"]:\\n                eff[(\\\"recolor\\\", x[1], x[2])] += 1\\n            for x in c[\\\"reshaped\\\"]:\\n                eff[(\\\"reshape\\\",)] += 1\\n            if c[\\\"appeared\\\"]:\\n                eff[(\\\"spawn\\\",)] += 1\\n            if c[\\\"disappeared\\\"]:\\n                eff[(\\\"remove\\\",)] += 1\\n\\n    def extend(self, ts: Iterable[Transition]) -> None:\\n        for t in ts:\\n            self.add(t)\\n\\n    @classmethod\\n    def from_log(cls, ts: Iterable[Transition]) -> \\\"ClickMap\\\":\\n        m = cls(); m.extend(ts); return m\\n\\n    # \\u2500\\u2500 queries \\u2500\\u2500\\n    def status(self, key: str) -> str:\\n        \\\"\\\"\\\"'untried' | 'responsive' | 'inert'.\\\"\\\"\\\"\\n        if self.n[key] == 0:\\n            return \\\"untried\\\"\\n        return \\\"responsive\\\" if self.responsive[key] > 0 else \\\"inert\\\"\\n\\n    def total(self) -> int:\\n        return sum(self.n.values())\\n\\n    def rank(self, scene: Scene, *, level: Optional[int] = None, include_inert: bool = False, limit: int = 64,\\n             extra: Iterable[tuple[int, int]] = ()) -> list[Action]:\\n        \\\"\\\"\\\"Click candidates on this scene in priority order: untried classes, then responsive classes (one object per\\n        class, largest first), then background cells never tried; inert classes only with `include_inert`, and then\\n        only if not yet clicked on this level (one re-check per level: a submit button may wake up).\\\"\\\"\\\"\\n        strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n        tried_here = self.tried_level.get(level, set()) if level is not None else set()\\n        untried: list[Action] = []; resp: list[Action] = []; inert: list[Action] = []\\n        seen_keys: set[str] = set()\\n        for o in sorted(scene.objects, key=lambda o: (-o.area, o.id)):\\n            if o.region in strips or o.area < 2:\\n                continue\\n            key = object_key(o)\\n            if key in seen_keys:\\n                continue\\n            seen_keys.add(key)\\n            a = Action.click(*o.center)\\n            st = self.status(key)\\n            if st == \\\"untried\\\":\\n                if key in self.skip_untried:\\n                    continue\\n                untried.append(a)\\n            elif st == \\\"responsive\\\":\\n                resp.append(a)\\n            elif include_inert and key not in tried_here:\\n                inert.append(a)\\n        # responsive classes with several objects (four arrow buttons of one colour): every object of the class matters\\n        for o in sorted(scene.objects, key=lambda o: (-o.area, o.id)):\\n            key = object_key(o)\\n            if o.region in strips or o.area < 2 or self.status(key) != \\\"responsive\\\":\\n                continue\\n            a = Action.click(*o.center)\\n            if a not in resp:\\n                resp.append(a)\\n        bg: list[Action] = []\\n        for r in scene.regions:\\n            if r.kind_hint == \\\"ui_strip\\\":\\n                continue\\n            a = Action.click(*r.center)\\n            if self.status(click_key(scene, a.row, a.col)) == \\\"untried\\\" and not any(x == a for x in untried + resp) and not self.skip_untried:\\n                bg.append(a)\\n        for rc in extra:\\n            a = Action.click(int(rc[0]), int(rc[1]))\\n            if a not in resp and a not in untried:\\n                resp.append(a)\\n        return (untried + resp + bg + inert)[:limit]\\n\\n    # \\u2500\\u2500 persistence \\u2500\\u2500\\n    def to_json(self) -> dict:\\n        return {\\\"n\\\": dict(self.n), \\\"responsive\\\": dict(self.responsive),\\n                \\\"effects\\\": {k: [[list(e) if isinstance(e, tuple) else e, c] for e, c in v.items()] for k, v in self.effects.items()},\\n                \\\"tried_level\\\": {str(l): sorted(v) for l, v in self.tried_level.items()}}\\n\\n    @classmethod\\n    def from_json(cls, d: dict) -> \\\"ClickMap\\\":\\n        m = cls()\\n        m.n = Counter(d.get(\\\"n\\\", {})); m.responsive = Counter(d.get(\\\"responsive\\\", {}))\\n        m.effects = {k: Counter({tuple(_tuplify(e)): c for e, c in v}) for k, v in d.get(\\\"effects\\\", {}).items()}\\n        m.tried_level = {int(l): set(v) for l, v in d.get(\\\"tried_level\\\", {}).items()}\\n        return m\\n\\n    def summary(self) -> str:\\n        parts = []\\n        for k in sorted(self.n, key=lambda k: (-self.responsive[k], -self.n[k])):\\n            parts.append(f\\\"{k}:{self.responsive[k]}/{self.n[k]}\\\")\\n        return \\\" \\\".join(parts[:12])\\n\\n\\ndef _tuplify(x):\\n    return tuple(_tuplify(y) for y in x) if isinstance(x, list) else x\\n\", \"pbg/probe/hypotheses.py\": \"\\\"\\\"\\\"probe/hypotheses.py \\u2014 look first, then test: role and relation hypotheses read off the FIRST frame, and the fewest\\nclicks that confirm or refute them.\\n\\nRoles are guessed from appearance (a set of identical small squares = buttons; a solid bar attached to a wall or border =\\na bar/tank; a small multi-colour piece = a mover; a small mark whose colour a mover carries = a marker; a thin long line\\n= a wall). Relations follow from geometry (a button controls the bars nearest to it; a mover rides the bar it stands on;\\na marker belongs to the mover of its colour). The probe then clicks ONE button candidate at a time: a change in the\\npredicted bars confirms the hypothesis (and one observation is enough to instantiate its effect rule); no change or a\\nchange elsewhere refutes it and the roles are revised. Objects hypothesised as non-interactive are not clicked at all\\nunless every trigger hypothesis fails.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom dataclasses import dataclass, field\\nfrom typing import Optional\\n\\nfrom ..core.types import Action, Object, Scene, Transition\\nfrom .clickmap import object_key\\nfrom .semantics import core_diff\\n\\n\\n@dataclass\\nclass RoleHyp:\\n    role: str\\n    p: float\\n    why: str\\n    confirmed: Optional[bool] = None\\n\\n\\ndef _solid(o: Object) -> bool:\\n    return o.mask is not None and min(o.height, o.width) >= 2 and float(o.mask.mean()) >= 0.75\\n\\n\\ndef _gap(a: tuple, b: tuple) -> int:\\n    \\\"\\\"\\\"Manhattan gap between two bboxes (0 when they touch or overlap).\\\"\\\"\\\"\\n    dr = max(0, max(a[0], b[0]) - min(a[2], b[2])); dc = max(0, max(a[1], b[1]) - min(a[3], b[3]))\\n    return dr + dc\\n\\n\\nclass BoardHypotheses:\\n    def __init__(self, scene: Scene):\\n        self.scene = scene\\n        self.roles: dict[int, RoleHyp] = {}\\n        self.controls: dict[int, list[int]] = {}      # button -> bars it probably moves (nearest first)\\n        self.rides: dict[int, int] = {}               # mover -> bar it stands on\\n        self.marker_of: dict[int, int] = {}           # marker -> mover with the same colour\\n        self.tested: dict[int, str] = {}              # button -> \\\"confirmed\\\" | \\\"refuted\\\"\\n        self.controls_other: dict[int, bool] = {}     # button -> its test click changed objects other than itself\\n        # ids can be re-assigned when the level is re-parsed after a test click (a panel turned out to be a bar):\\n        # objects are recognised by colour + bbox, which the same frame keeps\\n        self._key2id = {(o.color, tuple(o.bbox)): o.id for o in scene.objects}\\n        self._infer()\\n\\n    def _orig(self, o: Object) -> int:\\n        return self._key2id.get((o.color, tuple(o.bbox)), o.id)\\n\\n    # \\u2500\\u2500 appearance -> roles \\u2500\\u2500\\n    def _infer(self) -> None:\\n        s = self.scene\\n        strips = {r.id for r in s.regions if r.kind_hint == \\\"ui_strip\\\"}\\n        objs = [o for o in s.objects if o.region not in strips and o.area >= 2]\\n        shape_count = Counter((o.color, o.shape_sig) for o in objs)\\n        multi_colors = {c for o in objs if len(o.colors) > 1 for c in o.colors}\\n        small_single = [o for o in objs if len(o.colors) == 1 and o.area <= 12]\\n        for o in objs:\\n            single = len(o.colors) == 1\\n            if single and _solid(o) and o.area <= 36 and shape_count[(o.color, o.shape_sig)] >= 2 and abs(o.height - o.width) <= 2:\\n                self.roles[o.id] = RoleHyp(\\\"button\\\", 0.8, f\\\"{shape_count[(o.color, o.shape_sig)]} identical small squares of colour {o.color}\\\")\\n            elif not single and o.area <= 60:\\n                self.roles[o.id] = RoleHyp(\\\"mover\\\", 0.7, \\\"small multi-colour piece\\\")\\n            elif single and o.area <= 12 and (o.color in multi_colors or any(q.color == o.color and q.id != o.id and (q.height, q.width) != (o.height, o.width) for q in small_single)):\\n                self.roles[o.id] = RoleHyp(\\\"marker\\\", 0.7, f\\\"small mark of colour {o.color} that a piece also carries\\\")\\n            elif single and min(o.height, o.width) <= 4 and max(o.height, o.width) >= 12 and not _solid(o) is False:\\n                self.roles[o.id] = RoleHyp(\\\"wall\\\", 0.6, \\\"thin long line\\\")\\n            elif single and _solid(o) and o.area >= 12:\\n                r0, c0, r1, c1 = o.bbox\\n                border = r0 == 0 or c0 == 0 or r1 == s.grid_shape[0] or c1 == s.grid_shape[1]\\n                self.roles[o.id] = RoleHyp(\\\"bar\\\", 0.6 if border else 0.4, \\\"solid block\\\" + (\\\" attached to the border\\\" if border else \\\"\\\"))\\n            elif single and _solid(o) and o.area <= 36:\\n                self.roles[o.id] = RoleHyp(\\\"button\\\", 0.4, \\\"lone small square\\\")\\n            else:\\n                self.roles[o.id] = RoleHyp(\\\"decoration\\\", 0.3, \\\"no pattern\\\")\\n        # walls that are thin lines vs bars: a bar next to a wall keeps 'bar'\\n        by = {o.id: o for o in objs}\\n        bars = [i for i, h in self.roles.items() if h.role == \\\"bar\\\"]\\n        for i, h in self.roles.items():\\n            o = by[i]\\n            if h.role == \\\"button\\\":\\n                near = sorted(bars, key=lambda b: _gap(o.bbox, by[b].bbox))\\n                self.controls[i] = [b for b in near if _gap(o.bbox, by[b].bbox) <= 12][:2]\\n            elif h.role == \\\"mover\\\":\\n                touching = [b for b in bars if _gap(o.bbox, by[b].bbox) == 0]\\n                if touching:\\n                    self.rides[i] = touching[0]\\n            elif h.role == \\\"marker\\\":\\n                mv = [m for m, hm in self.roles.items() if hm.role == \\\"mover\\\" and o.color in by[m].colors]\\n                if mv:\\n                    self.marker_of[i] = mv[0]\\n\\n    # \\u2500\\u2500 what to test \\u2500\\u2500\\n    def test_actions(self) -> list[Action]:\\n        \\\"\\\"\\\"One click per button CLASS first (23 identical pieces are one hypothesis, not 23); the other members of a class\\n        are tested only after its representative proved to control OTHER objects (arrow buttons each move their own bar),\\n        never when clicking it changed the clicked object itself (pieces, pattern cells).\\\"\\\"\\\"\\n        by = {o.id: o for o in self.scene.objects}\\n        cands = [(h.p + (0.1 if self.controls.get(i) else 0.0), i) for i, h in self.roles.items() if h.role == \\\"button\\\" and i not in self.tested and i in by]\\n        cands.sort(reverse=True)\\n        out = []; seen_class: set = set()\\n        for _, i in cands:\\n            cls = (by[i].color, by[i].shape_sig)\\n            if cls in seen_class:\\n                continue\\n            tested_same = [j for j in self.tested if j in by and (by[j].color, by[j].shape_sig) == cls]\\n            if tested_same and not any(self.controls_other.get(j) for j in tested_same):\\n                continue          # the class representative changed itself or nothing: no per-object testing\\n            seen_class.add(cls) if not tested_same else None\\n            out.append(Action.click(*by[i].center))\\n        return out\\n\\n    def fallback_actions(self) -> list[Action]:\\n        \\\"\\\"\\\"Only when no trigger hypothesis held: the other object classes, one each, decorations last.\\\"\\\"\\\"\\n        by = {o.id: o for o in self.scene.objects}\\n        order = {\\\"marker\\\": 0, \\\"mover\\\": 1, \\\"bar\\\": 2, \\\"wall\\\": 3, \\\"decoration\\\": 4}\\n        seen: set[str] = set(); out = []\\n        for i, h in sorted(self.roles.items(), key=lambda x: order.get(x[1].role, 5)):\\n            o = by.get(i)\\n            if o is None or h.role == \\\"button\\\" or object_key(o) in seen:\\n                continue\\n            seen.add(object_key(o)); out.append(Action.click(*o.center))\\n        return out\\n\\n    def non_interactive_classes(self) -> set[str]:\\n        by = {o.id: o for o in self.scene.objects}\\n        return {object_key(by[i]) for i, h in self.roles.items() if h.role in (\\\"bar\\\", \\\"wall\\\", \\\"marker\\\", \\\"mover\\\", \\\"decoration\\\") and i in by}\\n\\n    # \\u2500\\u2500 evidence \\u2500\\u2500\\n    def observe(self, t: Transition) -> Optional[str]:\\n        \\\"\\\"\\\"Update the hypotheses with one probe transition; returns a one-line verdict for the event log.\\\"\\\"\\\"\\n        a = t.action\\n        if a.type != \\\"CLICK\\\":\\n            return None\\n        by = {o.id: o for o in t.before.objects}\\n        hit = next((o for o in t.before.objects if o.bbox[0] <= a.row < o.bbox[2] and o.bbox[1] <= a.col < o.bbox[3] and o.mask[a.row - o.bbox[0], a.col - o.bbox[1]]), None)\\n        if hit is None:\\n            return None\\n        c = core_diff(t)\\n        changed_ids = {i for i, _ in c[\\\"moved\\\"]} | {x[0] for x in c[\\\"recolored\\\"] + c[\\\"reshaped\\\"]} | {o.id for o in c[\\\"disappeared\\\"]}\\n        changed = {self._orig(by[i]) for i in changed_ids if i in by}\\n        hid = self._orig(hit)\\n        h = self.roles.get(hid)\\n        if h is None:\\n            return None\\n        if h.role == \\\"button\\\":\\n            if changed:\\n                expected = set(self.controls.get(hid, [])) | {m for m, b in self.rides.items() if b in self.controls.get(hid, [])}\\n                agree = bool(expected & changed)\\n                self.tested[hid] = \\\"confirmed\\\"; h.confirmed = True; h.p = 1.0\\n                self.controls_other[hid] = bool(changed - {hid})\\n                if not agree and expected:\\n                    # it is a trigger, but of other objects than the geometry suggested: keep the observed targets\\n                    self.controls[hid] = [i for i in changed if self.roles.get(i, RoleHyp(\\\"\\\", 0, \\\"\\\")).role == \\\"bar\\\"]\\n                orig_by = {o.id: o for o in self.scene.objects}\\n                what = \\\", \\\".join(f\\\"{orig_by[i].color}@{orig_by[i].bbox[0]},{orig_by[i].bbox[1]}\\\" for i in sorted(changed) if i in orig_by)[:80]\\n                return f\\\"button {hid} confirmed ({'as predicted' if agree else 'other targets'}): changed {what}\\\"\\n            self.tested[hid] = \\\"refuted\\\"; h.confirmed = False; h.p *= 0.3\\n            return f\\\"button {hid} refuted: no change (blocked or not a trigger)\\\"\\n        if changed:\\n            # a supposedly passive object reacted: it is a trigger after all\\n            self.roles[hid] = RoleHyp(\\\"button\\\", 0.9, f\\\"reacted when clicked (was {h.role})\\\", confirmed=True)\\n            self.tested[hid] = \\\"confirmed\\\"\\n            return f\\\"{h.role} {hid} reacted -> promoted to button\\\"\\n        return f\\\"{h.role} {hid} inert as expected\\\"\\n\\n    def confirmed_triggers(self) -> set[tuple]:\\n        by = {o.id: o for o in self.scene.objects}\\n        return {tuple(by[i].bbox) for i, v in self.tested.items() if v == \\\"confirmed\\\" and i in by}\\n\\n    def summary(self) -> str:\\n        cnt = Counter(h.role for h in self.roles.values())\\n        rel = f\\\"controls={len([b for b in self.controls.values() if b])} rides={len(self.rides)} markers={len(self.marker_of)}\\\"\\n        return f\\\"roles {dict(cnt)}; {rel}; tested {dict(Counter(self.tested.values()))}\\\"\\n\", \"pbg/probe/protocol.py\": \"\\\"\\\"\\\"probe/protocol.py \\u2014 Probe (spec \\u00a77): a fixed exploration protocol, not random search, within a budget cap.\\n\\nInitial protocol (game start):\\n  1. every button once in id order; 2. every non-noop button once more (consistency); 3. clicks: each object centre once,\\n  each region background centre once; 4. if nothing changed: triple-taps and ordered pairs; 5. RESET once if available\\n  (does it cost budget, what does it restore).\\nLocal protocol (new level / after a prediction error): move next to novel objects and press each direction once;\\nreproduce the mismatching (state, action) once. Cap: 5% of the total budget.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass, field\\nfrom typing import Callable, Optional\\n\\nfrom ..core.types import Action, Scene, Transition\\nfrom .semantics import ActionSemantics, classify_actions\\n\\nBUTTON_ORDER = (1, 2, 3, 4, 5, 7)\\n\\n\\n@dataclass\\nclass ProbeResult:\\n    hypotheses: object = None       # BoardHypotheses of the clicked frame (click games)\\n    verdicts: list = None           # one line per tested hypothesis\\n    transitions: list[Transition] = field(default_factory=list)\\n    semantics: ActionSemantics = field(default_factory=ActionSemantics)\\n    actions_used: int = 0\\n    stochastic: bool = False\\n    reset_costs_action: Optional[bool] = None\\n    reset_restores: Optional[bool] = None\\n    stopped_early: bool = False\\n    steps_done: list[str] = field(default_factory=list)\\n\\n\\nclass Probe:\\n    \\\"\\\"\\\"`session` must provide: act(action, kind) -> Transition | None (None = game ended / error), scene (current Scene),\\n    available_actions() -> list[Action], level, and status(). Budget caps are passed per run.\\\"\\\"\\\"\\n\\n    def __init__(self, session):\\n        self.s = session\\n\\n    # \\u2500\\u2500 helpers \\u2500\\u2500\\n    def _buttons(self) -> list[Action]:\\n        avail = {a.id for a in self.s.available_actions() if a.type == \\\"BUTTON\\\"}\\n        return [Action.button(i) for i in BUTTON_ORDER if i in avail]\\n\\n    def _has_click(self) -> bool:\\n        return any(a.type == \\\"CLICK\\\" for a in self.s.available_actions())\\n\\n    def _has_reset(self) -> bool:\\n        return getattr(self.s, \\\"reset_allowed\\\", True)\\n\\n    def _click_targets(self, scene: Scene, limit: int) -> list[Action]:\\n        \\\"\\\"\\\"Click targets in priority order: one object per (colour, shape) class (largest first), then region centres,\\n        then the remaining objects of already-seen classes. Independent of the budget size: the cheap, informative\\n        clicks always come first and `limit` only cuts the tail.\\\"\\\"\\\"\\n        seen: set[tuple] = set(); first: list[Action] = []; rest: list[Action] = []\\n        strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n        for o in sorted(scene.objects, key=lambda o: (-o.area, o.id)):\\n            if o.region in strips or o.area < 2:\\n                continue\\n            key = (o.color, o.shape_sig)\\n            (rest if key in seen else first).append(Action.click(*o.center))\\n            seen.add(key)\\n        regions = [Action.click(*r.center) for r in scene.regions if r.kind_hint != \\\"ui_strip\\\"]\\n        return (first + regions + rest)[:limit]\\n\\n    # \\u2500\\u2500 protocols \\u2500\\u2500\\n    def run_initial(self, budget_cap: int, *, do_reset_probe: bool = True) -> ProbeResult:\\n        res = ProbeResult()\\n        done_level = self.s.level\\n\\n        def act(a: Action, why: str) -> Optional[Transition]:\\n            if res.actions_used >= budget_cap or self.s.level != done_level or self.s.finished():\\n                res.stopped_early = True\\n                return None\\n            t = self.s.act(a, \\\"initial_probe\\\")\\n            if t is None:\\n                res.stopped_early = True\\n                return None\\n            res.actions_used += 1; res.transitions.append(t)\\n            return t\\n\\n        buttons = self._buttons()\\n        # 1. each button once\\n        first: dict[str, Transition] = {}\\n        for a in buttons:\\n            t = act(a, \\\"button-once\\\")\\n            if t is None:\\n                break\\n            first[a.key] = t\\n        res.steps_done.append(\\\"buttons\\\")\\n        # 2. every button once more (the state has changed since step 1, so a no-op may now act and vice versa)\\n        for a in buttons:\\n            if act(a, \\\"button-repeat\\\") is None:\\n                break\\n        res.steps_done.append(\\\"repeat\\\")\\n        # 3. clicks: look first, then test -- role/relation hypotheses from the frame, one click per trigger candidate;\\n        #    passive candidates (bars, markers, walls, movers) are clicked only if no trigger hypothesis holds\\n        if self._has_click():\\n            click_cap = min(budget_cap - res.actions_used - 2, int(getattr(self.s, \\\"initial_click_cap\\\", 16)))\\n            self._hypothesis_clicks(res, act, max(1, click_cap))\\n            res.steps_done.append(\\\"clicks\\\")\\n        # 4. nothing changed so far -> accumulation / combination mechanisms\\n        if res.transitions and all(t.diff.is_noop for t in res.transitions):\\n            for a in buttons:\\n                for _ in range(3):\\n                    if act(a, \\\"triple\\\") is None:\\n                        break\\n            for a in buttons:\\n                for b in buttons:\\n                    if a.key != b.key:\\n                        if act(a, \\\"pair\\\") is None or act(b, \\\"pair\\\") is None:\\n                            break\\n            res.steps_done.append(\\\"combos\\\")\\n        # 5. reset once\\n        if do_reset_probe and self._has_reset() and not res.stopped_early and res.actions_used < budget_cap and self.s.level == done_level:\\n            before_hash = res.transitions[0].before.frame_hash if res.transitions else None\\n            used_before = self.s.actions_used()\\n            t = self.s.act(Action.reset(), \\\"initial_probe\\\")\\n            if t is not None:\\n                res.actions_used += 1\\n                res.reset_costs_action = self.s.actions_used() > used_before\\n                res.reset_restores = (before_hash is not None and t.after.frame_hash == before_hash)\\n                res.steps_done.append(\\\"reset\\\")\\n        res.semantics = classify_actions(res.transitions)\\n        res.stochastic = self.s.status().stochasticity_score > 0.0\\n        return res\\n\\n    def _hypothesis_clicks(self, res: ProbeResult, act, cap: int) -> None:\\n        from .hypotheses import BoardHypotheses\\n        hyp = BoardHypotheses(self.s.scene)\\n        res.hypotheses = hyp; res.verdicts = []\\n        n = 0\\n        for a in hyp.test_actions():\\n            if n >= cap:\\n                break\\n            t = act(a, \\\"hypothesis-test\\\")\\n            if t is None:\\n                return\\n            n += 1\\n            v = hyp.observe(t)\\n            if v:\\n                res.verdicts.append(v)\\n        if not hyp.confirmed_triggers():\\n            for a in hyp.fallback_actions():\\n                if n >= cap:\\n                    break\\n                t = act(a, \\\"hypothesis-fallback\\\")\\n                if t is None:\\n                    return\\n                n += 1\\n                v = hyp.observe(t)\\n                if v:\\n                    res.verdicts.append(v)\\n\\n    def run_hypothesis_probe(self, budget_cap: int) -> ProbeResult:\\n        \\\"\\\"\\\"Level start: role/relation hypotheses on the new board and the clicks that test them (no walk).\\\"\\\"\\\"\\n        res = ProbeResult(); level = self.s.level\\n\\n        def act(a: Action, why: str) -> Optional[Transition]:\\n            if res.actions_used >= budget_cap or self.s.level != level or self.s.finished():\\n                res.stopped_early = True\\n                return None\\n            t = self.s.act(a, \\\"reprobe\\\")\\n            if t is None:\\n                res.stopped_early = True\\n                return None\\n            res.actions_used += 1; res.transitions.append(t)\\n            return t\\n        if self._has_click():\\n            self._hypothesis_clicks(res, act, budget_cap)\\n        res.steps_done.append(\\\"hypotheses\\\")\\n        res.semantics = classify_actions(res.transitions)\\n        return res\\n\\n    def run_walk(self, budget_cap: int, *, max_per_dir: int = 12, agent_ids: set = frozenset(), dirs: Optional[dict] = None,\\n                 bumped: Optional[set] = None, clicked: Optional[set] = None, click_map=None) -> ProbeResult:\\n        \\\"\\\"\\\"Re-exploration after STUCK / no plan (spec \\u00a77 budget row 'reprobe'): first bump every object adjacent to the agent\\n        once (passability may have changed: doors, keys), then press each move button repeatedly until the board stops\\n        changing, then untried clicks. Reveals walls, reach and side effects.\\\"\\\"\\\"\\n        res = ProbeResult()\\n        level = self.s.level\\n        bumped = bumped if bumped is not None else set()\\n\\n        def act(a: Action) -> Optional[Transition]:\\n            if res.actions_used >= budget_cap or self.s.level != level or self.s.finished():\\n                res.stopped_early = True\\n                return None\\n            t = self.s.act(a, \\\"reprobe\\\")\\n            if t is None:\\n                res.stopped_early = True\\n                return None\\n            res.actions_used += 1; res.transitions.append(t)\\n            return t\\n\\n        buttons = self._buttons()\\n        # 1. bump: step once toward each adjacent object not bumped before on this level (a door that opened, a key)\\n        if agent_ids and dirs:\\n            scene = self.s.scene\\n            agents = [o for o in scene.objects if o.id in agent_ids]\\n            for a in buttons:\\n                if a.id not in dirs:\\n                    continue\\n                dr, dc = dirs[a.id]\\n                for ag in agents:\\n                    ghost = ag.moved(dr, dc)\\n                    for o in scene.objects:\\n                        if o.id == ag.id or o.id in agent_ids or not ghost.overlaps(o):\\n                            continue\\n                        key = (level, o.identity())\\n                        if key in bumped:\\n                            continue\\n                        bumped.add(key)\\n                        if act(a) is None:\\n                            break\\n        for a in buttons:\\n            for _ in range(max_per_dir):\\n                t = act(a)\\n                if t is None or t.diff.is_noop or t.status_change:\\n                    break\\n        if self._has_click():\\n            scene = self.s.scene\\n            if click_map is not None:\\n                # click response map: untried classes, then classes known to react, then untried background; classes\\n                # known to be inert at most once per level. The list is NOT padded to the cap: nothing left = stop.\\n                targets = click_map.rank(scene, level=level, include_inert=True, limit=64)\\n                for a in targets[:max(0, budget_cap - res.actions_used)]:\\n                    t = act(a)\\n                    if t is None:\\n                        break\\n                    click_map.add(t)\\n            else:\\n                # objects never clicked on this level first (a submit button after the pieces are cleared), then the rest\\n                def ident(a: Action):\\n                    for o in scene.objects:\\n                        r0, c0, r1, c1 = o.bbox\\n                        if r0 <= a.row < r1 and c0 <= a.col < c1:\\n                            return (o.color, o.shape_sig)\\n                    return (\\\"bg\\\", a.row // 8, a.col // 8)\\n                targets = self._click_targets(scene, 64)\\n                fresh = [a for a in targets if clicked is None or ident(a) not in clicked]\\n                rest = [a for a in targets if a not in fresh]\\n                for a in (fresh + rest)[:max(0, budget_cap - res.actions_used)]:\\n                    if clicked is not None:\\n                        clicked.add(ident(a))\\n                    if act(a) is None:\\n                        break\\n        res.steps_done.append(\\\"walk\\\")\\n        res.semantics = classify_actions(res.transitions)\\n        return res\\n\\n    def run_local(self, budget_cap: int, *, novel_ids: list[int] = (), mismatch: Optional[Transition] = None,\\n                  navigate: Optional[Callable[[int], list[Action]]] = None) -> ProbeResult:\\n        res = ProbeResult()\\n        level = self.s.level\\n\\n        def act(a: Action) -> Optional[Transition]:\\n            if res.actions_used >= budget_cap or self.s.level != level or self.s.finished():\\n                res.stopped_early = True\\n                return None\\n            t = self.s.act(a, \\\"level_probe\\\")\\n            if t is None:\\n                res.stopped_early = True\\n                return None\\n            res.actions_used += 1; res.transitions.append(t)\\n            return t\\n\\n        buttons = self._buttons()\\n        for oid in list(novel_ids)[:3]:\\n            if navigate is not None:\\n                path = navigate(oid) or []\\n                for a in path[:max(0, budget_cap - res.actions_used - len(buttons))]:\\n                    if act(a) is None:\\n                        break\\n            for a in buttons:\\n                if act(a) is None:\\n                    break\\n        if mismatch is not None and self.s.scene.frame_hash == mismatch.before.frame_hash:\\n            act(mismatch.action)\\n        res.steps_done.append(\\\"local\\\")\\n        res.semantics = classify_actions(res.transitions)\\n        return res\\n\", \"pbg/probe/semantics.py\": \"\\\"\\\"\\\"probe/semantics.py \\u2014 ActionSemantics: deterministic classification of each action key from its Diff patterns\\n(spec \\u00a77). Classes: MOVE, TOGGLE, SPAWN, REMOVE, SELECT, NOOP, UNKNOWN.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom typing import Iterable, Optional\\n\\nfrom ..core.types import Scene, Transition\\n\\nCLASSES = (\\\"MOVE\\\", \\\"TRANSFORM\\\", \\\"TOGGLE\\\", \\\"SPAWN\\\", \\\"REMOVE\\\", \\\"SELECT\\\", \\\"NOOP\\\", \\\"UNKNOWN\\\")\\n\\n\\ndef _ui_ids(scene: Scene) -> set[int]:\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    return {o.id for o in scene.objects if o.region in strips}\\n\\n\\ndef core_diff(t: Transition) -> dict:\\n    \\\"\\\"\\\"Diff with ui-strip objects removed (counters change on every action and would hide the real effect).\\\"\\\"\\\"\\n    ui = _ui_ids(t.before) | _ui_ids(t.after)\\n    d = t.diff\\n    return {\\\"moved\\\": [(i, v) for i, v in d.moved if i not in ui], \\\"recolored\\\": [x for x in d.recolored if x[0] not in ui],\\n            \\\"reshaped\\\": [x for x in d.reshaped if x[0] not in ui], \\\"appeared\\\": [o for o in d.appeared if o.id not in ui],\\n            \\\"disappeared\\\": [o for o in d.disappeared if o.id not in ui],\\n            \\\"counters\\\": [i for i, _ in d.moved if i in ui] + [x[0] for x in d.recolored + d.reshaped if x[0] in ui]}\\n\\n\\ndef classify_one(t: Transition) -> tuple[str, dict]:\\n    c = core_diff(t)\\n    if not (c[\\\"moved\\\"] or c[\\\"recolored\\\"] or c[\\\"reshaped\\\"] or c[\\\"appeared\\\"] or c[\\\"disappeared\\\"]):\\n        return (\\\"NOOP\\\", {\\\"counters\\\": c[\\\"counters\\\"]}) if t.diff.is_noop or c[\\\"counters\\\"] or t.diff.pixel_changes == 0 else (\\\"UNKNOWN\\\", {\\\"pixels\\\": t.diff.pixel_changes})\\n    reshaped_ids = {x[0] for x in c[\\\"reshaped\\\"]}\\n    if reshaped_ids and all(i in reshaped_ids for i, _ in c[\\\"moved\\\"]) and not c[\\\"appeared\\\"] and not c[\\\"disappeared\\\"]:\\n        # every moved object also changed shape: a rotation / flip / re-drawing, not a translation\\n        return \\\"TRANSFORM\\\", {\\\"targets\\\": sorted(reshaped_ids), \\\"counters\\\": c[\\\"counters\\\"]}\\n    if c[\\\"moved\\\"] and not c[\\\"appeared\\\"] and not c[\\\"disappeared\\\"]:\\n        disp = Counter(v for _, v in c[\\\"moved\\\"])\\n        (dr, dc), n = disp.most_common(1)[0]\\n        return \\\"MOVE\\\", {\\\"displacement\\\": (dr, dc), \\\"movers\\\": [i for i, v in c[\\\"moved\\\"] if v == (dr, dc)], \\\"others\\\": len(c[\\\"moved\\\"]) - n,\\n                        \\\"recolored\\\": [x[0] for x in c[\\\"recolored\\\"]], \\\"counters\\\": c[\\\"counters\\\"]}\\n    if c[\\\"recolored\\\"] and not c[\\\"moved\\\"] and not c[\\\"appeared\\\"] and not c[\\\"disappeared\\\"]:\\n        if t.action.type == \\\"CLICK\\\":\\n            return \\\"SELECT\\\", {\\\"targets\\\": [x[0] for x in c[\\\"recolored\\\"]], \\\"counters\\\": c[\\\"counters\\\"]}\\n        return \\\"TOGGLE\\\", {\\\"targets\\\": [(i, a, b) for i, a, b in c[\\\"recolored\\\"]], \\\"counters\\\": c[\\\"counters\\\"]}\\n    if c[\\\"appeared\\\"] and not c[\\\"disappeared\\\"] and not c[\\\"moved\\\"]:\\n        return \\\"SPAWN\\\", {\\\"ids\\\": [o.id for o in c[\\\"appeared\\\"]], \\\"counters\\\": c[\\\"counters\\\"]}\\n    if c[\\\"disappeared\\\"] and not c[\\\"appeared\\\"] and not c[\\\"moved\\\"]:\\n        return \\\"REMOVE\\\", {\\\"ids\\\": [o.id for o in c[\\\"disappeared\\\"]], \\\"counters\\\": c[\\\"counters\\\"]}\\n    if t.action.type == \\\"CLICK\\\" and (c[\\\"reshaped\\\"] or c[\\\"recolored\\\"]):\\n        return \\\"SELECT\\\", {\\\"targets\\\": [x[0] for x in c[\\\"reshaped\\\"] + c[\\\"recolored\\\"]], \\\"counters\\\": c[\\\"counters\\\"]}\\n    if c[\\\"moved\\\"] and (c[\\\"appeared\\\"] or c[\\\"disappeared\\\"]):\\n        disp = Counter(v for _, v in c[\\\"moved\\\"])\\n        (dr, dc), n = disp.most_common(1)[0]\\n        return \\\"MOVE\\\", {\\\"displacement\\\": (dr, dc), \\\"movers\\\": [i for i, v in c[\\\"moved\\\"] if v == (dr, dc)], \\\"others\\\": len(c[\\\"moved\\\"]) - n,\\n                        \\\"removed\\\": [o.id for o in c[\\\"disappeared\\\"]], \\\"spawned\\\": [o.id for o in c[\\\"appeared\\\"]], \\\"counters\\\": c[\\\"counters\\\"]}\\n    return \\\"UNKNOWN\\\", {\\\"kinds\\\": sorted(t.diff.kinds()), \\\"counters\\\": c[\\\"counters\\\"]}\\n\\n\\nclass ActionSemantics(dict):\\n    \\\"\\\"\\\"{action_key: {\\\"class\\\", \\\"effect\\\", \\\"consistency\\\", ...}} plus \\\"_counters\\\" (ui objects that tick on actions).\\\"\\\"\\\"\\n\\n    def moves(self) -> dict[str, tuple[int, int]]:\\n        return {k: tuple(v[\\\"displacement\\\"]) for k, v in self.items() if not k.startswith(\\\"_\\\") and v.get(\\\"class\\\") == \\\"MOVE\\\" and v.get(\\\"displacement\\\")}\\n\\n    def mover_ids(self) -> set[int]:\\n        out: set[int] = set()\\n        for k, v in self.items():\\n            if not k.startswith(\\\"_\\\") and v.get(\\\"class\\\") == \\\"MOVE\\\":\\n                out |= set(v.get(\\\"movers\\\", []))\\n        return out\\n\\n    def noop_keys(self) -> set[str]:\\n        return {k for k, v in self.items() if not k.startswith(\\\"_\\\") and v.get(\\\"class\\\") == \\\"NOOP\\\"}\\n\\n    def to_json(self) -> dict:\\n        return {k: v for k, v in self.items()}\\n\\n\\ndef classify_actions(log: Iterable[Transition], prior: Optional[ActionSemantics] = None) -> ActionSemantics:\\n    obs: dict[str, list[tuple[str, dict, Transition]]] = {}\\n    for t in log:\\n        if t.action.type == \\\"RESET\\\":\\n            continue\\n        cls, info = classify_one(t)\\n        obs.setdefault(t.action.key, []).append((cls, info, t))\\n    sem = ActionSemantics(prior or {})\\n    counters: Counter = Counter()\\n    for key, items in obs.items():\\n        classes = Counter(c for c, _, _ in items)\\n        non_noop = [(c, i, t) for c, i, t in items if c != \\\"NOOP\\\"]\\n        for _, i, _ in items:\\n            counters.update(i.get(\\\"counters\\\", []))\\n        if not non_noop:\\n            states = {t.before.frame_hash for _, _, t in items}\\n            cls = \\\"NOOP\\\" if len(items) >= 2 and len(states) >= 2 else \\\"UNKNOWN\\\"\\n            sem[key] = {\\\"class\\\": cls, \\\"effect\\\": None, \\\"consistency\\\": f\\\"{len(items)}/{len(items)}\\\", \\\"n\\\": len(items), \\\"noop_count\\\": len(items),\\n                        \\\"note\\\": None if cls == \\\"NOOP\\\" else \\\"no change so far, but only observed from one state\\\"}\\n            continue\\n        cls, n = Counter(c for c, _, _ in non_noop).most_common(1)[0]\\n        entry: dict = {\\\"class\\\": cls, \\\"consistency\\\": f\\\"{n}/{len(items)}\\\", \\\"n\\\": len(items), \\\"noop_count\\\": classes.get(\\\"NOOP\\\", 0)}\\n        if cls == \\\"MOVE\\\":\\n            disp = Counter(tuple(i[\\\"displacement\\\"]) for c, i, _ in non_noop if c == \\\"MOVE\\\")\\n            (dr, dc), k = disp.most_common(1)[0]\\n            movers = Counter()\\n            for c, i, _ in non_noop:\\n                if c == \\\"MOVE\\\" and tuple(i[\\\"displacement\\\"]) == (dr, dc):\\n                    movers.update(i[\\\"movers\\\"])\\n            entry.update(displacement=[dr, dc], movers=[m for m, _ in movers.most_common(4)], effect=f\\\"obj {[m for m, _ in movers.most_common(2)]} by ({dr:+d},{dc:+d})\\\",\\n                         displacement_consistency=f\\\"{k}/{len(non_noop)}\\\")\\n        elif cls == \\\"TRANSFORM\\\":\\n            tg = Counter(x for c, i, _ in non_noop if c == \\\"TRANSFORM\\\" for x in i[\\\"targets\\\"])\\n            entry.update(effect=f\\\"transforms objects {[t for t, _ in tg.most_common(3)]} (rotation/flip/redraw)\\\", targets=[t for t, _ in tg.most_common(6)])\\n        elif cls == \\\"TOGGLE\\\":\\n            pairs = Counter((a, b) for c, i, _ in non_noop if c == \\\"TOGGLE\\\" for _, a, b in i[\\\"targets\\\"])\\n            entry.update(effect=\\\"recolor \\\" + \\\", \\\".join(f\\\"{a}->{b}\\\" for (a, b), _ in pairs.most_common(3)), targets=sorted({x[0] for c, i, _ in non_noop if c == \\\"TOGGLE\\\" for x in i[\\\"targets\\\"]}))\\n        elif cls == \\\"SELECT\\\":\\n            responsive = sorted({x for c, i, _ in non_noop if c == \\\"SELECT\\\" for x in i[\\\"targets\\\"]})\\n            entry.update(effect=f\\\"targets {responsive[:6]} respond\\\", responsive_targets=responsive)\\n        elif cls in (\\\"SPAWN\\\", \\\"REMOVE\\\"):\\n            entry.update(effect=f\\\"{cls.lower()} {sorted({x for c, i, _ in non_noop if c == cls for x in i['ids']})[:6]}\\\")\\n        else:\\n            entry.update(effect=\\\"mixed changes\\\")\\n        if key == \\\"CLICK\\\":\\n            resp, unresp = [], []\\n            for c, i, t in items:\\n                tgt = (t.action.row, t.action.col)\\n                (resp if c != \\\"NOOP\\\" else unresp).append(tgt)\\n            entry[\\\"responsive_coords\\\"] = resp[:20]; entry[\\\"unresponsive_coords\\\"] = unresp[:20]\\n        sem[key] = entry\\n    sem[\\\"_counters\\\"] = [i for i, _ in counters.most_common(6)]\\n    return sem\\n\", \"pbg/tools/__init__.py\": \"\", \"pbg/tools/build_reaction_dataset.py\": \"\\\"\\\"\\\"tools/build_reaction_dataset.py \\u2014 enumerate every level of every game (engine set_level) and record the initial map\\nplus the one-step reaction of each action. This is the raw material for distilling a level-start \\\"objective / patterns\\\"\\nmodel (teacher -> small model).\\n\\n    .venv/bin/python -m pbg.tools.build_reaction_dataset --games r11l,vc33 --out experiments/pbg/datasets/reactions\\n    .venv/bin/python -m pbg.tools.build_reaction_dataset --all --out experiments/pbg/datasets/reactions\\n\\nFor each (game, level): set_level(L) gives the pristine initial state (no need to solve prior levels). We render the\\ninitial 64x64 grid, segment it with our Perception, then for every candidate action -- the simple buttons the frame\\noffers, and one CLICK at the centre of every detected object -- we re-set the level, apply that single action, and record\\nthe structured object diff (moved / appeared / disappeared / recoloured), whether the board changed at all, and whether\\nthe action completed the level. One-step reactions only: stateful dynamics (selection, counters, sequence locks) need\\naction sequences and are out of scope for this first pass.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport os\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\nos.environ.setdefault(\\\"ARC_API_KEY\\\", \\\"local-dev\\\")\\nROOT = Path(__file__).resolve().parents[2]\\nSIMPLE = [1, 2, 3, 4, 5, 7]     # ACTION ids that need no coordinates (ACTION6 is the click)\\n\\n\\ndef _game_of(raw):\\n    for attr in (\\\"_game\\\", \\\"game\\\", \\\"_env\\\", \\\"env\\\"):\\n        g = getattr(raw, attr, None)\\n        if g is not None and hasattr(g, \\\"set_level\\\") and hasattr(g, \\\"perform_action\\\"):\\n            return g\\n    return None\\n\\n\\ndef _render(game) -> np.ndarray:\\n    return np.asarray(game.camera.render(game.current_level.get_sprites()), dtype=np.int8)\\n\\n\\ndef _num_levels(game) -> int:\\n    for attr in (\\\"_levels\\\", \\\"levels\\\"):\\n        v = getattr(game, attr, None)\\n        if v is not None:\\n            try:\\n                return len(v)\\n            except TypeError:\\n                pass\\n    return 1\\n\\n\\ndef _diff_summary(perc, before_grid, after_grid, level) -> dict:\\n    from ..core.types import Frame\\n    bg = np.asarray(before_grid); ag = np.asarray(after_grid)\\n    b = perc.parse(Frame(before_grid, level, 0), None)\\n    a = perc.parse(Frame(after_grid, level, 1), b)\\n    d = perc.diff(b, a, before_grid, after_grid)\\n    bo = {o.id: o for o in b.objects}; ao = {o.id: o for o in a.objects}\\n    # resize / reshape: bbox + area before -> after (vc33 bars grow, cd82 stamps) \\u2014 not captured by \\\"moved\\\"\\n    reshaped = []\\n    for i, _, _ in d.reshaped[:12]:\\n        p, q = bo.get(i), ao.get(i)\\n        if p is not None and q is not None:\\n            reshaped.append({\\\"id\\\": int(i), \\\"bbox_from\\\": [int(x) for x in p.bbox], \\\"bbox_to\\\": [int(x) for x in q.bbox],\\n                             \\\"area_from\\\": int(p.area), \\\"area_to\\\": int(q.area)})\\n    mask = bg != ag\\n    box = None\\n    if mask.any():\\n        ys, xs = np.nonzero(mask)\\n        box = [int(ys.min()), int(xs.min()), int(ys.max()) + 1, int(xs.max()) + 1]\\n    return {\\n        \\\"changed\\\": bool(mask.any()),\\n        \\\"pixel_changes\\\": int(mask.sum()),\\n        \\\"changed_bbox\\\": box,\\n        \\\"moved\\\": [[int(i), [int(dr), int(dc)]] for i, (dr, dc) in d.moved[:12]],\\n        \\\"reshaped\\\": reshaped,\\n        \\\"recolored\\\": [[int(i), int(o), int(n)] for i, o, n in d.recolored[:12]],\\n        \\\"appeared\\\": [{\\\"color\\\": int(o.color), \\\"bbox\\\": [int(x) for x in o.bbox]} for o in d.appeared[:12]],\\n        \\\"disappeared\\\": [{\\\"color\\\": int(o.color), \\\"bbox\\\": [int(x) for x in o.bbox]} for o in d.disappeared[:12]],\\n    }\\n\\n\\ndef _objects(perc, grid, level) -> list[dict]:\\n    from ..core.types import Frame\\n    s = perc.parse(Frame(grid, level, 0), None)\\n    out = []\\n    for o in sorted(s.objects, key=lambda o: -o.area):\\n        out.append({\\\"id\\\": int(o.id), \\\"color\\\": int(o.color), \\\"colors\\\": [int(c) for c in o.colors],\\n                    \\\"bbox\\\": [int(x) for x in o.bbox], \\\"area\\\": int(o.area),\\n                    \\\"center\\\": [int((o.bbox[0] + o.bbox[2]) // 2), int((o.bbox[1] + o.bbox[3]) // 2)]})\\n    return out\\n\\n\\ndef build_game(arc, game_id: str, *, max_click_objects: int = 40, log=print) -> dict:\\n    from pbg.env.wrapper import ArcadeEnv                       # noqa: F401  (ensures engine action enums import)\\n    from pbg.perception import Perception\\n    from arcengine import ActionInput, GameAction\\n    raw = arc.make(game_id)\\n    if raw is None:\\n        return {\\\"game_id\\\": game_id, \\\"error\\\": \\\"could not create env\\\"}\\n    game = _game_of(raw)\\n    if game is None:\\n        return {\\\"game_id\\\": game_id, \\\"error\\\": \\\"no underlying game with set_level\\\"}\\n    nlev = _num_levels(game)\\n\\n    def run(action_input):\\n        \\\"\\\"\\\"Apply an engine ActionInput (render-space coords, no scaling) from the current state; return (grid, state, completed).\\\"\\\"\\\"\\n        fr = game.perform_action(action_input, raw=True)\\n        grid = np.asarray(fr.frame[-1], dtype=np.int8) if fr.frame else None\\n        return grid, str(fr.state), int(getattr(fr, \\\"levels_completed\\\", 0) or 0)\\n\\n    def obj_at(objs, row, col):\\n        for o in objs:\\n            r0, c0, r1, c1 = o[\\\"bbox\\\"]\\n            if r0 <= row < r1 and c0 <= col < c1:\\n                return o[\\\"id\\\"]\\n        return None\\n\\n    levels = []\\n    for L in range(nlev):\\n        game.set_level(L)\\n        perc = Perception()\\n        init = _render(game)\\n        base_completed = int(getattr(game, \\\"_score\\\", 0) or 0)\\n        objs = _objects(perc, init, L)\\n        avail = [int(a.value if hasattr(a, \\\"value\\\") else a) for a in getattr(game, \\\"_available_actions\\\", []) or []]\\n        # authoritative reactive actions: the buttons + the EXACT clickable coordinates the engine offers (render space)\\n        try:\\n            valid = list(game._get_valid_actions())\\n        except Exception:\\n            valid = []\\n        seen_click = {(int(a.data[\\\"y\\\"]), int(a.data[\\\"x\\\"])) for a in valid if getattr(a, \\\"data\\\", None) and \\\"x\\\" in a.data}\\n        # click-anywhere games (ACTION6) react to arbitrary cells but expose only a few \\\"valid\\\" ones: also click the\\n        # centre of every object so each object's reaction is recorded (deduped against the engine's own click cells)\\n        extra = []\\n        if (not avail) or 6 in avail:\\n            for o in objs[:max_click_objects]:\\n                r, c = o[\\\"center\\\"]\\n                if (int(r), int(c)) not in seen_click:\\n                    extra.append(ActionInput(id=GameAction.ACTION6, data={\\\"x\\\": int(c), \\\"y\\\": int(r)}))\\n        reactions = []\\n        for a in valid + extra:\\n            game.set_level(L)\\n            after, state, completed = run(a)\\n            if after is None:\\n                continue\\n            data = getattr(a, \\\"data\\\", None) or {}\\n            aid = a.id.value if hasattr(a.id, \\\"value\\\") else int(a.id)\\n            entry = {\\\"action\\\": f\\\"ACTION{aid}\\\"}\\n            if \\\"x\\\" in data:\\n                row, col = int(data[\\\"y\\\"]), int(data[\\\"x\\\"])\\n                entry.update({\\\"action\\\": \\\"CLICK\\\", \\\"at\\\": [row, col], \\\"on_object\\\": obj_at(objs, row, col)})\\n            entry.update(_diff_summary(perc, init, after, L))\\n            entry.update({\\\"level_up\\\": completed > base_completed, \\\"state\\\": state})\\n            reactions.append(entry)\\n        n_changed = sum(1 for x in reactions if x[\\\"changed\\\"])\\n        levels.append({\\\"level\\\": L, \\\"grid\\\": init.tolist(), \\\"n_objects\\\": len(objs), \\\"objects\\\": objs,\\n                       \\\"available_actions\\\": avail, \\\"n_reactions\\\": len(reactions), \\\"n_changed\\\": n_changed,\\n                       \\\"reactions\\\": reactions})\\n        log(f\\\"  {game_id} L{L}: {len(objs)} objects, {len(reactions)} actions swept, {n_changed} changed the board\\\")\\n    return {\\\"game_id\\\": game_id, \\\"n_levels\\\": nlev, \\\"levels\\\": levels}\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--games\\\", default=\\\"\\\")\\n    ap.add_argument(\\\"--all\\\", action=\\\"store_true\\\")\\n    ap.add_argument(\\\"--out\\\", default=\\\"experiments/pbg/datasets/reactions\\\")\\n    args = ap.parse_args()\\n    from pbg.harness.online_runner import make_arcade\\n    arc = make_arcade()\\n    if args.all:\\n        games = sorted(p.name for p in (ROOT / \\\"environment_files\\\").iterdir() if p.is_dir())\\n    else:\\n        games = [g for g in args.games.split(\\\",\\\") if g]\\n    out = ROOT / args.out\\n    out.mkdir(parents=True, exist_ok=True)\\n    index = []\\n    for g in games:\\n        print(f\\\"[{g}] building reaction dataset ...\\\", flush=True)\\n        try:\\n            data = build_game(arc, g)\\n        except Exception as e:\\n            import traceback\\n            traceback.print_exc()\\n            data = {\\\"game_id\\\": g, \\\"error\\\": f\\\"{type(e).__name__}: {e}\\\"}\\n        (out / f\\\"{g}.json\\\").write_text(json.dumps(data, separators=(\\\",\\\", \\\":\\\")))\\n        nlev = data.get(\\\"n_levels\\\", 0)\\n        chg = sum(l.get(\\\"n_changed\\\", 0) for l in data.get(\\\"levels\\\", []))\\n        index.append({\\\"game_id\\\": g, \\\"n_levels\\\": nlev, \\\"error\\\": data.get(\\\"error\\\"), \\\"total_changed_reactions\\\": chg})\\n        print(f\\\"[{g}] -> {out / (g + '.json')}  ({nlev} levels)\\\", flush=True)\\n    (out / \\\"index.json\\\").write_text(json.dumps(index, indent=1))\\n    print(f\\\"index: {out / 'index.json'}\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"pbg/tools/collect_human_log.py\": \"#!/usr/bin/env python\\n\\\"\\\"\\\"tools/collect_human_log.py \\u2014 record a human play log (raw.jsonl with every frame) in the terminal (spec \\u00a75/\\u00a716).\\n\\n    .venv/bin/python pbg/tools/collect_human_log.py ls20 [--out pbg/data/human_logs/<player>/ls20/raw.jsonl] [--player me]\\n\\nKeys: w/a/s/d or arrows = ACTION1..4 (up/down/left/right), space = ACTION5, e = ACTION7, c r c = click at (row, col)\\ntyped as `c 12 34`, R = reset, q = quit. The board is printed as hex digits after every action.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport sys\\nfrom pathlib import Path\\n\\nROOT = Path(__file__).resolve().parents[2]\\nsys.path.insert(0, str(ROOT))\\n\\nfrom pbg.core.types import Action, grid_to_rows  # noqa: E402\\nfrom pbg.env.wrapper import ArcadeEnv  # noqa: E402\\nfrom pbg.harness.online_runner import make_arcade  # noqa: E402\\n\\nKEYS = {\\\"w\\\": 1, \\\"s\\\": 2, \\\"a\\\": 3, \\\"d\\\": 4, \\\" \\\": 5, \\\"e\\\": 7, \\\"1\\\": 1, \\\"2\\\": 2, \\\"3\\\": 3, \\\"4\\\": 4, \\\"5\\\": 5, \\\"7\\\": 7}\\n\\n\\ndef show(frame, status) -> None:\\n    rows = grid_to_rows(frame.grid)\\n    print(\\\"\\\\n\\\".join(f\\\"{i:2d} {r}\\\" for i, r in enumerate(rows)))\\n    print(f\\\"level {status.level}/{status.levels_total} state {status.state} actions {status.actions_used}\\\")\\n\\n\\ndef main(argv=None) -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"game\\\"); ap.add_argument(\\\"--player\\\", default=\\\"human\\\"); ap.add_argument(\\\"--out\\\")\\n    a = ap.parse_args(argv)\\n    out = Path(a.out) if a.out else ROOT / \\\"pbg\\\" / \\\"data\\\" / \\\"human_logs\\\" / a.player / a.game\\n    out.mkdir(parents=True, exist_ok=True)\\n    if (out / \\\"raw.jsonl\\\").exists():\\n        (out / \\\"raw.jsonl\\\").unlink()\\n    arc = make_arcade()\\n    env = ArcadeEnv(arc.make(a.game), a.game, log_dir=out)\\n    frame = env.reset(); show(frame, env.status())\\n    print(\\\"keys: w/a/s/d=1/3/2/4  space=5  e=7  'c r c'=click  R=reset  q=quit\\\")\\n    while True:\\n        try:\\n            line = input(\\\"> \\\")\\n        except EOFError:\\n            break\\n        if not line:\\n            continue\\n        if line == \\\"q\\\":\\n            break\\n        if line == \\\"R\\\":\\n            frame = env.reset(); show(frame, env.status()); continue\\n        if line.startswith(\\\"c \\\"):\\n            _, r, c = line.split()[:3]\\n            act = Action.click(int(r), int(c))\\n        elif line[0] in KEYS:\\n            act = Action.button(KEYS[line[0]])\\n        else:\\n            print(\\\"?\\\"); continue\\n        rt = env.step(act)\\n        if rt.error:\\n            print(\\\"error:\\\", rt.error); continue\\n        show(rt.after, rt.status)\\n        if rt.status_change:\\n            print(\\\"***\\\", rt.status_change)\\n        if rt.status.state in (\\\"WIN\\\", \\\"GAME_OVER\\\"):\\n            print(\\\"game\\\", rt.status.state); break\\n    print(\\\"saved\\\", out / \\\"raw.jsonl\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"pbg/tools/import_run_logs.py\": \"#!/usr/bin/env python\\n\\\"\\\"\\\"tools/import_run_logs.py \\u2014 turn recorded agent runs (experiments/rulebook/results/logs/<run>/<game>.actions.jsonl)\\ninto raw.jsonl play logs by re-executing the recorded actions on the offline engine (deterministic), so the replay\\nharness has logs before human logs exist.\\n\\n    .venv/bin/python pbg/tools/import_run_logs.py --run 20260928-030547-16108 [--games ls20,tn36] [--out pbg/data/human_logs/agent]\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom pathlib import Path\\n\\nROOT = Path(__file__).resolve().parents[2]\\nsys.path.insert(0, str(ROOT))\\n\\nfrom pbg.core.types import Action  # noqa: E402\\nfrom pbg.env.wrapper import ArcadeEnv  # noqa: E402\\nfrom pbg.harness.online_runner import make_arcade  # noqa: E402\\n\\nMODEL_TO_ID = {\\\"UP\\\": 1, \\\"DOWN\\\": 2, \\\"LEFT\\\": 3, \\\"RIGHT\\\": 4, \\\"SPACE\\\": 5, \\\"ACTION7\\\": 7}\\n\\n\\ndef to_action(rec_action) -> Action:\\n    if isinstance(rec_action, dict):\\n        return Action.click(int(rec_action[\\\"row\\\"]), int(rec_action[\\\"col\\\"]))\\n    return Action.button(MODEL_TO_ID[rec_action])\\n\\n\\ndef import_one(arc, game: str, actions_path: Path, out_dir: Path, *, max_steps: int = 100000) -> dict:\\n    recs = [json.loads(l) for l in actions_path.read_text().splitlines() if l.strip()]\\n    out = out_dir / game; out.mkdir(parents=True, exist_ok=True)\\n    if (out / \\\"raw.jsonl\\\").exists():\\n        (out / \\\"raw.jsonl\\\").unlink()\\n    env = ArcadeEnv(arc.make(game), game, log_dir=out)\\n    env.reset(); n = 0; mismatches = 0\\n    for r in recs[:max_steps]:\\n        if r[\\\"kind\\\"] == \\\"reset\\\":\\n            if n > 0:\\n                env.reset()\\n            continue\\n        rt = env.step(to_action(r[\\\"action\\\"]))\\n        n += 1\\n        if rt.error:\\n            break\\n        if r.get(\\\"hash\\\") and rt.after is not None:\\n            import hashlib\\n            from pbg.core.types import grid_to_rows\\n            h = hashlib.sha1(\\\"\\\\n\\\".join(grid_to_rows(rt.after.grid)).encode()).hexdigest()[:12]\\n            if h != r[\\\"hash\\\"]:\\n                mismatches += 1\\n        if rt.status.state in (\\\"WIN\\\", \\\"GAME_OVER\\\"):\\n            break\\n    return {\\\"game\\\": game, \\\"steps\\\": n, \\\"hash_mismatches\\\": mismatches, \\\"levels\\\": env.status().level - 1, \\\"out\\\": str(out / \\\"raw.jsonl\\\")}\\n\\n\\ndef main(argv=None) -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--run\\\", required=True, help=\\\"run id under experiments/rulebook/results/logs/\\\")\\n    ap.add_argument(\\\"--games\\\"); ap.add_argument(\\\"--out\\\", default=str(ROOT / \\\"pbg\\\" / \\\"data\\\" / \\\"human_logs\\\" / \\\"agent\\\"))\\n    a = ap.parse_args(argv)\\n    log_dir = ROOT / \\\"experiments\\\" / \\\"rulebook\\\" / \\\"results\\\" / \\\"logs\\\" / a.run\\n    arc = make_arcade()\\n    games = [g.strip() for g in a.games.split(\\\",\\\")] if a.games else sorted(p.name.split(\\\".\\\")[0] for p in log_dir.glob(\\\"*.actions.jsonl\\\"))\\n    for g in games:\\n        p = log_dir / f\\\"{g}.actions.jsonl\\\"\\n        if p.exists():\\n            print(import_one(arc, g, p, Path(a.out)))\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"pbg/tools/lint_no_game_id.py\": \"#!/usr/bin/env python\\n\\\"\\\"\\\"tools/lint_no_game_id.py \\u2014 CI lint (spec \\u00a713): no module may branch on a game id string.\\n\\nFlags string literals that look like game ids (two letters + two digits, e.g. 'ls20') used in comparisons, `in` tests,\\ndict keys or conditionals anywhere under pbg/ (tests and data files excluded).\\n\\n    .venv/bin/python pbg/tools/lint_no_game_id.py [paths...]   -> exit 1 on findings\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport ast\\nimport re\\nimport sys\\nfrom pathlib import Path\\n\\nGAME_ID = re.compile(r\\\"^[a-z]{2}\\\\d{2}$\\\")\\nROOT = Path(__file__).resolve().parents[2]\\nDEFAULT = [ROOT / \\\"pbg\\\"]\\nEXCLUDE = {\\\"tests\\\", \\\"data\\\", \\\"tools\\\"}\\n\\n\\ndef check_file(path: Path) -> list[str]:\\n    out = []\\n    try:\\n        tree = ast.parse(path.read_text())\\n    except SyntaxError as e:\\n        return [f\\\"{path}: syntax error {e}\\\"]\\n    for node in ast.walk(tree):\\n        if isinstance(node, (ast.Compare, ast.If, ast.IfExp, ast.Match if hasattr(ast, \\\"Match\\\") else ast.If, ast.Dict, ast.Subscript)):\\n            for sub in ast.walk(node):\\n                if isinstance(sub, ast.Constant) and isinstance(sub.value, str) and GAME_ID.match(sub.value):\\n                    out.append(f\\\"{path.relative_to(ROOT)}:{sub.lineno}: game id literal '{sub.value}' in a branch/lookup\\\")\\n                    break\\n    return out\\n\\n\\ndef main(argv=None) -> int:\\n    paths = [Path(p) for p in (argv if argv is not None else sys.argv[1:])] or DEFAULT\\n    findings = []\\n    for base in paths:\\n        files = [base] if base.is_file() else sorted(base.rglob(\\\"*.py\\\"))\\n        for f in files:\\n            if any(part in EXCLUDE for part in f.relative_to(ROOT).parts):\\n                continue\\n            findings += check_file(f)\\n    for line in findings:\\n        print(line)\\n    print(f\\\"no-game-id-branch: {len(findings)} finding(s)\\\")\\n    return 1 if findings else 0\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    sys.exit(main())\\n\", \"pbg/wml/__init__.py\": \"from .evaluate import evaluate, EvalResult  # noqa: F401\\nfrom .sandbox import Sandbox, StaticCheckError  # noqa: F401\\nfrom .refine import WorldModelLab  # noqa: F401\\nfrom .experiment import most_informative_action  # noqa: F401\\n\", \"pbg/wml/context.py\": \"\\\"\\\"\\\"wml/context.py \\u2014 context selection for LLM prompts (spec \\u00a714): never the whole log. Fill up to `cap` transitions:\\n1. all violated transitions (newest first); 2. three representative transitions per action class (distinct diffs first);\\n3. transitions right before/after a level change; 4. the most recent ones.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Iterable, Optional\\n\\nfrom ..core.types import Transition\\nfrom ..perception.summarize import summarize, transform_crops\\n\\n\\ndef select_context(log: list[Transition], violations: Iterable[str] = (), cap: int = 40) -> list[Transition]:\\n    viol = set(violations)\\n    chosen: list[Transition] = []; ids: set[str] = set()\\n\\n    def add(t: Transition) -> bool:\\n        if len(chosen) >= cap or t.id in ids or t.action.type == \\\"RESET\\\":\\n            return False\\n        chosen.append(t); ids.add(t.id)\\n        return True\\n\\n    for t in reversed(log):\\n        if t.id in viol:\\n            add(t)\\n    by_class: dict[str, list[Transition]] = {}\\n    for t in log:\\n        by_class.setdefault(t.action.key, []).append(t)\\n    for key, ts in by_class.items():\\n        seen_diffs: set[tuple] = set(); n = 0\\n        for t in reversed(ts):\\n            sig = (tuple(sorted(t.diff.kinds())), tuple(sorted(v for _, v in t.diff.moved)), t.diff.is_noop)\\n            if sig in seen_diffs and n >= 1:\\n                continue\\n            seen_diffs.add(sig)\\n            if add(t):\\n                n += 1\\n            if n >= 3:\\n                break\\n    for i, t in enumerate(log):\\n        if t.status_change in (\\\"LEVEL_UP\\\", \\\"WIN\\\"):\\n            add(t)\\n            if i + 1 < len(log):\\n                add(log[i + 1])\\n    for t in reversed(log):\\n        if len(chosen) >= cap:\\n            break\\n        add(t)\\n    chosen.sort(key=lambda t: (t.level, t.step_idx))\\n    return chosen\\n\\n\\ndef observation_lines(ts: list[Transition], perception=None, max_crops: int = 6) -> list[str]:\\n    \\\"\\\"\\\"One summary line per transition; the first transitions that reshape/recolour an object per action key also get\\n    before/after pixel crops (a summary line cannot show a rotation or a stamp).\\\"\\\"\\\"\\n    out = []; shown: set[tuple[str, int]] = set()\\n    for t in ts:\\n        out.append(f\\\"[{t.id}] \\\" + summarize(t))\\n        ids = [x[0] for x in t.diff.reshaped] + [x[0] for x in t.diff.recolored]\\n        key = (t.action.key, ids[0]) if ids else None\\n        if key and key not in shown and len(shown) < max_crops:\\n            crops = transform_crops(t)\\n            if crops:\\n                shown.add(key); out += crops\\n    return out\\n\", \"pbg/wml/effects.py\": \"\\\"\\\"\\\"wml/effects.py \\u2014 effect-table world model: every trigger (a click target object, or a button) is described by the\\nJOINT change it produced in the frame (movers shifted, bars lengthened/shortened, objects appearing/vanishing),\\nlearned from the log and applied as one rule per trigger. Resizes carry a built-in guard (a bar cannot shrink below\\nzero), which is what makes \\\"feed the middle bar before pulling from it\\\" plannable: the game refuses the click, and\\nso does the model.\\n\\nComponents of an effect (all level-specific positions, class-level signature in `meta` for the knowledge asset):\\n  (\\\"move\\\",   color, sig, dr, dc)                   every object of that colour+shape shifts\\n  (\\\"resize\\\", color, axis, lo, hi, anchor, delta)   the rectangle of `color` whose fixed-axis extent is (lo, hi) grows /\\n                                                   shrinks by `delta` along the other axis; `anchor` = the fixed edge\\n                                                   coordinate (a bar attached to the left border keeps c0 = anchor)\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter, defaultdict\\nfrom typing import Optional\\n\\nimport numpy as np\\n\\nfrom ..core.contracts import Rule, RuleModel\\nfrom ..core.types import Action, Object, Scene, Transition, shape_signature\\nfrom ..env.calibrate import object_under\\nfrom ..probe.semantics import core_diff\\n\\n\\ndef _is_rect(o: Object, scene: Optional[Scene] = None) -> bool:\\n    \\\"\\\"\\\"A bar/panel: fills most of its bbox, or fills it together with the objects drawn over it (buttons on a bar).\\\"\\\"\\\"\\n    if o.mask is None or o.height < 2 or o.width < 2:\\n        return False\\n    fill = float(o.mask.mean())\\n    if fill >= 0.75:\\n        return True\\n    if fill < 0.4 or scene is None:\\n        return False\\n    r0, c0, r1, c1 = o.bbox\\n    cov = o.mask.copy()\\n    for q in scene.objects:\\n        if q.id == o.id:\\n            continue\\n        a0, b0, a1, b1 = q.bbox\\n        rr0, cc0, rr1, cc1 = max(r0, a0), max(c0, b0), min(r1, a1), min(c1, b1)\\n        if rr1 > rr0 and cc1 > cc0:\\n            cov[rr0 - r0:rr1 - r0, cc0 - c0:cc1 - c0] |= q.mask[rr0 - a0:rr1 - a0, cc0 - b0:cc1 - b0]\\n    return float(cov.mean()) >= 0.95\\n\\n\\ndef _effective_bbox(o: Object, scene: Optional[Scene]) -> tuple:\\n    \\\"\\\"\\\"A bar's bbox extended over objects drawn on top of its ends (a button that covers the first rows of a column):\\n    the bar geometrically continues under them.\\\"\\\"\\\"\\n    r0, c0, r1, c1 = o.bbox\\n    if scene is None:\\n        return (r0, c0, r1, c1)\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    for q in scene.objects:\\n        if q.id == o.id or q.region in strips or q.area > 0.25 * o.area or max(q.height, q.width) > 8:\\n            continue          # a cap is a small object (button, marker); a wall spanning the bar's width is not\\n        a0, b0, a1, b1 = q.bbox\\n        # a cap drawn over the bar's end spans the bar's full thickness (a button on a column); a mover parked at the\\n        # bar's end does not, and must not be counted as bar\\n        cap_rows = (a1 == r0 or a0 == r1) and (b0, b1) == (c0, c1)\\n        cap_cols = (b1 == c0 or b0 == c1) and (a0, a1) == (r0, r1)\\n        if cap_rows or cap_cols:\\n            r0, c0, r1, c1 = min(r0, a0), min(c0, b0), max(r1, a1), max(c1, b1)\\n    return (r0, c0, r1, c1)\\n\\n\\ndef _resize_component(ob: Object, oa: Optional[Object], sb: Optional[Scene] = None, sa: Optional[Scene] = None):\\n    \\\"\\\"\\\"Describe a one-edge resize of a rectangle (or its disappearance) as a component, else None.\\\"\\\"\\\"\\n    if not _is_rect(ob, sb):\\n        return None\\n    r0, c0, r1, c1 = _effective_bbox(ob, sb)\\n    if oa is None:\\n        # vanished: shrunk to zero along the free axis -- we do not know the axis; caller resolves with other observations\\n        return (\\\"vanish\\\", ob.color, (r0, c0, r1, c1))\\n    if not _is_rect(oa, sa) or oa.color != ob.color:\\n        return None\\n    s0, d0, s1, d1 = _effective_bbox(oa, sa)\\n    # delta is always the LENGTH change (positive = grows); anchor is the edge that stayed\\n    if (r0, r1) == (s0, s1) and (c0, c1) != (d0, d1):\\n        if c0 == d0:\\n            return (\\\"resize\\\", ob.color, \\\"row\\\", r0, r1, c0, (d1 - d0) - (c1 - c0))       # right edge moved\\n        if c1 == d1:\\n            return (\\\"resize\\\", ob.color, \\\"row\\\", r0, r1, c1, (d1 - d0) - (c1 - c0))       # left edge moved\\n    if (c0, c1) == (d0, d1) and (r0, r1) != (s0, s1):\\n        if r0 == s0:\\n            return (\\\"resize\\\", ob.color, \\\"col\\\", c0, c1, r0, (s1 - s0) - (r1 - r0))\\n        if r1 == s1:\\n            return (\\\"resize\\\", ob.color, \\\"col\\\", c0, c1, r1, (s1 - s0) - (r1 - r0))\\n    return None\\n\\n\\ndef observe_effect(t: Transition) -> Optional[list[tuple]]:\\n    \\\"\\\"\\\"Components of the change in one transition (None when nothing outside ui strips changed).\\\"\\\"\\\"\\n    c = core_diff(t)\\n    if not (c[\\\"moved\\\"] or c[\\\"recolored\\\"] or c[\\\"reshaped\\\"] or c[\\\"appeared\\\"] or c[\\\"disappeared\\\"]):\\n        return None\\n    b = {o.id: o for o in t.before.objects}; a = {o.id: o for o in t.after.objects}\\n    comps: list[tuple] = []\\n    reshaped = {x[0] for x in c[\\\"reshaped\\\"]}\\n    for oid, (dr, dc) in c[\\\"moved\\\"]:\\n        if oid in reshaped:\\n            continue\\n        ob = b.get(oid)\\n        if ob is not None:\\n            # which object of the class moves is fixed by its extent across the movement axis (the mover on THIS column)\\n            fixed = (ob.bbox[1], ob.bbox[3]) if dr else (ob.bbox[0], ob.bbox[2])\\n            comps.append((\\\"move\\\", ob.color, ob.shape_sig[:8], dr, dc, fixed[0], fixed[1]))\\n    for oid in reshaped:\\n        ob, oa = b.get(oid), a.get(oid)\\n        if ob is None:\\n            continue\\n        rc = _resize_component(ob, oa, t.before, t.after)\\n        if rc is not None:\\n            comps.append(rc)\\n    for o in c[\\\"disappeared\\\"]:\\n        rc = _resize_component(o, None, t.before, None)\\n        if rc is not None:\\n            comps.append(rc)\\n    for o in c[\\\"appeared\\\"]:\\n        if _is_rect(o, t.after):\\n            comps.append((\\\"appear\\\", o.color, _effective_bbox(o, t.after)))\\n    return comps\\n\\n\\ndef _trigger_key(t: Transition):\\n    a = t.action\\n    if a.type == \\\"BUTTON\\\":\\n        return (\\\"BUTTON\\\", a.id)\\n    if a.type == \\\"CLICK\\\":\\n        oid = object_under(t.before, a.row, a.col)\\n        o = t.before.get(oid) if oid is not None else None\\n        if o is None:\\n            return None\\n        return (\\\"CLICK\\\", o.color, o.shape_sig[:8], tuple(o.bbox))\\n    return None\\n\\n\\ndef _resolve_vanish_appear(comps_by_obs: list[list[tuple]], global_resizes: Optional[list] = None) -> list[list[tuple]]:\\n    \\\"\\\"\\\"A bar that vanished / appeared is the extreme case of a resize: rewrite it with the axis, extent and delta of a\\n    resize of the same band seen anywhere in the log; failing that, with the axis and magnitude of the resizes in the\\n    same observation (a two-row bar that vanishes when its neighbours change by 2 shrank by 2).\\\"\\\"\\\"\\n    resizes = list(global_resizes or []) + [c for obs in comps_by_obs for c in obs if c[0] == \\\"resize\\\"]\\n    out = []\\n    for obs in comps_by_obs:\\n        new = []\\n        for c in obs:\\n            if c[0] in (\\\"vanish\\\", \\\"appear\\\"):\\n                color, bbox = c[1], c[2]\\n                r0, c0, r1, c1 = bbox\\n                match = None\\n                same_obs = [x for x in obs if x[0] == \\\"resize\\\"]\\n                if not match and same_obs:\\n                    axis = same_obs[0][2]; mag = abs(same_obs[0][6]); anchors = {x[5] for x in same_obs}\\n                    if axis == \\\"col\\\" and (r1 - r0) == mag and (r0 in anchors or r1 in anchors):\\n                        match = (\\\"resize\\\", color, \\\"col\\\", c0, c1, r1 if r1 in anchors else r0, -mag if c[0] == \\\"vanish\\\" else mag)\\n                    if axis == \\\"row\\\" and (c1 - c0) == mag and (c0 in anchors or c1 in anchors):\\n                        match = (\\\"resize\\\", color, \\\"row\\\", r0, r1, c1 if c1 in anchors else c0, -mag if c[0] == \\\"vanish\\\" else mag)\\n                for rz in ([] if match else resizes):\\n                    _, col, axis, lo, hi, anchor, delta = rz\\n                    if col != color:\\n                        continue\\n                    if axis == \\\"row\\\" and (lo, hi) == (r0, r1) and (c1 - c0) == abs(delta) and anchor in (c0, c1):\\n                        match = (\\\"resize\\\", color, \\\"row\\\", lo, hi, anchor, -abs(delta) if c[0] == \\\"vanish\\\" else abs(delta))\\n                    if axis == \\\"col\\\" and (lo, hi) == (c0, c1) and (r1 - r0) == abs(delta) and anchor in (r0, r1):\\n                        match = (\\\"resize\\\", color, \\\"col\\\", lo, hi, anchor, -abs(delta) if c[0] == \\\"vanish\\\" else abs(delta))\\n                    if match:\\n                        break\\n                if match:\\n                    new.append(match)\\n            else:\\n                new.append(c)\\n        out.append(new)\\n    return out\\n\\n\\ndef learn_effect_table(log: list[Transition], min_obs: int = 2, prior_classes: Optional[dict] = None, confirmed: Optional[set] = None) -> dict:\\n    \\\"\\\"\\\"trigger key -> list of components (the consistent joint effect), plus per-trigger evidence counts.\\\"\\\"\\\"\\n    obs: dict = defaultdict(list); noops: Counter = Counter()\\n    for t in log:\\n        if t.action.type == \\\"RESET\\\" or t.status_change:\\n            continue\\n        k = _trigger_key(t)\\n        if k is None:\\n            continue\\n        comps = observe_effect(t)\\n        if comps is None:\\n            noops[k] += 1\\n        else:\\n            obs[k].append(comps)\\n    table: dict = {}\\n    global_resizes = [c for lst in obs.values() for comps in lst for c in comps if c[0] == \\\"resize\\\"]\\n    for k, lst in obs.items():\\n        lst = _resolve_vanish_appear(lst, global_resizes)\\n        need = min_obs\\n        if prior_classes and k[0] == \\\"CLICK\\\" and f\\\"c{k[1]}:{k[2]}\\\" in prior_classes:\\n            need = 1\\n        if confirmed and k[0] == \\\"CLICK\\\" and k[3] in confirmed:\\n            need = 1          # a trigger hypothesis confirmed by its test click: the observed effect is the rule\\n        if len(lst) < need:\\n            continue\\n        cnt: Counter = Counter()\\n        for comps in lst:\\n            for c in set(comps):\\n                cnt[c] += 1\\n        # components present in at least 80% of the observations (and in >= `need` of them)\\n        keep = [c for c, n in cnt.items() if n >= need and n >= 0.5 * len(lst)]   # majority: partial effects are guard-blocked cases (a mover at the edge)\\n        if not keep:\\n            continue\\n        table[k] = {\\\"components\\\": sorted(keep, key=str), \\\"n\\\": len(lst), \\\"noops\\\": noops.get(k, 0)}\\n    return table\\n\\n\\ndef _apply_components(scene: Scene, comps: list[tuple]) -> Optional[Scene]:\\n    \\\"\\\"\\\"Apply a joint effect. The guard: a resize that would shrink a bar below zero (or a bar that is not there to\\n    shrink) blocks the WHOLE effect -> the scene is returned unchanged (the game refuses the click).\\\"\\\"\\\"\\n    objs = list(scene.objects)\\n    # guard first\\n    for c in comps:\\n        if c[0] != \\\"resize\\\" or c[6] >= 0:\\n            continue\\n        _, color, axis, lo, hi, anchor, delta = c\\n        tgt = _find_bar(objs, color, axis, lo, hi, scene)\\n        if tgt is None or (tgt.width if axis == \\\"row\\\" else tgt.height) < abs(delta):\\n            return scene\\n    new_objs: list[Object] = []; consumed: set[int] = set(); extra: list[Object] = []\\n    others = list(objs)\\n    for c in comps:\\n        if c[0] == \\\"move\\\":\\n            color, sig, dr, dc = c[1], c[2], c[3], c[4]\\n            fixed = (c[5], c[6]) if len(c) >= 7 else None\\n            for o in objs:\\n                if o.id in consumed or o.color != color or o.shape_sig[:8] != sig:\\n                    continue\\n                if fixed is not None:\\n                    ext = (o.bbox[1], o.bbox[3]) if dr else (o.bbox[0], o.bbox[2])\\n                    if ext != fixed:\\n                        continue\\n                new_objs.append(o.moved(dr, dc)); consumed.add(o.id)\\n        elif c[0] == \\\"resize\\\":\\n            _, color, axis, lo, hi, anchor, delta = c\\n            tgt = _find_bar(objs, color, axis, lo, hi, scene)\\n            if tgt is None:\\n                if delta > 0:\\n                    extra.append(_make_bar(color, axis, lo, hi, anchor, delta, scene))\\n                continue\\n            consumed.add(tgt.id)\\n            r0, c0, r1, c1 = _effective_bbox(tgt, scene)\\n            if axis == \\\"row\\\":\\n                nb = (r0, c0, r1, c1 + delta) if anchor == c0 else (r0, c0 - delta, r1, c1)\\n            else:\\n                nb = (r0, c0, r1 + delta, c1) if anchor == r0 else (r0 - delta, c0, r1, c1)\\n            if nb[2] > nb[0] and nb[3] > nb[1]:\\n                res = _trim(tgt.with_mask(_bar_mask(nb, [q for q in others if q.id != tgt.id]), None, (nb[0], nb[1])))\\n                if res is not None:\\n                    new_objs.append(res)\\n    rest = [o for o in objs if o.id not in consumed]\\n    return scene.copy(objects=sorted(rest + new_objs + extra, key=lambda o: o.id))\\n\\n\\ndef _bar_mask(nb: tuple, covering: list[Object]) -> np.ndarray:\\n    \\\"\\\"\\\"Full rectangle minus the cells of objects drawn over it (buttons, markers).\\\"\\\"\\\"\\n    m = np.ones((nb[2] - nb[0], nb[3] - nb[1]), dtype=bool)\\n    for q in covering:\\n        a0, b0, a1, b1 = q.bbox\\n        rr0, cc0, rr1, cc1 = max(nb[0], a0), max(nb[1], b0), min(nb[2], a1), min(nb[3], b1)\\n        if rr1 > rr0 and cc1 > cc0:\\n            m[rr0 - nb[0]:rr1 - nb[0], cc0 - nb[1]:cc1 - nb[1]] &= ~q.mask[rr0 - a0:rr1 - a0, cc0 - b0:cc1 - b0]\\n    return m\\n\\n\\ndef _trim(o: Object) -> Optional[Object]:\\n    \\\"\\\"\\\"Tight bbox around the mask (an empty mask means the bar is gone).\\\"\\\"\\\"\\n    if not o.mask.any():\\n        return None\\n    ys, xs = np.nonzero(o.mask)\\n    r0, c0 = int(ys.min()), int(xs.min()); r1, c1 = int(ys.max()) + 1, int(xs.max()) + 1\\n    if (r0, c0, r1, c1) == (0, 0, o.mask.shape[0], o.mask.shape[1]):\\n        return o\\n    return o.with_mask(o.mask[r0:r1, c0:c1], None, (o.bbox[0] + r0, o.bbox[1] + c0))\\n\\n\\ndef _resized_mask(o: Object, nb: tuple) -> np.ndarray:\\n    \\\"\\\"\\\"The object's mask cropped / extended (with True cells) to the new bbox: holes it already has stay where they are.\\\"\\\"\\\"\\n    r0, c0, r1, c1 = o.bbox\\n    m = np.ones((nb[2] - nb[0], nb[3] - nb[1]), dtype=bool)\\n    rr0, cc0 = max(r0, nb[0]), max(c0, nb[1]); rr1, cc1 = min(r1, nb[2]), min(c1, nb[3])\\n    if rr1 > rr0 and cc1 > cc0:\\n        m[rr0 - nb[0]:rr1 - nb[0], cc0 - nb[1]:cc1 - nb[1]] = o.mask[rr0 - r0:rr1 - r0, cc0 - c0:cc1 - c0]\\n    return m\\n\\n\\ndef _find_bar(objs: list[Object], color: int, axis: str, lo: int, hi: int, scene: Optional[Scene] = None) -> Optional[Object]:\\n    for o in objs:\\n        if o.color != color or o.height < 2 or o.width < 2:\\n            continue\\n        r0, c0, r1, c1 = _effective_bbox(o, scene) if scene is not None else o.bbox\\n        if axis == \\\"row\\\" and (r0, r1) == (lo, hi):\\n            return o\\n        if axis == \\\"col\\\" and (c0, c1) == (lo, hi):\\n            return o\\n    return None\\n\\n\\ndef _make_bar(color: int, axis: str, lo: int, hi: int, anchor: int, length: int, scene: Scene) -> Object:\\n    if axis == \\\"row\\\":\\n        bbox = (lo, anchor, hi, anchor + length) if anchor + length <= scene.grid_shape[1] else (lo, anchor - length, hi, anchor)\\n    else:\\n        bbox = (anchor, lo, anchor + length, hi) if anchor + length <= scene.grid_shape[0] else (anchor - length, lo, anchor, hi)\\n    m = np.ones((bbox[2] - bbox[0], bbox[3] - bbox[1]), dtype=bool)\\n    nid = max([o.id for o in scene.objects] + [0]) + 1000\\n    return Object(nid, color, (color,), bbox, m, int(m.sum()), shape_signature(m), \\\"R0\\\")\\n\\n\\ndef class_signature(comps: list[tuple]) -> list:\\n    \\\"\\\"\\\"Level-independent description of an effect: kinds, colours and magnitudes without positions.\\\"\\\"\\\"\\n    out = []\\n    for c in comps:\\n        if c[0] == \\\"move\\\":\\n            out.append([\\\"move\\\", c[1], abs(c[3]) + abs(c[4])])      # class signature ignores which object of the class\\n        elif c[0] == \\\"resize\\\":\\n            out.append([\\\"resize\\\", c[1], abs(c[6])])\\n    return sorted(out)\\n\\n\\ndef induce_effect_model(log: list[Transition], semantics: dict, available: list[Action], knowledge=None,\\n                        click_map=None) -> list[tuple[str, RuleModel]]:\\n    prior = getattr(knowledge, \\\"effect_rules\\\", None) if knowledge is not None else None\\n    table = learn_effect_table(log, prior_classes=prior, confirmed=getattr(knowledge, \\\"hyp_triggers\\\", None))\\n    if not table:\\n        return []\\n    cmap = click_map if click_map is not None else getattr(knowledge, \\\"clicks\\\", None)\\n    rules: list[Rule] = []\\n    for k, entry in table.items():\\n        comps = entry[\\\"components\\\"]\\n        if k[0] == \\\"BUTTON\\\":\\n            bid = k[1]\\n            applies = (lambda s, a, bid=bid: a.type == \\\"BUTTON\\\" and a.id == bid)\\n            name = f\\\"effect_ACTION{bid}\\\"\\n        else:\\n            _, color, sig, bbox = k\\n            def applies(s, a, bbox=bbox):\\n                if a.type != \\\"CLICK\\\":\\n                    return False\\n                r0, c0, r1, c1 = bbox\\n                return r0 <= a.row < r1 and c0 <= a.col < c1\\n            name = f\\\"effect_click_at_{bbox[0]}_{bbox[1]}\\\"\\n        r = Rule(name, applies, (lambda s, a, comps=comps: _apply_components(s, comps)), source=\\\"induced\\\")\\n        r.meta = {\\\"kind\\\": \\\"effect\\\", \\\"trigger_class\\\": (f\\\"c{k[1]}:{k[2]}\\\" if k[0] == \\\"CLICK\\\" else f\\\"ACTION{k[1]}\\\"), \\\"signature\\\": class_signature(comps),\\n                  \\\"components\\\": [list(c) for c in comps]}\\n        rules.append(r)\\n    explained_boxes = {k[3] for k in table if k[0] == \\\"CLICK\\\"}\\n    explained_buttons = {k[1] for k in table if k[0] == \\\"BUTTON\\\"}\\n\\n    def unknown(scene: Scene, action: Action):\\n        # a trigger the click map knows to react but the table does not explain -> UNKNOWN; everything else is a no-op\\n        if action.type == \\\"BUTTON\\\":\\n            return None if (action.id not in explained_buttons and any(a.type == \\\"BUTTON\\\" and a.id == action.id for a in available)) else scene\\n        if action.type == \\\"CLICK\\\" and cmap is not None:\\n            from ..probe.clickmap import click_key\\n            oid = object_under(scene, action.row, action.col)\\n            o = scene.get(oid) if oid is not None else None\\n            if o is not None and tuple(o.bbox) in explained_boxes:\\n                return scene\\n            if cmap.status(click_key(scene, action.row, action.col)) == \\\"responsive\\\":\\n                return None\\n        return scene\\n    rules.append(Rule(\\\"effect_unknown\\\", lambda s, a: True, unknown, source=\\\"induced\\\"))\\n\\n    def role_fn(scene: Scene) -> dict[int, str]:\\n        roles = {}\\n        for o in scene.objects:\\n            if tuple(o.bbox) in explained_boxes:\\n                roles[o.id] = f\\\"custom:trigger_at_{o.bbox[0]}_{o.bbox[1]}\\\"\\n        return roles\\n    model = RuleModel(rules, role_fn, default=\\\"noop\\\", name=\\\"induced:effects\\\")\\n    model.level_scoped = True          # positions of triggers and bars are level-specific\\n    return [(\\\"induced:effects\\\", model)]\\n\", \"pbg/wml/evaluate.py\": \"\\\"\\\"\\\"wml/evaluate.py \\u2014 verification of a world model against the WHOLE transition log (spec \\u00a78).\\n\\nscore = exactly predicted transitions / all transitions; coverage = transitions with a non-UNKNOWN prediction.\\nPer-rule confidence: 1 - violations attributed to the rule / times the rule applied.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\n\\nimport hashlib\\nfrom dataclasses import dataclass, field\\nfrom typing import Iterable, Optional\\n\\nfrom ..core.contracts import Hypothesis, Rule, RuleModel, WorldModel, scene_equal\\nfrom ..core.types import Transition\\n\\n\\n@dataclass\\nclass EvalResult:\\n    score: float\\n    coverage: float\\n    violations: list[str] = field(default_factory=list)\\n    unknown: list[str] = field(default_factory=list)\\n    rule_confidence: dict[str, float] = field(default_factory=dict)\\n    prediction_key: str = \\\"\\\"\\n    n: int = 0\\n    per_class: dict[str, tuple[int, int]] = field(default_factory=dict)   # action key -> (correct, total)\\n    change_score: float = 0.0   # accuracy on transitions where the board changed (a \\\"nothing happens\\\" model scores 0 here)\\n    changes_seen: int = 0       # number of visible-change transitions in the log (0 -> change_score is vacuous, not evidence)\\n\\n\\ndef _applied_rules(model: WorldModel, t: Transition) -> list[str]:\\n    if not isinstance(model, RuleModel):\\n        return []\\n    s = model.with_roles(t.before)\\n    names = []\\n    for r in model.rules():\\n        try:\\n            if r.applies(s, t.action):\\n                names.append(r.name)\\n        except Exception:\\n            pass\\n    return names\\n\\n\\ndef evaluate(model: WorldModel, log: Iterable[Transition], *, skip_reset: bool = True, ignore_ui: bool = True) -> EvalResult:\\n    log = [t for t in log if not (skip_reset and t.action.type == \\\"RESET\\\")]\\n    if log and getattr(model, \\\"level_scoped\\\", False):\\n        last = max(t.level for t in log)\\n        log = [t for t in log if t.level == last]\\n    if not log:\\n        return EvalResult(0.0, 0.0, n=0)\\n    correct = covered = 0; ch_ok = 0\\n    viol: list[str] = []; unk: list[str] = []\\n    applied_total: dict[str, int] = {}; applied_bad: dict[str, int] = {}\\n    keys = []\\n    per_class: dict[str, list[int]] = {}\\n    for t in log:\\n        try:\\n            pred = model.predict(t.before, t.action)\\n        except Exception:\\n            pred = None\\n        cls = per_class.setdefault(t.action.key, [0, 0]); cls[1] += 1\\n        if pred is None:\\n            unk.append(t.id); keys.append(\\\"?\\\")\\n            continue\\n        covered += 1\\n        ok = _equal(pred, t.after, ignore_ui, t, model)\\n        if not t.diff.is_noop and not _equal(t.before, t.after, ignore_ui, t, model):\\n            ch_ok += int(ok)      # a change the evaluator can see (not a strip tick / 1-2 px mark)\\n        keys.append(hashlib.sha1(repr(sorted(o.identity() for o in pred.objects)).encode()).hexdigest()[:8])\\n        for name in _applied_rules(model, t):\\n            applied_total[name] = applied_total.get(name, 0) + 1\\n            if not ok:\\n                applied_bad[name] = applied_bad.get(name, 0) + 1\\n        if ok:\\n            correct += 1; cls[0] += 1\\n        else:\\n            viol.append(t.id)\\n    conf = {n: 1.0 - applied_bad.get(n, 0) / c for n, c in applied_total.items()}\\n    if isinstance(model, RuleModel):\\n        for r in model.rules():\\n            if r.name in conf:\\n                r.confidence = conf[r.name]\\n                r.evidence = [t.id for t in log if t.id not in viol and r.name in _applied_rules(model, t)][:50]\\n                r.violated_by = [t.id for t in log if t.id in viol and r.name in _applied_rules(model, t)][:50]\\n    unk_set = set(unk)\\n    ch_total_all = sum(1 for t in log if t.id not in unk_set and not t.diff.is_noop and not _equal(t.before, t.after, ignore_ui, t, model))\\n    return EvalResult(correct / len(log), covered / len(log), viol, unk, conf, hashlib.sha1(\\\"|\\\".join(keys).encode()).hexdigest()[:12], len(log),\\n                      {k: (v[0], v[1]) for k, v in per_class.items()}, change_score=(ch_ok / ch_total_all) if ch_total_all else 1.0,\\n                      changes_seen=ch_total_all)\\n\\n\\ndef _is_rect(o, scene):\\n    from .effects import _is_rect as f\\n    return f(o, scene)\\n\\n\\ndef _effective_bbox(o, scene):\\n    from .effects import _effective_bbox as f\\n    return f(o, scene)\\n\\n\\ndef _equal(pred, obs, ignore_ui: bool, t: Transition, model=None) -> bool:\\n    \\\"\\\"\\\"Object-level equality; objects in ui_strip regions and objects the model calls 'indicator' are compared by\\n    existence only (display elements may recolour/tick without being part of the mechanics).\\\"\\\"\\\"\\n    if not ignore_ui:\\n        return scene_equal(pred, obs)\\n    bands = [r.bbox for r in list(obs.regions) + list(pred.regions) if r.kind_hint == \\\"ui_strip\\\"]\\n    loose_ids: set[int] = set()\\n    if model is not None:\\n        try:\\n            roles = model.assign_roles(obs)\\n            loose_ids = {i for i, r in roles.items() if r == \\\"indicator\\\"}\\n        except Exception:\\n            loose_ids = set()\\n\\n    def in_band(o) -> bool:      # geometry, not region ids: region numbering can differ between the two scenes\\n        r0, c0, r1, c1 = o.bbox\\n        return any(b[0] <= r0 and b[1] <= c0 and r1 <= b[2] and c1 <= b[3] for b in bands)\\n\\n    def covered(o, scene) -> bool:      # fully under another (larger) object: unobservable in the frame\\n        r0, c0, r1, c1 = o.bbox\\n        return any(p.id != o.id and p.area > o.area and p.bbox[0] <= r0 and p.bbox[1] <= c0 and r1 <= p.bbox[2] and c1 <= p.bbox[3]\\n                   for p in scene.objects)\\n\\n    def strict(o, scene):\\n        return not in_band(o) and o.id not in loose_ids and o.area > 2 and not covered(o, scene)   # 1-2 px marks/ticks are display elements\\n\\n    def key(o, scene):\\n        # a bar/panel is compared by colour and extent (extended under the caps drawn over its ends): a button drawn\\n        # on top of it, or a piece cut out of it, is not a different bar\\n        if _is_rect(o, scene) and max(o.height, o.width) >= 8:\\n            return (int(o.color), tuple(_effective_bbox(o, scene)), \\\"bar\\\")\\n        return o.identity()\\n    pa = [o for o in pred.objects if strict(o, pred)]; ob_ = [o for o in obs.objects if strict(o, obs)]\\n    a = sorted(key(o, pred) for o in pa); b = sorted(key(o, obs) for o in ob_)\\n    if a == b:\\n        return True\\n    ca, cb = Counter(a), Counter(b)\\n    only_a = list((ca - cb).elements()); only_b = list((cb - ca).elements())\\n    ka = {key(o, pred): o for o in pa}; kb = {key(o, obs): o for o in ob_}\\n\\n    def hidden(o, other):        # under an object of the OTHER scene (a bar drawn over a button)\\n        r0, c0, r1, c1 = o.bbox\\n        return any(p.area > o.area and p.color != o.color and p.bbox[0] <= r0 and p.bbox[1] <= c0 and r1 <= p.bbox[2] and c1 <= p.bbox[3]\\n                   for p in other.objects)\\n    only_a = [k for k in only_a if not (k in ka and hidden(ka[k], obs))]\\n    only_b = [k for k in only_b if not (k in kb and hidden(kb[k], pred))]\\n    return not only_a and not only_b\\n\\n\\ndef promotable(res: EvalResult, min_transitions: int = 20, min_per_class: int = 3) -> bool:\\n    \\\"\\\"\\\"Promotion condition (spec \\u00a78): score == 1.0, >= 20 transitions, >= 3 observations per action class, AND the log\\n    contained at least one visible change the model got right. Without that last clause a \\\"nothing happens\\\" model on a\\n    change-less log verifies vacuously (change_score falls back to 1.0), gets promoted to memory, and is transferred to\\n    other games where it explains nothing yet bypasses the quality gate -> no plan -> the planner spins.\\\"\\\"\\\"\\n    if res.score < 1.0 or res.n < min_transitions or res.changes_seen < 1:\\n        return False\\n    return all(tot >= min_per_class for _, tot in res.per_class.values())\\n\\n\\ndef make_hypothesis(model: WorldModel, res: EvalResult, code: str = \\\"\\\", name: str = \\\"\\\", origin: str = \\\"llm\\\", recent_mismatches: int = 0) -> Hypothesis:\\n    return Hypothesis(model, res.score, res.coverage, list(res.violations), code, name, promotable(res), res.prediction_key, origin, recent_mismatches,\\n                      change_score=res.change_score)\\n\\n\\ndef dedupe(H: list[Hypothesis]) -> list[Hypothesis]:\\n    seen: dict[str, Hypothesis] = {}\\n    for h in H:\\n        # prior-composed variants are deduplicated by name only: an LLM model that predicts the same log can still\\n        # generalise differently (a permutation table vs a uniform shift), and the composed one must stay available\\n        key = f\\\"induced:{h.name}\\\" if h.origin == \\\"induced\\\" else (h.prediction_key or h.name)\\n        if key in seen:\\n            keep = h if h.sort_key() > seen[key].sort_key() else seen[key]\\n            keep.recent_mismatches = max(h.recent_mismatches, seen[key].recent_mismatches)   # live mispredictions are not forgotten\\n            seen[key] = keep\\n        else:\\n            seen[key] = h\\n    return list(seen.values())\\n\", \"pbg/wml/experiment.py\": \"\\\"\\\"\\\"wml/experiment.py \\u2014 information-gain experiment selection (spec \\u00a78): pick the action on which the hypotheses'\\npredictions disagree most (entropy of the prediction partition, weighted by hypothesis score).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom typing import Optional\\n\\nfrom ..core.contracts import Hypothesis\\nfrom ..core.types import Action, Scene\\n\\n\\ndef entropy_of_partition(preds: list, weights: list[float]) -> float:\\n    groups: dict[str, float] = {}\\n    for p, w in zip(preds, weights):\\n        key = \\\"?\\\" if p is None else repr(sorted(o.identity() for o in p.objects)) + repr(sorted((k, repr(v)) for k, v in p.aux.items() if not k.startswith(\\\"_\\\")))\\n        groups[key] = groups.get(key, 0.0) + max(w, 1e-6)\\n    tot = sum(groups.values())\\n    if tot <= 0 or len(groups) < 2:\\n        return 0.0\\n    return -sum(v / tot * math.log(v / tot, 2) for v in groups.values())\\n\\n\\ndef candidate_actions(scene: Scene, available: list[Action], semantics=None, extra_clicks: list[tuple[int, int]] = (), click_map=None) -> list[Action]:\\n    out: list[Action] = []\\n    for a in available:\\n        if a.type == \\\"BUTTON\\\":\\n            out.append(a)\\n        elif a.type == \\\"CLICK\\\" and click_map is not None and click_map.total() >= 3:\\n            out.extend(click_map.rank(scene, include_inert=False, limit=40, extra=extra_clicks))\\n        elif a.type == \\\"CLICK\\\":\\n            seen = set()\\n            strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n            for o in sorted(scene.objects, key=lambda o: -o.area):\\n                if o.region in strips:\\n                    continue\\n                c = o.center\\n                if c not in seen:\\n                    seen.add(c); out.append(Action.click(*c))\\n            for r in scene.regions:\\n                if r.kind_hint != \\\"ui_strip\\\" and r.center not in seen:\\n                    seen.add(r.center); out.append(Action.click(*r.center))\\n            for rc in extra_clicks:\\n                if tuple(rc) not in seen:\\n                    seen.add(tuple(rc)); out.append(Action.click(*rc))\\n    return out[:64]\\n\\n\\ndef most_informative_action(H: list[Hypothesis], scene: Scene, available: list[Action], *, tau: float = 0.3, semantics=None,\\n                            extra_clicks=(), exclude: Optional[set] = None, state_key=None, click_map=None) -> Optional[Action]:\\n    \\\"\\\"\\\"`exclude` = {(frame_hash, action label)} already run: an experiment is never repeated from the same state.\\\"\\\"\\\"\\n    if len(H) < 2:\\n        return None\\n    best, best_gain = None, 0.0\\n    weights = [max(h.score, 0.05) for h in H]\\n    for a in candidate_actions(scene, available, semantics, extra_clicks, click_map):\\n        if exclude and ((state_key if state_key is not None else scene.frame_hash), a.label()) in exclude:\\n            continue\\n        preds = []\\n        for h in H:\\n            try:\\n                preds.append(h.model.predict(scene, a))\\n            except Exception:\\n                preds.append(None)\\n        gain = entropy_of_partition(preds, weights)\\n        if gain > best_gain:\\n            best, best_gain = a, gain\\n    return best if best_gain > tau else None\\n\", \"pbg/wml/induce.py\": \"\\\"\\\"\\\"wml/induce.py \\u2014 deterministic hypothesis induction from ActionSemantics + prior mechanisms (no LLM).\\n\\nThe spec generates candidates with the LLM; this module provides the prior-driven baseline the lab always tries first\\n(and the no-LLM fallback): movers under button actions become `agent`, consistent displacements become move rules,\\ncancelled moves become wall / floor rules, disappearances on overlap become collect rules, ui strips become shrink rules.\\nEvery candidate is still verified on the whole log by evaluate().\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom typing import Optional\\n\\nimport numpy as np\\n\\nfrom ..core.contracts import Rule, RuleModel\\nfrom ..core.types import Action, Scene, Transition\\nfrom ..memory.priors.mechanisms import (cancel_move_if_off_floor, cancel_move_if_outside, cancel_move_if_overlap, collect_on_overlap,\\n                                        move_on_click, move_role, noop_for, permute_on_click, push_role, recolor_on_click, remove_on_click, unknown_for)\\nfrom ..env.calibrate import object_under\\n\\n\\ndef infer_roles_static(log: list[Transition], semantics: dict) -> dict:\\n    \\\"\\\"\\\"Role hints from the log: colours/shapes of movers -> agent; objects that stopped a move -> wall; objects that vanished under the agent -> collectible.\\\"\\\"\\\"\\n    hints: dict = {\\\"agent\\\": set(), \\\"wall_colors\\\": set(), \\\"collectible_colors\\\": set(), \\\"floor_colors\\\": set()}\\n    movers = set()\\n    for k, v in semantics.items():\\n        if not k.startswith(\\\"_\\\") and isinstance(v, dict) and v.get(\\\"class\\\") == \\\"MOVE\\\":\\n            movers |= set(v.get(\\\"movers\\\", []))\\n    for t in log:\\n        for o in t.before.objects:\\n            if o.id in movers:\\n                hints[\\\"agent\\\"].add((o.colors, o.shape_sig))\\n    hints[\\\"agent_colors\\\"] = {cs for cs, _ in hints[\\\"agent\\\"]}\\n    # indicators: objects in non-board panels (or outside every panel) that recoloured but never moved\\n    recolored: set[tuple] = set(); moved_keys: set[tuple] = set()\\n    for t in log:\\n        for oid, _, _ in t.diff.recolored:\\n            o = t.before.get(oid)\\n            if o is not None:\\n                recolored.add((o.region, tuple(o.bbox)))\\n        reshaped = {x[0] for x in t.diff.reshaped}\\n        for oid, _ in t.diff.moved:\\n            o = t.before.get(oid)\\n            if o is not None and oid not in reshaped:      # a shrink/grow is not a move\\n                moved_keys.add((o.region, tuple(o.bbox)))\\n    hints[\\\"indicator_boxes\\\"] = {k for k in recolored if k not in moved_keys}\\n    # regions (by bg colour) an agent-class object has ever been in; objects that change inside OTHER non-board regions are\\n    # display elements (a direction compass in a side panel)\\n    agent_regions: set = set()\\n    for t in log:\\n        for o in t.before.objects:\\n            if (o.colors, o.shape_sig) in hints[\\\"agent\\\"]:\\n                reg = t.before.region(o.region)\\n                if reg is not None:\\n                    agent_regions.add(reg.bg_color)\\n    changed_boxes: set = set()\\n    for t in log:\\n        for oid, _ in t.diff.moved:\\n            o = t.before.get(oid)\\n            if o is not None:\\n                reg = t.before.region(o.region)\\n                if reg is not None and reg.id != \\\"R0\\\" and reg.kind_hint != \\\"ui_strip\\\" and reg.bg_color not in agent_regions and (o.colors, o.shape_sig) not in hints[\\\"agent\\\"]:\\n                    changed_boxes.add((o.region, tuple(o.bbox)))\\n    hints[\\\"indicator_boxes\\\"] |= changed_boxes\\n    # object classes that move while no agent is anywhere near them react to the button, not to the agent: indicators\\n    # (ls20's direction compass lives outside every panel)\\n    far_moves: Counter = Counter(); near_moves: Counter = Counter()\\n    for t in log:\\n        agents_before = [o for o in t.before.objects if (o.colors, o.shape_sig) in hints[\\\"agent\\\"]]\\n        for oid, (dr, dc) in t.diff.moved:\\n            o = t.before.get(oid)\\n            if o is None or (o.colors, o.shape_sig) in hints[\\\"agent\\\"]:\\n                continue\\n            reg = t.before.region(o.region)\\n            if reg is not None and reg.kind_hint == \\\"ui_strip\\\":\\n                continue\\n            moved_box = o.moved(dr, dc)\\n            touched = any(a.overlaps(o) or a.overlaps(moved_box) or _near(a, o, 6) for a in agents_before)\\n            (near_moves if touched else far_moves)[(o.colors, o.shape_sig)] += 1\\n    hints[\\\"indicator_classes\\\"] = {k for k, n in far_moves.items() if n >= 2 and n > 2 * near_moves.get(k, 0)}\\n    hints[\\\"indicator_regions\\\"] = set()\\n    if agent_regions:\\n        for t in log[-1:]:\\n            for reg in t.after.regions:\\n                if reg.id != \\\"R0\\\" and reg.kind_hint == \\\"panel\\\" and reg.bg_color not in agent_regions:\\n                    hints[\\\"indicator_regions\\\"].add(reg.bg_color)\\n    # floor: colours of the pixels the agent actually moved onto (rendered scene without the agent)\\n    for t in log:\\n        moved = dict(t.diff.moved)\\n        agents = [o for o in t.before.objects if o.id in movers and o.id in moved]\\n        if not agents:\\n            continue\\n        floor = t.before.copy(objects=[o for o in t.before.objects if o.id not in {a.id for a in agents}]).render()\\n        h, w = floor.shape\\n        for a in agents:\\n            g = a.moved(*moved[a.id]); r0, c0, r1, c1 = g.bbox\\n            if 0 <= r0 and 0 <= c0 and r1 <= h and c1 <= w:\\n                hints[\\\"floor_colors\\\"] |= {int(v) for v in floor[r0:r1, c0:c1][g.mask]}\\n    return hints\\n\\n\\ndef _near(a, b, d: int) -> bool:\\n    ar0, ac0, ar1, ac1 = a.bbox; br0, bc0, br1, bc1 = b.bbox\\n    return not (ar1 + d <= br0 or br1 + d <= ar0 or ac1 + d <= bc0 or bc1 + d <= ac0)\\n\\n\\ndef _role_fn_factory(agent_keys: set, wall_colors: set, collectible_colors: set, agent_colors: set, pushable_colors: set = frozenset(),\\n                     indicator_boxes: set = frozenset(), indicator_regions: set = frozenset(), indicator_classes: set = frozenset()):\\n    ind_boxes = {b for _, b in indicator_boxes}\\n    ind_regions = set(indicator_regions or ())\\n    agent_palette = set()\\n    for cs in agent_colors:\\n        agent_palette |= set(cs)\\n\\n    def agent_similarity(o) -> float:\\n        \\\"\\\"\\\"Colour-overlap similarity with the known agent appearances (an animated avatar changes pattern with facing).\\\"\\\"\\\"\\n        cs = set(o.colors)\\n        if not cs & agent_palette:\\n            return 0.0\\n        return len(cs & agent_palette) / len(cs | agent_palette)\\n\\n    def role_fn(scene: Scene) -> dict[int, str]:\\n        roles = {}\\n        strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n        for o in scene.objects:\\n            reg = scene.region(o.region)\\n            if o.region in strips or (tuple(o.bbox) in ind_boxes and (o.colors, o.shape_sig) not in agent_keys) or \\\\\\n               (reg is not None and reg.bg_color in ind_regions and (o.colors, o.shape_sig) not in agent_keys) or \\\\\\n               ((o.colors, o.shape_sig) in indicator_classes and (o.colors, o.shape_sig) not in agent_keys):\\n                roles[o.id] = \\\"indicator\\\"\\n            elif (o.colors, o.shape_sig) in agent_keys or (o.colors in agent_colors and len(o.colors) > 1):\\n                roles[o.id] = \\\"agent\\\"\\n            elif o.color in collectible_colors:\\n                roles[o.id] = \\\"collectible\\\"\\n            elif o.color in pushable_colors:\\n                roles[o.id] = \\\"pushable\\\"\\n            elif o.color in wall_colors:\\n                roles[o.id] = \\\"wall\\\"\\n            else:\\n                roles[o.id] = \\\"unknown\\\"\\n        if \\\"agent\\\" not in roles.values():\\n            # appearance changed (facing/animation): the multi-colour object most similar to the known agent palette,\\n            # else the single-colour object of an agent colour\\n            cands = [(agent_similarity(o), len(o.colors), o.area, o) for o in scene.objects if o.region not in strips and roles.get(o.id) != \\\"indicator\\\"]\\n            cands = [c for c in cands if c[0] > 0]\\n            if cands:\\n                cands.sort(key=lambda c: (-c[0], -c[1], -c[2]))\\n                roles[cands[0][3].id] = \\\"agent\\\"\\n        return roles\\n    return role_fn\\n\\n\\ndef _panel_color(scene: Scene, o) -> int:\\n    \\\"\\\"\\\"Background colour of the smallest region containing the object whose colour differs from the global background\\n    (the panel a body lives in); the global background when there is none.\\\"\\\"\\\"\\n    gb = scene.aux.get(\\\"_global_bg\\\", scene.regions[0].bg_color if scene.regions else -1)\\n    best = None\\n    r, c = o.center\\n    for reg in scene.regions:\\n        if reg.id == \\\"R0\\\" or reg.kind_hint == \\\"ui_strip\\\" or reg.bg_color == gb:\\n            continue\\n        r0, c0, r1, c1 = reg.bbox\\n        if r0 <= r < r1 and c0 <= c < c1 and (best is None or reg.area < best.area):\\n            best = reg\\n    return best.bg_color if best is not None else gb\\n\\n\\ndef induce_multi_agent(log: list[Transition], semantics: dict, available: list[Action]) -> list[tuple[str, RuleModel]]:\\n    \\\"\\\"\\\"Several bodies that respond to the same buttons with different displacements (mirrored twin in m0r0/ar25):\\n    group movers by (colours, shape, region); each group gets its own direction table and role agent / agent2 / ...\\\"\\\"\\\"\\n    groups: dict[tuple, dict[int, Counter]] = {}\\n    keys_of: dict[tuple, set] = {}\\n    for t in log:\\n        if t.action.type != \\\"BUTTON\\\":\\n            continue\\n        reshaped = {x[0] for x in t.diff.reshaped}\\n        for i, (dr, dc) in t.diff.moved:\\n            o = t.before.get(i)\\n            if o is None or i in reshaped:\\n                continue\\n            reg = t.before.region(o.region)\\n            if reg is None or reg.kind_hint == \\\"ui_strip\\\":\\n                continue\\n            gk = (o.colors, o.shape_sig, i)      # identical twins are told apart only by their tracking id\\n            groups.setdefault(gk, {}).setdefault(t.action.id, Counter())[(dr, dc)] += 1\\n    # keep groups that move under >= 2 buttons; direction = modal displacement per button\\n    tables = {}\\n    for gk, per_button in groups.items():\\n        tab = {b: c.most_common(1)[0][0] for b, c in per_button.items() if c.most_common(1)[0][1] >= 1}\\n        if len(tab) >= 2:\\n            tables[gk] = tab\\n    if len(tables) < 2:\\n        return []\\n    # merge groups whose direction tables agree (the same body under a re-assigned id); keep at most 4 distinct bodies\\n    merged: list[tuple[dict, list]] = []\\n    for gk in sorted(tables, key=lambda k: k[2]):\\n        tab = tables[gk]\\n        for mtab, ids in merged:\\n            if all(mtab.get(b) == d for b, d in tab.items() if b in mtab) and any(b in mtab for b in tab):\\n                mtab.update({b: d for b, d in tab.items() if b not in mtab}); ids.append(gk[2]); break\\n        else:\\n            merged.append((dict(tab), [gk[2]]))\\n    if len(merged) < 2:\\n        return []\\n    merged = merged[:4]\\n    tables = {i: tab for i, (tab, _) in enumerate(merged)}\\n    id_to_group = {oid: i for i, (_, ids) in enumerate(merged) for oid in ids}\\n    appearance = {(gk[0], gk[1]) for gk in groups}\\n    roles = {i: (\\\"agent\\\" if i == 0 else f\\\"agent{i + 1}\\\") for i in tables}\\n    hints = infer_roles_static(log, semantics)\\n    wall_colors = set(); collectible_colors = set()\\n    buttons = sorted({a.id for a in available if a.type == \\\"BUTTON\\\"})\\n    used = set().union(*[set(t) for t in tables.values()])\\n    other = tuple(b for b in buttons if b not in used)\\n\\n    def role_fn(scene: Scene) -> dict[int, str]:\\n        out = {}\\n        strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n        bodies = [o for o in scene.objects if (o.colors, o.shape_sig) in appearance and o.region not in strips]\\n        taken = set()\\n        for o in bodies:                                   # known tracking ids first\\n            if o.id in id_to_group and id_to_group[o.id] not in taken:\\n                out[o.id] = roles[id_to_group[o.id]]; taken.add(id_to_group[o.id])\\n        for o in sorted(bodies, key=lambda o: o.center[1]):   # unknown ids: left-to-right into the free groups\\n            if o.id not in out:\\n                free = [i for i in tables if i not in taken]\\n                if free:\\n                    out[o.id] = roles[free[0]]; taken.add(free[0])\\n        for o in scene.objects:\\n            if o.id in out:\\n                continue\\n            out[o.id] = \\\"indicator\\\" if (o.region in strips or tuple(o.bbox) in {b for _, b in hints[\\\"indicator_boxes\\\"]}) else \\\"unknown\\\"\\n        return out\\n\\n    floor = tuple(sorted(hints[\\\"floor_colors\\\"]))\\n\\n    def build(with_floor: bool) -> RuleModel:\\n        rules = []\\n        for gi, tab in tables.items():\\n            step = max(abs(dr) + abs(dc) for dr, dc in tab.values()) or 1\\n            unit = {b: (int(np.sign(dr)), int(np.sign(dc))) for b, (dr, dc) in tab.items()}\\n            role = roles[gi]\\n            rules.append(Rule(f\\\"move_{role}\\\", lambda s, a, tab=tab: a.type == \\\"BUTTON\\\" and a.id in tab, move_role(role, unit, step), source=\\\"induced\\\"))\\n            if with_floor and floor:\\n                rules.append(Rule(f\\\"{role}_on_floor\\\", lambda s, a, tab=tab: a.type == \\\"BUTTON\\\" and a.id in tab, cancel_move_if_off_floor(role, floor), source=\\\"induced\\\"))\\n            else:\\n                rules.append(Rule(f\\\"{role}_stays_inside\\\", lambda s, a, tab=tab: a.type == \\\"BUTTON\\\" and a.id in tab, cancel_move_if_outside(role), source=\\\"induced\\\"))\\n        if other:\\n            rules.append(Rule(\\\"unknown_buttons\\\", lambda s, a: a.type == \\\"BUTTON\\\" and a.id in other, unknown_for(other), source=\\\"induced\\\"))\\n        name = \\\"induced:multi_agent\\\" + (\\\"+floor\\\" if with_floor else \\\"\\\")\\n        return RuleModel(rules, role_fn, default=\\\"unknown\\\", name=name)\\n\\n    return [(\\\"induced:multi_agent+floor\\\", build(True)), (\\\"induced:multi_agent\\\", build(False))]\\n\\n\\ndef induce_hypotheses(log: list[Transition], semantics: dict, available: list[Action], knowledge=None) -> list[tuple[str, RuleModel]]:\\n    \\\"\\\"\\\"Return (name, model) candidates built from priors. Several variants so evaluate() can pick the best.\\\"\\\"\\\"\\n    if not log:\\n        return []\\n    def consistent(v) -> bool:      # a translation must repeat its displacement; rotations mislabelled as moves do not\\n        dc = v.get(\\\"displacement_consistency\\\", \\\"1/1\\\").split(\\\"/\\\")\\n        return int(dc[0]) >= 2 and int(dc[0]) * 2 >= int(dc[1])\\n    moves = {k: tuple(v[\\\"displacement\\\"]) for k, v in semantics.items()\\n             if not k.startswith(\\\"_\\\") and isinstance(v, dict) and v.get(\\\"class\\\") == \\\"MOVE\\\" and v.get(\\\"displacement\\\") and consistent(v)}\\n    hints = infer_roles_static(log, semantics)\\n    agent_keys = set(hints[\\\"agent\\\"]); agent_colors = set(hints[\\\"agent_colors\\\"])\\n    from .effects import induce_effect_model\\n    effects = induce_effect_model(log, semantics, available, knowledge)     # joint-effect table: every trigger, with resize guards\\n    from .point import induce_point_hypotheses\\n    points = induce_point_hypotheses(log, semantics, available, knowledge)  # click = destination, the piece steps toward it\\n    if not moves or not agent_keys:\\n        return induce_click_hypotheses(log, semantics, available, knowledge) + effects + points\\n    dirs = {int(k.replace(\\\"ACTION\\\", \\\"\\\")): d for k, d in moves.items() if k.startswith(\\\"ACTION\\\")}\\n    if not dirs:\\n        return induce_click_hypotheses(log, semantics, available, knowledge) + effects + points\\n    multi = induce_multi_agent(log, semantics, available)\\n    buttons = sorted({a.id for a in available if a.type == \\\"BUTTON\\\"})\\n    noop_buttons = tuple(b for b in buttons if f\\\"ACTION{b}\\\" in semantics and semantics[f\\\"ACTION{b}\\\"].get(\\\"class\\\") == \\\"NOOP\\\")\\n    other_buttons = tuple(b for b in buttons if b not in dirs and b not in noop_buttons)\\n    # candidate wall colours: colours of objects the agent overlapped when a move did not happen\\n    wall_votes: Counter = Counter(); collect_votes: Counter = Counter(); push_votes: Counter = Counter()\\n    for t in log:\\n        if t.action.type != \\\"BUTTON\\\" or t.action.id not in dirs:\\n            continue\\n        dr, dc = dirs[t.action.id]\\n        agents = [o for o in t.before.objects if (o.colors, o.shape_sig) in agent_keys or o.colors in agent_colors]\\n        moved_ids = {i for i, _ in t.diff.moved}\\n        for a in agents:\\n            ghost = a.moved(dr, dc)\\n            if a.id not in moved_ids:\\n                for o in t.before.objects:\\n                    if o.id != a.id and ghost.overlaps(o) and o.colors not in agent_colors:\\n                        wall_votes[o.color] += 1\\n            else:\\n                for o in t.diff.disappeared:\\n                    if ghost.overlaps(o):\\n                        collect_votes[o.color] += 1\\n                for i, d in t.diff.moved:\\n                    if i != a.id and d == (dr, dc):\\n                        ob = t.before.get(i)\\n                        if ob is not None and ob.colors not in agent_colors:\\n                            push_votes[ob.color] += 1\\n    wall_colors = {c for c, n in wall_votes.items() if n >= 1}\\n    collectible_colors = {c for c, n in collect_votes.items() if n >= 1}\\n    pushable_colors = {c for c, n in push_votes.items() if n >= 2}\\n    floor_colors = tuple(sorted(hints[\\\"floor_colors\\\"]))\\n    step_sizes = {abs(dr) + abs(dc) for dr, dc in dirs.values()}\\n    unit_dirs = {k: (int(np.sign(dr)), int(np.sign(dc))) for k, (dr, dc) in dirs.items()}\\n    step = max(step_sizes) if step_sizes else 1\\n    # assumed directions for arrow buttons not yet observed to move (ARC-AGI-3 convention 1=up 2=down 3=left 4=right);\\n    # kept as a separate hypothesis so an information-gain experiment can test it\\n    from ..memory.priors.mechanisms import DIRS4\\n    assumed = {b: DIRS4[b] for b in buttons if b in DIRS4 and b not in unit_dirs and b not in noop_buttons}\\n    other_buttons = tuple(b for b in other_buttons if b not in assumed) if assumed else other_buttons\\n\\n    def base_rules(with_walls: bool, with_floor: bool, with_collect: bool, with_push: bool, use_assumed: bool = False) -> list[Rule]:\\n        d = dict(unit_dirs); d.update(assumed if use_assumed else {})\\n        rules = [Rule(\\\"move_agent\\\", lambda s, a, d=d: a.type == \\\"BUTTON\\\" and a.id in d, move_role(\\\"agent\\\", d, step), source=\\\"induced\\\")]\\n        if assumed and not use_assumed:\\n            rules.append(Rule(\\\"unassumed_dirs\\\", lambda s, a: a.type == \\\"BUTTON\\\" and a.id in assumed, unknown_for(tuple(assumed)), source=\\\"induced\\\"))\\n        if with_push and pushable_colors:\\n            rules.append(Rule(\\\"push\\\", lambda s, a: a.type == \\\"BUTTON\\\" and a.id in dirs, push_role(\\\"agent\\\", \\\"pushable\\\"), source=\\\"induced\\\"))\\n        if with_walls:\\n            rules.append(Rule(\\\"wall_blocks\\\", lambda s, a: a.type == \\\"BUTTON\\\" and a.id in dirs, cancel_move_if_overlap(\\\"agent\\\", \\\"wall\\\"), source=\\\"induced\\\"))\\n        if with_floor and floor_colors:\\n            rules.append(Rule(\\\"stay_on_floor\\\", lambda s, a: a.type == \\\"BUTTON\\\" and a.id in dirs, cancel_move_if_off_floor(\\\"agent\\\", floor_colors), source=\\\"induced\\\"))\\n        else:\\n            rules.append(Rule(\\\"stay_inside\\\", lambda s, a: a.type == \\\"BUTTON\\\" and a.id in dirs, cancel_move_if_outside(\\\"agent\\\"), source=\\\"induced\\\"))\\n        if with_collect and collectible_colors:\\n            rules.append(Rule(\\\"collect\\\", lambda s, a: a.type == \\\"BUTTON\\\" and a.id in dirs, collect_on_overlap(\\\"agent\\\", \\\"collectible\\\"), source=\\\"induced\\\"))\\n        if noop_buttons:\\n            rules.append(Rule(\\\"noop_buttons\\\", lambda s, a: a.type == \\\"BUTTON\\\" and a.id in noop_buttons, noop_for(noop_buttons), source=\\\"induced\\\"))\\n        if other_buttons:\\n            rules.append(Rule(\\\"unknown_buttons\\\", lambda s, a: a.type == \\\"BUTTON\\\" and a.id in other_buttons, unknown_for(other_buttons), source=\\\"induced\\\"))\\n        return rules\\n\\n    out = []\\n    variants = [(\\\"induced:move+walls+floor+collect+push\\\", True, True, True, True), (\\\"induced:move+floor+collect\\\", False, True, True, False),\\n                (\\\"induced:move+walls+collect\\\", True, False, True, False), (\\\"induced:move+floor\\\", False, True, False, False), (\\\"induced:move\\\", False, False, False, False),\\n                (\\\"induced:move+collect\\\", False, False, True, False)]     # region boundary blocks, no colour rule: corridors of the background colour are walkable\\n    for name, w, f, c, p in variants:\\n        role_fn = _role_fn_factory(agent_keys, wall_colors if w else set(), collectible_colors if c else set(), agent_colors, pushable_colors if p else set(),\\n                                   hints[\\\"indicator_boxes\\\"], hints.get(\\\"indicator_regions\\\", set()), hints.get(\\\"indicator_classes\\\", set()))\\n        m = RuleModel(base_rules(w, f, c, p), role_fn, default=\\\"unknown\\\", name=name)\\n        out.append((name, m))\\n        if assumed:\\n            m2 = RuleModel(base_rules(w, f, c, p, use_assumed=True), role_fn, default=\\\"unknown\\\", name=name + \\\"+assumed_dirs\\\")\\n            out.append((name + \\\"+assumed_dirs\\\", m2))\\n    return out + multi\\n\\n\\ndef induce_click_hypotheses(log: list[Transition], semantics: dict, available: list[Action], knowledge=None) -> list[tuple[str, RuleModel]]:\\n    \\\"\\\"\\\"Click games: what happens to the clicked object, by (colour) class -> recolour cycle / removal / no-op.\\\"\\\"\\\"\\n    clicks = [t for t in log if t.action.type == \\\"CLICK\\\"]\\n    if not clicks:\\n        return []\\n    effects: dict[int, Counter] = {}     # colour of clicked object -> Counter of effect keys\\n    cycles: dict[int, Counter] = {}\\n\\n    def strip_ids(t: Transition) -> set[int]:\\n        strips = {r.id for r in t.before.regions if r.kind_hint == \\\"ui_strip\\\"} | {r.id for r in t.after.regions if r.kind_hint == \\\"ui_strip\\\"}\\n        return {o.id for o in t.before.objects + t.after.objects if o.region in strips}\\n\\n    for t in clicks:\\n        oid = object_under(t.before, t.action.row, t.action.col)\\n        o = t.before.get(oid) if oid is not None else None\\n        color = o.color if o is not None else -1\\n        eff = effects.setdefault(color, Counter())\\n        rec = [x for x in t.diff.recolored if x[0] == oid]\\n        if oid is not None and rec:\\n            eff[\\\"recolor\\\"] += 1; cycles.setdefault(color, Counter())[(rec[0][1], rec[0][2])] += 1\\n        elif oid is not None and any(x.id == oid for x in t.diff.disappeared):\\n            eff[\\\"remove\\\"] += 1\\n        elif t.diff.is_noop or not (set(i for i, _ in t.diff.moved) | {x[0] for x in t.diff.recolored + t.diff.reshaped} | {o.id for o in t.diff.appeared + t.diff.disappeared}) - strip_ids(t):\\n            eff[\\\"noop\\\"] += 1\\n        else:\\n            eff[\\\"other\\\"] += 1\\n    # click on a trigger object moves other objects (arrows / controls): trigger (colour, shape) -> displacement of movers\\n    trig: dict[tuple, Counter] = {}; trig_movers: dict[tuple, Counter] = {}\\n    for t in clicks:\\n        oid = object_under(t.before, t.action.row, t.action.col)\\n        o = t.before.get(oid) if oid is not None else None\\n        if o is None or not t.diff.moved:\\n            continue\\n        key = (o.color, o.shape_sig)\\n        ui = strip_ids(t)\\n        disp = Counter(v for i, v in t.diff.moved if i != oid and i not in ui)\\n        if disp:\\n            (d, n) = disp.most_common(1)[0]\\n            trig.setdefault(key, Counter())[d] += 1\\n            for i, v in t.diff.moved:\\n                if v == d:\\n                    ob = t.before.get(i)\\n                    if ob is not None:\\n                        trig_movers.setdefault(key, Counter())[ob.color] += 1\\n    click_moves = {k: c.most_common(1)[0][0] for k, c in trig.items() if c.most_common(1)[0][1] >= 2}\\n    # triggers whose displacement is (nearly) always the same: a uniform shift generalises to positions never seen\\n    # (the winning one), which a learned permutation table cannot\\n    trig_box: dict[tuple, Counter] = {}       # per trigger OBJECT (two identical arrow buttons move in opposite directions)\\n    for t in clicks:\\n        oid = object_under(t.before, t.action.row, t.action.col)\\n        o = t.before.get(oid) if oid is not None else None\\n        if o is None or not t.diff.moved:\\n            continue\\n        ui = strip_ids(t)\\n        disp = Counter(v for i, v in t.diff.moved if i != oid and i not in ui)\\n        if disp:\\n            trig_box.setdefault((o.color, o.shape_sig, tuple(o.bbox)), Counter())[disp.most_common(1)[0][0]] += 1\\n    uniform = {k: c.most_common(1)[0][0] for k, c in trig_box.items() if c.most_common(1)[0][1] >= 2 and c.most_common(1)[0][1] >= 0.8 * sum(c.values())}\\n    # knowledge asset (earlier levels): a trigger class whose displacement set is known needs ONE consistent observation\\n    prior_rules = getattr(knowledge, \\\"click_rules\\\", {}) if knowledge is not None else {}\\n    for k, c in trig_box.items():\\n        if k in uniform:\\n            continue\\n        pr = prior_rules.get(f\\\"c{k[0]}:{k[1][:8]}\\\")\\n        d, n = c.most_common(1)[0]\\n        if pr and list(d) in pr.get(\\\"displacements\\\", []) and n >= 1 and n >= 0.8 * sum(c.values()):\\n            uniform[k] = d\\n    # learned permutations: per trigger class, the top-left -> top-left mapping of every moved object, if consistent\\n    votes: dict[tuple, dict] = {}\\n    last_level = max(t.level for t in clicks)\\n    move_counts: Counter = Counter()          # (colour, shape) classes that move at least 3 times are pieces; others are static marks\\n    for t in [x for x in clicks if x.level == last_level and not x.status_change]:\\n        for i, _ in t.diff.moved:\\n            ob = t.before.get(i)\\n            if ob is not None:\\n                move_counts[(ob.color, ob.shape_sig)] += 1\\n    piece_classes = {k for k, n in move_counts.items() if n >= 3}\\n    for t in [x for x in clicks if x.level == last_level and not x.status_change]:   # level-specific positions; a level-up frame is the next board\\n        oid = object_under(t.before, t.action.row, t.action.col)\\n        o = t.before.get(oid) if oid is not None else None\\n        if o is None or not t.diff.moved:\\n            continue\\n        key = (o.color, o.shape_sig, tuple(o.bbox)); ui = strip_ids(t)     # a permutation belongs to ONE trigger object\\n        reshaped = {x[0] for x in t.diff.reshaped}\\n        m = votes.setdefault(key, {})\\n        occupied_before = {(o.bbox[0], o.bbox[1]) for o in t.before.objects}\\n        for i, (dr, dc) in t.diff.moved:\\n            ob = t.before.get(i)\\n            if ob is None or i in ui or i == oid or i in reshaped or (ob.color, ob.shape_sig) not in piece_classes:\\n                continue\\n            src = (ob.bbox[0], ob.bbox[1]); dst = (ob.bbox[0] + dr, ob.bbox[1] + dc)\\n            m.setdefault(src, []).append((dst, frozenset(occupied_before)))\\n    # per source position: the nearest destination is the one-slot move; a farther destination is explained by the\\n    # one-slot target being FREE at click time (pieces slide into gaps). Otherwise majority vote.\\n    perms: dict[tuple, dict] = {}\\n    for key, m in votes.items():\\n        mapping = {}; agree = total = 0\\n        for src, obs_list in m.items():\\n            c = Counter(d for d, _ in obs_list)\\n            total += len(obs_list)\\n            if len(c) == 1:\\n                mapping[src] = next(iter(c)); agree += len(obs_list); continue\\n            # identical adjacent pieces make the tracker report \\\"one piece moved two slots, the other stayed\\\" for what is\\n            # really \\\"both moved one slot\\\": the nearest well-supported destination is the true one-slot move\\n            supported = [d for d, n in c.items() if n >= 0.25 * len(obs_list)]\\n            near = min(supported or c, key=lambda d: abs(d[0] - src[0]) + abs(d[1] - src[1]))\\n            mapping[src] = near; agree += c[near]\\n        if len(mapping) >= 2 and agree >= 0.6 * total:\\n            perms[key] = mapping\\n    if not any(eff.get(\\\"recolor\\\") or eff.get(\\\"remove\\\") for eff in effects.values()) and not click_moves and not perms:\\n        return []\\n    recolor_colors = {c for c, eff in effects.items() if eff.get(\\\"recolor\\\") and eff[\\\"recolor\\\"] >= max(eff.get(\\\"other\\\", 0), 1)}\\n    remove_colors = {c for c, eff in effects.items() if eff.get(\\\"remove\\\") and eff[\\\"remove\\\"] >= max(eff.get(\\\"other\\\", 0), 1)}\\n    # colour cycles: chain a->b, b->c ...\\n    cycle: list[int] = []\\n    pairs = Counter()\\n    for c in recolor_colors:\\n        pairs.update(cycles.get(c, {}))\\n    nxt = {a: b for (a, b), _ in pairs.most_common()}\\n    if nxt:\\n        start = next(iter(nxt)); cur = start; cycle = [start]\\n        while cur in nxt and nxt[cur] not in cycle and len(cycle) < 16:\\n            cur = nxt[cur]; cycle.append(cur)\\n        if cur in nxt and nxt[cur] == start:\\n            pass\\n        else:\\n            cycle.append(nxt.get(cur, cur)) if nxt.get(cur) not in cycle and nxt.get(cur) is not None else None\\n    strips_role = \\\"indicator\\\"\\n\\n    trigger_roles = {k: f\\\"custom:trigger{i}\\\" for i, k in enumerate(click_moves)}\\n    mover_colors = {k: {c for c, _ in trig_movers.get(k, Counter()).most_common(3)} for k in click_moves}\\n    perm_boxes: dict = {k[2]: f\\\"custom:trigger_at_{k[2][0]}_{k[2][1]}\\\" for k in perms}\\n\\n    def role_fn(scene: Scene) -> dict[int, str]:\\n        roles = {}\\n        strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n        for o in scene.objects:\\n            if o.region in strips:\\n                roles[o.id] = strips_role\\n            elif tuple(o.bbox) in perm_boxes:\\n                roles[o.id] = perm_boxes[tuple(o.bbox)]\\n            elif (o.color, o.shape_sig) in trigger_roles:\\n                roles[o.id] = trigger_roles[(o.color, o.shape_sig)]\\n            elif any(o.color in mc for mc in mover_colors.values()):\\n                roles[o.id] = \\\"custom:moved\\\"\\n            elif o.color in recolor_colors or (cycle and o.color in cycle):\\n                roles[o.id] = \\\"custom:toggle\\\"\\n            elif o.color in remove_colors:\\n                roles[o.id] = \\\"collectible\\\"\\n            else:\\n                roles[o.id] = \\\"unknown\\\"\\n        return roles\\n\\n    rules = []\\n    perm_rules = []\\n    for n, (k, m) in enumerate(perms.items()):\\n        role = perm_boxes[k[2]]\\n        perm_rules.append(Rule(f\\\"click_perm_{n}\\\", lambda s, a: a.type == \\\"CLICK\\\", permute_on_click(role, dict(m)), source=\\\"induced\\\"))\\n    perm_classes = {(k[0], k[1]) for k in perms}\\n    for k, (dr, dc) in click_moves.items():\\n        if k in perm_classes:\\n            continue      # permutations explain these triggers better than a uniform shift\\n        rules.append(Rule(f\\\"click_move_{trigger_roles[k]}\\\", lambda s, a: a.type == \\\"CLICK\\\", move_on_click(trigger_roles[k], \\\"custom:moved\\\", dr, dc), source=\\\"induced\\\"))\\n    rules = perm_rules + rules\\n    if cycle and len(cycle) >= 2:\\n        rules.append(Rule(\\\"click_recolor\\\", lambda s, a: a.type == \\\"CLICK\\\", recolor_on_click(\\\"custom:toggle\\\", tuple(cycle)), source=\\\"induced\\\"))\\n    if remove_colors:\\n        rules.append(Rule(\\\"click_remove\\\", lambda s, a: a.type == \\\"CLICK\\\", remove_on_click(\\\"collectible\\\"), source=\\\"induced\\\"))\\n    buttons = tuple(a.id for a in available if a.type == \\\"BUTTON\\\")\\n    if buttons:\\n        rules.append(Rule(\\\"unknown_buttons\\\", lambda s, a: a.type == \\\"BUTTON\\\", unknown_for(buttons), source=\\\"induced\\\"))\\n    if not rules:\\n        return []\\n    model = RuleModel(rules, role_fn, default=\\\"noop\\\", name=\\\"induced:click\\\")\\n    model.level_scoped = bool(perms)      # learned position permutations only hold for the level they were seen on\\n    model.position_graph = {k[2]: {src: (d[\\\"near\\\"] if isinstance(d, dict) else d) for src, d in m.items()} for k, m in perms.items()}   # trigger bbox -> {top-left -> top-left}\\n    out = [(\\\"induced:click\\\", model)]\\n    shift_keys = {k: uniform[k] for k in perms if k in uniform}\\n    if shift_keys:\\n        # variant: uniform shifts instead of permutation tables for the consistent triggers (a slider moved by arrow buttons)\\n        rules2 = [r for r in rules if not r.name.startswith(\\\"click_perm_\\\")]\\n        rules2 += [Rule(f\\\"click_perm_{n}\\\", lambda s, a: a.type == \\\"CLICK\\\", permute_on_click(perm_boxes[k[2]], dict(m)), source=\\\"induced\\\")\\n                   for n, (k, m) in enumerate(perms.items()) if k not in shift_keys]\\n        for k, (dr, dc) in shift_keys.items():\\n            r_ = Rule(f\\\"click_shift_{perm_boxes[k[2]]}\\\", lambda s, a: a.type == \\\"CLICK\\\", move_on_click(perm_boxes[k[2]], \\\"custom:moved\\\", dr, dc), source=\\\"induced\\\")\\n            r_.meta = {\\\"kind\\\": \\\"click_shift\\\", \\\"trigger_class\\\": f\\\"c{k[0]}:{k[1][:8]}\\\", \\\"displacement\\\": [int(dr), int(dc)],\\n                       \\\"mover_colors\\\": sorted(mover_colors.get((k[0], k[1]), set()))}\\n            rules2.insert(0, r_)\\n        if knowledge is not None and getattr(knowledge, \\\"clicks\\\", None) is not None:\\n            # clicks on classes the click map knows to react but no rule explains -> UNKNOWN, not \\\"nothing happens\\\"\\n            cmap = knowledge.clicks\\n            explained = {perm_boxes[k[2]] for k in shift_keys} | {perm_boxes[k[2]] for k in perms if k not in shift_keys}\\n            def unknown_click(scene: Scene, action: Action):\\n                if action.type != \\\"CLICK\\\":\\n                    return scene\\n                from ..probe.clickmap import click_key\\n                key = click_key(scene, action.row, action.col)\\n                oid = object_under(scene, action.row, action.col)\\n                o = scene.get(oid) if oid is not None else None\\n                if cmap.status(key) == \\\"responsive\\\" and (o is None or o.role not in explained):\\n                    return None\\n                return scene\\n            rules2.append(Rule(\\\"unknown_responsive_click\\\", lambda s, a: a.type == \\\"CLICK\\\", unknown_click, source=\\\"induced\\\"))\\n        m2 = RuleModel(rules2, role_fn, default=\\\"noop\\\", name=\\\"induced:click-shift\\\")\\n        m2.level_scoped = any(k not in shift_keys for k in perms)\\n        m2.position_graph = {k[2]: {src: (d[\\\"near\\\"] if isinstance(d, dict) else d) for src, d in m.items()} for k, m in perms.items() if k not in shift_keys}\\n        out.append((\\\"induced:click-shift\\\", m2))\\n    return out\\n\\n\\ndef induced_code(name: str) -> str:\\n    \\\"\\\"\\\"Pseudo-source recorded for induced models (they are rebuilt from semantics, not from code).\\\"\\\"\\\"\\n    return f\\\"# induced model: {name}\\\\n# (built by pbg.wml.induce.induce_hypotheses from action semantics + priors)\\\\n\\\"\\n\", \"pbg/wml/point.py\": \"\\\"\\\"\\\"wml/point.py \\u2014 POINT-select games (docs/029 priority 3): a click is a destination, the piece steps toward it.\\n\\nNeither the LLM nor the click inducers wrote r11l's rule (\\\"click a cell -> the piece moves one step toward it\\\"); the\\nmechanism `step_toward_click` existed but nothing instantiated it. This inducer reads the click transitions of the\\nlog: for every object that moved on a click, does its displacement point from its centre toward the click? A colour/\\nshape class that does so on >= 2 clicks and on most of the clicks that moved it becomes `mover`; the step length and\\nthe axis rule (4-connected vs diagonal) are read off the displacements. Two variants are returned so evaluate() can\\npick: free movement, and static objects as walls (the step is cancelled on overlap).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter, defaultdict\\nfrom typing import Optional\\n\\nfrom ..core.contracts import Rule, RuleModel\\nfrom ..core.types import Action, Scene, Transition\\nfrom ..memory.priors.mechanisms import jump_to_click, step_toward_click\\nfrom ..probe.clickmap import object_key\\nfrom ..probe.semantics import core_diff\\n\\nMIN_OBS = 2\\nMIN_SHARE = 0.6\\n\\n\\nMODES = (\\\"4\\\", \\\"diag\\\", \\\"jump\\\")     # step along the major axis / step on both axes / the centre lands on the click\\n\\n\\ndef _expected(o, action: Action, step: int, mode: str) -> tuple[int, int]:\\n    if mode == \\\"jump\\\":\\n        cr = (o.bbox[0] + o.bbox[2] - 1) // 2; cc = (o.bbox[1] + o.bbox[3] - 1) // 2\\n        return (int(action.row) - cr, int(action.col) - cc)\\n    cr = (o.bbox[0] + o.bbox[2] - 1) / 2.0; cc = (o.bbox[1] + o.bbox[3] - 1) / 2.0\\n    drr = action.row - cr; dcc = action.col - cc\\n    sr = (drr > 0) - (drr < 0); sc = (dcc > 0) - (dcc < 0)\\n    if mode == \\\"diag\\\":\\n        return (sr * step, sc * step)\\n    return (sr * step, 0) if abs(drr) >= abs(dcc) else (0, sc * step)\\n\\n\\ndef induce_point_hypotheses(log: list[Transition], semantics: dict, available: list[Action], knowledge=None) -> list[tuple[str, RuleModel]]:\\n    clicks = [t for t in log if t.action.type == \\\"CLICK\\\"]\\n    if len(clicks) < MIN_OBS:\\n        return []\\n    moved_by_class: dict[str, int] = Counter()\\n    match: dict[tuple[str, int, str], int] = Counter()        # (class, step, mode) -> clicks where the displacement matched the mode\\n    for t in clicks:\\n        before = {o.id: o for o in t.before.objects}\\n        cd = core_diff(t)\\n        moves = [(oid, d, before.get(oid)) for oid, d in cd[\\\"moved\\\"]]\\n        # a jump longer than the tracker's matching radius shows up as one object of a class vanishing and one appearing\\n        gone = {}; came = {}\\n        for o in cd[\\\"disappeared\\\"]:\\n            gone.setdefault(object_key(o), []).append(o)\\n        for o in cd[\\\"appeared\\\"]:\\n            came.setdefault(object_key(o), []).append(o)\\n        for kk in gone:\\n            if len(gone[kk]) == 1 and len(came.get(kk, [])) == 1:\\n                g, c = gone[kk][0], came[kk][0]\\n                moves.append((g.id, (c.bbox[0] - g.bbox[0], c.bbox[1] - g.bbox[1]), g))\\n        for oid, (dr, dc), o in moves:\\n            if o is None:\\n                continue\\n            k = object_key(o); moved_by_class[k] += 1\\n            step = max(abs(dr), abs(dc))\\n            for mode in MODES:\\n                if (dr, dc) == _expected(o, t.action, step, mode):\\n                    match[(k, 0 if mode == \\\"jump\\\" else step, mode)] += 1\\n    out: list[tuple[str, RuleModel]] = []\\n    for (k, step, mode), n in sorted(match.items(), key=lambda kv: -kv[1]):\\n        if n < MIN_OBS or n < MIN_SHARE * moved_by_class[k]:\\n            continue\\n        static = {object_key(o) for t in log for o in t.before.objects} - {k}      # classes that never were the mover\\n        never_moved = static - {object_key(before_o) for t in log for i, _ in core_diff(t)[\\\"moved\\\"] for before_o in [dict((o.id, o) for o in t.before.objects).get(i)] if before_o is not None}\\n\\n        def role_fn_factory(walls: frozenset):\\n            def role_fn(scene: Scene) -> dict[int, str]:\\n                roles = {}\\n                strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n                for o in scene.objects:\\n                    kk = object_key(o)\\n                    roles[o.id] = \\\"mover\\\" if kk == k else (\\\"wall\\\" if kk in walls and o.region not in strips else \\\"unknown\\\")\\n                return roles\\n            return role_fn\\n        for variant, walls in ((\\\"free\\\", frozenset()), (\\\"walls\\\", frozenset(never_moved))):\\n            if variant == \\\"walls\\\" and not walls:\\n                continue\\n            effect = jump_to_click(\\\"mover\\\", (\\\"wall\\\",)) if mode == \\\"jump\\\" else step_toward_click(\\\"mover\\\", (\\\"wall\\\",), step=step, diagonal=(mode == \\\"diag\\\"))\\n            rule = Rule(f\\\"point_{mode}_{variant}\\\", lambda s, a: a.type == \\\"CLICK\\\", effect, source=\\\"induced\\\",\\n                        meta={\\\"class\\\": k, \\\"step\\\": step, \\\"mode\\\": mode, \\\"walls\\\": sorted(walls)})\\n            model = RuleModel([rule], role_fn_factory(walls), default=\\\"noop\\\", name=f\\\"point_{mode}_s{step}_{variant}\\\")\\n            model.point_game = True; model.point_step = max(1, step); model.point_mode = mode\\n            out.append((model.name, model))\\n        break            # the best-supported (class, step, axis rule) only; its two variants compete in evaluate()\\n    return out\\n\\n\\ndef point_click_candidates(scene: Scene, step: int = 1, reach: int = 6) -> list[Action]:\\n    \\\"\\\"\\\"Clicks that steer each mover: a cell `reach` away in each of the four directions (and the diagonals). For a jump\\n    model the planner also needs the centres of the other objects (the places a piece can be put).\\\"\\\"\\\"\\n    out: list[Action] = []\\n    for o in scene.objects:\\n        if o.role != \\\"mover\\\":\\n            continue\\n        r, c = o.center\\n        for dr, dc in ((-reach, 0), (reach, 0), (0, -reach), (0, reach), (-reach, -reach), (-reach, reach), (reach, -reach), (reach, reach)):\\n            rr, cc = max(0, min(63, r + dr)), max(0, min(63, c + dc))\\n            a = Action.click(rr, cc)\\n            if a not in out:\\n                out.append(a)\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    for o in scene.objects:\\n        if o.role != \\\"mover\\\" and o.region not in strips and o.area >= 2:\\n            a = Action.click(*o.center)\\n            if a not in out:\\n                out.append(a)\\n    return out[:40]\\n\", \"pbg/wml/prompts/__init__.py\": \"\\\"\\\"\\\"wml/prompts \\u2014 prompt builders for the World Model Lab and Goal Inference (spec \\u00a714).\\n\\nSystem prompt = core/types.py + core/contracts.py verbatim + the rule-writing contract; user prompt = prior signatures,\\nthe current model code (violated rules marked), selected observation summaries, and the instruction.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nCORE = Path(__file__).resolve().parents[2] / \\\"core\\\"\\n\\nCONTRACT = \\\"\\\"\\\"You write a WORLD MODEL for an unknown pixel board game as executable Python.\\nHard rules:\\n- Define `def build_model() -> RuleModel`. Build it from a list of `Rule(name, applies, effect, confidence)` objects and a\\n  role-assignment function: `RuleModel(rules, role_fn, default=\\\"unknown\\\")`.\\n- Rules reference objects by ROLE (agent, wall, target, key, door, pushable, collectible, hazard, indicator, decoration,\\n  or custom:<name>), never by object id, never by absolute coordinates. Read every constant (colours, step sizes,\\n  regions) from the Scene when you can; the role function decides which object is which.\\n- The role function `role_fn(scene) -> dict[obj_id, role]` may use colour, shape, size, region and position relative\\n  to regions (e.g. 'the object that moved' is not available: infer from appearance).\\n- Prefer the prior mechanisms listed below (already imported: call them by name). Effects compose in rule order:\\n  a move rule first, then collision/cancel rules, then counters. `scene.aux['_before']` holds the pre-action scene.\\n- Return UNKNOWN (an effect returning None, or `unknown_for(buttons)`) for transitions you cannot explain; never\\n  invent behaviour to force coverage.\\n- Rules for ui_strip regions (gauges/counters) are optional: evaluation ignores objects in ui_strip regions and objects\\n  whose role is 'indicator'. Do not write rules for them.\\n- Region ids (R0, R1, ...) are re-numbered every frame: identify a region by kind_hint, bg colour and size, never by id.\\n  Object ids are tracking ids and can change when an object reshapes: identify objects by colour/shape/region.\\n- Every button/click the log shows must get a prediction from your rules (an unexplained action = UNKNOWN only for\\n  that action). A catch-all UNKNOWN rule, if any, must come last and must not swallow explained actions.\\n- Prediction is checked object by object (colour, bbox, shape) against the real next frame, so displacements and\\n  cancel conditions must be exact (step size in pixels, blocked by which colours/roles, board bounds).\\n- No imports except numpy, math, itertools, dataclasses, typing, collections. No file/network access. Do not compare\\n  game ids. Do not hard-code lists of coordinates.\\n- CODE ONLY: no analysis, no reasoning, no observation-by-observation walkthrough anywhere in the reply \\u2014 not in prose,\\n  not in comments, not in docstrings (at most one short comment per rule). Work out the mechanics silently and write\\n  the final rules. Keep the whole reply under 120 lines of code.\\n- Do not model gauges / counters / timers in ui_strip regions: evaluation ignores them and they waste rules.\\nReply with exactly ONE ```python code block (the model) and optionally ONE ```json block {\\\"notes\\\": \\\"...\\\"}.\\n\\n# Worked example (a different game: a blue 4x4 agent moves 4 px per button on a black board, grey blocks are walls,\\n# red 2x2 pieces vanish when the agent steps on them, button 5 does nothing). Adapt the pattern, never the numbers.\\n```python\\ndef role_fn(scene):\\n    roles = {}\\n    strips = {r.id for r in scene.regions if r.kind_hint == \\\"ui_strip\\\"}\\n    for o in scene.objects:\\n        if o.region in strips:\\n            roles[o.id] = \\\"indicator\\\"\\n        elif o.color == 1 and o.area >= 12:\\n            roles[o.id] = \\\"agent\\\"\\n        elif o.color == 5:\\n            roles[o.id] = \\\"wall\\\"\\n        elif o.color == 2:\\n            roles[o.id] = \\\"collectible\\\"\\n        else:\\n            roles[o.id] = \\\"decoration\\\"\\n    return roles\\n\\ndef build_model():\\n    dirs = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}\\n    move_buttons = lambda s, a: a.type == \\\"BUTTON\\\" and a.id in dirs\\n    rules = [\\n        Rule(\\\"move_agent\\\", move_buttons, move_role(\\\"agent\\\", dirs, step=4)),\\n        Rule(\\\"wall_blocks\\\", move_buttons, cancel_move_if_overlap(\\\"agent\\\", \\\"wall\\\")),\\n        Rule(\\\"stay_on_board\\\", move_buttons, cancel_move_if_off_floor(\\\"agent\\\", (0,))),   # floor colour read from the scene\\n        Rule(\\\"collect\\\", move_buttons, collect_on_overlap(\\\"agent\\\", \\\"collectible\\\")),\\n        Rule(\\\"noop_button\\\", lambda s, a: a.type == \\\"BUTTON\\\" and a.id == 5, noop_for((5,))),\\n    ]\\n    return RuleModel(rules, role_fn, default=\\\"unknown\\\")\\n```\\n# Hidden state example: clicking a board block selects it (a cursor appears), button 7 recolours the selected block.\\n# State that is not visible as an object goes into scene.aux and is read back by later rules:\\n```python\\n    rules = [\\n        Rule(\\\"select\\\", lambda s, a: a.type == \\\"CLICK\\\", set_flag_on_click(\\\"piece\\\", key=\\\"selected\\\")),\\n        Rule(\\\"recolour_selected\\\", lambda s, a: a.type == \\\"BUTTON\\\" and a.id == 7, recolour_selected_effect),  # reads s.aux[\\\"selected\\\"]\\n        Rule(\\\"key_then_door\\\", lambda s, a: a.type == \\\"BUTTON\\\", key_opens_door(\\\"agent\\\", \\\"key\\\", \\\"door\\\", flag=\\\"has_key\\\")),\\n    ]\\n```\\\"\\\"\\\"\\n\\n\\ndef core_sources() -> str:\\n    parts = []\\n    for name in (\\\"types.py\\\", \\\"contracts.py\\\"):\\n        p = CORE / name\\n        if p.exists():\\n            parts.append(f\\\"# ==== core/{name} ====\\\\n\\\" + p.read_text())\\n    return \\\"\\\\n\\\\n\\\".join(parts)\\n\\n\\nAPI_SUMMARY = '''\\n# Data model (summary of core/types.py, core/contracts.py)\\nAction: .type in {\\\"BUTTON\\\",\\\"CLICK\\\",\\\"RESET\\\"}; .id (BUTTON 1..7); .row/.col (CLICK, grid cells); .key -> \\\"ACTION1\\\"..\\\"CLICK\\\"\\nRegion: .id \\\"R0\\\".. (re-numbered every frame), .bg_color, .bbox (r0,c0,r1,c1 exclusive), .kind_hint in {board,panel,ui_strip,unknown}, .area, .center\\nObject: .id (tracking id), .color, .colors (tuple, multi-colour objects), .bbox, .mask (bbox-sized bool), .area, .shape_sig (translation-invariant hash),\\n        .region (Region.id), .role (set by role_fn), .center, .height, .width; .color_mask (bbox-sized int8, -1 outside; multi-colour objects),\\n        .composite (adjacent parts fused into one rectangle: a pattern, a canvas, a framed button);\\n        .moved(dr,dc) -> copy shifted; .recolored(c) -> copy; .rotated(k) -> copy turned k*90deg clockwise about its centre; .flipped(axis) -> mirror;\\n        .with_mask(mask, color_mask=None, origin=(r0,c0)) -> copy with a new pixel layout (stamping, painting); .overlaps(other) -> bool\\nScene: .regions, .objects, .grid_shape, .aux (dict: hidden state you may add; scene.aux[\\\"_before\\\"] = pre-action scene inside effects);\\n       .by_role(role) -> objects; .get(id); .region(id); .in_region(id); .copy(objects=..., aux=...); .render() -> grid (numpy int8)\\nRule(name, applies(scene, action)->bool, effect(scene, action)->Scene|None, confidence=0.5)\\nRuleModel(rules, role_fn, default=\\\"unknown\\\")   # predict() applies matching rules in order; None == UNKNOWN\\nPrediction check: multiset of (color, bbox, shape_sig, colour layout) of objects not in ui_strip regions and not role \\\"indicator\\\" must equal the real next frame.\\nObservations may include pixel crops (hex digits, '.' outside) of objects whose shape or colour layout changed: read them to see rotations, flips, stamps.\\n'''\\n\\n\\ndef system_prompt() -> str:\\n    return CONTRACT + \\\"\\\\n\\\" + API_SUMMARY\\n\\n\\ndef world_model_prompt(*, signatures: str, current_code: Optional[str], violated_rules: list[str], observations: list[str],\\n                       semantics_text: str, scene_text: str, hidden_state_hint: bool = False, level_note: str = \\\"\\\",\\n                       diagnosis: Optional[str] = None) -> list[dict]:\\n    user = [\\\"# Prior mechanisms (call by name; signatures only)\\\", signatures, \\\"\\\"]\\n    user += [\\\"# Action semantics measured by the probe\\\", semantics_text, \\\"\\\"]\\n    user += [\\\"# Current scene (regions and objects)\\\", scene_text, \\\"\\\"]\\n    if current_code:\\n        code = current_code\\n        for r in violated_rules:\\n            code = code.replace(f'\\\"{r}\\\"', f'\\\"{r}\\\"  # VIOLATED').replace(f\\\"'{r}'\\\", f\\\"'{r}'  # VIOLATED\\\")\\n        user += [\\\"# Current model code (rules marked # VIOLATED failed on the log)\\\", \\\"```python\\\", code, \\\"```\\\", \\\"\\\"]\\n    user += [\\\"# Observations (action -> object-level diff), newest last\\\", *[f\\\"- {o}\\\" for o in observations], \\\"\\\"]\\n    if level_note:\\n        user += [\\\"# Level note\\\", level_note, \\\"\\\"]\\n    if diagnosis:\\n        user += [\\\"# Diagnosis of the last violation\\\", diagnosis, \\\"\\\"]\\n    if hidden_state_hint:\\n        user += [\\\"# Hidden state\\\", \\\"The same scene and action gave different results more than once while the environment is deterministic. \\\"\\n                 \\\"Introduce state variables in scene.aux (e.g. the last k actions, the selected object, a counter) and update them in your rules.\\\", \\\"\\\"]\\n    if current_code:\\n        user += [\\\"# Instruction\\\", \\\"Make the MINIMAL change that explains the violations: fix only the violated rules or add a new rule. \\\"\\n                 \\\"Do not create new constants; read them from the Scene. Return UNKNOWN for transitions you cannot explain.\\\"]\\n    else:\\n        user += [\\\"# Instruction\\\", \\\"Write a rule-based model that reproduces every observation above. Assign roles from appearance. \\\"\\n                 \\\"Return UNKNOWN for transitions you cannot explain.\\\"]\\n    return [{\\\"role\\\": \\\"system\\\", \\\"content\\\": system_prompt()}, {\\\"role\\\": \\\"user\\\", \\\"content\\\": \\\"\\\\n\\\".join(user)}]\\n\\n\\ndef goal_template_prompt(scene_text: str, observations: list[str], rejected: Optional[list[str]] = None) -> list[dict]:\\n    sys = (\\\"You propose the WIN CONDITION of an unknown pixel board game as executable Python. Define `def build_goal()` returning an \\\"\\n           \\\"object with `name: str`, `is_goal(scene) -> bool` and `progress(scene) -> float in [0,1]` (progress must grow as the player gets \\\"\\n           \\\"closer; it is used as a search heuristic). Read colours/regions/objects from the Scene; no absolute coordinates; no comments. \\\"\\n           \\\"The condition may be procedural (a pattern to reproduce, a sequence of states, a count) as long as it is a function of the scene. \\\"\\n           \\\"Reply with exactly ONE ```python block.\\\\n\\\" + API_SUMMARY)\\n    user = \\\"# Scene\\\\n\\\" + scene_text + \\\"\\\\n\\\\n# Observations (pixel crops show shape/colour changes)\\\\n\\\" + \\\"\\\\n\\\".join(f\\\"- {o}\\\" for o in observations)\\n    if rejected:\\n        user += \\\"\\\\n\\\\n# Predicates that held WITHOUT completing the level (so they are NOT the win condition)\\\\n\\\" + \\\"\\\\n\\\".join(f\\\"- {r}\\\" for r in rejected)\\n    user += \\\"\\\\n\\\\nNo goal template explains this level. Propose the most plausible win condition as code.\\\"\\n    return [{\\\"role\\\": \\\"system\\\", \\\"content\\\": sys}, {\\\"role\\\": \\\"user\\\", \\\"content\\\": user}]\\n\\n\\ndef diagnose_prompt(rule_code: str, violation_summaries: list[str], scene_text: str) -> list[dict]:\\n    sys = \\\"You diagnose why a rule of a pixel-board-game world model failed after a level change. Reply with ONE ```json block {\\\\\\\"rule\\\\\\\": ..., \\\\\\\"cause\\\\\\\": ..., \\\\\\\"fix_hint\\\\\\\": ...}.\\\"\\n    user = \\\"# Violated rule\\\\n```python\\\\n\\\" + rule_code + \\\"\\\\n```\\\\n\\\\n# Violating transitions\\\\n\\\" + \\\"\\\\n\\\".join(f\\\"- {v}\\\" for v in violation_summaries) + \\\"\\\\n\\\\n# Scene\\\\n\\\" + scene_text\\n    return [{\\\"role\\\": \\\"system\\\", \\\"content\\\": sys}, {\\\"role\\\": \\\"user\\\", \\\"content\\\": user}]\\n\\n\\ndef critique_prompt(previous: list[dict], score: float, coverage: float, mismatches: list[str], unknowns: list[str]) -> list[dict]:\\n    lines = [f\\\"Your model was verified on the whole log: accuracy {score:.2f}, coverage {coverage:.2f}.\\\"]\\n    if mismatches:\\n        lines += [\\\"Wrong predictions (predicted vs observed):\\\", *[f\\\"- {m}\\\" for m in mismatches[:6]]]\\n    if unknowns:\\n        lines += [\\\"UNKNOWN (no prediction) for transitions such as:\\\", *[f\\\"- {u}\\\" for u in unknowns[:4]]]\\n    lines.append(\\\"Fix the rules so these transitions are predicted exactly; keep what already works. Return the corrected full code in ONE ```python block.\\\")\\n    return previous + [{\\\"role\\\": \\\"user\\\", \\\"content\\\": \\\"\\\\n\\\".join(lines)}]\\n\\n\\ndef repair_prompt(previous: list[dict], error: str) -> list[dict]:\\n    return previous + [{\\\"role\\\": \\\"user\\\", \\\"content\\\": f\\\"Your code failed to load or run:\\\\n```\\\\n{error[-1200:]}\\\\n```\\\\nReturn the corrected full code in ONE ```python block.\\\"}]\\n\", \"pbg/wml/refine.py\": \"\\\"\\\"\\\"wml/refine.py \\u2014 World Model Lab (spec \\u00a78): generate candidate models (priors-driven induction first, then K LLM\\ncandidates), verify each on the WHOLE log, keep N hypotheses with score/coverage/violations, detect hidden state.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport threading\\nimport time\\nfrom typing import Optional\\n\\nfrom ..core.contracts import Hypothesis, RuleModel\\nfrom ..core.types import Action, Scene, Transition\\nfrom ..perception.summarize import scene_text, summarize\\nfrom .context import observation_lines, select_context\\nfrom .evaluate import dedupe, evaluate, make_hypothesis, promotable\\nfrom .experiment import most_informative_action as _mia\\nfrom .induce import induce_hypotheses, induced_code\\nfrom .prompts import critique_prompt, diagnose_prompt, repair_prompt, world_model_prompt\\nfrom .sandbox import Sandbox\\n\\n\\nclass WorldModelLab:\\n    def __init__(self, *, llm=None, sandbox: Optional[Sandbox] = None, memory=None, K: int = 4, N: int = 5, tau: float = 0.3, log=None,\\n                 validate_in_subprocess: bool = True):\\n        self.llm, self.sandbox, self.memory = llm, sandbox or Sandbox(), memory\\n        self.K, self.N, self.tau = K, N, tau\\n        self.log = log or (lambda *a, **k: None)\\n        self.validate_in_subprocess = validate_in_subprocess\\n        self.llm_calls = 0\\n        self.hidden_state_suspected = False\\n        self.last_context: list[dict] = []\\n        self.generation_log: list[dict] = []\\n        self._job: Optional[threading.Thread] = None      # background LLM generation (spec \\u00a715: keep planning meanwhile)\\n        self._job_result: list[Hypothesis] = []\\n        self._job_started = 0.0\\n        self.async_rounds = 0\\n\\n    # \\u2500\\u2500 public \\u2500\\u2500\\n    def refine(self, log: list[Transition], current: list[Hypothesis], priors: Optional[dict], *, scene: Scene, semantics: dict,\\n               available: list[Action], level_note: str = \\\"\\\", use_llm: bool = True, K: Optional[int] = None, knowledge=None) -> list[Hypothesis]:\\n        H: list[Hypothesis] = []\\n        # 1. re-verify what we already have on the (grown) log; prior-composed models are rebuilt from the full log\\n        #    below (a stale permutation table must not linger), their live misprediction penalties carry over by name\\n        induced_pen = {h.name: (h.recent_mismatches, h.prediction_key) for h in current if h.origin == \\\"induced\\\"}\\n        for h in current:\\n            if h.origin == \\\"induced\\\":\\n                continue\\n            res = evaluate(h.model, log)\\n            H.append(make_hypothesis(h.model, res, h.code, h.name, h.origin, recent_mismatches=h.recent_mismatches))\\n        # 2. deterministic induction from semantics + priors\\n        for name, model in induce_hypotheses(log, semantics, available, knowledge):\\n            res = evaluate(model, log)\\n            pen, pkey = induced_pen.get(name, (0, None))\\n            # a rebuilt model that predicts the log differently is a new model: it does not inherit the old one's mispredictions\\n            H.append(make_hypothesis(model, res, induced_code(name), name, \\\"induced\\\", recent_mismatches=pen if pkey == res.prediction_key else 0))\\n        H = dedupe(H); H.sort(key=lambda h: h.sort_key(), reverse=True)\\n        best = H[0] if H else None\\n        # 3. LLM candidates when nothing is perfect \\u2014 synchronously here, or in the background via start_llm_job()\\n        if use_llm and self.llm is not None and (best is None or best.score < 1.0 or best.coverage < 1.0):\\n            H += self._llm_candidates(log, best, priors, scene=scene, semantics=semantics, level_note=level_note, K=K or self.K)\\n            H = dedupe(H); H.sort(key=lambda h: h.sort_key(), reverse=True)\\n        # merge candidates a finished background job produced (re-verified on the current, larger log); a candidate whose\\n        # predictions equal an already-penalised hypothesis inherits its live mispredictions (dedupe keeps the max)\\n        for h in self.collect_job():\\n            res = evaluate(h.model, log)\\n            H.append(make_hypothesis(h.model, res, h.code, h.name, h.origin))\\n        H = dedupe(H); H.sort(key=lambda h: h.sort_key(), reverse=True)\\n        self._detect_hidden_state(log)\\n        # live misprediction penalties follow the CODE (a re-delivered LLM candidate with the same source is the same model)\\n        penalty_by_code: dict[str, int] = {}\\n        for h in list(current) + H:\\n            if h.code:\\n                penalty_by_code[h.code] = max(penalty_by_code.get(h.code, 0), h.recent_mismatches)\\n        for h in H:\\n            if h.code and penalty_by_code.get(h.code, 0) > h.recent_mismatches:\\n                h.recent_mismatches = penalty_by_code[h.code]\\n        for h in H:\\n            h.verified = promotable(evaluate(h.model, log)) if h.score >= 1.0 else False\\n        # keep the hypothesis set diverse: the best prior-composed model stays even when LLM candidates outscore it\\n        # (a high-scoring LLM model can still be useless for planning; the composed one rarely is)\\n        top = H[:self.N]\\n        # every prior-composed variant stays (a permutation table verifies at 1.0 but cannot reach unseen positions; the\\n        # uniform-shift variant scores lower because it says UNKNOWN where it has no rule, yet it is the one that plans)\\n        induced = [h for h in H if h.origin == \\\"induced\\\" and h not in top]\\n        if induced:\\n            top = top[:max(1, self.N - len(induced))] + induced\\n        # the working set is ordered usable-first: a stale memory model with a high score but no change accuracy must not\\n        # sit at H[0] and block the gate while a usable composed model is available\\n        top.sort(key=lambda h: (h.usable(), h.sort_key()), reverse=True)\\n        return top\\n\\n    # \\u2500\\u2500 background generation \\u2500\\u2500\\n    def job_running(self) -> bool:\\n        return self._job is not None and self._job.is_alive()\\n\\n    def start_llm_job(self, log: list[Transition], best: Optional[Hypothesis], priors, *, scene: Scene, semantics: dict, level_note: str = \\\"\\\",\\n                      K: Optional[int] = None) -> bool:\\n        \\\"\\\"\\\"Generate LLM candidates in a thread on a snapshot of the log; the orchestrator keeps acting on the current\\n        hypotheses and merges the result at a later HYPOTHESIZE (spec \\u00a715 'LLM latency -> plan on the top hypothesis').\\\"\\\"\\\"\\n        if self.llm is None or self.job_running() or self.llm.exhausted():\\n            return False\\n        snapshot = list(log)\\n        def run():\\n            try:\\n                self._job_result = self._llm_candidates(snapshot, best, priors, scene=scene, semantics=dict(semantics), level_note=level_note, K=K or self.K)\\n            except Exception as e:\\n                self.log(f\\\"llm job failed: {e!r}\\\")\\n                self._job_result = []\\n        self._job_result = []\\n        self._job = threading.Thread(target=run, daemon=True, name=\\\"wml-llm\\\")\\n        self._job_started = time.time()\\n        self._job.start(); self.async_rounds += 1\\n        return True\\n\\n    def collect_job(self) -> list[Hypothesis]:\\n        if self._job is None or self._job.is_alive():\\n            return []\\n        out, self._job_result, self._job = self._job_result, [], None\\n        if out:\\n            self.log(f\\\"llm job delivered {len(out)} candidate(s) after {time.time() - self._job_started:.0f}s\\\")\\n        return out\\n\\n    def wait_job(self, timeout: float) -> bool:\\n        \\\"\\\"\\\"Block up to `timeout` seconds for a running job (used only when nothing useful can be done meanwhile).\\\"\\\"\\\"\\n        if not self.job_running():\\n            return True\\n        self._job.join(timeout)\\n        return not self.job_running()\\n\\n    def most_informative_action(self, H: list[Hypothesis], scene: Scene, available: list[Action], *, semantics=None, extra_clicks=(),\\n                                exclude: Optional[set] = None, state_key=None, click_map=None) -> Optional[Action]:\\n        return _mia(H, scene, available, tau=self.tau, semantics=semantics, extra_clicks=extra_clicks, exclude=exclude, state_key=state_key, click_map=click_map)\\n\\n    def diagnose(self, h: Hypothesis, log: list[Transition], scene: Scene) -> Optional[dict]:\\n        \\\"\\\"\\\"Violation diagnosis after a level change (spec \\u00a714, JSON {rule, cause, fix_hint}).\\\"\\\"\\\"\\n        if self.llm is None or not h.violations:\\n            return None\\n        viol = [t for t in log if t.id in set(h.violations)][-5:]\\n        code = h.code or \\\"\\\"\\n        try:\\n            reply = self.llm.chat(diagnose_prompt(code[:4000], [summarize(t) for t in viol], scene_text(scene)), purpose=\\\"diagnose\\\")\\n            self.llm_calls += 1\\n            return self.llm.extract_json(reply)\\n        except Exception as e:\\n            self.log(f\\\"diagnose failed: {e!r}\\\")\\n            return None\\n\\n    # \\u2500\\u2500 internals \\u2500\\u2500\\n    def _llm_candidates(self, log, best, priors, *, scene, semantics, level_note, K) -> list[Hypothesis]:\\n        out: list[Hypothesis] = []\\n        ctx = select_context(log, best.violations if best else [], cap=40)\\n        obs = observation_lines(ctx)\\n        self.last_context = [{\\\"id\\\": t.id, \\\"summary\\\": summarize(t)} for t in ctx]\\n        sigs = (priors or {}).get(\\\"signatures\\\", \\\"\\\") if priors else \\\"\\\"\\n        current_code = best.code if best and best.origin == \\\"llm\\\" and best.code else None\\n        violated = []\\n        if best and isinstance(best.model, RuleModel):\\n            violated = [r.name for r in best.model.rules() if r.violated_by]\\n        sem_text = json.dumps({k: v for k, v in semantics.items() if not k.startswith(\\\"_\\\")}, default=str)[:3000]\\n        msgs = world_model_prompt(signatures=sigs, current_code=current_code, violated_rules=violated, observations=obs, semantics_text=sem_text,\\n                                  scene_text=scene_text(scene), hidden_state_hint=self.hidden_state_suspected, level_note=level_note)\\n        log_json = [self._t_json(t) for t in log]\\n        # K candidates are generated in parallel threads (each: one generation + at most one follow-up)\\n        results: list[list[Hypothesis]] = [[] for _ in range(K)]\\n        workers = [threading.Thread(target=self._one_candidate, args=(k, msgs, log, log_json, results), daemon=True) for k in range(K)]\\n        for w in workers:\\n            w.start()\\n        for w in workers:\\n            w.join()\\n        for r in results:\\n            out.extend(r)\\n        return out\\n\\n    def _one_candidate(self, k: int, msgs: list[dict], log: list[Transition], log_json: list[dict], results: list) -> None:\\n        out: list[Hypothesis] = []\\n        try:\\n            self._generate_candidate(k, msgs, log, log_json, out)\\n        except Exception as e:\\n            self.log(f\\\"llm candidate {k} crashed: {e!r}\\\")\\n        results[k] = out\\n\\n    def _generate_candidate(self, k: int, msgs: list[dict], log: list[Transition], log_json: list[dict], out: list[Hypothesis]) -> None:\\n        for _ in range(1):\\n            if self.llm.exhausted():\\n                break\\n            try:\\n                # candidate k>0: higher temperature; models without sampling parameters get a prompt nudge instead\\n                kmsgs = msgs if k == 0 else msgs + [{\\\"role\\\": \\\"user\\\", \\\"content\\\": f\\\"Alternative {k}: decompose the mechanics differently from the most obvious reading \\\"\\n                                                    f\\\"(different roles or rule granularity); still return ONE complete ```python block.\\\"}]\\n                reply = self.llm.chat(kmsgs, purpose=\\\"world_model\\\", use_cache=(k == 0), override={\\\"temperature\\\": 0.6 + 0.1 * k} if k else None)\\n                self.llm_calls += 1\\n            except Exception as e:\\n                self.log(f\\\"llm world_model failed: {e!r}\\\")\\n                break\\n            code = self.llm.extract_code(reply)\\n            attempts = 0\\n            while attempts < 2:      # generate + at most ONE follow-up (repair or critique) per candidate\\n                if not code:\\n                    self.generation_log.append({\\\"k\\\": k, \\\"error\\\": \\\"no code block\\\"})\\n                    break\\n                if self.validate_in_subprocess:\\n                    v = self.sandbox.validate(code, log_json)\\n                else:\\n                    m = self.sandbox.load(code)\\n                    v = {\\\"ok\\\": m is not None, \\\"error\\\": \\\"load failed\\\"} if m is None else {\\\"ok\\\": True, **evaluate(m, log).__dict__}\\n                if v.get(\\\"ok\\\"):\\n                    model = self.sandbox.load(code)\\n                    if model is None:\\n                        break\\n                    res = evaluate(model, log)\\n                    h = make_hypothesis(model, res, code, f\\\"llm:{len(self.generation_log)}\\\", \\\"llm\\\")\\n                    out.append(h)\\n                    self.generation_log.append({\\\"k\\\": k, \\\"score\\\": res.score, \\\"coverage\\\": res.coverage, \\\"n_rules\\\": len(model.rules())})\\n                    self.log(f\\\"llm candidate {k}: score={res.score:.2f} cov={res.coverage:.2f} rules={len(model.rules())}\\\")\\n                    # one critique round when the verification is imperfect (spec \\u00a714 'violation diagnosis' feedback)\\n                    if res.score < 1.0 and attempts < 1 and not self.llm.exhausted():\\n                        attempts += 1\\n                        mism = self._mismatch_lines(model, log, res.violations[:6])\\n                        unk = [summarize(t) for t in log if t.id in set(res.unknown[:4])]\\n                        try:\\n                            reply = self.llm.chat(critique_prompt(msgs + [{\\\"role\\\": \\\"assistant\\\", \\\"content\\\": reply}], res.score, res.coverage, mism, unk),\\n                                                  purpose=\\\"world_model\\\", use_cache=False)\\n                            self.llm_calls += 1\\n                            code = self.llm.extract_code(reply)\\n                            continue\\n                        except Exception as e:\\n                            self.log(f\\\"llm critique failed: {e!r}\\\")\\n                    break\\n                attempts += 1\\n                self.generation_log.append({\\\"k\\\": k, \\\"attempt\\\": attempts, \\\"error\\\": str(v.get(\\\"error\\\"))[:300]})\\n                self.log(f\\\"llm candidate {k} rejected: {str(v.get('error'))[:200]}\\\")\\n                if attempts >= 2 or self.llm.exhausted():\\n                    break\\n                try:\\n                    reply = self.llm.chat(repair_prompt(msgs + [{\\\"role\\\": \\\"assistant\\\", \\\"content\\\": reply}], str(v.get(\\\"error\\\"))), purpose=\\\"world_model\\\", use_cache=False)\\n                    self.llm_calls += 1\\n                    code = self.llm.extract_code(reply)\\n                except Exception as e:\\n                    self.log(f\\\"llm repair failed: {e!r}\\\")\\n                    break\\n\\n    @staticmethod\\n    def _mismatch_lines(model, log: list[Transition], ids: list[str]) -> list[str]:\\n        from ..core.contracts import scene_mismatch\\n        out = []\\n        for t in log:\\n            if t.id not in ids:\\n                continue\\n            try:\\n                pred = model.predict(t.before, t.action)\\n            except Exception:\\n                pred = None\\n            if pred is None:\\n                out.append(f\\\"[{t.id}] {summarize(t, 120)} | predicted: UNKNOWN\\\")\\n                continue\\n            mm = scene_mismatch(pred, t.after)\\n            out.append(f\\\"[{t.id}] observed: {summarize(t, 120)} | predicted-only objects (id,colour,bbox): {mm['only_predicted'][:3]} | observed-only: {mm['only_observed'][:3]}\\\")\\n        return out\\n\\n    @staticmethod\\n    def _t_json(t: Transition) -> dict:\\n        return {\\\"id\\\": t.id, \\\"level\\\": t.level, \\\"step_idx\\\": t.step_idx, \\\"action\\\": t.action.to_json(), \\\"before\\\": t.before.to_json(), \\\"after\\\": t.after.to_json(),\\n                \\\"status_change\\\": t.status_change}\\n\\n    def _detect_hidden_state(self, log: list[Transition]) -> None:\\n        \\\"\\\"\\\"Same frame hash + same action -> different results twice or more (spec \\u00a78 hidden state).\\\"\\\"\\\"\\n        seen: dict[tuple, set] = {}\\n        for t in log:\\n            if t.action.type == \\\"RESET\\\":\\n                continue\\n            seen.setdefault((t.before.frame_hash, t.action.label()), set()).add(t.after.frame_hash)\\n        conflicts = sum(1 for v in seen.values() if len(v) > 1)\\n        self.hidden_state_suspected = conflicts >= 2\\n\", \"pbg/wml/sandbox.py\": \"\\\"\\\"\\\"wml/sandbox.py \\u2014 loading LLM-written model / goal code safely (spec \\u00a78).\\n\\n* static checks: game-id string comparisons, >= 20 hard-coded coordinate tuples, eval/exec, disallowed imports\\n* validation run in a subprocess with resource limits (CPU 2 s, 256 MB, no network/file access \\u2014 no socket/open in\\n  the namespace) that builds the model and evaluates it on the log\\n* after validation the same code is loaded in-process (restricted builtins) so the planner can call predict() cheaply\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport ast\\nimport json\\nimport os\\nimport re\\nimport subprocess\\nimport sys\\nimport textwrap\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nALLOWED_IMPORTS = {\\\"numpy\\\", \\\"dataclasses\\\", \\\"typing\\\", \\\"itertools\\\", \\\"math\\\", \\\"core\\\", \\\"core.types\\\", \\\"core.contracts\\\", \\\"core.mechanisms\\\",\\n                   \\\"pbg.core\\\", \\\"pbg.core.types\\\", \\\"pbg.core.contracts\\\", \\\"pbg.memory.priors.mechanisms\\\", \\\"collections\\\", \\\"functools\\\",\\n                   \\\"mechanism\\\", \\\"mechanisms\\\"}     # the prompt lists the mechanism library; models import it under either name\\nAPI_ROOTS = {\\\"core\\\", \\\"pbg\\\", \\\"mechanism\\\", \\\"mechanisms\\\"}\\nGAME_ID_RE = re.compile(r\\\"^[a-z]{2}\\\\d{2}$\\\")\\nCOORD_RE = re.compile(r\\\"\\\\(\\\\s*\\\\d+\\\\s*,\\\\s*\\\\d+\\\\s*\\\\)\\\")\\nROOT = Path(__file__).resolve().parents[2]\\n\\n\\nclass StaticCheckError(ValueError):\\n    pass\\n\\n\\ndef static_check(code: str, allow_getattr: bool = False) -> None:\\n    try:\\n        tree = ast.parse(code)\\n    except SyntaxError as e:\\n        raise StaticCheckError(f\\\"syntax error: {e}\\\")\\n    for node in ast.walk(tree):\\n        if allow_getattr and isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in (\\\"getattr\\\", \\\"hasattr\\\"):\\n            names = [a.value for a in node.args[1:2] if isinstance(a, ast.Constant) and isinstance(a.value, str)]\\n            if any(n.startswith(\\\"__\\\") for n in names):\\n                raise StaticCheckError(\\\"dunder access not allowed: getattr\\\")\\n            continue\\n        if isinstance(node, (ast.Import, ast.ImportFrom)):\\n            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or \\\"\\\"]\\n            for n in names:\\n                root = n.split(\\\".\\\")[0]\\n                if n not in ALLOWED_IMPORTS and root not in {\\\"numpy\\\", \\\"dataclasses\\\", \\\"typing\\\", \\\"itertools\\\", \\\"math\\\", \\\"collections\\\", \\\"functools\\\"} | API_ROOTS:\\n                    raise StaticCheckError(f\\\"import not allowed: {n}\\\")\\n        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in (\\\"eval\\\", \\\"exec\\\", \\\"compile\\\", \\\"open\\\", \\\"__import__\\\", \\\"getattr\\\", \\\"globals\\\", \\\"locals\\\"):\\n            raise StaticCheckError(f\\\"call not allowed: {node.func.id}\\\")\\n        if isinstance(node, ast.Attribute) and node.attr.startswith(\\\"__\\\") and node.attr not in (\\\"__name__\\\",):\\n            raise StaticCheckError(f\\\"dunder access not allowed: {node.attr}\\\")\\n        if isinstance(node, ast.Constant) and isinstance(node.value, str) and GAME_ID_RE.match(node.value):\\n            raise StaticCheckError(f\\\"game id literal '{node.value}' (no-game-id-branch)\\\")\\n    n_coords = sum(1 for node in ast.walk(tree) if isinstance(node, ast.Tuple) and len(node.elts) == 2\\n                   and all(isinstance(e, ast.Constant) and isinstance(e.value, int) for e in node.elts))\\n    if n_coords >= 20:\\n        raise StaticCheckError(\\\"too many hard-coded coordinate tuples in code (>= 20)\\\")\\n    if \\\"def build_model\\\" not in code and \\\"def build_goal\\\" not in code:\\n        raise StaticCheckError(\\\"code must define build_model() or build_goal()\\\")\\n\\n\\nSAFE_BUILTINS = {k: __builtins__[k] if isinstance(__builtins__, dict) else getattr(__builtins__, k) for k in (\\n    \\\"abs\\\", \\\"all\\\", \\\"any\\\", \\\"bool\\\", \\\"dict\\\", \\\"enumerate\\\", \\\"filter\\\", \\\"float\\\", \\\"frozenset\\\", \\\"int\\\", \\\"isinstance\\\", \\\"len\\\", \\\"list\\\", \\\"map\\\", \\\"max\\\", \\\"min\\\",\\n    \\\"range\\\", \\\"reversed\\\", \\\"round\\\", \\\"set\\\", \\\"sorted\\\", \\\"str\\\", \\\"sum\\\", \\\"tuple\\\", \\\"zip\\\", \\\"print\\\", \\\"Exception\\\", \\\"ValueError\\\", \\\"KeyError\\\", \\\"IndexError\\\",\\n    \\\"TypeError\\\", \\\"None\\\", \\\"True\\\", \\\"False\\\", \\\"hasattr\\\", \\\"setattr\\\", \\\"object\\\", \\\"staticmethod\\\", \\\"property\\\", \\\"type\\\", \\\"repr\\\", \\\"divmod\\\", \\\"pow\\\", \\\"iter\\\", \\\"next\\\")}\\n\\n\\ndef _namespace() -> dict:\\n    import numpy as np\\n    from ..core import contracts, types\\n    from ..memory.priors import mechanisms\\n    builtins = dict(SAFE_BUILTINS)\\n    builtins[\\\"__build_class__\\\"] = __build_class__     # class statements (a goal object with is_goal/progress) need it\\n    ns: dict = {\\\"__builtins__\\\": builtins, \\\"__name__\\\": \\\"hypothesis\\\", \\\"np\\\": np, \\\"numpy\\\": np}   # class statements read __name__\\n    for mod in (types, contracts, mechanisms):\\n        for k in getattr(mod, \\\"__all__\\\", None) or [n for n in dir(mod) if not n.startswith(\\\"_\\\")]:\\n            ns[k] = getattr(mod, k)\\n    ns[\\\"Rule\\\"], ns[\\\"RuleModel\\\"], ns[\\\"Scene\\\"], ns[\\\"Action\\\"], ns[\\\"Object\\\"] = contracts.Rule, contracts.RuleModel, types.Scene, types.Action, types.Object\\n    ns[\\\"Optional\\\"] = Optional\\n    import dataclasses, itertools, math, typing, collections, functools\\n    ns.update({\\\"dataclasses\\\": dataclasses, \\\"itertools\\\": itertools, \\\"math\\\": math, \\\"typing\\\": typing, \\\"collections\\\": collections, \\\"functools\\\": functools,\\n               \\\"dataclass\\\": dataclasses.dataclass, \\\"field\\\": dataclasses.field})\\n    real_import = __import__\\n\\n    class _Api:\\n        \\\"\\\"\\\"One module that exposes every API name: `from core.types import RuleModel` and `from pbg.core.contracts import\\n        Scene` both work, whichever module the model believes a name lives in.\\\"\\\"\\\"\\n        pass\\n    api = _Api()\\n    for mod in (types, contracts, mechanisms):\\n        for k in getattr(mod, \\\"__all__\\\", None) or [n for n in dir(mod) if not n.startswith(\\\"_\\\")]:\\n            setattr(api, k, getattr(mod, k))\\n    for k in (\\\"Rule\\\", \\\"RuleModel\\\", \\\"Scene\\\", \\\"Action\\\", \\\"Object\\\", \\\"Region\\\", \\\"Transition\\\"):\\n        if hasattr(types, k) or hasattr(contracts, k):\\n            setattr(api, k, getattr(contracts, k, getattr(types, k, None)))\\n    api.types = api.contracts = api.mechanisms = api; api.core = api\\n\\n    def safe_import(name, globals=None, locals=None, fromlist=(), level=0):\\n        root = name.split(\\\".\\\")[0]\\n        if root in API_ROOTS or name in (\\\"core\\\", \\\"core.types\\\", \\\"core.contracts\\\", \\\"core.mechanisms\\\"):\\n            return api\\n        if name.split(\\\".\\\")[0] in (\\\"numpy\\\", \\\"dataclasses\\\", \\\"typing\\\", \\\"itertools\\\", \\\"math\\\", \\\"collections\\\", \\\"functools\\\"):\\n            return real_import(name, globals, locals, fromlist, level)\\n        raise ImportError(f\\\"import not allowed in sandbox: {name}\\\")\\n    ns[\\\"__builtins__\\\"][\\\"__import__\\\"] = safe_import\\n    return ns\\n\\n\\nclass _CoreShim:\\n    def __init__(self, types, contracts, mechanisms):\\n        self.types, self.contracts, self.mechanisms = types, contracts, mechanisms\\n\\n\\ndef load_in_process(code: str, allow_getattr: bool = False):\\n    \\\"\\\"\\\"Execute model code in a restricted namespace; returns the namespace (build_model / build_goal available).\\\"\\\"\\\"\\n    static_check(code, allow_getattr=allow_getattr)\\n    ns = _namespace()\\n    if allow_getattr:\\n        ns[\\\"__builtins__\\\"] = dict(ns[\\\"__builtins__\\\"]) if isinstance(ns.get(\\\"__builtins__\\\"), dict) else ns.get(\\\"__builtins__\\\")\\n        if isinstance(ns[\\\"__builtins__\\\"], dict):\\n            ns[\\\"__builtins__\\\"][\\\"getattr\\\"] = getattr; ns[\\\"__builtins__\\\"][\\\"hasattr\\\"] = hasattr\\n    exec(compile(code, \\\"<model>\\\", \\\"exec\\\"), ns)   # noqa: S102 \\u2014 after static checks, restricted builtins\\n    return ns\\n\\n\\nclass Sandbox:\\n    def __init__(self, cpu_seconds: int = 2, memory_mb: int = 256, python: Optional[str] = None, log=None, allow_getattr: bool = False):\\n        self.cpu_seconds, self.memory_mb = cpu_seconds, memory_mb\\n        self.python = python or sys.executable\\n        self.log = log or (lambda *a, **k: None)\\n        self.allow_getattr = allow_getattr\\n        self.last_error: str = \\\"\\\"\\n\\n    def load_namespace(self, code: str):\\n        \\\"\\\"\\\"Validated in-process load; returns the module namespace or None (error logged).\\\"\\\"\\\"\\n        try:\\n            self.last_error = \\\"\\\"\\n            return load_in_process(code, allow_getattr=self.allow_getattr)\\n        except Exception as e:\\n            self.last_error = f\\\"{type(e).__name__}: {str(e)[:200]}\\\"\\n            self.log(f\\\"sandbox load failed: {self.last_error}\\\")\\n            return None\\n\\n    def validate(self, code: str, log_json: list[dict]) -> dict:\\n        \\\"\\\"\\\"Run static checks + build + evaluate in a resource-limited subprocess. Returns {ok, score, coverage, violations, error}.\\\"\\\"\\\"\\n        try:\\n            static_check(code)\\n        except StaticCheckError as e:\\n            return {\\\"ok\\\": False, \\\"error\\\": f\\\"static: {e}\\\"}\\n        payload = json.dumps({\\\"code\\\": code, \\\"log\\\": log_json, \\\"cpu\\\": self.cpu_seconds, \\\"mem\\\": self.memory_mb})\\n        env = {\\\"PATH\\\": os.environ.get(\\\"PATH\\\", \\\"\\\"), \\\"PYTHONPATH\\\": str(ROOT), \\\"OPENBLAS_NUM_THREADS\\\": \\\"1\\\", \\\"OMP_NUM_THREADS\\\": \\\"1\\\", \\\"MKL_NUM_THREADS\\\": \\\"1\\\"}\\n        try:\\n            r = subprocess.run([self.python, \\\"-m\\\", \\\"pbg.wml.sandbox_worker\\\"], input=payload, capture_output=True, text=True, timeout=self.cpu_seconds * 5 + 10, env=env, cwd=str(ROOT))\\n        except subprocess.TimeoutExpired:\\n            return {\\\"ok\\\": False, \\\"error\\\": \\\"timeout (wall)\\\"}\\n        if r.returncode != 0:\\n            return {\\\"ok\\\": False, \\\"error\\\": f\\\"worker exit {r.returncode}: {(r.stderr or r.stdout)[-800:]}\\\"}\\n        try:\\n            return json.loads(r.stdout.strip().splitlines()[-1])\\n        except Exception:\\n            return {\\\"ok\\\": False, \\\"error\\\": f\\\"bad worker output: {r.stdout[-400:]} {r.stderr[-400:]}\\\"}\\n\\n    def load(self, code: str):\\n        \\\"\\\"\\\"In-process load after validation. Returns the WorldModel or None (error logged).\\\"\\\"\\\"\\n        try:\\n            ns = load_in_process(code, allow_getattr=self.allow_getattr)\\n            if \\\"build_model\\\" not in ns:\\n                return None\\n            model = ns[\\\"build_model\\\"]()\\n            return model\\n        except Exception as e:\\n            self.log(f\\\"sandbox load failed: {type(e).__name__}: {e}\\\")\\n            return None\\n\\n    def load_goal(self, code: str):\\n        try:\\n            ns = load_in_process(code)\\n            if \\\"build_goal\\\" not in ns:\\n                return None\\n            g = ns[\\\"build_goal\\\"]()\\n            from ..goal.templates import GoalInstance\\n            if isinstance(g, GoalInstance):\\n                return g\\n            if hasattr(g, \\\"is_goal\\\") and hasattr(g, \\\"progress\\\"):\\n                return GoalInstance(getattr(g, \\\"name\\\", \\\"llm_goal\\\"), \\\"llm\\\", {}, g.is_goal, g.progress, origin=\\\"llm\\\", code=code)\\n        except Exception as e:\\n            self.log(f\\\"sandbox goal load failed: {type(e).__name__}: {e}\\\")\\n        return None\\n\", \"pbg/wml/sandbox_worker.py\": \"\\\"\\\"\\\"Subprocess worker for Sandbox.validate: resource limits, then build_model() + evaluate on the log (JSON in/out).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport resource\\nimport sys\\nimport traceback\\n\\n\\ndef main() -> None:\\n    payload = json.loads(sys.stdin.read())\\n    cpu, mem = int(payload.get(\\\"cpu\\\", 2)), int(payload.get(\\\"mem\\\", 256))\\n    try:\\n        resource.setrlimit(resource.RLIMIT_CPU, (cpu, cpu + 1))\\n        resource.setrlimit(resource.RLIMIT_AS, (mem * 2 ** 20, mem * 2 ** 20))\\n        resource.setrlimit(resource.RLIMIT_NOFILE, (16, 16))\\n    except Exception:\\n        pass\\n    try:\\n        from pbg.core.types import Action, Scene, Transition, Diff\\n        from pbg.wml.evaluate import evaluate\\n        from pbg.wml.sandbox import load_in_process\\n        ns = load_in_process(payload[\\\"code\\\"])\\n        if \\\"build_model\\\" not in ns:\\n            print(json.dumps({\\\"ok\\\": False, \\\"error\\\": \\\"no build_model\\\"})); return\\n        model = ns[\\\"build_model\\\"]()\\n        log = []\\n        for d in payload[\\\"log\\\"]:\\n            t = Transition(Scene.from_json(d[\\\"before\\\"]), Action.from_json(d[\\\"action\\\"]), Scene.from_json(d[\\\"after\\\"]), Diff(), d.get(\\\"status_change\\\"), [], d.get(\\\"id\\\", \\\"\\\"),\\n                           int(d.get(\\\"level\\\", 1)), int(d.get(\\\"step_idx\\\", 0)))\\n            log.append(t)\\n        res = evaluate(model, log)\\n        rules = [{\\\"name\\\": r.name, \\\"confidence\\\": r.confidence} for r in model.rules()] if hasattr(model, \\\"rules\\\") else []\\n        print(json.dumps({\\\"ok\\\": True, \\\"score\\\": res.score, \\\"coverage\\\": res.coverage, \\\"violations\\\": res.violations[:50], \\\"unknown\\\": res.unknown[:50],\\n                          \\\"rule_confidence\\\": res.rule_confidence, \\\"prediction_key\\\": res.prediction_key, \\\"n\\\": res.n, \\\"rules\\\": rules,\\n                          \\\"per_class\\\": {k: list(v) for k, v in res.per_class.items()}}))\\n    except MemoryError:\\n        print(json.dumps({\\\"ok\\\": False, \\\"error\\\": \\\"memory limit\\\"}))\\n    except Exception:\\n        print(json.dumps({\\\"ok\\\": False, \\\"error\\\": traceback.format_exc()[-1500:]}))\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"scripts/eval_viewer/__init__.py\": \"\\\"\\\"\\\"Local evaluation viewer: replays recorded rulebook runs level by level and serves them to a browser (scripts/serve_eval.py).\\\"\\\"\\\"\\n\", \"scripts/eval_viewer/ids.py\": \"\\\"\\\"\\\"Stable ids for recorded games.\\n\\ngame_hash(run_id, game_id): 10 hex chars, sha1 of \\\"<run id>/<game id>\\\". One value per game execution \\u2014 a run id is unique\\n(utc timestamp + pid) and a game is played once per run \\u2014 so it can be computed for any run, old or new, from the\\nsame two strings. Runners write it into the run summary and logs/<run>/run.json; the Replay viewer resolves it with\\nGET /api/find/<hash> (or the #h/<hash> URL) and shows it next to every game. Stdlib only: imported by the runners.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\n\\nHASH_LEN = 10\\n\\n\\ndef game_hash(run_id: str, game_id: str) -> str:\\n    return hashlib.sha1(f\\\"{run_id}/{game_id}\\\".encode()).hexdigest()[:HASH_LEN]\\n\"}")
for rel, body in FILES.items():
    os.makedirs(os.path.dirname(f'{WORK}/{rel}'), exist_ok=True)
    open(f'{WORK}/{rel}', 'w').write(body)
sys.path.insert(0, WORK)
os.environ['PBG_LLM_CONFIG'] = f'{WORK}/pbg/llm/llm-kaggle.yaml'
import arc_agi, yaml, pbg.harness.online_runner, pbg.hypothesis.policy
print(f'pbg embedded ({len(FILES)} files) | arc_agi ok |', f'{time.time()-T0:.0f}s')


In [ ]:
import glob, json, os, subprocess, sys, tarfile, time, urllib.request
WORK = '/kaggle/working'
def _find(ref):
    owner, slug = ref.split('/', 1)
    for p in (f'/kaggle/input/{slug}', f'/kaggle/input/datasets/{owner}/{slug}'):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(ref)
RUNTIME = _find('keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1')
_mp = glob.glob('/kaggle/input/models/**/config.json', recursive=True)
MODEL_PATH = os.path.dirname(_mp[0]) if _mp else _find('nvidia/qwen3-8-flash-next-nvfp4/pyTorch/v1/1')
cfgs = glob.glob(MODEL_PATH + '/**/config.json', recursive=True)
MODEL_PATH = os.path.dirname(cfgs[0]) if cfgs else MODEL_PATH
print('runtime:', RUNTIME, '| model:', MODEL_PATH)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

# Extract the exported image layers (manifest order; later layers overlay earlier ones) into a scratch root.
ROOT = '/tmp/vllm-image'
if not os.path.exists(ROOT + '/.done'):
    man = json.load(open(RUNTIME + '/runtime-manifest.json'))
    layers = sorted(man['selected_layers'], key=lambda l: l['index'])
    os.makedirs(ROOT, exist_ok=True)
    for l in layers:
        cands = [RUNTIME + '/' + l['file'] + '.blob', RUNTIME + '/' + l['file']] + glob.glob(RUNTIME + '/layer-%d-*' % l['index'])
        blob = next(p for p in cands if os.path.exists(p))
        print('extract', os.path.basename(blob), f"({l['size']//(1<<20)} MiB) ...", flush=True)
        with tarfile.open(blob, 'r:gz') as tf:
            for _mem in tf:   # per-member so one odd entry (device, absolute symlink, .wh. whiteout) never kills the layer
                try:
                    tf.extract(_mem, ROOT, filter='tar')
                except Exception:
                    pass
    open(ROOT + '/.done', 'w').write('ok')
print('image root size MiB:', subprocess.run(['du', '-sm', ROOT], capture_output=True, text=True).stdout.split()[0])

# Locate the image's python site-packages (the dir that holds vllm). Its native extensions must match this kernel's
# python ABI (both cp312 for vllm/vllm-openai + Kaggle); warn otherwise.
_v = (glob.glob(ROOT + '/**/site-packages/vllm/__init__.py', recursive=True) or
      glob.glob(ROOT + '/**/dist-packages/vllm/__init__.py', recursive=True) or   # vllm/vllm-openai installs into dist-packages
      glob.glob(ROOT + '/**/vllm/__init__.py', recursive=True))
_v = [p for p in _v if os.path.isdir(os.path.dirname(p) + '/entrypoints')] or _v   # a real install, not a nested copy
if not _v:
    raise RuntimeError('no vllm in the extracted image. python dirs: ' + str(glob.glob(ROOT + '/**/python3*/', recursive=True)[:20]))
SITE = os.path.dirname(os.path.dirname(_v[0]))
import re as _re
_m = _re.search(r'python3\.(\d+)', SITE)
if _m and int(_m.group(1)) != sys.version_info.minor:
    print(f'WARNING: image python 3.{_m.group(1)} != kernel python 3.{sys.version_info.minor} (ABI mismatch likely)')
print('image vllm site-packages:', SITE)
# CUDA libs: torch ships its own under site-packages/nvidia/*/lib; add them plus the Kaggle driver stub.
_nvlibs = ':'.join(sorted(set(os.path.dirname(p) for p in glob.glob(SITE + '/nvidia/**/lib', recursive=True)) |
                          set(glob.glob(SITE + '/nvidia/*/lib'))))
# The image's own CUDA toolkit (nvcc) must be used: FlashInfer JIT-compiles the NVFP4 MoE kernel for sm120 at
# startup, and Kaggle's base nvcc is too old ("No supported CUDA architectures found for major versions [12]").
_nvcc = glob.glob(ROOT + '/**/cuda*/bin/nvcc', recursive=True) or glob.glob(ROOT + '/**/bin/nvcc', recursive=True)
CUDA_HOME = os.path.dirname(os.path.dirname(_nvcc[0])) if _nvcc else '/usr/local/cuda'
print('image nvcc:', _nvcc[0] if _nvcc else 'NOT FOUND (JIT will likely fail)', '| CUDA_HOME:', CUDA_HOME)
_ld = '/usr/local/nvidia/lib64:' + CUDA_HOME + '/lib64:' + _nvlibs + ':' + SITE + '/torch/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
_ram_gb = round(int(next(l.split()[1] for l in open('/proc/meminfo') if l.startswith('MemTotal'))) / (1 << 20), 1)
print(f'host RAM: {_ram_gb} GiB (the FP8 PLE n-gram table offloaded to CPU needs ~48 GiB + headroom; recipe wants >= 64)')
# Qwen3.8-Flash-Next has a 51B-param PLE n-gram embedding. On one 96 GB GPU it must be offloaded to host RAM
# (VLLM_PLE_CPU_OFFLOAD=1): NVFP4 experts then use ~89 GiB VRAM. Recipe: max-model-len 32768, max-num-seqs 36.
# sm120 FlashInfer JIT needs the arch declared explicitly (FLASHINFER_CUDA_ARCH_LIST=12.0f, FLASHINFER_FORCE_SM=120f).
env = dict(os.environ, PYTHONPATH=SITE, USE_TF='0', TRANSFORMERS_NO_TF='1', TRANSFORMERS_NO_TORCHVISION='1',
           VLLM_NO_USAGE_STATS='1', VLLM_PLE_CPU_OFFLOAD='1', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
           VLLM_PLE_OFFLOAD_READY_TIMEOUT='5400',   # the offload worker reads the ~48 GiB PLE table over slow NFS; the 600 s default can expire first
           FLASHINFER_CUDA_ARCH_LIST='12.0f', FLASHINFER_FORCE_SM='120f', TORCH_CUDA_ARCH_LIST='12.0',
           CUDA_HOME=CUDA_HOME, PATH=CUDA_HOME + '/bin:' + os.environ.get('PATH', ''),
           LIBRARY_PATH='/usr/local/nvidia/lib64:' + CUDA_HOME + '/lib64:' + os.environ.get('LIBRARY_PATH', ''), LD_LIBRARY_PATH=_ld)
# --distributed-executor-backend mp is load-bearing on ONE GPU (vllm#53960): the default single-GPU executor never
# spawns the PLE n-gram offload worker, so the warmup forward waits on it forever -- v12/v13/v14 all went silent right
# after "Free memory on device" and never bound the port. --no-enable-flashinfer-autotune: autotuned MoE tactics
# corrupt output on sm120 (flashinfer#4841).
cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--model', MODEL_PATH, '--served-model-name', 'local-model',
       '--host', '127.0.0.1', '--port', '1234', '--max-model-len', '32768', '--max-num-seqs', '36', '--gpu-memory-utilization', '0.93',
       '--distributed-executor-backend', 'mp', '--no-enable-flashinfer-autotune',
       '--enable-auto-tool-choice', '--enable-prefix-caching', '--generation-config', 'vllm'] + ['--tool-call-parser', 'qwen3_coder', '--reasoning-parser', 'qwen3']
log = open(f'{WORK}/vllm-server.log', 'w')
VLLM = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)
t0 = time.time(); _hb = 0
while True:
    if VLLM.poll() is not None:
        _log = open(f'{WORK}/vllm-server.log').read()
        _err = [l for l in _log.splitlines() if any(k in l for k in ('FAILED', 'error:', 'Error', 'cannot find', 'RuntimeError', 'assert', 'not recognize'))]
        raise RuntimeError('vLLM exited. Error lines:\n' + '\n'.join(_err[:40]) + '\n--- tail ---\n' + _log[-3000:])
    try:
        urllib.request.urlopen('http://127.0.0.1:1234/v1/models', timeout=5).read(); break
    except Exception:
        # 10-24 min weight load (NFS speed varies) plus the PLE worker's ~48 GiB table load before the port opens;
        # give startup 90 min. A heartbeat every 5 min shows the last server log line, so a silent stall is visible
        # without waiting for the timeout.
        _el = time.time() - t0
        if _el - _hb >= 300:
            _hb = _el
            _lines = [l for l in open(f'{WORK}/vllm-server.log').read().splitlines() if l.strip()]
            print(f'[wait {_el/60:.0f} min] last: {(_lines[-1] if _lines else "")[:200]}', flush=True)
        if _el > 5400:
            raise TimeoutError(open(f'{WORK}/vllm-server.log').read()[-4000:])
        time.sleep(5)
print(f'vLLM ready after {time.time()-t0:.0f}s')
req = urllib.request.Request('http://127.0.0.1:1234/v1/chat/completions', data=json.dumps({'model': 'local-model', 'max_tokens': 400,
      'messages': [{'role': 'user', 'content': 'Say hello in five words.'}], **{'chat_template_kwargs': {'enable_thinking': False}}}).encode(),
      headers={'Content-Type': 'application/json'})
try:
    _msg = json.loads(urllib.request.urlopen(req, timeout=300).read())['choices'][0]['message']
    print('smoke:', repr((_msg.get('content') or '')[:200]))
except Exception as _e:
    print('smoke chat failed:', repr(_e)[:300])


In [ ]:
import math, os, sys, time, urllib.request, logging
from pathlib import Path
sys.path.insert(0, '/kaggle/working')
logging.basicConfig(level=logging.WARNING)
from pbg.harness.online_runner import make_arcade, run
cfg = {'policy': 'hypothesis', 'explore': 20, 'no_llm': False, 'verbose': False, 'max_levels': 20, 'budget': None,
       'llm_calls_per_game': 600}
out_dir = Path('/kaggle/working/pbg-results'); mem = Path('/kaggle/working/pbg-memory')
if RERUN:
    os.environ['ARC_API_KEY'] = 'test-key-123'
    deadline = time.time() + 60
    while True:  # wait for the gateway sidecar
        try:
            urllib.request.urlopen('http://gateway:8001/api/games', timeout=10).read(); break
        except Exception as e:
            if time.time() > deadline:
                raise RuntimeError(f'gateway not ready: {e!r}')
            time.sleep(5)
    arc = make_arcade(competition=True, base_url='http://gateway:8001/')
    games = [e.game_id for e in arc.get_environments()]
    waves = max(1, math.ceil(len(games) / 12))
    total_left = 470 - (time.time() - T0) / 60
    # per game: its wave's share of the time left, minus the 5-min hang allowance of the runner
    cfg.update(jobs=12, max_minutes=max(10, min(120, total_left / waves - 6)))
    print(f'rerun: {len(games)} games, {cfg["jobs"]} concurrent, {cfg["max_minutes"]:.0f} min/game, waves={waves}')
    run(games, cfg, out_dir=out_dir, memory_root=mem, tag='kaggle-rerun', arc=arc)   # the gateway records actions and emits submission.parquet
else:
    arc = make_arcade(environments_dir='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files')
    cfg.update(jobs=2, max_minutes=12)
    res = run(['r11l', 'vc33'], cfg, out_dir=out_dir, memory_root=mem, tag='kaggle-smoke', arc=arc)
    print('smoke errors:', [(g['game_id'], g.get('error')) for g in res['games'] if g.get('error')])
    import pandas as pd   # commit mode: dummy submission so the commit succeeds
    pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score']).to_parquet('/kaggle/working/submission.parquet', index=False)
try:
    VLLM.terminate()
except Exception:
    pass
print(f'done in {(time.time()-T0)/60:.1f} min')
